# E7 · Top-10% head ablation in fixed depth bands, 17 models

Does a model whose head activity is more concentrated (higher top-10% share) depend more on its most
active heads? E4 compared one pair of 70B models; this runs the paper's 17 models with
`ablation_window.py`, on the four serial tasks (fixed suites, wording w0, test split).

**Design, fixed before running** (module docstring):
* Items halved by k as in E4 (same seed): a selection half and a measurement half.
* **Selection half:** per-head activity ‖z_h W_O^h‖ at the answer position on every layer, the
  heads' mean outputs (mean-ablation values), heads ranked per layer by mean activity, and each
  layer's top-10% share.
* **Heads:** q = ⌈0.1 H⌉ per layer (the paper's top-10% group); random baseline: q mid-ranked heads
  per layer (25th–75th percentile of activity, top heads excluded).
* **Layer sets, independent of how layers respond to k:** the 40–80% window (E4's), and five depth
  bands of 20%. All layers of a set are ablated together.
* **Conditions:** clean; per set top-q mean-ablation, top-q zero-ablation, random-q (5 draws for the
  window, 2 per band); for the window also top-1 and top-3 (E4's sizes). 30 conditions, run in one
  batched forward per item.
* **Outcome:** gold-minus-competitor logit difference at the answer position (E4's), and every E4
  verification (per-head split, intended-input and o_proj delta checks on the first item, KV-cache
  checks, repeat check).

**Analysis (step 4):** per model and task, the top-10% share of the window against the *excess drop*
(top-q minus random, fraction of clean, clean-correct items, depth-balanced). Within each task across
the 17 models (Spearman, permutation null shuffling models within task), and within each model across
its four tasks.

**Runtime:** H100 80GB (A100 80GB works; 40 GB is enough for all but Mistral-Small-24B), standard RAM.
Computation is ~1–3 min per suite; downloading ~240 GB of weights dominates, so the next model is
downloaded while the current one runs. All 17 models about 1.5–2 h in one runtime, or about 1 h with
the two half notebooks in two runtimes.
**Outputs (Drive):** `next_runs/E7_ablation_window/bf16_<kernel>/…__abwin.npz`, `…/report/`, `logs/`.

## 0 · Setup

Run these cells in order at the start of every session (after a disconnect too:
finished work is skipped further down).

- **0.1** clones the code (or pulls it if this runtime already has it), installs the
  packages, and records the clone's commit in `NEXT_RUNS_COMMIT`. If pip says numpy
  was upgraded and asks for a restart, restart the runtime and run again from the top.
- **0.2** writes the modules that are not on GitHub yet. They are generated from the
  tested files by `colab/make_notebooks.py`; do not edit them here.
- **0.3** stops unless the written modules are exactly the embedded ones and the
  clone's `deep_probe.py`, `active_subgraph_dot.py` and `path_analyzer.py` are the
  versions they were tested with, then appends the module hashes to
  `NEXT_RUNS_COMMIT` (every output records it). *Check:* the printed versions.

In [ ]:
# 0.1 Code and packages
import os, subprocess, sys
REPO_URL = "https://github.com/JohnnyPunchinello/LLM_path_analysis.git"
REPO_DIR = "/content/LLM_path_analysis"
PIP_PACKAGES = "transformers accelerate networkx scipy statsmodels"   # pin here if needed, e.g. "transformers==5.12.1 ..."
EMBEDDED = ["serial_tasks.py", "sweep_probe.py", "signature_analysis.py", "ablation_probe.py", "head_alignment_probe.py", "ablation_window.py"]

def git(*args):
    r = subprocess.run(["git", *args], cwd=REPO_DIR, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # 0.2 rewrites the embedded modules; remove this session's copies first so a pull
    # never stops on them (untracked now, perhaps tracked upstream later)
    for f in EMBEDDED:
        if os.path.exists(os.path.join(REPO_DIR, f)):
            tracked = subprocess.run(["git", "ls-files", "--error-unmatch", f], cwd=REPO_DIR,
                                     capture_output=True).returncode == 0
            git("checkout", "--", f) if tracked else os.remove(os.path.join(REPO_DIR, f))
    git("pull", "-q", "--ff-only")
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
!pip install -q -U {PIP_PACKAGES}
if globals().get("_exit_code"):         # IPython's exit status of the last ! command
    raise RuntimeError("pip install failed (read its output above); nothing further would run "
                       "with the intended package versions")
CLONE_COMMIT = git("rev-parse", "HEAD")
os.environ["NEXT_RUNS_COMMIT"] = CLONE_COMMIT          # 0.3 appends the module hashes
# weights, and the login token load_model stores, stay on this VM's disk, never on Drive
os.environ["HF_HOME"] = "/content/hf"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("code:", REPO_DIR, "at commit", CLONE_COMMIT)

### 0.2 Modules not yet on GitHub
Generated from the tested local files; the next cell checks them byte for byte.

In [ ]:
%%writefile serial_tasks.py
#!/usr/bin/env python3
"""
Serial-depth task suites with task-agnostic item metadata.

Every probe downstream (sweep_probe, ablation_probe, signature_analysis) reads
depth, answer set and echo from the items written here instead of parsing a
prompt template, so any family below can go through the same pipeline.

A suite is one JSON file

    {"name", "family", "wording", "setting", "chance", "items": [item, ...]}

and an item is

    id          "<suite name>/<k>/<i>", unique and stable given the seed
    family      pointer | arith | binding | relational | fixed_digit | ladder
    wording     w0 | w1 (paraphrase frames, WORDINGS)
    k           serial depth (fixed_digit: effective depth, see below)
    label       "D{k}N{i:02d}", for the existing parsers
    text        the prompt; the answer follows as " <digit>"
    gold        the answer digit
    answer_set  admissible answers; competitors = answer_set - gold
    k_written   whether the numeral of k appears in the prompt
    echo        that numeral, if written; echoing it is never correct
    split       calib (about 30%, stratified by k) | test
    setting     item-level generator parameters (start, table, ...)

Families (FAMILIES documents each, WORDINGS holds the templates the generator
formats, so the two cannot drift apart):

    pointer      random n-cycle, k written ("Take k steps"); w0 is the
                 serial_depth_fs template.  Worked examples are drawn per item
                 with balanced answers (shot_mode="per_item") or shared by all
                 items (shot_mode="shared", reproducing serial_depth_fs).
    arith        add d, k times, mod `modulus` (default 10); k written.
    binding      variable chains rooted in digit literals, one link longer than
                 the deepest query (no queried variable is a chain's leaf); k is
                 the hop distance of the queried variable.  Numeral-free.
    relational   "The boss of Q is M" chains rooted in digit bosses, built like
                 binding; k is the number of hops from the queried person to
                 the top boss.  Numeral-free.
    fixed_digit  the numeral control: written step count fixed, rule table
                 varied (one c-cycle through the start plus fixed points), so
                 k = distinct states traversed changes while the numeral does
                 not.

Within a family, items are drawn from one seeded stream that no template
consumes, so w0 and w1 suites built with the same seed and settings hold the
same instances (tables, starts, answers, split): wordings are compared on
paired items.  Different families never share instances: pointer draws from
Random(seed) (the serial_depth_fs reproduction needs it), every other family
from Random("<family>:<seed>").  seed must be an int.  Per k, gold digits are
balanced over the admissible answers (counts differ by at most one); the calib
split takes about calib_frac of every depth, round-robin over gold digits, and
keeps a matched family's body (shared across k) on one side.

Final problems are drawn without replacement within each depth and gold digit,
so a depth cell repeats a problem only when it holds more items than the
settings have problems (e.g. arith modulus=7: 36 per depth), and then as evenly
as possible; identical final problems always fall on one side of the split, so
calibration stays held out.  No worked example is the item's own query: at the
item's depth and gold an example could otherwise be the query with its answer
shown, which a copying model solves for free, and only at the shallow depths
the examples use.

Worked examples (shot_mode="per_item") come in a random depth order, and at
every depth and example position their answers are balanced over the gold's own
admissible set: never the item's written numeral (echo), never the example's own
step count, distinct within a prompt as far as possible.  Copying any one
example is then right 1/(n-1) of the time at every k (n admissible states), so a
copy bias cannot make a depth effect.

Guarantees, checked on every suite built or loaded (abort, never write a wrong
item): the gold answer, depth, answer set, written numeral (echo, k_written)
and start/table/query metadata are recomputed from the prompt text alone by an
independent parser (solve_text), including every worked example; gold is in the
answer set and never equals the echo; chance = 1/|answer set|; ids and labels
agree and are unique; the suite has items; every table, increment and chain
has the family's form (pointer: one n-cycle; fixed_digit: one cycle through
the start; arith: k*d != 0 mod m).  check_suite reports the rest: per
tokenizer, constant token count across k and deep_probe.align_answer on every
item; per k, answer balance, worked-example balance per position and the
accuracy of copying each example, identical prompts and worked examples equal
to the item's own query (both failures); distractor digits in every prompt;
and, as warnings, design limits the analysis must handle (shortest lookup path
below k, arith offsets fixed by k, repeated final problems, a queried
definition whose distance to the query trends with k, calib split too small
for the min-correct rule, prompts above a tokenizer's model_max_length).

    python3 serial_tasks.py --family pointer arith binding relational fixed_digit \\
        --wording w0 w1 --out suites [--k 1 6] [--n-per 40] [--seed 0] \\
        [--set shots=5 --set shot_mode=per_item] [--tokenizer gpt2 Qwen/Qwen2.5-0.5B]
    python3 serial_tasks.py --ladder --out suites
    python3 serial_tasks.py --check suites/*.json --tokenizer gpt2 Qwen/Qwen2.5-0.5B
"""
import argparse, collections, hashlib, json, math, operator, random, re, sys
from pathlib import Path

DIGITS = "0123456789"
# Bumped whenever the generator draws different items for the same arguments; it
# enters every automatic suite name, so a regenerated suite never takes the name
# (and hence the item ids and sweep_probe output file) of an older version.
GENERATOR_VERSION = 3
# Re-draws of a worked example that came out as the item's own query before its
# depth-answer pair is taken to have no other problem (then another answer).
_TRIES = 64
# Every entity name follows a space, and a space plus one letter is one token in
# the gpt2 and Qwen2.5 tokenizers (check_suite verifies this per suite), so
# swapping which entity is queried cannot change the token count.  "a"/"A" and
# "i"/"I" are dropped: they read as English words.
LOWER = [c for c in "abcdefghijklmnopqrstuvwxyz" if c not in "ai"]
UPPER = [c for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZ" if c not in "AI"]
EXAMPLE_END = " {ans}.\n\n"          # a solved instance + this = one worked example
ITEM_KEYS = ("id", "family", "wording", "k", "label", "text", "gold", "answer_set",
             "k_written", "echo", "split", "setting")
SUITE_KEYS = ("name", "family", "wording", "setting", "chance", "items")


# ────────────────────────────── templates ───────────────────────────────────
# Placeholders: a,b rule source/target  s start  k written step count
# d increment  m modulus  x,y statement subject/object  v queried entity.
_POINTER_WORDINGS = {
    "w0": {"head": "Rules: ", "rule": "{a} goes to {b}.",
           "query": "Start at {s}. Take {k} steps. You end at"},
    "w1": {"head": "Map: ", "rule": "{a} leads to {b}.",
           "query": "From {s}, make {k} moves. You finish at"},
}
WORDINGS = {
    "pointer": _POINTER_WORDINGS,
    # same frames as pointer: the control must differ from it only in the table
    "fixed_digit": _POINTER_WORDINGS,
    # "query" is the mod-10 frame; "query_mod" is used for any other modulus
    "arith": {
        "w0": {"query": "Rule: add {d} and keep only the last digit. Start at {s}. "
                        "Apply the rule {k} times. You end at",
               "query_mod": "Rule: add {d} and keep the remainder after dividing by {m}. "
                            "Start at {s}. Apply the rule {k} times. You end at"},
        "w1": {"query": "x = {s}. Repeat {k} times: x = (x + {d}) mod 10. Now x =",
               "query_mod": "x = {s}. Repeat {k} times: x = (x + {d}) mod {m}. Now x ="},
    },
    "binding": {
        "w0": {"head": "Assignments: ", "stmt": "{x} = {y}.", "query": "Therefore {v} ="},
        "w1": {"head": "Facts: ", "stmt": "Let {x} be {y}.", "query": "The value of {v} is"},
    },
    "relational": {
        "w0": {"head": "Chart: ", "stmt": "The boss of {x} is {y}.",
               "query": "The top boss above {v} is"},
        "w1": {"head": "Chart: ", "stmt": "{x} reports to {y}.",
               "query": "{v} ultimately reports to"},
    },
    # the original single-prompt ladder (active_subgraph_dot serial_depth)
    "ladder": {"w0": _POINTER_WORDINGS["w0"]},
}

_COMMON = {"shots": 3, "shot_mode": "per_item", "shot_depths": [1, 2, 3], "calib_frac": 0.3}
FAMILIES = {
    "pointer": {
        "doc": "Pointer chase on a random single n-cycle over states 0..n_states-1 with a "
               "random start; each item draws its own cycle.  Gold never equals k.",
        "k": "number of steps, written as a numeral", "k_written": True, "matched": False,
        "answer_set": "all states 0..n_states-1 (chance 1/n_states)",
        "balance": "per k, gold uniform over the states other than k",
        "defaults": dict(_COMMON, n_states=7, balance=True),
        "notes": "Seeded from Random(seed) (the other families from Random('<family>:"
                 "<seed>')).  shot_mode='shared' draws one worked-example prefix exactly as "
                 "_gen_serial_depth_rep does; with balance=False (unbalanced legacy item "
                 "draw) and seed 0 it reproduces serial_depth_fs byte for byte.  Llama-3.1-70B "
                 "copied the shared prefix's answer '6' (2 of 3 examples) on 119/240 items, "
                 "hence per_item examples by default: random depth order, and per depth and "
                 "example position answers balanced over the states other than k (never the "
                 "example's own step count either), so copying any example scores "
                 "1/(n_states-1) at every k.  On one n-cycle, k steps forward land where "
                 "n_states-k steps backward do, so the shortest lookup path is "
                 "min(k, n_states-k) (1,2,3,3,2,1 for the 7-state default); it is recorded "
                 "per item as setting.min_hops and check_suite warns.  Use it as a covariate, "
                 "restrict to k <= n_states//2, or raise n_states (e.g. n_states=10, k 1..5).  "
                 "With balance=True no cycle repeats within a (k, gold) cell until all "
                 "(n_states-1)! are used, and no final problem equals a shared example.",
    },
    "arith": {
        "doc": "Composed arithmetic: start digit s, add d (from increments) k times, keep "
               "the remainder mod `modulus` (default 10: 'keep only the last digit').  "
               "Gold = (s + k*d) mod modulus.",
        "k": "number of additions, written as a numeral", "k_written": True, "matched": False,
        "answer_set": "0..modulus-1 (chance 1/modulus; 0..9 and 0.1 by default)",
        "balance": "per k, gold uniform over 0..modulus-1 minus k; s is solved for",
        "defaults": dict(_COMMON, modulus=10, increments=list(range(1, 10))),
        "notes": "d with k*d = 0 mod modulus is never used, so the answer never equals the "
                 "start digit (copying the start would otherwise score at k=5 for 4 of 9 d).  "
                 "Mod 10 leaks: at k=5 every allowed d gives offset 5 (gold = start+5 without "
                 "composing anything) and even k keep the start's parity; check_suite reports "
                 "the offsets per k and warns.  A prime modulus above max k removes both: "
                 "modulus=7 (increments default to 1..modulus-1) gives offsets uniform over "
                 "1..6 at every k in 1..6, chance 1/7.  Per (k, gold) the increment cycles "
                 "through the allowed ones, so a final problem repeats only when a gold has "
                 "more items than allowed increments: modulus=7 holds 36 problems per depth "
                 "(6 golds x 6 increments), so n_per above 36 repeats some (kept on one side "
                 "of the split; check_suite warns, and fails identical prompts, i.e. shots=0).",
    },
    "binding": {
        "doc": "n_chains chains of single-letter variables, each rooted in a distinct digit "
               "literal (x = 3. y = x. ...), all of max(k_range)+1 links, in one body.  The "
               "same body is queried at every depth.  The last link of a chain is never "
               "queried, so every queried variable is also the value of the next link: it "
               "is mentioned twice at every k, never as a leaf.",
        "k": "hop distance of the queried variable to its literal (1 = assigned the digit)",
        "k_written": False, "matched": True,
        "answer_set": "the n_chains root digits (chance 1/n_chains)",
        "balance": "gold uniform over 0..9 across bodies (identical at every k)",
        "defaults": dict(_COMMON, n_chains=None, order="shuffled"),
        "notes": "Numeral-free.  Items with the same i share their body and worked "
                 "examples, so prompts at different k differ only in the queried letter.  "
                 "Names: n_chains x (max k + 1) of 24 letters; n_chains=None takes 4 where "
                 "that fits (max k <= 5), else as many as fit (3 for k up to 6).  "
                 "order='shuffled' (default) permutes all statements, with the queried "
                 "chain's statement order balanced across bodies (every link takes every "
                 "rank equally often), so the distance from the queried definition to the "
                 "query (setting.def_distance, statements, 1 = the last) does not trend with "
                 "k.  order='interleaved' keeps each chain in order (definition before use), "
                 "so deeper links sit later and def_distance falls with k (about 22 -> 4 for "
                 "k 1..6): a recency covariate confounded with depth; check_suite warns.  "
                 "setting.query_refs counts the statements that use the queried entity as a "
                 "value (1 at every k).  binding and relational draw from separate seeded "
                 "streams, so the two families never hold the same instances (RQ1 counts "
                 "them as two families).",
    },
    "relational": {
        "doc": "n_chains reporting chains (The boss of Q is M.) of capital-letter people, "
               "each topped by a distinct digit boss, all of max(k_range)+1 links (the last "
               "never queried, as in binding); query 'The top boss above X is'.",
        "k": "hops from the queried person to the top boss", "k_written": False,
        "matched": True,
        "answer_set": "the n_chains top-boss digits; these are the only digits in the "
                      "prompt, so 'root digits' and 'all digits used' coincide "
                      "(chance 1/n_chains)",
        "balance": "gold uniform over 0..9 across bodies (identical at every k)",
        "defaults": dict(_COMMON, n_chains=None, order="shuffled"),
        "notes": "Numeral-free and matched, with the same chain sizes, orders and "
                 "def_distance / query_refs records as binding.  Non-root people are "
                 "letters, not digits, because ten digit labels cannot hold chains of depth "
                 "6 plus distractor chains.",
    },
    "fixed_digit": {
        "doc": "Numeral control on n_states-state tables: the written step count is fixed "
               "(written_k); the table is one c-cycle through the start plus fixed points, "
               "c = k+1, so the walk traverses k distinct new states (k=0: identity, "
               "1: transposition, ..., n_states-1: the full cycle).",
        "k": "effective depth = distinct states visited after the start "
             "(len(set(s_0..s_K)) - 1)",
        "k_written": True, "matched": False,
        "answer_set": "all states 0..n_states-1 (chance 1/n_states)",
        "balance": "per k, gold uniform over the states other than written_k",
        "defaults": dict(_COMMON, n_states=7, written_k=None),
        "notes": "echo = written_k for every item.  When (k+1) divides written_k the walk "
                 "ends on its start, so copying the start is correct there (unavoidable with "
                 "permutation tables, as in the original); flagged per item as "
                 "setting.gold_eq_start.  written_k=None picks the numeral >= max(k_range) "
                 "with the fewest such cells: 7 for k 1..6 (only k=6), where 6 would give "
                 "three (k=1, 2, 5).  Worked examples are ordinary pointer examples.  The "
                 "original's identity cell is k=0 and is opt-in: pass k_range=(0, 6) to keep "
                 "it (the shared default k_range=(1, 6) leaves it out; k=0 is a pure "
                 "copy-the-start cell).  setting.min_hops = the shortest lookup path from "
                 "start to gold, forward or backward (1,1,1,2,1,0 for k 1..6 with "
                 "written_k 7): k counts states traversed, not lookups needed.  Tables are "
                 "drawn without replacement per (k, gold): k=1 has n_states-1 "
                 "transpositions per gold (42 problems per depth on 7 states, so n_per up to "
                 "42 without repeats), the k=0 identity cell one problem per gold (with "
                 "shots=0 its prompts repeat exactly, which check_suite fails).",
    },
    "ladder": {
        "doc": "The original single-prompt ladder TASK_SUITES['serial_depth'] (fixed "
               "5-cycle, start 0, k=1..6), wrapped by ladder_suite().",
        "k": "number of steps, written", "k_written": True, "matched": False,
        "answer_set": "the 5 states (chance 0.2)", "balance": "one item per k",
        "defaults": {}, "notes": "split 'test' only; label D{k}N00 (original D{k}W1 kept "
                                 "in setting.orig_label).",
    },
}


# ────────────────────────────── primitives ──────────────────────────────────
def _rand_cycle(rng, n):
    """Random single n-cycle on 0..n-1 -> dict.  Same draws as
    active_subgraph_dot._rand_cycle, which serial_depth_fs reproduction needs."""
    order = list(range(n))
    rng.shuffle(order)
    return {order[i]: order[(i + 1) % n] for i in range(n)}


def _walk(tbl, s, k):
    for _ in range(k):
        s = tbl[s]
    return s


def _back(tbl, g, k):
    """The state k steps before g (tables are permutations)."""
    inv = {b: a for a, b in tbl.items()}
    for _ in range(k):
        g = inv[g]
    return g


def _cycles(tbl):
    """The cycles of a permutation table, each as a list of states."""
    seen, out = set(), []
    for x in sorted(tbl):
        if x not in seen:
            cyc = [x]
            seen.add(x)
            while tbl[cyc[-1]] != x:
                cyc.append(tbl[cyc[-1]])
                seen.add(cyc[-1])
            out.append(cyc)
    return out


def _fresh(draw, key, used, space, clash=None):
    """draw() without replacement within a cell and away from worked examples.

    Returns a draw whose key is not in `used` (a set, then updated) and that
    does not `clash` (reproduce a worked example), as far as possible.  `space`
    is the number of distinct keys draw() can return: once all are used a new
    round starts, so a cell repeats a problem only when it holds more items
    than the settings have problems, and then as evenly as possible.  Rejection
    sampling with 50 tries per possible key finds a free problem whenever one
    exists (the spaces that can run out are small); failing that it takes a
    repeat, and a clash only when every problem clashes (degenerate settings
    such as a 2-state pointer; check_suite fails the suite then).
    """
    if len(used) >= space:
        used.clear()
    tries = int(min(50 * space, 20000))
    for fresh_only in (True, False):
        for _ in range(tries):
            x = draw()
            if not (clash and clash(x)) and not (fresh_only and key(x) in used):
                used.add(key(x))
                return x
        used.clear()                              # nothing new clear of the examples: repeat
    x = draw()
    used.add(key(x))
    return x


def _example_blocks(prefix):
    """The worked examples of a prefix without their answers: the blocks no
    final problem may equal."""
    return {re.sub(r" \d\.$", "", b) for b in prefix.split("\n\n")[:-1]}


def _build_avoiding(build, kd, a, forbid):
    """build(kd, a), re-drawn while it is one of the `forbid` blocks; None when
    every try is (that depth-answer pair then has no other problem)."""
    for _ in range(_TRIES if forbid else 1):
        t = build(kd, a)
        if t not in forbid:
            return t
    return None


def _min_hops(tbl, s, g):
    """Fewest lookups from s to g walking the table forward or backward: on a
    cycle of length c with g f steps ahead, min(f, c - f).  (A 7-cycle's k=6 is
    one backward lookup.)"""
    ring, x = [s], tbl[s]
    while x != s:
        ring.append(x)
        x = tbl[x]
    f = ring.index(g)
    return min(f, len(ring) - f)


def _balanced(pool, n, rng):
    """n draws from pool with counts differing by at most one, in random order."""
    pool = list(pool)
    if not pool:
        raise ValueError("no admissible answers to balance over")
    out = []
    while len(out) < n:
        blk = pool[:]
        rng.shuffle(blk)
        out += blk
    out = out[:n]
    rng.shuffle(out)
    return out


def _pointer_body(W, tbl, s, k):
    return (W["head"] + " ".join(W["rule"].format(a=a, b=tbl[a]) for a in sorted(tbl))
            + " " + W["query"].format(s=s, k=k))


def _chain_body_text(W, stmts, v):
    return (W["head"] + " ".join(W["stmt"].format(x=x, y=y) for x, y in stmts)
            + " " + W["query"].format(v=v))


def _new_tally():
    """Worked-example answer counts: one Counter per example position, plus
    the pooled count used to break ties."""
    return {"slot": [], "all": collections.Counter()}


def _max_distinct(cands, taken):
    """How many of the positions (candidate lists) can get pairwise distinct
    answers outside `taken`: a maximum bipartite matching (at most ~10 x 10)."""
    owner = {}

    def augment(j, seen):
        for d in cands[j]:
            if d in taken or d in seen:
                continue
            seen.add(d)
            if d not in owner or augment(owner[d], seen):
                owner[d] = j
                return True
        return False

    return sum(augment(j, set()) for j in range(len(cands)))


def _examples(build, pool, S, rng, tally, avoid=(), own=True, shuffle=True, forbid=()):
    """Worked-example prefix with balanced answers.

    The example depths (shot_depths cycled to `shots`) come in random order when
    `shuffle`, so no position is tied to one depth.  An example's answer is never
    in `avoid` (the item's written numeral: the gold's own exclusion) nor, with
    `own`, the example's own step count.  Answers are distinct within the prefix
    as far as the exclusions allow (a maximum matching is kept reachable), and
    each is the least-used digit so far at its position (tally["slot"][j]), then
    overall (tally["all"]), ties at random.  One tally per depth makes every
    position balanced over the gold's admissible set at every k, so copying any
    example scores the same at every k.  build(kd, answer) returns the solved
    instance's text.

    No example is one of the `forbid` blocks (the item's own final problem):
    at the item's depth and gold an example could otherwise be the query with
    its answer shown.  Such an example is re-drawn with the same answer (the
    balance is kept); only when that depth-answer pair has no other problem
    (e.g. a single allowed increment) is another answer taken, and only when
    every answer does (degenerate settings) is the clash kept, for check_suite
    to fail.
    """
    depths = [S["shot_depths"][j % len(S["shot_depths"])] for j in range(S["shots"])]
    if shuffle:
        rng.shuffle(depths)
    avoid = {str(a) for a in avoid}
    cands = [[d for d in pool if str(d) not in avoid and not (own and str(d) == str(kd))]
             for kd in depths]
    if any(not c for c in cands):
        raise ValueError("a worked example has no admissible answer")
    while len(tally["slot"]) < len(cands):
        tally["slot"].append(collections.Counter())
    goal = _max_distinct(cands, set())
    used, answers, parts = set(), [], []
    for j, (kd, cand) in enumerate(zip(depths, cands)):
        rest = cands[j + 1:]
        fresh = [d for d in cand if d not in used
                 and len(used) + 1 + _max_distinct(rest, used | {d}) >= goal]
        opts = fresh or cand          # no fresh digit fits: a repeat cannot cost distinctness
        cnt = tally["slot"][j]

        def least_used(live):
            best = min((cnt[d], tally["all"][d]) for d in live)
            return rng.choice([d for d in live if (cnt[d], tally["all"][d]) == best])

        a = text = None
        for tier in (opts, [d for d in cand if d not in opts]):
            live = list(tier)
            while live and text is None:
                a = least_used(live)
                text = _build_avoiding(build, kd, a, forbid)
                live.remove(a)
            if text is not None:
                break
        if text is None:              # every answer at this depth is the query itself
            a = least_used(opts)
            text = build(kd, a)
        cnt[a] += 1
        tally["all"][a] += 1
        used.add(a)
        answers.append(a)
        parts.append(text + EXAMPLE_END.format(ans=a))
    return "".join(parts), answers, depths


# ────────────────────────────── generators ──────────────────────────────────
# Each returns a list of partial items: k, i, text, gold, answer_set, echo, setting.

def _gen_pointer(W, ks, n_per, rng, S):
    n = S["n_states"]
    states = list(range(n))

    def build(kd, a):
        tbl = _rand_cycle(rng, n)
        return _pointer_body(W, tbl, _back(tbl, a, kd), kd)

    shared = S["shot_mode"] == "shared"
    ex_blocks = set()
    if shared:
        # _gen_serial_depth_rep's prefix, draw for draw (answers not balanced:
        # this mode exists to reproduce that condition)
        parts, sh_ans, sh_k = [], [], []
        for j in range(S["shots"]):
            kd = S["shot_depths"][j % len(S["shot_depths"])]
            tbl, s = _rand_cycle(rng, n), rng.randrange(n)
            a = _walk(tbl, s, kd)
            parts.append(_pointer_body(W, tbl, s, kd) + EXAMPLE_END.format(ans=a))
            sh_ans.append(a); sh_k.append(kd)
        prefix = "".join(parts)
        ex_blocks = _example_blocks(prefix)
    items = []
    key = lambda t: tuple(t[d] for d in states)   # noqa: E731
    for k in ks:
        tally = _new_tally()                       # example answers, balanced within k
        targets = (_balanced([d for d in states if d != k], n_per, rng)
                   if S["balance"] else None)
        used = collections.defaultdict(set)        # cycles drawn per gold: no repeats
        for i in range(n_per):
            if targets is not None:
                g = targets[i]
                clash = ((lambda t: _pointer_body(W, t, _back(t, g, k), k) in ex_blocks)
                         if ex_blocks else None)
                tbl = _fresh(lambda: _rand_cycle(rng, n), key, used[g],
                             math.factorial(n - 1), clash)
                s = _back(tbl, g, k)
            else:                                  # legacy draw: resample gold == k
                while True:
                    tbl, s = _rand_cycle(rng, n), rng.randrange(n)
                    if _walk(tbl, s, k) != k:
                        break
            gold = _walk(tbl, s, k)
            final = _pointer_body(W, tbl, s, k)
            if not shared:                         # drawn after the final: never equal to it
                prefix, sh_ans, sh_k = _examples(build, states, S, rng, tally, avoid={k},
                                                 forbid={final})
            items.append(dict(k=k, i=i, text=prefix + final,
                              gold=str(gold), answer_set=[str(d) for d in states],
                              echo=str(k),
                              setting={"start": s, "table": [tbl[d] for d in states],
                                       "min_hops": _min_hops(tbl, s, gold),
                                       "shot_answers": list(sh_ans), "shot_ks": list(sh_k)}))
    return items


def _gen_fixed_digit(W, ks, n_per, rng, S):
    n, K = S["n_states"], S["written_k"]
    states = list(range(n))

    def build(kd, a):                              # ordinary pointer examples
        tbl = _rand_cycle(rng, n)
        return _pointer_body(W, tbl, _back(tbl, a, kd), kd)

    shared = S["shot_mode"] == "shared"
    ex_blocks = set()
    if shared:
        prefix, sh_ans, sh_k = _examples(build, states, S, rng, _new_tally(), avoid={K},
                                         shuffle=False)
        ex_blocks = _example_blocks(prefix)
    items = []
    key = lambda t: tuple(t[d] for d in states)   # noqa: E731
    for e in ks:
        tally = _new_tally()
        targets = _balanced([d for d in states if str(d) != str(K)], n_per, rng)
        c = e + 1                                  # cycle length through the start
        # c-cycles through a given gold: the other c-1 members in every cyclic order
        space = math.comb(n - 1, c - 1) * math.factorial(c - 1)
        used = collections.defaultdict(set)        # tables drawn per gold: no repeats
        for i in range(n_per):
            g = targets[i]

            def draw():
                members = rng.sample([d for d in states if d != g], c - 1) + [g]
                rng.shuffle(members)
                tbl = {d: d for d in states}
                for j in range(c):
                    tbl[members[j]] = members[(j + 1) % c]
                return tbl
            clash = ((lambda t: _pointer_body(W, t, _back(t, g, K), K) in ex_blocks)
                     if ex_blocks else None)
            tbl = _fresh(draw, key, used[g], space, clash)
            s = _back(tbl, g, K)
            final = _pointer_body(W, tbl, s, K)
            if not shared:                         # drawn after the final: never equal to it
                prefix, sh_ans, sh_k = _examples(build, states, S, rng, tally, avoid={K},
                                                 forbid={final})
            items.append(dict(k=e, i=i, text=prefix + final,
                              gold=str(g), answer_set=[str(d) for d in states], echo=str(K),
                              setting={"start": s, "table": [tbl[d] for d in states],
                                       "written_k": K, "cycle_len": c, "gold_eq_start": s == g,
                                       "min_hops": _min_hops(tbl, s, g),
                                       "shot_answers": list(sh_ans), "shot_ks": list(sh_k)}))
    return items


def _arith_query(W, m):
    """The frame for modulus m: the mod-10 template, else the general one."""
    return W["query"] if m == 10 else W["query_mod"]


def _gen_arith(W, ks, n_per, rng, S):
    incs, m = S["increments"], S["modulus"]
    Q = _arith_query(W, m)

    def allowed(k):                                # never back to the start: k*d != 0 mod m
        return sorted({d for d in incs if (k * d) % m})

    def build(kd, a):
        d = rng.choice(allowed(kd))
        return Q.format(s=(a - kd * d) % m, d=d, k=kd, m=m)

    digits = list(range(m))
    shared = S["shot_mode"] == "shared"
    ex_blocks = set()
    if shared:                                     # one prefix for every k: no echo to avoid
        prefix, sh_ans, sh_k = _examples(build, digits, S, rng, _new_tally(), shuffle=False)
        ex_blocks = _example_blocks(prefix)
    items = []
    for k in ks:
        tally = _new_tally()
        targets = _balanced([d for d in digits if d != k], n_per, rng)
        ds = allowed(k)
        used = collections.defaultdict(set)        # increments drawn per gold: no repeats
        for i in range(n_per):
            g = targets[i]
            clash = ((lambda d: Q.format(s=(g - k * d) % m, d=d, k=k, m=m) in ex_blocks)
                     if ex_blocks else None)
            d = _fresh(lambda: rng.choice(ds), lambda d: d, used[g], len(ds), clash)
            s = (g - k * d) % m
            final = Q.format(s=s, d=d, k=k, m=m)
            if not shared:                         # drawn after the final: never equal to it
                prefix, sh_ans, sh_k = _examples(build, digits, S, rng, tally, avoid={k},
                                                 forbid={final})
            items.append(dict(k=k, i=i, text=prefix + final,
                              gold=str(targets[i]), answer_set=[str(x) for x in digits],
                              echo=str(k),
                              setting={"start": s, "increment": d,
                                       "shot_answers": list(sh_ans), "shot_ks": list(sh_k)}))
    return items


def _chain_body(rng, target, C, H, pool, order, ranks=None):
    """C chains of H names, each rooted in a distinct digit; the target chain
    (root = target) sits at a random position.  Statement slots are dealt to the
    chains at random; order='interleaved' fills each chain's slots in chain
    order (definition before use), 'shuffled' in a random order, except that
    `ranks` (ranks[j] = rank of link j among its chain's statements) fixes the
    target chain's order, which _gen_chain balances across bodies."""
    roots = [target] + rng.sample([d for d in range(10) if d != target], C - 1)
    t = rng.randrange(C)
    roots[0], roots[t] = roots[t], roots[0]
    names = rng.sample(pool, C * H)
    chains = [names[c * H:(c + 1) * H] for c in range(C)]
    per = [[(ch[0], str(r))] + [(ch[j], ch[j - 1]) for j in range(1, H)]
           for ch, r in zip(chains, roots)]
    seq = [c for c in range(C) for _ in range(H)]  # which chain fills each slot
    rng.shuffle(seq)
    orders = []
    for c in range(C):
        if order == "interleaved":
            orders.append(list(range(H)))
        elif c == t and ranks is not None:
            orders.append(sorted(range(H), key=ranks.__getitem__))
        else:
            o = list(range(H))
            rng.shuffle(o)
            orders.append(o)
    pos, stmts = [0] * C, []
    for c in seq:
        stmts.append(per[c][orders[c][pos[c]]]); pos[c] += 1
    return {"stmts": stmts, "chains": chains, "roots": roots, "target": t}


def _gen_chain(family, W, ks, n_per, rng, S):
    # Chains one link longer than the deepest query: the last link is never
    # queried, so every queried entity is also the next link's value (mentioned
    # twice) at every k.  With chains exactly max(k) long the deepest cell alone
    # queried a leaf (mentioned once), a surface property confounded with depth.
    C, H, order = S["n_chains"], max(ks) + 1, S["order"]
    pool = LOWER if family == "binding" else UPPER

    def build(kd, a):
        b = _chain_body(rng, a, C, H, pool, order)
        return _chain_body_text(W, b["stmts"], b["chains"][b["target"]][kd - 1])

    def finals(b):                                # the body's final block at every k
        return [_chain_body_text(W, b["stmts"], b["chains"][b["target"]][k - 1]) for k in ks]

    digits = list(range(10))
    tally = _new_tally()                          # bodies are shared across k
    shared = S["shot_mode"] == "shared"
    ex_blocks = set()
    if shared:                                    # no numerals: nothing to avoid
        prefix, sh_ans, sh_k = _examples(build, digits, S, rng, tally, own=False,
                                         shuffle=False)
        ex_blocks = _example_blocks(prefix)
    targets = _balanced(digits, n_per, rng)
    bodies, used, ranks = [], set(), None
    for i in range(n_per):
        if order == "shuffled":
            # Within each block of H bodies link j takes every rank once among
            # the target chain's statements (a random permutation composed with a
            # rotation, so each body's order is still uniform): the queried
            # definition's distance to the query does not trend with k.
            if i % H == 0:
                perm = list(range(H))
                rng.shuffle(perm)
            ranks = [perm[(j + i) % H] for j in range(H)]
        clash = (lambda b: any(f in ex_blocks for f in finals(b))) if ex_blocks else None
        b = _fresh(lambda: _chain_body(rng, targets[i], C, H, pool, order, ranks),
                   lambda b: tuple(b["stmts"]), used, math.inf, clash)
        if not shared:                            # drawn after the body: never equal to it
            prefix, sh_ans, sh_k = _examples(build, digits, S, rng, tally, own=False,
                                             forbid=set(finals(b)))
        bodies.append((prefix, list(sh_ans), list(sh_k), b))
    items = []
    for k in ks:
        for i, (pre, sa, sk, b) in enumerate(bodies):
            v = b["chains"][b["target"]][k - 1]
            xs = [x for x, _ in b["stmts"]]
            items.append(dict(k=k, i=i, text=pre + _chain_body_text(W, b["stmts"], v),
                              gold=str(targets[i]),
                              answer_set=sorted(str(r) for r in b["roots"]), echo=None,
                              setting={"body": i, "query": v, "roots": list(b["roots"]),
                                       "target_chain": b["target"],
                                       "def_distance": len(xs) - xs.index(v),
                                       "query_refs": sum(y == v for _, y in b["stmts"]),
                                       "shot_answers": sa, "shot_ks": sk}))
    return items


_GEN = {
    "pointer": _gen_pointer,
    "fixed_digit": _gen_fixed_digit,
    "arith": _gen_arith,
    "binding": lambda W, ks, n, rng, S: _gen_chain("binding", W, ks, n, rng, S),
    "relational": lambda W, ks, n, rng, S: _gen_chain("relational", W, ks, n, rng, S),
}


# ────────────────────────────── settings ────────────────────────────────────
def _settings(family, lo, hi, given):
    """Merge defaults, reject unknown keys, validate ranges."""
    D = FAMILIES[family]["defaults"]
    unknown = set(given) - set(D)
    if unknown:
        raise ValueError(f"{family}: unknown setting(s) {sorted(unknown)}; "
                         f"known: {sorted(D)}")
    S = json.loads(json.dumps(D))                 # deep copy, JSON types only
    try:                                          # tuples -> lists, so save/load round-trips
        S.update(json.loads(json.dumps(_plain(given))))
    except (TypeError, ValueError) as e:
        raise ValueError(f"{family}: settings must be JSON values (numbers, strings, "
                         f"lists): {e}") from None
    if not isinstance(S["shots"], int) or S["shots"] < 0:
        raise ValueError("shots must be an int >= 0")
    if S["shot_mode"] not in ("per_item", "shared"):
        raise ValueError("shot_mode must be 'per_item' or 'shared'")
    if not 0 <= S["calib_frac"] < 1:
        raise ValueError("calib_frac must be in [0, 1)")
    if lo > hi:
        raise ValueError(f"k_range {lo, hi}: low above high")
    if lo < (0 if family == "fixed_digit" else 1):
        raise ValueError(f"{family}: k must be >= {0 if family == 'fixed_digit' else 1}")

    if family in ("pointer", "fixed_digit"):
        n = S["n_states"]
        if not 2 <= n <= 10:
            raise ValueError("n_states must be 2..10 (single-digit states)")
        dmax = n - 1                              # worked examples are n-cycle walks
        if family == "pointer":
            if hi >= n:
                raise ValueError(f"pointer: k must stay below n_states ({hi} >= {n}); a "
                                 f"longer walk revisits states")
        else:
            if S["written_k"] is None:
                S["written_k"] = _auto_written_k(lo, hi)
            K = S["written_k"]
            if not isinstance(K, int) or not 1 <= K <= 9:
                raise ValueError("written_k must be a single digit 1..9")
            if hi > min(K, n - 1):
                raise ValueError(f"fixed_digit: effective depth {hi} exceeds min(written_k, "
                                 f"n_states-1) = {min(K, n - 1)}")
    elif family == "arith":
        if hi > 9:
            raise ValueError("arith: k must be a single digit")
        m = S["modulus"]
        if isinstance(m, bool) or not isinstance(m, int) or not 2 <= m <= 10:
            raise ValueError("modulus must be an int in 2..10 (single-digit answers)")
        if "increments" not in given:
            S["increments"] = list(range(1, m))
        incs = S["increments"]
        if not incs or any(isinstance(d, bool) or not isinstance(d, int) or not 1 <= d < m
                           for d in incs):
            raise ValueError(f"increments must be ints in 1..{m - 1} (modulus {m})")
        dmax = 9
    else:
        pool = LOWER if family == "binding" else UPPER
        links = hi + 1                            # the last link is never queried
        fit = len(pool) // links
        if S["n_chains"] is None:                 # 4 where they fit, else as many as fit
            if fit < 2:
                raise ValueError(f"{family}: k up to {hi} needs chains of {links} names; "
                                 f"{len(pool)} single-letter names hold fewer than 2 such "
                                 f"chains (max k is {len(pool) // 2 - 1})")
            S["n_chains"] = min(4, fit)
        C = S["n_chains"]
        if isinstance(C, bool) or not isinstance(C, int) or not 2 <= C <= 10:
            raise ValueError("n_chains must be an int 2..10 (distinct root digits, >= 1 "
                             "distractor), or None for the most that fit up to 4")
        if C * links > len(pool):
            raise ValueError(f"{family}: {C} chains of {links} names (max k {hi} plus the "
                             f"unqueried last link) need {C * links}; only {len(pool)} "
                             f"single-letter names (n_chains <= {fit} fits, or lower max k)")
        if S["order"] not in ("interleaved", "shuffled"):
            raise ValueError("order must be 'interleaved' or 'shuffled'")
        dmax = hi                                 # examples use the same chain length

    if "shot_depths" in given:
        bad = [d for d in S["shot_depths"] if not isinstance(d, int) or not 1 <= d <= dmax]
        if bad or not S["shot_depths"]:
            raise ValueError(f"{family}: shot_depths must be ints in 1..{dmax}")
    else:                                         # defaults: keep those that fit
        fits = (lambda d: d <= dmax and any((d * i) % S["modulus"] for i in S["increments"])
                ) if family == "arith" else (lambda d: d <= dmax)
        S["shot_depths"] = [d for d in S["shot_depths"] if fits(d)] or [max(lo, 1)]
    if family == "arith":
        # an increment with k*d = 0 mod m returns to the start; none may remain
        m = S["modulus"]
        for k in list(range(lo, hi + 1)) + (S["shot_depths"] if S["shots"] else []):
            if not any((k * d) % m for d in S["increments"]):
                raise ValueError(f"arith: every increment gives k*d = 0 mod {m} at k={k}")
    if family == "pointer" and not isinstance(S["balance"], bool):
        raise ValueError("balance must be a bool")
    return S


def _plain(v):
    """A setting value in plain JSON types: numpy scalars and arrays (a grid
    built with np.arange) become the equal int/float/list, tuples lists, so
    the name, equality and save/load all see one value."""
    if v is None or isinstance(v, (bool, str, int, float)):
        return v
    if isinstance(v, dict):
        return {k: _plain(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [_plain(x) for x in v]
    if hasattr(v, "tolist"):                      # numpy scalar or array
        return _plain(v.tolist())
    return v


def _copy_start_depths(lo, hi, K):
    """Effective depths at which a K-step walk ends on its start: the cycle
    through the start has length e+1, and the walk wraps home when e+1 divides K."""
    return [e for e in range(lo, hi + 1) if e < K and K % (e + 1) == 0]


def _auto_written_k(lo, hi):
    """The single-digit numeral >= hi with the fewest copy-the-start cells
    (ties: the smallest).  For k 1..6 that is 7 (only e=6 wraps home); 6 would
    make e=1, 2 and 5 all answerable by copying the start."""
    return min(range(max(hi, 1), 10), key=lambda K: (len(_copy_start_depths(lo, hi, K)), K))


def _auto_name(family, wording, lo, hi, n_per, seed, S):
    """Readable main knobs plus a hash of the full setting, so every point of a
    settings grid gets its own file name."""
    knob = {"pointer": lambda: f"_st{S['n_states']}",
            "fixed_digit": lambda: f"_st{S['n_states']}K{S['written_k']}",
            "arith": lambda: "" if S["modulus"] == 10 else f"_m{S['modulus']}",
            "binding": lambda: f"_ch{S['n_chains']}",
            "relational": lambda: f"_ch{S['n_chains']}"}[family]()
    canon = json.dumps({"family": family, "wording": wording, "k_range": [lo, hi],
                        "n_per": n_per, "seed": seed, "generator": GENERATOR_VERSION, **S},
                       sort_keys=True)
    h = hashlib.sha256(canon.encode()).hexdigest()[:6]
    mode = "p" if S["shot_mode"] == "per_item" else "s"
    return (f"{family}_{wording}_k{lo}-{hi}{knob}_sh{S['shots']}{mode}"
            f"_n{n_per}_seed{seed}_{h}")


# ────────────────────────────── split ───────────────────────────────────────
def _stratified_pick(golds, sizes, n_pick, rng):
    """Indices of the units (groups of sizes[j] items, one gold each) that make
    up n_pick items, spread over gold digits as the singleton rule would: items
    taken round-robin across golds, larger golds first, give each gold a quota,
    so the calib and test parts both stay answer-balanced.  Units are indivisible;
    when their sizes cannot meet the quotas exactly, the total nearest n_pick
    and then the counts nearest the quotas win."""
    groups = {}
    for idx, g in enumerate(golds):
        groups.setdefault(g, []).append(idx)
    for v in groups.values():
        rng.shuffle(v)
    keys = sorted(groups)
    rng.shuffle(keys)
    tot = {g: sum(sizes[j] for j in groups[g]) for g in keys}
    keys.sort(key=lambda g: -tot[g])              # stable: ties stay in random order
    quota, left = dict.fromkeys(keys, 0), n_pick
    while left > 0 and any(quota[g] < tot[g] for g in keys):
        for g in keys:
            if left > 0 and quota[g] < tot[g]:
                quota[g] += 1
                left -= 1
    # per gold, one subset of its units for every reachable item count (the first
    # found in the shuffled order: for single-item units a random subset)
    reach = {}
    for g in keys:
        r = {0: []}
        for j in groups[g]:
            for s, sub in list(r.items()):
                r.setdefault(s + sizes[j], sub + [j])
        reach[g] = r
    best = {0: (0, [])}                           # total -> (deviation from quotas, choice)
    for g in keys:
        nxt = {}
        for t, (dev, ch) in best.items():
            for s in reach[g]:
                d = dev + abs(s - quota[g])
                if t + s not in nxt or d < nxt[t + s][0]:
                    nxt[t + s] = (d, ch + [(g, s)])
        best = nxt
    t = min(best, key=lambda t: (abs(t - n_pick), best[t][0], t))
    return {j for g, s in best[t][1] for j in reach[g][s]}


def _calib_members(members, key, gold, frac, rng):
    """The members for calib: round(frac * len(members)), answer-balanced, with
    members of one key (the same final problem) kept together on one side."""
    units = {}
    for m in members:
        units.setdefault(key(m), []).append(m)
    units = list(units.values())
    pick = _stratified_pick([gold(u[0]) for u in units], [len(u) for u in units],
                            round(frac * len(members)), rng)
    return {m for j in pick for m in units[j]}


def _assign_split(items, frac, seed, family, matched):
    """calib / test per item.  Identical problems go to one side whole: the same
    final problem in both would let calibration (E1) see test items (E2)."""
    rng = random.Random(f"split:{family}:{seed}")  # own stream: split never moves items
    last = lambda it: it["text"].split("\n\n")[-1]  # noqa: E731
    if matched:
        # a body is shared across k, so the whole body goes to one side
        first = {}
        for it in items:
            first.setdefault(it["i"], it)
        calib = _calib_members(sorted(first), lambda b: last(first[b]),
                               lambda b: first[b]["gold"], frac, rng)
        for it in items:
            it["split"] = "calib" if it["i"] in calib else "test"
        return
    for k in sorted({it["k"] for it in items}):
        at = [j for j, it in enumerate(items) if it["k"] == k]
        calib = _calib_members(at, lambda j: last(items[j]), lambda j: items[j]["gold"],
                               frac, rng)
        for j in at:
            items[j]["split"] = "calib" if j in calib else "test"


# ────────────────────────────── public API ──────────────────────────────────
class _DefaultInt(int):
    """An int default that make_suite can tell apart from the same value passed
    explicitly (the ladder takes no arguments; a grid loop must not slip past)."""


class _DefaultRange(tuple):
    """The tuple counterpart of _DefaultInt."""


_K_RANGE, _N_PER, _SEED = _DefaultRange((1, 6)), _DefaultInt(40), _DefaultInt(0)


def make_suite(family, wording="w0", k_range=_K_RANGE, n_per=_N_PER, seed=_SEED, **setting):
    """Build a suite dict (see the module docstring).  k_range is inclusive
    (default (1, 6)), n_per items per k (default 40), seed an int (default 0;
    anything else, None included, raises: a suite name promises reproducible
    items).  Settings not given take FAMILIES[family]['defaults']; unknown ones
    raise; `name=` overrides the automatic name.  family="ladder" returns
    ladder_suite() and raises if given any other argument, since the original
    ladder has no knobs."""
    if family == "ladder":
        given = [n for n, v, d in (("k_range", k_range, _K_RANGE), ("n_per", n_per, _N_PER),
                                   ("seed", seed, _SEED)) if v is not d] + sorted(setting)
        if wording != "w0" or given:
            raise ValueError(f"ladder: the original single-prompt ladder is fixed (w0, k 1..6, "
                             f"one item per k) and takes no {given or ['wording']}; call "
                             f"ladder_suite(), or use family 'pointer' for a many-item ladder")
        return ladder_suite()
    if family not in _GEN:
        raise ValueError(f"unknown family {family!r}; one of {sorted(_GEN)}")
    if wording not in WORDINGS[family]:
        raise ValueError(f"{family}: unknown wording {wording!r}; one of "
                         f"{sorted(WORDINGS[family])}")
    try:
        seed = operator.index(seed)               # ints, numpy ints; not None, floats, str
    except TypeError:
        raise ValueError(f"seed must be an int, got {seed!r}: a random seed would give "
                         f"different items under one name and one set of ids") from None
    name = setting.pop("name", None)
    try:                                          # (lo, hi); a list of depths is not one
        lo, hi = (operator.index(x) for x in k_range)
    except (TypeError, ValueError):
        raise ValueError(f"k_range must be an inclusive (lo, hi) pair of ints, got "
                         f"{k_range!r}") from None
    if int(n_per) < 1:
        raise ValueError("n_per must be >= 1")
    n_per = int(n_per)
    S = _settings(family, lo, hi, setting)
    name = name or _auto_name(family, wording, lo, hi, n_per, seed, S)
    if "/" in name:
        raise ValueError("suite name must not contain '/' (it is part of every item id)")

    # pointer keeps Random(seed) (serial_depth_fs is drawn from it); every other
    # family has its own stream, so no two families share instances
    rng = random.Random(seed if family == "pointer" else f"{family}:{seed}")
    raw = _GEN[family](WORDINGS[family][wording], list(range(lo, hi + 1)), n_per, rng, S)
    _assign_split(raw, S["calib_frac"], seed, family, FAMILIES[family]["matched"])
    items = [{"id": f"{name}/{r['k']}/{r['i']}", "family": family, "wording": wording,
              "k": r["k"], "label": f"D{r['k']}N{r['i']:02d}", "text": r["text"],
              "gold": r["gold"], "answer_set": r["answer_set"],
              "k_written": FAMILIES[family]["k_written"], "echo": r["echo"],
              "split": r["split"], "setting": r["setting"]} for r in raw]
    sizes = {len(it["answer_set"]) for it in items}
    suite = {"name": name, "family": family, "wording": wording,
             "setting": {"k_range": [lo, hi], "n_per": n_per, "seed": seed, **S},
             "chance": 1 / max(sizes), "items": items}
    verify_suite(suite)
    return suite


def ladder_suite():
    """The original single-prompt ladder, active_subgraph_dot.TASK_SUITES
    ['serial_depth'], as a suite (texts and answers unchanged)."""
    from active_subgraph_dot import TASK_SUITES, _PERM
    src = TASK_SUITES["serial_depth"]
    n = len(_PERM)
    items = []
    for text, label, gold in zip(src["tasks"], src["labels"], src["_answers"]):
        k = int(re.match(r"D(\d+)", label).group(1))
        _, info = solve_text("ladder", "w0", text)
        items.append({"id": f"ladder/{k}/0", "family": "ladder", "wording": "w0", "k": k,
                      "label": f"D{k}N00", "text": text, "gold": str(gold),
                      "answer_set": [str(d) for d in range(n)], "k_written": True,
                      "echo": str(k), "split": "test",
                      "setting": {"start": info["start"],
                                  "table": [_PERM[d] for d in range(n)],
                                  "min_hops": info["min_hops"], "orig_label": label}})
    ks = [it["k"] for it in items]
    suite = {"name": "ladder", "family": "ladder", "wording": "w0",
             "setting": {"source": "active_subgraph_dot.TASK_SUITES['serial_depth']",
                         "k_range": [min(ks), max(ks)], "n_per": 1, "n_states": n},
             "chance": 1 / n, "items": items}
    verify_suite(suite)
    return suite


def save_suite(suite, path):
    """Verify, then write atomically (a Colab disconnect must not leave half a file)."""
    verify_suite(suite)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(suite, indent=1))
    tmp.replace(path)
    return path


def load_suite(path):
    suite = json.loads(Path(path).read_text())
    verify_suite(suite)
    return suite


# ────────────────────────────── verification ────────────────────────────────
_PH = {"a": r"\d", "b": r"\d", "s": r"\d", "k": r"\d", "d": r"\d", "m": r"\d",
       "x": r"(?<![A-Za-z0-9])[A-Za-z]", "v": r"(?<![A-Za-z0-9])[A-Za-z]",
       "y": r"(?<![A-Za-z0-9])[A-Za-z0-9]"}


def _tmpl_re(t):
    out, pos = "", 0
    for m in re.finditer(r"\{(\w+)\}", t):
        out += re.escape(t[pos:m.start()]) + f"(?P<{m.group(1)}>{_PH[m.group(1)]})"
        pos = m.end()
    return out + re.escape(t[pos:])


def solve_text(family, wording, text):
    """Answer the final query of `text` from the text alone.

    Independent of the generators: parses the last block with the WORDINGS
    templates, checks that re-rendering the parse gives the block back exactly,
    and computes the answer.  Returns (answer digit, info) where info has
    'depth' (the k the item should carry), 'answer_set' (the admissible answers
    the text defines: its states for pointer-like and arith, the digit literals
    of the final block for binding/relational), 'written' if the text writes a
    step count, and what can be read off the text (start, table, min_hops,
    increment, modulus, query).
    """
    W = WORDINGS[family][wording]
    block = text.split("\n\n")[-1]
    if family == "arith":
        for key, m in (("query", 10), ("query_mod", None)):
            q = re.fullmatch(_tmpl_re(W[key]), block)
            if q is not None:
                break
        else:
            raise ValueError(f"arith/{wording}: query not found in {block[-80:]!r}")
        m = m or int(q["m"])
        s, d, K = int(q["s"]), int(q["d"]), int(q["k"])
        if not (2 <= m <= 10 and 0 <= s < m and 1 <= d < m):
            raise ValueError(f"arith/{wording}: start {s} / increment {d} outside mod {m}: "
                             f"{block!r}")
        return str((s + K * d) % m), {"depth": K, "written": K, "start": s, "increment": d,
                                      "modulus": m,
                                      "answer_set": [str(x) for x in range(m)]}
    q = re.search((" " if "head" in W else "") + _tmpl_re(W["query"]) + "$", block)
    if q is None or not block.startswith(W.get("head", "")):
        raise ValueError(f"{family}/{wording}: query not found in {block[-80:]!r}")
    mid = block[len(W.get("head", "")):q.start()]
    if family in ("pointer", "fixed_digit", "ladder"):
        tbl = {int(m["a"]): int(m["b"]) for m in re.finditer(_tmpl_re(W["rule"]), mid)}
        s, K = int(q["s"]), int(q["k"])
        if (sorted(tbl) != list(range(len(tbl))) or sorted(tbl.values()) != sorted(tbl)
                or s not in tbl or _pointer_body(W, tbl, s, K) != block):
            raise ValueError(f"{family}/{wording}: rule table does not parse as a "
                             f"permutation: {block[:120]!r}")
        seen = [s]
        for _ in range(K):
            seen.append(tbl[seen[-1]])
        eff = len(set(seen)) - 1
        return str(seen[-1]), {"depth": eff if family == "fixed_digit" else K,
                               "written": K, "effective": eff, "start": s,
                               "table": [tbl[x] for x in range(len(tbl))],
                               "min_hops": _min_hops(tbl, s, seen[-1]),
                               "answer_set": [str(x) for x in range(len(tbl))]}
    stmts = [(m["x"], m["y"]) for m in re.finditer(_tmpl_re(W["stmt"]), mid)]
    mp = dict(stmts)
    if len(mp) != len(stmts) or _chain_body_text(W, stmts, q["v"]) != block:
        raise ValueError(f"{family}/{wording}: statements do not parse: {block[:120]!r}")
    v, hops = q["v"], 0
    while not v.isdigit():
        if v not in mp or hops > len(mp):
            raise ValueError(f"{family}/{wording}: {q['v']} has no root")
        v, hops = mp[v], hops + 1
    xs = [x for x, _ in stmts]
    return v, {"depth": hops, "query": q["v"],
               "def_distance": len(xs) - xs.index(q["v"]),
               "query_refs": sum(y == q["v"] for _, y in stmts),
               "answer_set": sorted({y for _, y in stmts if y.isdigit()})}


def _is_digit_str(x):
    return isinstance(x, str) and len(x) == 1 and x in DIGITS


def verify_suite(suite):
    """Schema and answer checks; raises ValueError on the first failing suite
    rather than let a wrong gold, answer set, echo or chance reach a
    measurement.  Everything an item claims about its prompt is recomputed from
    the text by solve_text: gold, depth, answer set (competitors downstream),
    whether and which numeral is written (echo, k_written), start / table /
    min_hops / increment / query, and every worked example; chance must be
    1/|answer set|; the suite must have items."""
    probs = []
    miss = [k for k in SUITE_KEYS if k not in suite]
    if miss:
        raise ValueError(f"suite lacks {miss}")
    fam, wd = suite["family"], suite["wording"]
    if fam not in WORDINGS or wd not in WORDINGS[fam]:
        raise ValueError(f"unknown family/wording {fam}/{wd}")
    if not isinstance(suite["items"], list) or not suite["items"]:
        raise ValueError(f"suite {suite['name']} has no items")
    spec = FAMILIES[fam]
    ss = suite["setting"] if isinstance(suite["setting"], dict) else {}
    ids, sizes = set(), []
    for it in suite["items"]:
        miss = [k for k in ITEM_KEYS if k not in it]
        if miss:
            probs.append(f"{it.get('id')}: lacks {miss}"); continue
        tag = it["id"]
        if it["id"] in ids:
            probs.append(f"{tag}: duplicate id")
        ids.add(it["id"])
        m_id = re.fullmatch(re.escape(suite["name"]) + rf"/{it['k']}/(\d+)", str(it["id"]))
        if not m_id:
            probs.append(f"{tag}: id is not <name>/<k>/<i>")
        elif it["label"] != f"D{it['k']}N{int(m_id.group(1)):02d}":
            probs.append(f"{tag}: label {it['label']!r} does not carry the id's k and i")
        if it["family"] != fam or it["wording"] != wd:
            probs.append(f"{tag}: family/wording differ from the suite's")
        if it["split"] not in ("calib", "test"):
            probs.append(f"{tag}: split {it['split']!r}")
        A, g = it["answer_set"], it["gold"]
        if (not isinstance(A, list) or not A or not all(map(_is_digit_str, A))
                or len(set(A)) != len(A)):
            probs.append(f"{tag}: answer_set {A!r} is not a list of distinct digits")
        else:
            sizes.append(len(A))
        if not _is_digit_str(g) or not isinstance(A, list) or g not in A:
            probs.append(f"{tag}: gold {g!r} is not a digit in the answer set")
        if it["k_written"] is not spec["k_written"]:
            probs.append(f"{tag}: k_written {it['k_written']!r}, but {fam} "
                         f"{'writes' if spec['k_written'] else 'never writes'} its step count")
        if it["k_written"] != (it["echo"] is not None):
            probs.append(f"{tag}: echo {it['echo']!r} inconsistent with k_written")
        if it["echo"] is not None and it["echo"] == g:
            probs.append(f"{tag}: gold equals the echoed numeral {g}")
        if not isinstance(it["setting"], dict):
            probs.append(f"{tag}: setting is not a dict")
        st = it["setting"] if isinstance(it["setting"], dict) else {}
        try:
            ans, info = solve_text(fam, wd, it["text"])
            if ans != g:
                probs.append(f"{tag}: text solves to {ans}, gold says {g}")
            if info["depth"] != it["k"]:
                probs.append(f"{tag}: text has depth {info['depth']}, item says k={it['k']}")
            if ("written" in info) != bool(it["k_written"]):
                probs.append(f"{tag}: the text {'writes' if 'written' in info else 'writes no'} "
                             f"step count, item says k_written={it['k_written']!r}")
            if "written" in info and str(info["written"]) != it["echo"]:
                probs.append(f"{tag}: written numeral {info['written']} != echo {it['echo']!r}")
            if A != info["answer_set"]:
                probs.append(f"{tag}: answer_set {A!r}, but the text admits "
                             f"{info['answer_set']}")
            for key in ("start", "table", "min_hops", "increment", "query", "def_distance",
                        "query_refs"):
                if key in st and key in info and st[key] != info[key]:
                    probs.append(f"{tag}: setting.{key} {st[key]!r}, the text gives "
                                 f"{info[key]!r}")
            # the family's form, without which k is not a serial depth
            if fam in ("pointer", "ladder") and len(_cycles(dict(enumerate(info["table"])))) != 1:
                probs.append(f"{tag}: the rule table is not one {len(info['table'])}-cycle")
            if fam == "fixed_digit":
                moved = [c for c in _cycles(dict(enumerate(info["table"]))) if len(c) > 1]
                if len(moved) > 1 or (moved and info["start"] not in moved[0]):
                    probs.append(f"{tag}: the rule table is not one cycle through the start "
                                 f"plus fixed points")
            if fam == "arith" and (info["depth"] * info["increment"]) % info["modulus"] == 0:
                probs.append(f"{tag}: k*d = 0 mod {info['modulus']}: the walk returns to its "
                             f"start, so k is not a serial depth")
            if "written_k" in st and st["written_k"] != info.get("written"):
                probs.append(f"{tag}: setting.written_k {st['written_k']!r} != written "
                             f"{info.get('written')!r}")
            if "gold_eq_start" in st and st["gold_eq_start"] != (g == str(info.get("start"))):
                probs.append(f"{tag}: setting.gold_eq_start {st['gold_eq_start']!r} is wrong")
            if "roots" in st and sorted(str(r) for r in st["roots"]) != info["answer_set"]:
                probs.append(f"{tag}: setting.roots {st['roots']!r} are not the text's roots")
            if fam == "arith" and ss.get("modulus", 10) != info["modulus"]:
                probs.append(f"{tag}: text is mod {info['modulus']}, suite says "
                             f"{ss.get('modulus', 10)}")
            n_key = "n_chains" if fam in ("binding", "relational") else "n_states"
            if n_key in ss and fam != "arith" and len(info["answer_set"]) != ss[n_key]:
                probs.append(f"{tag}: {len(info['answer_set'])} admissible answers, suite "
                             f"{n_key} = {ss[n_key]}")
            # every worked example must be solved correctly as well
            efam = "pointer" if fam in ("fixed_digit", "ladder") else fam
            for b in it["text"].split("\n\n")[:-1]:
                m = re.search(r" (\d)\.$", b)
                if m is None or solve_text(efam, wd, b[:m.start()])[0] != m.group(1):
                    probs.append(f"{tag}: a worked example is wrong: {b[-60:]!r}")
        except ValueError as e:
            probs.append(f"{tag}: {e}")
        if len(probs) > 20:
            break
    c = suite["chance"]
    if sizes and (isinstance(c, bool) or not isinstance(c, (int, float))
                  or abs(c - 1 / max(sizes)) > 1e-9):
        probs.append(f"chance {c!r} != 1/{max(sizes)} (the largest answer set)")
    if probs:
        raise ValueError(f"suite {suite['name']} failed verification "
                         f"({len(probs)} problem(s)):\n  " + "\n  ".join(probs[:20]))
    return True


# ────────────────────────────── check_suite ─────────────────────────────────
def _named_tokenizers(tokenizers):
    """[(name, tokenizer)] from a dict, one tokenizer, a list of them, or
    tokenizer names (a name stays a str here; check_suite loads it as the CLI
    does)."""
    if tokenizers is None:
        return []
    if isinstance(tokenizers, str) or hasattr(tokenizers, "convert_ids_to_tokens"):
        tokenizers = [tokenizers]
    if isinstance(tokenizers, dict):
        return list(tokenizers.items())
    return [(t if isinstance(t, str) else getattr(t, "name_or_path", None) or f"tokenizer{j}", t)
            for j, t in enumerate(tokenizers)]


def _example_answers(text):
    """Worked-example answers of a prompt, in order (from the text)."""
    out = []
    for b in text.split("\n\n")[:-1]:
        m = re.search(r" (\d)\.$", b)
        if m:
            out.append(m.group(1))
    return out


def _query_among_examples(text):
    """Is the final problem one of the prompt's own worked examples (answer shown)?"""
    blocks = text.split("\n\n")
    return blocks[-1] in {re.sub(r" \d\.$", "", b) for b in blocks[:-1]}


def _corr(x, y):
    """Pearson correlation; 0 when either side is constant."""
    n = len(x)
    mx, my = sum(x) / n, sum(y) / n
    sxy = sum((a - mx) * (b - my) for a, b in zip(x, y))
    sxx, syy = sum((a - mx) ** 2 for a in x), sum((b - my) ** 2 for b in y)
    return sxy / math.sqrt(sxx * syy) if sxx and syy else 0.0


def check_suite(suite, tokenizers, min_correct=20):
    """Report (never raise) everything a run needs to trust the suite.

    Tokenizer-independent: verification (each failing item and why), per-k
    answer balance over the admissible digits (answer sets minus the echo;
    balanced = counts differ by at most one), fraction of gold == echo (must be
    0), fraction of items with a digit other than gold in the prompt (must be
    1), split counts per k, and how often gold equals the start state where
    there is one.  Per k, distinct final problems (rep["distinct_problems"]),
    identical prompts (rep["duplicate_prompts"]; must be 0: one measurement
    would count as several items) and items whose final problem is one of their
    own worked examples (rep["example_is_query"]; must be 0: copying solves
    them).
    Worked examples (rep["examples"][k]): per example position the answer
    counts, copy_expected (accuracy of always copying that example if it were
    independent of gold: sum_d P(gold=d) P(answer=d)) and copy_realised, plus
    echo_frac (items whose examples answer the item's written numeral; must be
    0 for shot_mode="per_item").
    Per tokenizer (rep["tokenizers"][name]; tokenizer objects, a dict of them,
    or names to load): token counts per k and whether one count holds for the
    whole suite (constant, n_tokens); the same for the fed length after
    deep_probe.align_answer (feed_counts, feed_constant); whether align_answer
    succeeds on every item (align_ok, align_failures); the appended token forms;
    for matched families, whether prompts of one body have one length and differ
    in at most one position across k (the queried entity); plus a copy of the
    tokenizer-independent checks above.
    `ok` is True only when every check passes; `problems` lists what failed.
    `warnings` lists design limits the analysis must handle, without failing:
    shortest lookup path below k (rep["min_hops"]), arith offsets that k fixes
    (rep["arith_offsets"]), repeated final problems (more items per depth than
    the settings have problems), a queried definition whose distance to the
    query trends with k (rep["def_distance"], binding/relational) or a queried
    entity whose use as a value varies with k (rep["query_refs"]), copy accuracy
    that varies with k (depths with >= 20 items only, below that it is noise), a
    calib split with fewer than `min_correct` items per depth (then no depth can
    pass eligible_depths), and prompts longer than a tokenizer's
    model_max_length (e.g. GPT-2's 1024 positions).
    """
    rep = {"name": None, "family": None, "wording": None, "n_items": 0, "n_per_k": {},
           "verified": False, "tokenizers": {}, "problems": [], "warnings": []}
    try:
        _check_suite(suite, tokenizers, min_correct, rep)
    except Exception as e:                        # a malformed suite: report it, never raise
        rep["problems"].append(f"check_suite could not finish on this suite: "
                               f"{type(e).__name__}: {e}")
    rep["ok"] = not rep["problems"]
    return rep


def _check_suite(suite, tokenizers, min_correct, rep):
    from deep_probe import align_answer
    P, Wn = rep["problems"], rep["warnings"]
    rep.update(name=suite.get("name"), family=suite.get("family"),
               wording=suite.get("wording"))
    items = suite.get("items") or []
    rep["n_items"] = len(items)
    if not items:                                 # e.g. a split filter left nothing
        P.append("suite has no items")
        return
    try:
        verify_suite(suite)
        rep["verified"] = True
    except ValueError as e:                       # keep which item failed, and why
        lines = str(e).splitlines()
        P.append(lines[0])
        P.extend(ln.strip() for ln in lines[1:])
    ks = sorted({it["k"] for it in items})
    by_k = {k: [it for it in items if it["k"] == k] for k in ks}
    rep["n_per_k"] = {k: len(v) for k, v in by_k.items()}

    bal = {}
    for k, its in by_k.items():
        echoes = {it["echo"] for it in its}
        adm = sorted(set().union(*(it["answer_set"] for it in its))
                     - (echoes if len(echoes) == 1 else set()))
        cnt = {d: sum(it["gold"] == d for it in its) for d in adm}
        spread = max(cnt.values()) - min(cnt.values()) if cnt else 0
        bal[k] = {"counts": cnt, "spread": spread, "balanced": spread <= 1}
    rep["answer_balance"] = bal
    rep["balanced"] = all(b["balanced"] for b in bal.values())
    if not rep["balanced"]:
        P.append("answer imbalance at k=" + ",".join(str(k) for k, b in bal.items()
                                                     if not b["balanced"]))
    rep["gold_eq_echo"] = sum(it["echo"] == it["gold"] for it in items) / len(items)
    if rep["gold_eq_echo"]:
        P.append(f"gold equals echo on {rep['gold_eq_echo']:.1%} of items")
    rep["distractor_frac"] = sum(bool(set(re.findall(r"\d", it["text"])) - {it["gold"]})
                                 for it in items) / len(items)
    if rep["distractor_frac"] < 1:
        P.append(f"gold is the only digit in {1 - rep['distractor_frac']:.1%} of prompts")
    rep["split"] = {k: {s: sum(it["split"] == s for it in its) for s in ("calib", "test")}
                    for k, its in by_k.items()}
    rep["gold_eq_start"] = {k: sum(str(it["setting"].get("start")) == it["gold"]
                                   for it in its) / len(its)
                            for k, its in by_k.items() if "start" in its[0]["setting"]}
    ss = suite.get("setting") or {}

    # repeated problems: one final problem is one problem, one prompt one measurement
    rep["distinct_problems"] = {k: len({it["text"].split("\n\n")[-1] for it in its})
                                for k, its in by_k.items()}
    rep["duplicate_prompts"] = {k: len(its) - len({it["text"] for it in its})
                                for k, its in by_k.items()}
    dup = [f"k={k}: {n}" for k, n in rep["duplicate_prompts"].items() if n]
    if dup:
        P.append(f"identical prompts ({', '.join(dup)} duplicate(s)): one measurement would "
                 f"count as several items; lower n_per to at most "
                 f"{min(rep['distinct_problems'].values())} or add worked examples")
    rpt = [f"k={k}: {n}/{len(by_k[k])}" for k, n in rep["distinct_problems"].items()
           if n < len(by_k[k])]
    if rpt:
        Wn.append(f"final problems repeat (distinct/items {', '.join(rpt)}): the settings hold "
                  f"fewer problems per depth than items, so repeats are not independent items "
                  f"(they stay on one side of the split)")
    # a worked example that is the item's own query hands a copying model the answer
    rep["example_is_query"] = {k: sum(_query_among_examples(it["text"]) for it in its)
                               for k, its in by_k.items()}
    selfq = [f"k={k}: {n}" for k, n in rep["example_is_query"].items() if n]
    if selfq:
        P.append(f"a worked example is the item's own query, answer shown ({', '.join(selfq)} "
                 f"item(s)): copying it solves the item")

    # worked examples: per position balance and what copying one would score
    ex, spread = {}, collections.defaultdict(list)
    for k, its in by_k.items():
        rows = [_example_answers(it["text"]) for it in its]
        slots = []
        for j in range(max(map(len, rows))):
            pairs = [(r[j], it["gold"]) for it, r in zip(its, rows) if len(r) > j]
            n = len(pairs)
            cnt = collections.Counter(a for a, _ in pairs)
            gcnt = collections.Counter(g for _, g in pairs)
            exp = sum(cnt[d] * gcnt[d] for d in cnt) / n / n
            slots.append({"counts": dict(sorted(cnt.items())), "copy_expected": round(exp, 4),
                          "copy_realised": round(sum(a == g for a, g in pairs) / n, 4)})
            if n >= 20:
                spread[j].append(exp)
        echo_frac = sum(it["echo"] is not None and it["echo"] in r
                        for it, r in zip(its, rows)) / len(its)
        if slots:
            ex[k] = {"slots": slots, "echo_frac": echo_frac}
    rep["examples"] = ex
    bad = [k for k, e in ex.items() if e["echo_frac"]]
    if bad and ss.get("shot_mode") == "per_item":
        P.append("worked examples answer the item's written numeral at k="
                 + ",".join(map(str, bad)))
    for j, v in sorted(spread.items()):
        if len(v) == len(by_k) > 1 and max(v) - min(v) > 0.03:
            Wn.append(f"copying worked example {j} would score {min(v):.3f}-{max(v):.3f} "
                      f"across k, so a copy bias would look like a depth effect")

    # shortest lookup path: k steps on an n-cycle = n-k steps backward
    if all("min_hops" in it["setting"] for it in items):
        rep["min_hops"] = {k: dict(sorted(collections.Counter(
            it["setting"]["min_hops"] for it in its).items())) for k, its in by_k.items()}
        short = [k for k, its in by_k.items() if any(it["setting"]["min_hops"] < k for it in its)]
        if short:
            Wn.append("the shortest lookup path (forward or backward) is below k at k="
                      + ",".join(map(str, short)) + "; use setting.min_hops as a covariate "
                      "or restrict k")

    # matched families: where the queried entity sits and how it is used, read
    # from the text (suites written before these were recorded are checked too)
    if suite.get("family") in ("binding", "relational"):
        try:
            info = {it["id"]: solve_text(suite["family"], suite["wording"], it["text"])[1]
                    for it in items}
        except ValueError:                        # verification has already failed
            info = None
        if info:
            dd = {k: [info[it["id"]]["def_distance"] for it in its] for k, its in by_k.items()}
            rep["def_distance"] = {k: round(sum(v) / len(v), 2) for k, v in dd.items()}
            r = _corr([it["k"] for it in items], [info[it["id"]]["def_distance"] for it in items])
            rep["def_distance_corr"] = round(r, 3)
            if abs(r) >= 0.2:
                # 'shuffled' balances the queried link's rank only per block of
                # max(k)+1 bodies and the other chains' slots stay random, so at
                # small n_per a trend is sampling noise; recommending 'shuffled'
                # there would send the user to the setting already in use
                fix = ("use setting.def_distance as a covariate, or raise n_per (the "
                       "shuffled order balances the queried link's rank per block of "
                       "max(k)+1 bodies; the rest is sampling noise)"
                       if ss.get("order", "shuffled") == "shuffled" else
                       "use order='shuffled' or setting.def_distance as a covariate")
                Wn.append(f"the queried definition's distance to the query (mean statements "
                          f"per k {rep['def_distance']}) trends with k (r = {r:.2f}): recency "
                          f"is confounded with depth; {fix}")
            rep["query_refs"] = {k: dict(sorted(collections.Counter(
                info[it["id"]]["query_refs"] for it in its).items())) for k, its in by_k.items()}
            if len({frozenset(v) for v in rep["query_refs"].values()}) > 1:
                Wn.append(f"how often the queried entity is used as a value varies with k "
                          f"({rep['query_refs']}): at some depth it is a chain's leaf, a "
                          f"surface property confounded with depth")

    # arith: which offsets (gold - start) a depth allows
    if suite.get("family") == "arith":
        m = ss.get("modulus", 10)
        incs = ss.get("increments")
        rep["arith_offsets"] = {
            k: (sorted({(k * d) % m for d in incs} - {0}) if incs else
                sorted({(int(it["gold"]) - it["setting"]["start"]) % m for it in its}))
            for k, its in by_k.items()}
        sizes = {k: len(o) for k, o in rep["arith_offsets"].items()}
        if len(set(sizes.values())) > 1 or 1 in sizes.values():
            Wn.append(f"arith mod {m}: the number of possible offsets start->gold varies "
                      f"with k ({sizes}); where it is 1 the answer needs no composition "
                      f"(a prime modulus above max k, e.g. modulus=7, avoids this)")

    # calibration size against the correct-item rule
    calib = {k: v["calib"] for k, v in rep["split"].items()}
    if any(calib.values()) and min(calib.values()) < min_correct:
        frac = ss.get("calib_frac")
        need = f"; calibration suites need n_per >= {math.ceil(min_correct / frac)}" if frac else ""
        Wn.append(f"the calib split has {min(calib.values())} items at some depth, fewer than "
                  f"the {min_correct} correct items eligible_depths needs{need}")

    matched = FAMILIES.get(suite["family"], {}).get("matched", False)
    for tname, tok in _named_tokenizers(tokenizers):
        if isinstance(tok, str):                  # a name, as the CLI takes them
            try:
                tok = _load_tokenizers([tok])[tok]
            except Exception as e:
                P.append(f"{tname}: tokenizer could not be loaded ({type(e).__name__}: {e})")
                continue
        r = {"token_counts": {}, "feed_counts": {}, "align_failures": [], "appended": []}
        ids_of = {}
        for k, its in by_k.items():
            n_tok, n_feed = set(), set()
            for it in its:
                ids = tok(it["text"])["input_ids"]
                ids_of[it["id"]] = ids
                n_tok.add(len(ids))
                try:
                    feed, _, app = align_answer(tok, it["text"])
                    n_feed.add(len(feed))
                    if app not in r["appended"]:
                        r["appended"].append(app)
                except SystemExit as e:           # align_answer aborts via SystemExit
                    r["align_failures"].append([it["id"], str(e)])
            r["token_counts"][k] = sorted(n_tok)
            r["feed_counts"][k] = sorted(n_feed)
        alln = set().union(*map(set, r["token_counts"].values()))
        allf = set().union(*map(set, r["feed_counts"].values()))
        r["constant"] = len(alln) == 1
        r["n_tokens"] = min(alln) if r["constant"] else None
        r["feed_constant"] = len(allf) == 1
        r["align_ok"] = not r["align_failures"]
        if not r["constant"]:
            P.append(f"{tname}: token count varies {sorted(alln)}")
        if not r["feed_constant"] and r["align_ok"]:
            P.append(f"{tname}: measured position varies {sorted(allf)}")
        if not r["align_ok"]:
            P.append(f"{tname}: align_answer fails on {len(r['align_failures'])} item(s)")
        # HF sets model_max_length to a huge sentinel when a tokenizer has no limit
        mml = getattr(tok, "model_max_length", None)
        r["model_max_length"] = mml if isinstance(mml, int) and mml < 10 ** 7 else None
        if r["model_max_length"] and max(alln) > r["model_max_length"]:
            Wn.append(f"{tname}: prompts reach {max(alln)} tokens, above the tokenizer's "
                      f"model_max_length {r['model_max_length']}: a model with that context "
                      f"cannot run this suite (use fewer worked examples)")
        if matched:
            worst, same_len, bodies = 0, True, {}
            for it in items:
                bodies.setdefault(it["setting"].get("body"), []).append(ids_of[it["id"]])
            for seqs in bodies.values():
                for s in seqs[1:]:
                    if len(s) != len(seqs[0]):
                        same_len = False
                    else:
                        worst = max(worst, sum(a != b for a, b in zip(s, seqs[0])))
            r["matched_same_length"] = same_len
            r["matched_max_diff"] = worst
            r["matched_ok"] = same_len and worst <= 1
            if not r["matched_ok"]:
                P.append(f"{tname}: prompts of one body differ across k beyond the queried "
                         f"entity (same length {same_len}, up to {worst} positions)")
        # the tokenizer-independent checks, repeated so each entry stands alone
        for key in ("answer_balance", "balanced", "gold_eq_echo", "distractor_frac"):
            r[key] = rep[key]
        rep["tokenizers"][tname] = r


def _summary(rep):
    lines = [f"{rep['name']}: {rep['n_items']} items, k={sorted(rep['n_per_k'])}, "
             f"{'OK' if rep['ok'] else 'FAILED'}"]
    for t, r in rep["tokenizers"].items():
        lines.append(f"    {t:<24} tokens {r['n_tokens'] if r['constant'] else r['token_counts']}"
                     f"   align {'ok' if r['align_ok'] else 'FAILS'}   appended {r['appended']}"
                     + (f"   matched diff {r['matched_max_diff']}" if "matched_ok" in r else ""))
    for p in rep["problems"]:
        lines.append(f"    ! {p}")
    for w in rep.get("warnings", []):
        lines.append(f"    ~ {w}")
    return "\n".join(lines)


# ────────────────────────────── CLI ─────────────────────────────────────────
def _parse_set(pairs):
    out = {}
    for p in pairs:
        if "=" not in p:
            raise SystemExit(f"--set expects KEY=VALUE, got {p!r}")
        key, val = p.split("=", 1)
        try:
            out[key] = json.loads(val)
        except json.JSONDecodeError:
            out[key] = val
    return out


def _load_tokenizers(names):
    import os
    from transformers import AutoTokenizer
    tok = os.environ.get("HF_TOKEN") or None
    return {n: AutoTokenizer.from_pretrained(n, token=tok) for n in names}


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("--family", nargs="+", choices=sorted(_GEN))
    ap.add_argument("--wording", nargs="+", default=["w0"])
    ap.add_argument("--k", nargs=2, type=int, default=[1, 6], metavar=("LO", "HI"))
    ap.add_argument("--n-per", type=int, default=40)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--set", action="append", default=[], metavar="KEY=VALUE",
                    help="family setting, JSON value (e.g. shots=5, shot_mode=shared)")
    ap.add_argument("--ladder", action="store_true", help="write the original ladder suite")
    ap.add_argument("--out", default="suites")
    ap.add_argument("--check", nargs="+", metavar="JSON", help="check existing suite files")
    ap.add_argument("--tokenizer", nargs="+", default=[],
                    help="run check_suite with these tokenizers")
    ap.add_argument("--show", type=int, default=0, help="print the first N item texts")
    a = ap.parse_args()
    if not (a.family or a.ladder or a.check):
        ap.error("--family, --ladder or --check required")
    fams, words = list(dict.fromkeys(a.family or [])), list(dict.fromkeys(a.wording))
    sets = _parse_set(a.set)
    if "name" in sets and len(fams) * len(words) > 1:
        # one name for several suites would write them all to one file (and one set of ids)
        ap.error(f"--set name=... names one suite, but {len(fams) * len(words)} "
                 f"(family, wording) pairs were requested")
    toks = _load_tokenizers(a.tokenizer) if a.tokenizer else {}
    suites, written = [], {}

    def _write(s, path):
        key = str(path.resolve())
        if key in written:
            raise SystemExit(f"{path}: already written in this run by suite {written[key]!r}; "
                             f"refusing to overwrite it with {s['name']!r}")
        written[key] = s["name"]
        print(f"  -> {save_suite(s, path)}")

    if a.check:
        suites += [load_suite(p) for p in a.check]
    for fam in fams:
        for wd in words:
            s = make_suite(fam, wd, tuple(a.k), a.n_per, a.seed, **sets)
            _write(s, Path(a.out) / (s["name"] + ".json"))
            suites.append(s)
    if a.ladder:
        s = ladder_suite()
        _write(s, Path(a.out) / "ladder.json")
        suites.append(s)
    failed = 0
    for s in suites:
        for it in s["items"][:a.show]:
            print(f"\n[{it['id']}] gold {it['gold']}  echo {it['echo']}\n{it['text']}")
        if toks:
            rep = check_suite(s, toks)
            print(_summary(rep))
            failed += not rep["ok"]
    sys.exit(1 if failed else 0)


In [ ]:
%%writefile sweep_probe.py
#!/usr/bin/env python3
"""
Per-item capture over a serial-task suite (serial_tasks.py JSON) for the E1-E3
runs of research_outline.tex.  One model, one or more suites, one .npz each.

Per item, at the measured position (the one whose next token is the answer
digit; deep_probe.align_answer):

  contribution  [L,H]  ||z_h W_O^h||: the head's slice of the attention output
                       projection before any norm (the paper's activity; same
                       quantity as deep_probe's `contribution`)
  write_norm    [L,H]  OLMo-2 / Gemma-2 only (layers that normalise the
                       attention output before the residual add): norm of the
                       head's post-norm write at the frozen norm scale, as
                       head_alignment_probe defines it
  pos0_attn     [L,H]  attention weight from the measured position to position
                       0 (--pos0; eager only), for the BOS / attention-sink
                       control.  Read from each attention module's returned
                       weights, the tensor transformers collects for
                       output_attentions, keeping only the measured row: all
                       L full [H,T,T] maps held at once are GBs at 80 x 64
                       heads.  The two sources are compared once per suite on
                       a short prefix.
  digit_logits  [10]   the model's logits of the ten answer-digit tokens,
                       index = digit
  predictions          pred_constrained (argmax over the item's answer_set),
                       gold_prob, digit_mass, correct_constrained; with
                       --generate a greedy KV-cached continuation (<= 16
                       tokens, stopping after a digit other than the item's
                       echo) giving pred_first_digit and correct_strict

An item is correct_constrained only when gold is the UNIQUE maximum over its
answer_set.  bf16 logits are coarse (0.0625 apart at |logit| 8-16), so digits
often tie exactly; a tie is no preference for gold, and without this rule the
verdict would depend on the order answer_set lists the digits.  A tie is
resolved against gold (pred_constrained = the lowest tied digit other than
gold, so pred_constrained == gold exactly when correct), n_tied_at_max records
it, and gold_rank counts tied tokens as ahead of gold.

has_bos: position 0 is the tokenizer's BOS or another special token (Qwen's
<|endoftext|> prepended for the sink control; Qwen has no bos_token), i.e. not
a content token.

Depth, answer set and echo come from the item metadata, never from parsing the
prompt, so every task family goes through the same code.

Every linearisation is checked on every item and the run aborts rather than
write numbers when one fails.  Errors are max |error| / max |reference| at the
measured position, and each tolerance follows from how many roundings a
correct computation goes through (bf16 unit roundoff 2^-8 = 3.9e-3):

  exact         the per-head split must sum to the output projection's map
                recomputed in fp32 from the same input and weights; only fp32
                summation order differs (EXACT_TOL 1e-4)
  kernel        the split must reproduce the projection's own output
                (split_err), which the model's GEMM rounded to bf16, and
                cuBLAS may also reduce split-K partial sums in bf16 (PyTorch's
                default), one more rounding each.  Correct OLMo-2-7B bf16 runs
                on an L4 reached 9.6e-3; 16-64 bf16-reduced slices reach
                1.3e-2 to 2.4e-2 at width 8192.  Above WARN_TOL 1e-2 the run
                logs and counts it; it aborts above TOL 5e-2 for bf16/fp16
                kernels (1e-2 for fp32 ones).  A wrong Linear / GPT-2 Conv1D
                layout, a transposed weight or permuted head inputs give
                5e-2 to O(1).  Only the module's output can reveal these, since
                the exact check holds under either layout, so this check
                also chooses the layout
  one rounding  the frozen post-attention norm must reproduce its norm's
                output (OLMo-2 and Gemma-2 norms compute in fp32 and round
                once), the norm's input must be that layer's projection output,
                and attention rows must sum to 1 (ROUNDING_TOL 1e-2)

Every hook must fire exactly once per layer, nothing may be NaN, and the model
must be in eval mode (dropout would make every number a random draw).  fp32
matmuls run at full precision during capture (no TF32), whatever the global
setting.  The split check verifies the weight layout and that the attention
output decomposes linearly over heads.  Any column partition of W_O sums to
the same output, though, so it cannot see head boundaries.  n_heads and
head_dim are therefore checked against the model's config and its attention
modules' head_dim (what their forward reshapes by) before anything runs.

    python3 sweep_probe.py --model Qwen/Qwen2.5-7B --suite suites/pointer_w0.json \\
        --suite suites/binding_w0.json --out runs/e2 --device cuda \\
        --attn-impl eager --precision bf16 --pos0 --generate --split test
    python3 sweep_probe.py --model meta-llama/Llama-3.2-1B --suite calib/*.json \\
        --out runs/e1 --accuracy-only --split calib

Output: OUT/<model with / -> _>__<suite name>.npz holding the arrays above
(stacked over items, plus per-item k, gold, answer_mask and prediction arrays)
and `meta`, a JSON string with provenance, checks and per-item records.  An
output that already exists is skipped, so a disconnected Colab session resumes
where it stopped; one written with different settings, or by a version of
this file whose measurement differs (MEASUREMENT_VERSION), aborts the run
instead, so a sweep never mixes measurements.  Every suite is validated, and output paths checked for
collisions, before the model is loaded; --device cuda is enforced (a CPU
fallback of load_model aborts instead of running for days).
"""
import argparse, functools, hashlib, json, math, os, subprocess, sys, time
from pathlib import Path
import numpy as np

from deep_probe import align_answer
from head_alignment_probe import fit_norm, norm_family, _eps, _find_proj, _rel_err

DIGITS = "0123456789"
# Tolerances, max |error| / max |reference| at the measured position (module
# docstring).  bf16 unit roundoff is 2^-8 = 3.9e-3.
EXACT_TOL = 1e-4      # two fp32 computations of one map: summation order only
ROUNDING_TOL = 1e-2   # at most one bf16 rounding: frozen norm fit, attention
                      # rows; also the kernel limit when the GEMM is fp32
WARN_TOL = 1e-2       # kernel error above this is logged and counted
# The projection's bf16 output against the exact map.  A single bf16 rounding
# is only the floor: split-K GEMMs reduce partial sums in bf16 (cuBLAS under
# PyTorch's default allow_bf16_reduced_precision_reduction=True).  The
# repository's correct OLMo-2-7B runs (L4, 240 items) reached 9.6e-3, so 1e-2
# would abort correct 70B sweeps.  Structural faults are 5e-2 to O(1).
TOL = 5e-2
MAX_NEW = 16
# prefix length for the once-per-suite check that the hooked attention rows
# are output_attentions' rows; short, so the check itself never holds L full
# [H,T,T] maps at the prompt's length
POS0_CHECK_TOKENS = 16
SPLITS = ("calib", "test", "all")
REQUIRED = ("id", "k", "text", "gold", "answer_set", "echo", "split")


# ────────────────────────────── helpers ─────────────────────────────────────
def _ok(err, tol=TOL):
    """NaN-safe: a NaN error must fail the check, and `nan > tol` is False."""
    return err is not None and err <= tol


def _kernel_tol(*dtypes):
    """Abort limit for a module's own output against its exact fp32 map: TOL
    when any operand or the output is a 16-bit float (bf16/fp16 rounding,
    possibly per split-K partial sum), ROUNDING_TOL for fp32 kernels."""
    import torch
    return TOL if any(d in (torch.bfloat16, torch.float16) for d in dtypes) else ROUNDING_TOL


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def suite_sha256(suite):
    """Content hash of a suite, so a resumed run can tell a regenerated suite
    with the same name from the one it already measured."""
    return hashlib.sha256(json.dumps(suite, sort_keys=True).encode()).hexdigest()


@functools.lru_cache(maxsize=None)
def _git_commit():
    # once per process: the subprocess forks, which the tokenizers library
    # warns about on every fork once its thread pool has been used
    here = Path(__file__).resolve().parent
    try:
        r = subprocess.run(["git", "rev-parse", "HEAD"], cwd=here,
                           capture_output=True, text=True, timeout=10)
        if r.returncode == 0 and r.stdout.strip():
            return r.stdout.strip(), "git"
    except Exception:
        pass
    env = os.environ.get("NEXT_RUNS_COMMIT")
    return (env, "env") if env else (None, None)


def _provenance():
    import torch, transformers
    here = Path(__file__).resolve()
    commit, src = _git_commit()
    deps = {}
    for name in ("deep_probe.py", "head_alignment_probe.py", "active_subgraph_dot.py"):
        p = here.parent / name
        deps[name] = _sha256(p) if p.exists() else None
    return {"torch_version": torch.__version__,
            "transformers_version": transformers.__version__,
            "numpy_version": np.__version__,
            "python_version": sys.version.split()[0],
            "git_commit": commit, "git_commit_source": src,
            "next_runs_commit_env": os.environ.get("NEXT_RUNS_COMMIT"),
            "module_sha256": _sha256(here), "dependency_sha256": deps,
            "measurement_version": MEASUREMENT_VERSION}


def _check_items(suite):
    """Abort on items the capture cannot score correctly."""
    items = suite.get("items")
    if not items:
        raise SystemExit(f"suite {suite.get('name')!r} has no items")
    seen = set()
    for t in items:
        miss = [f for f in REQUIRED if f not in t]
        if miss:
            raise SystemExit(f"item {t.get('id')!r} lacks {miss}")
        if t["id"] in seen:
            raise SystemExit(f"duplicate item id {t['id']!r}")
        seen.add(t["id"])
        g, A, e = t["gold"], t["answer_set"], t["echo"]
        if not (isinstance(g, str) and len(g) == 1 and g in DIGITS):
            raise SystemExit(f"item {t['id']!r}: gold {g!r} is not a single digit")
        if (not A or len(set(A)) != len(A)
                or any(not (isinstance(a, str) and len(a) == 1 and a in DIGITS) for a in A)):
            raise SystemExit(f"item {t['id']!r}: answer_set {A!r} is not a set of digits")
        if g not in A:
            raise SystemExit(f"item {t['id']!r}: gold {g!r} not in answer_set {A!r}")
        if e is not None and not (isinstance(e, str) and len(e) == 1 and e in DIGITS):
            raise SystemExit(f"item {t['id']!r}: echo {e!r} is not a single digit or None")
        # echoing the written numeral must never score as correct
        if e is not None and e == g:
            raise SystemExit(f"item {t['id']!r}: gold equals echo {e!r}")
        if t["split"] not in ("calib", "test"):
            raise SystemExit(f"item {t['id']!r}: split {t['split']!r} not calib/test")
        if isinstance(t["k"], bool) or not isinstance(t["k"], int):
            raise SystemExit(f"item {t['id']!r}: k {t['k']!r} is not an int")


def _select(items, split="all", limit=None):
    """Indices of the items to run: the split, then at most `limit` taken
    round-robin across k, so a smoke test still covers every depth."""
    if split not in SPLITS:
        raise SystemExit(f"split must be one of {SPLITS}, got {split!r}")
    idx = [i for i, t in enumerate(items) if split == "all" or t["split"] == split]
    if limit is not None and limit < len(idx):
        byk = {}
        for i in idx:
            byk.setdefault(items[i]["k"], []).append(i)
        queues = [list(v) for _, v in sorted(byk.items())]
        pick = []
        while len(pick) < limit:
            for q in queues:
                if q and len(pick) < limit:
                    pick.append(q.pop(0))
        idx = sorted(pick)
    if not idx:
        raise SystemExit(f"no items in split {split!r}")
    return idx


def _patch(module, factory, patched):
    # accelerate's offload hook moves weights onto the device and then calls
    # _old_forward, so that is where the weights are real (as in deep_probe)
    at = "_old_forward" if hasattr(module, "_old_forward") else "forward"
    had = at in module.__dict__
    orig = getattr(module, at)
    setattr(module, at, factory(orig))
    patched.append((module, at, orig, had))


def _unpatch(patched):
    for module, at, orig, had in reversed(patched):
        if had:
            setattr(module, at, orig)
        else:
            delattr(module, at)       # back to the class method, no stale wrapper
    patched.clear()


def _hook_device(module):
    """The execution device accelerate's hook on `module` moves weights and
    inputs to, or None (no hook, or none that says)."""
    import torch
    hook = getattr(module, "_hf_hook", None)
    for h in [hook] + list(getattr(hook, "hooks", None) or []):
        d = getattr(h, "execution_device", None)
        if d is not None:
            return torch.device(d) if not isinstance(d, int) else torch.device("cuda", d)
    return None


def _input_device(hf):
    """Where input ids must live: the embedding's device.  When the embedding
    is offloaded (accelerate cpu_offload, or a device_map putting it on
    cpu/disk), its weight is on 'meta' outside the forward, and so is
    hf.device; the execution device is then on its accelerate hook."""
    try:
        emb = hf.get_input_embeddings()
    except Exception:
        emb = None
    if emb is not None:
        d = _hook_device(emb)
        if d is not None and d.type != "meta":
            return d
        try:
            d = emb.weight.device
            if d.type != "meta":
                return d
        except Exception:
            pass
    d = _hook_device(hf)
    if d is not None and d.type != "meta":
        return d
    if hf.device.type == "meta":
        raise SystemExit("cannot tell which device the input ids belong on: the embedding "
                         "is offloaded with no accelerate execution device")
    return hf.device


def _precision(hf, load_info):
    if load_info.get("precision") == "nf4":
        return "nf4"
    for p in hf.parameters():
        if p.is_floating_point():
            return str(p.dtype).replace("torch.", "")
    return None


def _eos_ids(hf, tok):
    ids = set()
    for v in (getattr(tok, "eos_token_id", None),
              getattr(getattr(hf, "generation_config", None), "eos_token_id", None)):
        if isinstance(v, int):
            ids.add(v)
        elif isinstance(v, (list, tuple)):
            ids.update(int(x) for x in v)
    return ids


def _check_heads(model, layers, H, dh):
    """Abort unless the wrapper's n_heads / d_head are the model's own.  The
    split check cannot see head boundaries (every column partition of W_O sums
    to the same output), so 8 heads of 8 would pass where the model runs 4
    heads of 16 and give meaningless per-head scores.  The attention module's
    head_dim is what its forward reshapes by, so it is the ground truth."""
    cfg = model._model.config
    try:
        cfg = cfg.get_text_config()
    except Exception:
        pass
    seen = []
    nh = getattr(cfg, "num_attention_heads", None)
    if isinstance(nh, int) and not isinstance(nh, bool):
        if nh != H:
            raise SystemExit(f"per-head split does not reproduce the model's heads: the "
                             f"wrapper says n_heads={H}, the model config "
                             f"num_attention_heads={nh}; refusing to emit contribution scores")
        seen.append("config.num_attention_heads")
    for l, ly in enumerate(layers):
        at = model._get_attn(ly)
        for attr, want in (("head_dim", dh), ("num_heads", H)):
            v = getattr(at, attr, None)
            if isinstance(v, int) and not isinstance(v, bool):
                if v != want:
                    raise SystemExit(f"layer {l}: per-head split does not reproduce the "
                                     f"model's heads: the wrapper says {attr}={want}, the "
                                     f"attention module {attr}={v}; refusing to emit "
                                     f"contribution scores")
                if l == 0:
                    seen.append(f"attention.{attr}")
    # one of H or dh confirmed is enough: the split check enforces H*dh = the
    # projection's input width, which pins the other
    if not seen:
        raise SystemExit("cannot verify head boundaries: the model has neither "
                         "config.num_attention_heads nor attention head_dim/num_heads")
    return seen


def _special_start_ids(tok):
    """Token ids that make position 0 a start token rather than content: the
    BOS and the other special tokens (Qwen2.5 has no bos_token; the outline's
    sink control prepends <|endoftext|>).  unk is content that failed to
    tokenize, not a start token."""
    ids = set()
    try:
        ids.update(int(i) for i in (getattr(tok, "all_special_ids", None) or []))
    except Exception:
        pass
    unk, bos = getattr(tok, "unk_token_id", None), getattr(tok, "bos_token_id", None)
    if unk is not None and unk != bos:
        ids.discard(int(unk))
    if bos is not None:
        ids.add(int(bos))
    return ids


def _constrained(dl, answer_set, gold):
    """(pred, correct, tied digits) over the item's answer_set.  Correct only
    when gold is the unique maximum: bf16 logits tie often, and a tie is no
    preference for gold.  The tie is resolved against gold (lowest other tied
    digit), so pred == gold exactly when correct and nothing depends on the
    order in which answer_set lists the digits."""
    vals = {d: float(dl[int(d)]) for d in answer_set}
    top = max(vals.values())
    tied = sorted(d for d, v in vals.items() if v == top)
    correct = tied == [gold]
    pred = gold if correct else next(d for d in tied if d != gold)
    return pred, correct, tied


def _split(x_last, W, mode, H, dh):
    """(per-head outputs [H, d_out], the whole map [d_out]) at one position,
    both in fp32 and without the bias (it belongs to no head).  The whole map
    is a separate matrix-vector product, not the sum of the heads, so the two
    agree only if the per-head reshape is right.  Linear weights are
    [out, in]; GPT-2's Conv1D is [in, out]."""
    import torch
    Wm = (W if mode == "linear" else W.t()).float()          # one fp32 copy
    if Wm.dim() != 2 or Wm.shape[1] != H * dh:
        raise ValueError(f"projection weight {tuple(Wm.shape)} does not take "
                         f"n_heads*head_dim {H}*{dh} inputs")
    x = x_last.float()
    v = torch.einsum("hd,ohd->ho", x.reshape(H, dh), Wm.reshape(Wm.shape[0], H, dh))
    return v, Wm @ x


def _per_head(x_last, W, mode, H, dh):
    """Per-head outputs [H, d_out] at one position, bias excluded (it belongs
    to no head).  Linear weights are [out, in]; GPT-2's Conv1D is [in, out]."""
    return _split(x_last, W, mode, H, dh)[0]


def _greedy(hf, tok, first_logits, pkv, echo, dev, eos):
    """Greedy continuation reusing the probe forward's KV cache (its logits
    give the first token).  Stops at EOS, after MAX_NEW tokens, or once a
    digit other than the echo has appeared: echoing the written numeral is not
    an answer, and nothing after the first real digit is read."""
    import torch
    gen, logits = [], first_logits
    for step in range(MAX_NEW):
        nx = int(logits.argmax())
        gen.append(nx)
        if nx in eos:
            break
        if any(c in DIGITS and c != echo for c in tok.decode(gen)):
            break
        if step == MAX_NEW - 1:
            break
        with torch.no_grad():
            o = hf(input_ids=torch.tensor([[nx]], device=dev), past_key_values=pkv,
                   use_cache=True)
        logits, pkv = o.logits[0, -1].float(), o.past_key_values
    return gen


# ────────────────────────────── capture ─────────────────────────────────────
def capture_suite(model, suite, *, attn_impl, record_pos0=False, generate=False,
                  split="all", limit=None, log=print, accuracy_only=False):
    """Run the selected items of `suite` through `model` (an _HFHookedModel-like
    wrapper: .cfg.n_layers/.n_heads/.d_head/.model_name, ._layers(),
    ._get_attn(layer), ._model, ._tok) and return (arrays, meta).

    attn_impl is the kernel the model runs; it is recorded, so it is checked
    against the model's config.  accuracy_only skips activation capture
    (logits and predictions only, for E1 calibration).
    """
    import torch
    t_start = time.time()
    hf, tok = model._model, model._tok
    L, H, dh = model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_head
    name = getattr(model.cfg, "model_name", None)
    layers = model._layers()
    if len(layers) != L:
        raise SystemExit(f"config says {L} layers, model has {len(layers)}")
    _check_items(suite)
    if record_pos0 and accuracy_only:
        raise SystemExit("record_pos0 captures activations; it cannot be combined "
                         "with accuracy_only")
    actual = getattr(hf.config, "_attn_implementation", None)
    if actual is not None and actual != attn_impl:
        raise SystemExit(f"model runs {actual!r} attention but attn_impl={attn_impl!r}; "
                         f"the recorded kernel must be the one that ran")
    if record_pos0 and attn_impl != "eager":
        raise SystemExit("position-0 attention needs eager attention (other kernels "
                         "return no attention weights)")
    # dropout in train mode makes every logit and activity a random draw, and
    # no check below would notice
    training = [nm or type(hf).__name__ for nm, m in hf.named_modules() if m.training]
    if training:
        raise SystemExit(f"model is in train mode ({len(training)} modules, e.g. "
                         f"{training[0]!r}): dropout would randomise every number; "
                         f"call .eval() first")

    items_all = suite["items"]
    sel = _select(items_all, split, limit)
    items = [items_all[i] for i in sel]
    n = len(items)
    dev = _input_device(hf)
    eos = _eos_ids(hf, tok)
    asd = sys.modules.get("active_subgraph_dot")
    load_info = dict(getattr(asd, "LAST_LOAD_INFO", None) or {})
    if load_info.get("model") != name:
        load_info = {"note": "model not loaded through active_subgraph_dot.load_model "
                             "in this process"}

    # OLMo-2 and Gemma-2 normalise the attention output before the residual
    # add; they are the architectures that also carry a
    # post_feedforward_layernorm.  (In Llama/Qwen, post_attention_layernorm is
    # the pre-MLP norm.)  Model metadata, so recorded in both modes.
    sandwich = [l for l in range(L) if hasattr(layers[l], "post_feedforward_layernorm")]

    # ── hooks: output projection per layer, post-attention norm where present ──
    st = {"on": False, "contrib": {}, "pending": {}, "write": {}, "calls": {}, "ncalls": {},
          "attn_on": False, "attn": {}, "acalls": {}}
    layout, norm_kind = {}, {}
    # split_err: per-head split vs the projection's own output (kernel
    # rounding included); split_exact_err: vs its map recomputed in fp32;
    # post_norm_input_err: the norm's input vs the projection's output.
    # split_warn counts (item, layer) checks above WARN_TOL, per layer too.
    checks = {"split_err": 0.0, "split_exact_err": 0.0, "post_norm_err": 0.0,
              "post_norm_input_err": 0.0, "attn_row_err": 0.0,
              "split_tol": None, "split_warn": 0, "split_warn_layers": {}, "split_checks": 0}
    pname, patched, handles, head_checks = None, [], [], []

    def attn_hook(l):
        # the attention module's (output, weights) pair: transformers collects
        # output[1] of this module for output_attentions.  Only the measured
        # row is kept, as a copy (a view would keep the whole map alive), so
        # each layer's [H,T,T] map is freed as soon as the layer is done.
        def hook(module, args, output):
            if not st["attn_on"]:
                return None
            st["acalls"][l] = st["acalls"].get(l, 0) + 1
            w = output[1] if isinstance(output, (tuple, list)) and len(output) > 1 else None
            st["attn"][l] = (None if w is None or w.dim() != 4 else
                             w[0, :, -1, :].detach().to(torch.float32, copy=True))
            return None
        return hook

    def attn_rows(what, T):
        """[L, H, T] measured-position rows recorded by the hooks, or abort."""
        rows = [st["attn"].get(l) for l in range(L)]
        if any(r is None for r in rows):
            raise SystemExit(f"{what}: the model returned no attention weights for every "
                             f"layer (is attention eager?)")
        if any(st["acalls"].get(l) != 1 for l in range(L)):
            raise SystemExit(f"{what}: attention hook calls per layer "
                             f"{dict(sorted(st['acalls'].items()))}; expected exactly one")
        P = torch.stack(rows)
        if tuple(P.shape) != (L, H, T) or not torch.isfinite(P).all():
            raise SystemExit(f"{what}: attention weights of shape {tuple(P.shape)} "
                             f"(want {(L, H, T)}) or non-finite")
        return P

    def pos0_source_check(feed):
        """The rows come from hooks, the spec's source is output_attentions:
        on a short prefix of the first item, run once with
        output_attentions and require the two to be identical."""
        ids = torch.tensor([feed[:POS0_CHECK_TOKENS]], device=dev)
        st["attn"].clear(); st["acalls"].clear()
        st["attn_on"] = True
        try:
            with torch.no_grad():
                r = hf(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False,
                       output_attentions=True)
        finally:
            st["attn_on"] = False
        att = getattr(r, "attentions", None)
        if att is None or len(att) != L or any(a is None for a in att):
            raise SystemExit("pos0 source check: the model returned no attention weights "
                             "for every layer (is attention eager?)")
        P = attn_rows("pos0 source check", ids.shape[1])
        ref = torch.stack([a[0, :, -1, :].float() for a in att])
        del r, att
        if ref.shape != P.shape or not torch.equal(ref, P):
            err = float((ref - P).abs().max()) if ref.shape == P.shape else float("inf")
            raise SystemExit(f"attention weights returned by the attention modules differ "
                             f"from output_attentions (max |diff| {err:.2e}); refusing to "
                             f"emit pos0_attn")
        st["attn"].clear(); st["acalls"].clear()

    def proj_hook(l, module):
        def factory(orig):
            def f(*a, **kw):
                out = orig(*a, **kw)
                if not st["on"]:
                    return out
                st["calls"][l] = st["calls"].get(l, 0) + 1
                x = a[0] if a else next(iter(kw.values()))
                W = module.weight
                if x.dim() != 3 or W.device.type == "meta":
                    return out            # not recorded -> the fired-on-all check aborts
                ref = out[0, -1].float()
                if not torch.isfinite(ref).all():
                    raise SystemExit(f"layer {l}: non-finite attention output at the "
                                     f"measured position (NaN/inf in the model)")
                b = getattr(module, "bias", None)
                b = 0 if b is None else b.float()
                # bf16/fp16 GEMMs round their output and possibly each split-K
                # partial sum; an fp32 one only reorders the sum
                ktol = _kernel_tol(x.dtype, W.dtype, out.dtype)
                checks["split_tol"] = max(checks["split_tol"] or 0.0, ktol)
                if l not in layout:
                    # shape cannot tell the layouts apart when the matrix is
                    # square, and the exact check below holds under either
                    # (both sides use the same x and W): keep the layout that
                    # reproduces the module's own output
                    best = ("none", float("inf"))
                    for mode in ("linear", "conv1d"):
                        try:
                            e = _rel_err(_split(x[0, -1], W, mode, H, dh)[0].sum(0) + b, ref)
                        except Exception:
                            e = float("inf")
                        e = e if math.isfinite(e) else float("inf")
                        if e < best[1]:
                            best = (mode, e)
                    if not _ok(best[1], ktol):
                        raise SystemExit(
                            f"layer {l}: per-head split does not reproduce "
                            f"{type(module).__name__} under either weight layout (best "
                            f"relative error {best[1]:.2e} > {ktol}, n_heads={H}, "
                            f"head_dim={dh}). Wrong head_dim, or attention that does not "
                            f"decompose into per-head slices; refusing to emit "
                            f"contribution scores.")
                    layout[l] = best
                v, whole = _split(x[0, -1], W, layout[l][0], H, dh)
                tot = v.sum(0) + b
                # the scores are the exact per-head split of the module's map
                # (independent of how the model's kernel rounded it)
                e_exact = _rel_err(tot, whole + b)
                if not _ok(e_exact, EXACT_TOL):
                    raise SystemExit(f"layer {l}: per-head split does not sum to the "
                                     f"projection recomputed in fp32 (error {e_exact:.2e} > "
                                     f"{EXACT_TOL}); refusing")
                # ... and that map is what the module computed, up to rounding
                e = _rel_err(tot, ref)
                if not _ok(e, ktol):
                    raise SystemExit(f"layer {l}: per-head split error {e:.2e} > {ktol} "
                                     f"(layout {layout[l][0]}); refusing")
                checks["split_checks"] += 1
                if e > WARN_TOL:
                    checks["split_warn"] += 1
                    wl = checks["split_warn_layers"]
                    wl[str(l)] = wl.get(str(l), 0) + 1
                checks["split_err"] = max(checks["split_err"], e)
                checks["split_exact_err"] = max(checks["split_exact_err"], e_exact)
                st["contrib"][l] = v.norm(dim=-1).cpu()
                if l in sandwich:
                    # consumed by the post-norm hook, which checks that its
                    # input is this output (a copy: a view would keep the
                    # whole [T, d] output alive)
                    st["pending"][l] = (v, out[0, -1].detach().to(torch.float32, copy=True))
                return out
            return f
        return factory

    def norm_hook(l, module):
        fam = norm_family(module)
        def factory(orig):
            def f(*a, **kw):
                out = orig(*a, **kw)
                if not st["on"]:
                    return out
                st["ncalls"][l] = st["ncalls"].get(l, 0) + 1
                x = a[0] if a else next(iter(kw.values()))
                if x.dim() != 3 or module.weight.device.type == "meta" or l not in st["pending"]:
                    return out
                kind, g, center, _bias, e_fit = fit_norm(
                    x[0, -1], out[0, -1], module.weight, getattr(module, "bias", None),
                    _eps(module), fam)
                v, proj_out = st["pending"].pop(l)
                # the norm must act on this layer's attention output, else the
                # frozen map is applied to the wrong vector.  Compared with the
                # projection's own output, not the per-head sum, so the GEMM's
                # rounding (checked in the projection hook) does not enter:
                # the two are the same tensor, up to at most a dtype cast
                e_in = _rel_err(proj_out.to(x.device), x[0, -1].float())
                e_fit = e_fit if math.isfinite(e_fit) else float("inf")
                if not (_ok(e_fit, ROUNDING_TOL) and _ok(e_in, ROUNDING_TOL)):
                    raise SystemExit(
                        f"layer {l}: frozen post-attention norm check failed (fit "
                        f"{kind} error {e_fit:.2e}, input-vs-attention-output error "
                        f"{e_in:.2e}, tolerance {ROUNDING_TOL}); refusing to emit write_norm")
                if norm_kind.setdefault(l, kind) != kind:
                    raise SystemExit(f"layer {l}: post-attention norm fitted as "
                                     f"{norm_kind[l]} and {kind} on different items")
                checks["post_norm_err"] = max(checks["post_norm_err"], e_fit)
                checks["post_norm_input_err"] = max(checks["post_norm_input_err"], e_in)
                w = ((v - v.mean(-1, keepdim=True)) if center else v) * g.to(v.device)
                st["write"][l] = w.norm(dim=-1).cpu()
                return out
            return f
        return factory

    # the exact check compares two fp32 products at EXACT_TOL, and the stored
    # scores are fp32 products: TF32 (a global opt-in on GPUs) would round
    # both to 10 mantissa bits.  Restored afterwards.
    mm_prec = torch.get_float32_matmul_precision()
    torch.set_float32_matmul_precision("highest")
    try:
        if not accuracy_only:
            for l in sandwich:
                if not hasattr(layers[l], "post_attention_layernorm"):
                    raise SystemExit(f"layer {l}: post_feedforward_layernorm without "
                                     f"post_attention_layernorm; unsupported")
            head_checks = _check_heads(model, layers, H, dh)
            for l in range(L):
                o, nm = _find_proj(model._get_attn(layers[l]))
                if o is None:
                    raise SystemExit(f"layer {l}: no output projection; unsupported "
                                     f"architecture")
                pname = pname or nm
                _patch(o, proj_hook(l, o), patched)
                if l in sandwich:
                    _patch(layers[l].post_attention_layernorm,
                           norm_hook(l, layers[l].post_attention_layernorm), patched)
            if record_pos0:
                for l in range(L):
                    handles.append(model._get_attn(layers[l]).register_forward_hook(attn_hook(l)))
                pos0_source_check(align_answer(tok, items[0]["text"])[0])
            log(f"  output projection: {pname}   post-attention norm on "
                f"{len(sandwich)}/{L} layers   attention {attn_impl}   heads checked "
                f"against {', '.join(head_checks)}"
                + ("   pos0 (hooked rows = output_attentions)" if record_pos0 else "")
                + ("   generate" if generate else ""))
        else:
            log(f"  accuracy only: no activation hooks   attention {attn_impl}"
                + ("   generate" if generate else ""))

        A = {"k": np.zeros(n, np.int16), "gold": np.zeros(n, np.int8),
             "answer_mask": np.zeros((n, 10), bool),
             "digit_logits": np.zeros((n, 10), np.float32),
             "pred_constrained": np.zeros(n, np.int8),
             "correct_constrained": np.zeros(n, bool),
             "n_tied_at_max": np.zeros(n, np.int8),
             "gold_prob": np.zeros(n, np.float32), "digit_mass": np.zeros(n, np.float32),
             "n_tokens": np.zeros(n, np.int32)}
        if not accuracy_only:
            A["contribution"] = np.zeros((n, L, H), np.float32)
            if sandwich:
                A["write_norm"] = np.zeros((n, L, H), np.float32)
        if record_pos0:
            A["pos0_attn"] = np.zeros((n, L, H), np.float32)
        if generate:
            A["pred_first_digit"] = np.full(n, -1, np.int8)
            A["correct_strict"] = np.zeros(n, bool)

        start_ids = _special_start_ids(tok)
        recs, times = [], []
        for i, t in enumerate(items):
            t0 = time.time()
            log(f"  [{i+1}/{n}] {t['id']}")
            for d in ("contrib", "pending", "write", "calls", "ncalls", "attn", "acalls"):
                st[d].clear()
            feed, ANS, appended = align_answer(tok, t["text"])
            ids = torch.tensor([feed], device=dev)
            # no output_attentions: the attention hooks keep the measured rows
            kw = dict(input_ids=ids, attention_mask=torch.ones_like(ids),
                      use_cache=bool(generate))
            st["on"] = not accuracy_only
            st["attn_on"] = bool(record_pos0)
            try:
                with torch.no_grad():
                    res = hf(**kw)
            finally:
                # generation forwards must not overwrite anything
                st["on"] = st["attn_on"] = False
            lg = res.logits[0, -1].float()
            if not torch.isfinite(lg).all():
                raise SystemExit(f"item {t['id']}: non-finite logits at the measured position")

            if not accuracy_only:
                fired = sorted(st["contrib"])
                if len(fired) < L or any(st["calls"].get(l) != 1 for l in range(L)):
                    raise SystemExit(
                        f"item {t['id']}: output-projection hook recorded {len(fired)}/{L} "
                        f"layers (calls per layer {dict(sorted(st['calls'].items()))}); "
                        f"expected exactly one call on every layer")
                if (len(st["write"]) < len(sandwich)
                        or any(st["ncalls"].get(l) != 1 for l in sandwich)):
                    raise SystemExit(f"item {t['id']}: post-attention norm hook recorded "
                                     f"{len(st['write'])}/{len(sandwich)} layers")
                C = torch.stack([st["contrib"][l] for l in range(L)]).numpy()
                if not np.isfinite(C).all():
                    raise SystemExit(f"item {t['id']}: contribution has NaN/inf")
                A["contribution"][i] = C
                if sandwich:
                    Wn = C.copy()   # non-sandwich layers write v itself
                    for l in sandwich:
                        Wn[l] = st["write"][l].numpy()
                    if not np.isfinite(Wn).all():
                        raise SystemExit(f"item {t['id']}: write_norm has NaN/inf")
                    A["write_norm"][i] = Wn
            if record_pos0:
                P = attn_rows(f"item {t['id']}", len(feed))                   # [L, H, seq]
                row = float((P.sum(-1) - 1).abs().max())
                if not _ok(row, ROUNDING_TOL):
                    raise SystemExit(f"item {t['id']}: attention rows sum to 1 +/- {row:.2e} "
                                     f"(tolerance {ROUNDING_TOL})")
                checks["attn_row_err"] = max(checks["attn_row_err"], row)
                A["pos0_attn"][i] = P[:, :, 0].cpu().numpy()
                del P
                st["attn"].clear()

            # predictions at the measured position, over the exact answer tokens
            probs = torch.softmax(lg, -1)
            dl = torch.stack([lg[ANS[d]] for d in DIGITS]).cpu().numpy()
            gold, aset, echo = t["gold"], list(t["answer_set"]), t["echo"]
            pred, correct, tied = _constrained(dl, aset, gold)
            n_tied = len(tied)
            gold_prob = float(probs[ANS[gold]])
            digit_mass = float(sum(probs[ANS[d]] for d in DIGITS))
            tv, ti = probs.topk(5)
            toks = tok.convert_ids_to_tokens(feed)
            rec = {"id": t["id"], "suite_index": sel[i], "label": t.get("label"),
                   "k": int(t["k"]), "family": t.get("family"), "wording": t.get("wording"),
                   # fewest lookups from start to answer walking the table either
                   # way (serial_tasks); below k on a cycle (k and n-k coincide)
                   "min_hops": (t.get("setting") or {}).get("min_hops"),
                   "k_written": t.get("k_written"), "echo": echo, "split": t["split"],
                   "gold": gold, "answer_set": aset, "text": t["text"],
                   "n_tokens": len(feed), "measured_token": toks[-1], "appended": appended,
                   # position 0 is a start token (BOS, or a special token such
                   # as a prepended <|endoftext|>) rather than content; without
                   # one, attention to it is not a sink in the usual sense
                   "has_bos": int(feed[0]) in start_ids,
                   "pos0_token": toks[0],
                   "pred_constrained": pred, "correct_constrained": correct,
                   "n_tied_at_max": n_tied,
                   # scored incorrect only because of the tie
                   "gold_tied_at_max": n_tied > 1 and gold in tied,
                   "pred_digit": DIGITS[int(np.argmax(dl))],
                   # tokens tied with gold count as ahead of it, as in correctness
                   "gold_prob": gold_prob, "gold_rank": int((probs >= probs[ANS[gold]]).sum()),
                   "digit_mass": digit_mass,
                   "top5": [[tok.decode([int(j)]), round(float(v), 4)] for v, j in zip(tv, ti)]}
            if generate:
                gen = _greedy(hf, tok, lg, res.past_key_values, echo, dev, eos)
                text = tok.decode(gen)
                digs = [c for c in text if c in DIGITS]
                nonecho = [c for c in digs if c != echo]
                rec.update(gen_text=text, pred_first_digit=digs[0] if digs else None,
                           pred_nonecho=nonecho[0] if nonecho else None,
                           correct_strict=correct and bool(digs) and digs[0] == gold)
                A["pred_first_digit"][i] = int(digs[0]) if digs else -1
                A["correct_strict"][i] = rec["correct_strict"]
            del res

            A["k"][i], A["gold"][i] = int(t["k"]), int(gold)
            A["answer_mask"][i, [int(d) for d in aset]] = True
            A["digit_logits"][i] = dl
            A["pred_constrained"][i] = int(pred)
            A["correct_constrained"][i] = correct
            A["n_tied_at_max"][i] = n_tied
            A["gold_prob"][i], A["digit_mass"][i] = gold_prob, digit_mass
            A["n_tokens"][i] = len(feed)
            rec["time_s"] = round(time.time() - t0, 3)
            times.append(rec["time_s"])
            recs.append(rec)
            if dev.type == "cuda":
                torch.cuda.empty_cache()
    finally:
        st["on"] = st["attn_on"] = False
        st["attn"].clear()
        for h in handles:
            h.remove()
        _unpatch(patched)
        torch.set_float32_matmul_precision(mm_prec)

    if not np.isfinite(A["digit_logits"]).all():
        raise SystemExit("digit logits contain NaN/inf")
    if layout:
        log(f"  checks (max relative error): per-head split {checks['split_err']:.1e} vs the "
            f"module's output (abort {checks['split_tol']}), {checks['split_exact_err']:.1e} "
            f"vs its fp32 map (abort {EXACT_TOL}); layout "
            f"{sorted({m for m, _ in layout.values()})}"
            + (f"   post-attn norm {checks['post_norm_err']:.1e}, its input "
               f"{checks['post_norm_input_err']:.1e} (abort {ROUNDING_TOL})" if sandwich else "")
            + (f"   attention rows {checks['attn_row_err']:.1e} (abort {ROUNDING_TOL})"
               if record_pos0 else ""))
        if checks["split_warn"]:
            log(f"  WARNING: per-head split error above {WARN_TOL} on {checks['split_warn']} "
                f"of {checks['split_checks']} (item, layer) checks (per layer "
                f"{checks['split_warn_layers']}; max {checks['split_err']:.1e}, abort "
                f"{checks['split_tol']}).  Expected from bf16 split-K rounding only if "
                f"sporadic and well below the abort limit; the same layers on most items "
                f"point to a projection that is not the plain linear map of its weights")

    nt_by_k = {}
    for r in recs:
        nt_by_k.setdefault(str(r["k"]), set()).add(r["n_tokens"])
    measured = {}
    for r in recs:
        measured[r["measured_token"]] = measured.get(r["measured_token"], 0) + 1
    meta = {
        "model": name, "suite": suite.get("name"), "family": suite.get("family"),
        "wording": suite.get("wording"), "setting": suite.get("setting"),
        "chance": suite.get("chance"), "suite_sha256": suite_sha256(suite),
        "split": split, "limit": limit, "accuracy_only": bool(accuracy_only),
        "record_pos0": bool(record_pos0), "generate": bool(generate),
        "n_items": n, "n_layers": L, "n_heads": H, "d_head": dh,
        "load_info": load_info, "attn_impl": attn_impl,
        "precision": _precision(hf, load_info), "device": str(dev),
        **_provenance(),
        "proj_name": pname, "sandwich_layers": sandwich,
        "post_norm_kind": {str(l): k for l, k in sorted(norm_kind.items())},
        "head_layout": {str(l): m for l, (m, _) in sorted(layout.items())},
        "checks": {**checks,
                   # abort limits (module docstring); split_tol above is the
                   # kernel limit actually applied (TOL for bf16/fp16 GEMMs)
                   "tolerances": {"split": TOL, "split_fp32_kernel": ROUNDING_TOL,
                                  "split_exact": EXACT_TOL, "split_warn": WARN_TOL,
                                  "post_norm": ROUNDING_TOL, "post_norm_input": ROUNDING_TOL,
                                  "attn_row": ROUNDING_TOL},
                   "float32_matmul_precision": {"during": "highest", "outside": mm_prec},
                   "layers_hooked": 0 if accuracy_only else L,
                   "verified": not accuracy_only, "head_boundaries_checked": head_checks,
                   # hooked attention rows identical to output_attentions'
                   "pos0_source_checked": bool(record_pos0),
                   "pos0_source_check_tokens": POS0_CHECK_TOKENS if record_pos0 else None},
        "digit_order": DIGITS,
        # items whose answer_set maximum is shared by several digits (scored
        # incorrect, see the module docstring), and those among them where
        # gold is one of the tied digits
        "constrained_ties": {"items": sum(r["n_tied_at_max"] > 1 for r in recs),
                             "gold_among_tied": sum(r["gold_tied_at_max"] for r in recs)},
        "measured_tokens": measured,
        "appended_forms": [list(a) for a in sorted({tuple(r["appended"]) for r in recs})],
        "has_bos": sum(r["has_bos"] for r in recs),
        "pos0_tokens": sorted({r["pos0_token"] for r in recs}),
        "n_tokens_by_k": {k: sorted(v) for k, v in sorted(nt_by_k.items(), key=lambda kv: int(kv[0]))},
        "timing": {"total_s": round(time.time() - t_start, 2),
                   "per_item_mean_s": round(float(np.mean(times)), 3),
                   "per_item_max_s": round(float(np.max(times)), 3)},
        "items": recs,
    }
    return A, meta


# ────────────────────────────── outputs ─────────────────────────────────────
def output_path(out_dir, model_name, suite_name):
    return (Path(out_dir) /
            f"{model_name.replace('/', '_')}__{str(suite_name).replace('/', '_')}.npz")


def save_output(path, arrays, meta):
    """Write atomically: a disconnect mid-write must not leave a truncated file
    that the next session would skip as done."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".partial")
    with open(tmp, "wb") as f:
        np.savez_compressed(f, meta=np.array(json.dumps(meta)), **arrays)
    os.replace(tmp, path)


def load_output(path):
    """(arrays, meta) from one output file."""
    with np.load(path, allow_pickle=False) as z:
        meta = json.loads(str(z["meta"]))
        return {k: z[k] for k in z.files if k != "meta"}, meta


# An output is "done" only if it was measured the same way.  MEASUREMENT_VERSION
# is bumped by any change that can alter the recorded numbers; edits that cannot
# (messages, or a fix that only changes which models run, such as taking
# Gemma's (1 + w) norm form from its class) keep it, so a resumed sweep does not
# redo hours of finished work.  Outputs written before the field existed are
# mapped from their module_sha256 in _COMPATIBLE_MODULES.
MEASUREMENT_VERSION = 1
_COMPATIBLE_MODULES = {
    # Sep 24 (E3 and the first E2 runs); differs from version 1 only in how the
    # post-attention norm family is chosen, which a successful run never changes
    "4ac79510b759a3a28853167ad0f859cf96f1d334b1f89ec4785182b68dc1c1b7": 1,
}
RESUME_KEYS = ("model", "suite", "suite_sha256", "split", "limit", "accuracy_only",
               "record_pos0", "generate", "attn_impl", "requested_precision",
               "measurement_version")


def _measurement_version(meta):
    v = meta.get("measurement_version")
    return v if v is not None else _COMPATIBLE_MODULES.get(meta.get("module_sha256"))


def _resume_state(path, want):
    """None if the output is missing; else the list of settings that differ."""
    if not Path(path).exists():
        return None
    try:
        _, meta = load_output(path)
    except Exception as e:
        raise SystemExit(f"{path} exists but cannot be read ({e}); delete it to re-run")
    have = dict(meta, measurement_version=_measurement_version(meta))
    return [f"{k}: have {have.get(k)!r}, want {want[k]!r}"
            for k in RESUME_KEYS if have.get(k) != want[k]]


def _summary(meta, log=print):
    byk = {}
    for r in meta["items"]:
        byk.setdefault(r["k"], []).append(r)
    log("  constrained by k: " + "  ".join(
        f"k{k} {sum(r['correct_constrained'] for r in v)}/{len(v)}" for k, v in sorted(byk.items()))
        + f"   chance {meta.get('chance')}")
    if meta["generate"]:
        log("  strict by k:      " + "  ".join(
            f"k{k} {sum(r['correct_strict'] for r in v)}/{len(v)}" for k, v in sorted(byk.items())))
    ties = meta.get("constrained_ties") or {}
    log(f"  measured tokens {meta['measured_tokens']}   appended {meta['appended_forms']}   "
        f"BOS {meta['has_bos']}/{meta['n_items']}   ties at the constrained max "
        f"{ties.get('items')} (gold among the tied, scored incorrect: "
        f"{ties.get('gold_among_tied')})")


# ─────────────────────────────── CLI ────────────────────────────────────────
def run(args, log=print):
    suites = []
    for p in args.suite:
        with open(p) as f:
            s = json.load(f)
        if not isinstance(s, dict) or not isinstance(s.get("name"), str) or not s["name"]:
            raise SystemExit(f"{p}: not a suite with a name (serial_tasks.save_suite)")
        suites.append((p, s))
    # compare the files, not the names: 'grp/a' and 'grp_a' both write
    # MODEL__grp_a.npz, and case-insensitive disks (macOS, some Drive mounts)
    # also merge names differing only in case
    plan, by_file = [], {}
    for p, s in suites:
        f = output_path(args.out, args.model, s["name"])
        key = str(f).casefold()
        if key in by_file:
            raise SystemExit(f"suites {by_file[key]!r} and {p!r} would both write {f.name}; "
                             f"rename one")
        by_file[key] = p
        plan.append((p, s, f))
    common = dict(model=args.model, split=args.split, limit=args.limit,
                  accuracy_only=args.accuracy_only, record_pos0=args.pos0,
                  generate=args.generate, attn_impl=args.attn_impl,
                  requested_precision=args.precision,
                  measurement_version=MEASUREMENT_VERSION)
    todo = []
    for p, s, f in plan:
        want = dict(common, suite=s["name"], suite_sha256=suite_sha256(s))
        diff = _resume_state(f, want)
        if diff is None:
            todo.append((p, s, f))
        elif diff:
            hint = (" (written by a version of sweep_probe.py that measures differently)"
                    if any(d.startswith("measurement_version") for d in diff)
                    else "")
            raise SystemExit(f"{f} exists with different settings ({'; '.join(diff)}){hint}; "
                             f"use another --out or delete it")
        else:
            log(f"  skip {f.name}: already done")
    if not todo:
        log("  nothing to do")
        return []
    # validate every pending suite before the load (minutes to an hour for a
    # 70B) and before earlier suites spend their Colab time
    for p, s, f in todo:
        try:
            _check_items(s)
            _select(s["items"], args.split, args.limit)
        except SystemExit as e:
            raise SystemExit(f"suite {p}: {e}")

    _git_commit()                 # before any tokenizer threads exist
    # pin the load path; load_model reads these
    os.environ["ASD_FORCE_STRATEGY"] = "B"
    os.environ["ASD_FORCE_PRECISION"] = "bf16" if args.precision == "bf16" else ""
    os.environ["ASD_ATTN_IMPL"] = args.attn_impl
    import torch
    try:
        want_dev = torch.device(args.device).type
    except Exception:
        raise SystemExit(f"--device {args.device!r} is not a torch device")
    # load_model quietly falls back to the CPU when CUDA is missing (or a GPU
    # load fails): a 70B would then run for days or exhaust host RAM
    if want_dev == "cuda" and not torch.cuda.is_available():
        raise SystemExit("--device cuda but CUDA is not available: select a GPU runtime, "
                         "or pass --device cpu explicitly")
    if args.precision == "fp32" and args.device != "cpu" and torch.cuda.is_available():
        raise SystemExit("load_model supports fp32 on CPU only; use --device cpu")
    import active_subgraph_dot as asd
    t0 = time.time()
    model = asd.load_model(args.model, device=args.device)
    load_s = time.time() - t0
    if not hasattr(model, "_model"):
        raise SystemExit("needs the _HFHookedModel wrapper (ASD_FORCE_STRATEGY=B)")
    on = sorted({p.device.type for p in model._model.parameters()})
    if want_dev != "cpu" and want_dev not in on:
        raise SystemExit(f"--device {args.device} requested but the model's weights are on "
                         f"{on} (load_model fell back); refusing to run there")
    li = asd.LAST_LOAD_INFO
    want_prec = {"bf16": "bfloat16", "fp32": "float32"}[args.precision]
    if li.get("strategy") != "B" or li.get("precision") != want_prec:
        raise SystemExit(f"loaded with strategy {li.get('strategy')!r} precision "
                         f"{li.get('precision')!r}; requested B / {want_prec}")
    log(f"  {args.model}: n_layers={model.cfg.n_layers} n_heads={model.cfg.n_heads} "
        f"d_head={model.cfg.d_head}  load_info={dict(li)}")

    written = []
    for p, s, f in todo:
        log(f"\n  suite {s.get('name')} ({p})")
        A, meta = capture_suite(model, s, attn_impl=args.attn_impl, record_pos0=args.pos0,
                                generate=args.generate, split=args.split, limit=args.limit,
                                log=log, accuracy_only=args.accuracy_only)
        meta.update(model=args.model, requested_precision=args.precision,
                    suite_path=str(p), load_s=round(load_s, 1),
                    cli={k: v for k, v in vars(args).items()})
        save_output(f, A, meta)
        _summary(meta, log)
        log(f"  -> {f}  ({f.stat().st_size / 1e6:.1f} MB)")
        written.append(f)
    return written


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0],
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True)
    ap.add_argument("--suite", required=True, action="append", nargs="+",
                    help="suite JSON (serial_tasks.save_suite); repeatable")
    ap.add_argument("--out", required=True, help="output directory")
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--attn-impl", default="eager", choices=["eager", "sdpa"],
                    help="attention kernel; it changes the contribution metric, so it "
                         "is pinned and recorded")
    ap.add_argument("--precision", default="bf16", choices=["bf16", "fp32"],
                    help="fp32 is supported on CPU only (load_model)")
    ap.add_argument("--pos0", action="store_true",
                    help="record attention to position 0 (needs --attn-impl eager)")
    ap.add_argument("--generate", action="store_true",
                    help="greedy continuation for pred_first_digit / correct_strict")
    ap.add_argument("--split", default="all", choices=list(SPLITS))
    ap.add_argument("--limit", type=int, default=None,
                    help="at most N items, taken round-robin across k (smoke tests)")
    ap.add_argument("--accuracy-only", action="store_true",
                    help="logits and predictions only, no activation capture (E1)")
    a = ap.parse_args(argv)
    a.suite = [p for group in a.suite for p in group]
    if a.pos0 and a.attn_impl != "eager":
        ap.error("--pos0 needs --attn-impl eager")
    if a.pos0 and a.accuracy_only:
        ap.error("--pos0 records activations; drop it with --accuracy-only")
    if a.limit is not None and a.limit < 1:
        ap.error("--limit must be positive")
    return run(a)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile signature_analysis.py
#!/usr/bin/env python3
"""
Item-level analysis of the next runs (E1-E6 of paper_manuscript/research_outline.tex).

Turns per-item capture files into the paper's statistics.  Activity s(l,h) is
the per-head contribution ||z_h W_O^h|| at the position whose next token is the
answer digit (`contribution`; `write_norm`, the post-norm write, is reported
alongside for OLMo-2 / Gemma-2).

  primary     top-head share max_h s / sum_h s, averaged over layers
              floor(0.4(L-1))..floor(0.8(L-1)) ("40-80% depth"); its item-level
              OLS slope on the serial depth k with a permutation null, and the
              k=1 -> kmax change.  PR/H over the same window is secondary.
  signature   spread (share falls) / funnel (rises) / flat / mixed (share and
              PR/H point to opposite responses) / inconclusive (correct items
              and all items disagree in sign) / missing (no correct-item
              analysis: fewer than 3 depths with >= 20 correct items and
              accuracy >= 2c/(1+c))
  RQ1         invariance of a model's pre-registered sign across the four task
              families (pointer, arith, binding, relational) and wordings
              (stratified bootstrap CI, Holm across families), on each model's
              pinned configuration only; the numeral control (fixed_digit),
              the single-prompt ladder and kernel / precision re-runs are
              reported beside it, never counted as families or models
  RQ3         per-layer classes (BH within model and task, joint k-permutation
              null), layer map, final-20% region, contiguous PR band
  RQ4         pooled scale-normalised slopes under ONE relabelling of k shared
              by all models on the same items; paired differences
  checks      kernel comparison, power targets, BOS / attention-sink control,
              mass centroid, and the E1 choice of one setting per family and
              comparison group

Input files (load_run), all item-level:
  <model>__<suite>.npz   sweep_probe.py           arrays + meta (JSON string)
  <model>_align.npz      head_alignment_probe.py  contribution, items in meta
  <model>_deep.json      deep_probe.py            tasks[] with contribution
Older files carry no item metadata; depth, answer set and echo are then parsed
from the pointer-chase template (label D{k}.., "a goes to b.", "Take k steps").

Measured position: files written with deep_probe.align_answer record it per
item ('appended' / 'measured_token').  A file without that record whose own
predictions show a position that does not predict the answer digit (top-1
next token whitespace on most items, or median digit mass < 0.5; the 17
September deep_probe runs, which the outline says are "not evidence yet") is
rejected by the CLI; one with no evidence either way is analysed with a note.

Every analysis checks its input and refuses rather than report numbers from a
file that is inconsistent (NaN activity, arrays and item records that disagree,
correctness flags that contradict the stored predictions, the same item id
with different depths, prompts or answers in two runs).  The CLI lists such
files as rejected and reports nothing for them; an analysis that fails on one
file is listed as refused and the rest of the report is still written.

When several captures of one model, suite and configuration are globbed
(an E1 calibration run, a --limit smoke run and the E2 capture all share the
file name sweep_probe gives them), files with identical contents (a copy in
another folder) are rejected as duplicates and one capture is analysed: the
one with activity, then without --limit, then on the test split, then with the
most items; the others are listed as superseded.  Captures that tie on all of
that but differ in content (a re-run saved in another folder) are conflicts:
the files cannot tell which is current, so none of them is analysed.

Each run's permutations and bootstraps are drawn from a generator seeded by
(--seed, the run's model / suite / kernel / precision, the analysis), so a
run's p-values do not change when other files join the glob.

    python3 signature_analysis.py --runs 'runs/e2/*.npz' --out report \\
        --preregistered qwen2.5=funnel llama=spread OLMo-2-1124-7B-Instruct=spread
    python3 signature_analysis.py --calib 'runs/e1/*.npz' --groups groups.json --out e1

--preregistered MODEL=SIGN (SIGN funnel/spread, or none to leave a model out
of RQ1): MODEL occurs in the model name, case-insensitively; a post-trained
model (-Instruct, -SFT, -DPO, -it, -chat, -RLVR) is covered only by a pattern
that names its marker, so qwen2.5=funnel is "Qwen2.5 base: funnel" and does not
reach Qwen2.5-7B-Instruct (post-training moves the signature: OLMo).  A
pattern ending in $ is anchored at the end of the name; re:REGEX is a regular
expression with no marker rule.  An exact model name wins, then the longest
pattern.  --pinned ATTN PRECISION names E3's configuration; without it each
model's RQ1 uses the configuration holding most of its task-family cells.
"""
import argparse, glob, hashlib, json, math, os, re, subprocess, sys, time, warnings, zlib
from collections import Counter
from pathlib import Path
from statistics import NormalDist
import numpy as np

DIGITS = "0123456789"
CLASSES = ("spread", "funnel", "flat", "mixed", "inconclusive", "missing")
# Arrays not read from capture files: large and unused by these statistics.
_SKIP_ARRAYS = ("dla", "cos_top", "cos_sum", "top_idx", "sum_norm")
POST_TRAINING_SUFFIXES = ("-instruct", "-sft", "-dpo", "-it", "-chat", "-rlvr")
# post-training marker anywhere in a name (Mistral-7B-Instruct-v0.3, gemma-2-27b-it)
_POST_MARK = re.compile(r"(?:^|[-_. /])(instruct|sft|dpo|it|chat|rlvr)(?=$|[-_. /])", re.I)
# RQ1 tests invariance over these; anything else is reported beside it
TASK_FAMILIES = ("pointer", "arith", "binding", "relational")
CONTROL_FAMILIES = {"fixed_digit": "numeral control (fixed_digit)",
                    "ladder": "single-prompt ladder (one item per depth)"}


class RunError(ValueError):
    """A capture file that cannot be analysed without emitting wrong numbers."""


# ────────────────────────────── small helpers ───────────────────────────────
def _rng(rng):
    if isinstance(rng, np.random.Generator):
        return rng
    return np.random.default_rng(0 if rng is None else rng)


def _sub_rng(seed, *parts):
    """A generator that depends only on the seed and `parts` (a run's identity
    and the analysis): a run's p-values must not move when another file joins
    the glob, as one stream shared in file order would make them."""
    ent = [int(seed) % (1 << 63)] + [zlib.crc32(str(p).encode()) for p in parts]
    return np.random.default_rng(ent)


def _ge(null_abs, obs_abs):
    """|null| >= |obs| with a relative tolerance: a relabelling that reproduces
    the observed arrangement must count, whatever the floating-point order."""
    return null_abs >= obs_abs - (1e-9 * obs_abs + 1e-15)


def _perm_rows(x, n_perm, rng, batch=2000):
    """Yield [b, n] blocks whose rows are independent random permutations of x."""
    x = np.asarray(x, float)
    done = 0
    while done < n_perm:
        b = min(batch, n_perm - done)
        yield rng.permuted(np.tile(x, (b, 1)), axis=1)
        done += b


def _ols_slope(y, k):
    y, k = np.asarray(y, float), np.asarray(k, float)
    kc = k - k.mean()
    ss = kc @ kc
    return float(kc @ (y - y.mean()) / ss) if ss > 0 else float("nan")


def _row_slopes(Y, K):
    """OLS slope of each row of Y on the matching row of K."""
    Kc = K - K.mean(1, keepdims=True)
    Yc = Y - Y.mean(1, keepdims=True)
    ss = (Kc * Kc).sum(1)
    return np.where(ss > 0, (Kc * Yc).sum(1) / np.where(ss > 0, ss, 1), np.nan)


def _p_two_sided(obs, null):
    null = np.asarray(null, float)
    if not np.isfinite(obs) or len(null) == 0:
        return float("nan")
    return float((_ge(np.abs(null), abs(obs)).sum() + 1) / (len(null) + 1))


def _sign(x):
    return 0 if x is None or not np.isfinite(x) else int(np.sign(x))


def _nanmean(a, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(a, axis=axis)


def _short(model):
    return str(model).split("/")[-1]


def _as_run(r):
    return load_run(r) if isinstance(r, (str, Path)) else r


# ────────────────────────────── statistics ──────────────────────────────────
def window(L, lo=0.4, hi=0.8):
    """Layers floor(lo(L-1))..floor(hi(L-1)) inclusive: 10-21 of 28, 12-24 of
    32, 31-63 of 80.  The epsilon keeps an exact product such as 0.4*10 from
    rounding down to the layer below."""
    if L < 2:
        raise ValueError(f"window needs L >= 2, got {L}")
    return slice(math.floor(lo * (L - 1) + 1e-9), math.floor(hi * (L - 1) + 1e-9) + 1)


def _kept(S, exclude):
    """Activity with excluded heads zeroed (activity is a norm, so zero adds
    nothing to a max or a sum) and the number of heads kept per layer."""
    S = np.asarray(S, float)
    if exclude is None:
        return S, np.full(S.shape[:-1], S.shape[-1], float)
    ex = np.asarray(exclude, bool)
    if ex.shape != S.shape:
        raise ValueError(f"exclude mask {ex.shape} does not match activity {S.shape}")
    return np.where(ex, 0.0, S), (~ex).sum(-1).astype(float)


def top_share(S, exclude=None):
    """[n, L] top-head share max_h s / sum_h s (NaN where no head is kept)."""
    S, _ = _kept(S, exclude)
    tot = S.sum(-1)
    return np.where(tot > 0, S.max(-1) / np.where(tot > 0, tot, 1), np.nan)


def pr_h(S, exclude=None):
    """[n, L] participation ratio (sum s)^2 / sum s^2 over the kept heads,
    divided by their number (H when nothing is excluded)."""
    S, nk = _kept(S, exclude)
    s, s2 = S.sum(-1), (S * S).sum(-1)
    ok = (s2 > 0) & (nk > 0)
    return np.where(ok, s * s / np.where(ok, s2, 1) / np.where(ok, nk, 1), np.nan)


def top_share_window(S, L, lo=0.4, hi=0.8, exclude=None):
    """[n] the primary statistic: top share averaged over the 40-80% window."""
    S = np.asarray(S, float)
    if S.ndim != 3 or S.shape[1] != L:
        raise ValueError(f"activity of shape {S.shape} is not [n, {L}, H]")
    return _nanmean(top_share(S, exclude)[:, window(L, lo, hi)], 1)


def slope_perm(y, k, n_perm=10000, rng=None):
    """OLS slope of y on k and its two-sided permutation p-value (k shuffled
    across items), p = (count + 1) / (n_perm + 1).  Non-finite items dropped."""
    y, k = np.asarray(y, float), np.asarray(k, float)
    ok = np.isfinite(y) & np.isfinite(k)
    y, k = y[ok], k[ok]
    if len(y) < 3 or np.ptp(k) == 0:
        return float("nan"), float("nan")
    rng = _rng(rng)
    kc = k - k.mean()
    ss, yc = kc @ kc, y - y.mean()
    obs = float(kc @ yc / ss)
    cnt = 0
    for P in _perm_rows(kc, n_perm, rng):
        cnt += int(_ge(np.abs(P @ yc / ss), abs(obs)).sum())
    return obs, (cnt + 1) / (n_perm + 1)


def eligible_depths(k, correct, chance, min_correct=20):
    """Depths with >= min_correct correct items and accuracy >= 2c/(1+c).  If a
    fraction g of items is solved and the rest guessed at chance c, accuracy is
    a = g + (1-g)c, and g/a >= 1/2 exactly when a >= 2c/(1+c): at least half
    of the correct items are genuine solves."""
    k, ok = np.asarray(k), np.asarray(correct, bool)
    thr = 2 * chance / (1 + chance)
    out = []
    for d in np.unique(k):
        m = k == d
        nc = int(ok[m].sum())
        if nc >= min_correct and nc / m.sum() >= thr - 1e-12:
            out.append(int(d))
    return out


def holm(pvals):
    """Holm step-down adjusted p-values (NaN entries stay NaN, not counted)."""
    p = np.asarray(pvals, float)
    out = np.full(p.shape, np.nan)
    ok = np.isfinite(p)
    x = p[ok]
    m = len(x)
    if m:
        o = np.argsort(x, kind="stable")
        adj = np.minimum(np.maximum.accumulate((m - np.arange(m)) * x[o]), 1.0)
        r = np.empty(m)
        r[o] = adj
        out[ok] = r
    return out


def bh(pvals, q=0.05):
    """Benjamini-Hochberg rejections at FDR q (bool mask; NaN never rejected)."""
    p = np.asarray(pvals, float)
    rej = np.zeros(p.shape, bool)
    ok = np.isfinite(p)
    x = p[ok]
    m = len(x)
    if m:
        o = np.argsort(x, kind="stable")
        below = np.nonzero(x[o] <= q * np.arange(1, m + 1) / m)[0]
        if len(below):
            r = np.zeros(m, bool)
            r[o[:below[-1] + 1]] = True
            rej[ok] = r
    return rej


def _bh_rows(P, q):
    """bh() applied to every row of P at once."""
    R, m = P.shape
    o = np.argsort(P, 1, kind="stable")
    below = np.take_along_axis(P, o, 1) <= q * np.arange(1, m + 1) / m
    last = np.where(below.any(1), m - 1 - np.argmax(below[:, ::-1], 1), -1)
    rank = np.argsort(o, 1)
    return rank <= last[:, None]


def _joint_slopes(Y, k, n_perm, rng):
    """Per-column OLS slopes of Y [n, L] on k and their null [n_perm, L] under
    ONE shuffle of k per draw applied to every column, which keeps the
    correlation of sampling noise between columns (adjacent layers)."""
    Y, k = np.asarray(Y, float), np.asarray(k, float)
    if Y.ndim != 2 or len(k) != len(Y):
        raise ValueError(f"Y {Y.shape} and k {k.shape} do not match")
    if not np.isfinite(Y).all() or not np.isfinite(k).all():
        raise ValueError("non-finite values in Y or k; a joint null needs every item")
    kc = k - k.mean()
    ss = kc @ kc
    if ss == 0:
        raise ValueError("k does not vary")
    Yc = Y - Y.mean(0)
    obs = kc @ Yc / ss
    null = np.concatenate([P @ Yc / ss for P in _perm_rows(kc, n_perm, rng)])
    return obs, null


def _perm_p_cols(obs, null):
    return (_ge(np.abs(null), np.abs(obs)).sum(0) + 1) / (len(null) + 1)


def _null_p_rows(null):
    """Per-layer p-value of every null draw against the whole null set, so each
    draw can be classified exactly as the observed slopes are."""
    R, L = null.shape
    A = np.abs(null)
    P = np.empty_like(A)
    for l in range(L):
        s = np.sort(A[:, l])
        P[:, l] = (R - np.searchsorted(s, A[:, l] - (1e-9 * A[:, l] + 1e-15), "left")) / R
    return P


def classify_layers(Y, k, q=0.05, n_perm=5000, rng=None):
    """Per layer -1 (spreading: top-share slope < 0), +1 (funnelling) or 0.

    Y [n, L] per-layer top share.  Item-level OLS slopes per layer, two-sided
    p from a joint permutation null (one shuffle of k applied to all layers),
    Benjamini-Hochberg at q within the model and task."""
    obs, null = _joint_slopes(Y, k, n_perm, _rng(rng))
    rej = bh(_perm_p_cols(obs, null), q)
    return np.where(rej, np.sign(obs), 0).astype(int)


def _seq_stats(c):
    """(sign changes, longest same-sign run) among the classified layers, in
    layer order with unclassified layers skipped."""
    s = np.asarray(c)[np.asarray(c) != 0]
    if len(s) == 0:
        return 0, 0
    ch = np.nonzero(s[1:] != s[:-1])[0]
    bounds = np.concatenate([[-1], ch, [len(s) - 1]])
    return int(len(ch)), int(np.diff(bounds).max())


def _matched_classes(null, npos, nneg):
    """Classes of each null draw with exactly the observed numbers of
    funnelling and spreading layers: the npos layers with the largest and the
    nneg with the smallest standardised slope (slope / its null SD, so layers
    with noisier statistics are not favoured)."""
    R, L = null.shape
    npos, nneg = int(npos), int(nneg)
    if npos + nneg > L:
        raise ValueError("more classified layers than layers")
    sd = null.std(0)
    o = np.argsort(null / np.where(sd > 0, sd, 1), 1)
    C = np.zeros((R, L), int)
    rows = np.arange(R)[:, None]
    if nneg:
        C[rows, o[:, :nneg]] = -1
    if npos:
        C[rows, o[:, L - npos:]] = 1
    return C


def layer_map(classes, Y, k, n_perm=2000, rng=None, q=0.05, mass=None, final=0.8):
    """RQ3 layer map: are spreading and funnelling layers contiguous,
    interleaved or randomly ordered?

    Statistics among classified layers: number of sign changes, longest
    same-sign run.  Nulls:
      joint (primary)   ONE shuffle of k per draw for all layers, which keeps
                        the correlation of noise between adjacent layers.  It
                        shuffles k against each layer's residuals (Y minus its
                        fitted slope): shuffling raw Y would give every layer
                        with a true effect b_l the common null term
                        b_l*slope(k, shuffled k), so the null would copy the
                        observed block pattern.  Each draw is reclassified
                        with the observed numbers of spreading and funnelling
                        layers held fixed (the most negative / most positive
                        standardised slopes), so only the arrangement is tested
      joint_bh          raw-Y draws reclassified by BH at q, as the spec's text
                        literally says ("recompute classes under each
                        shuffle").  Degenerate: under the null BH classifies
                        almost nothing, so its null has ~0 sign changes and it
                        only tests whether ANY layer responds (p_more is then
                        1/(n_perm+1) for every responsive model).  Reported for
                        completeness; never read as an arrangement test
      layer_shuffle     the observed classes in random layer order (secondary)
    p_fewer = P(null changes <= observed) (contiguous), p_more = P(>=)
    (interleaved), p_longer = P(null run >= observed) (contiguous).

    `arrangement` holds the p-values to read.  Contiguity (p_contiguous,
    p_contiguous_run) is judged against the joint null: correlated noise
    between adjacent layers makes contiguous blocks by itself, and the joint
    null keeps that correlation (the layer shuffle would over-call contiguity).
    Interleaving (p_interleaved) is judged against the layer shuffle: the
    joint null's draws inherit the noise correlation, so a random arrangement
    of true signs looks more interleaved than they are (AR(1) layer noise with
    rho 0.8: joint p_more <= 0.05 in ~60% of random arrangements, layer
    shuffle ~1%; with independent noise both hold their level).  This splits
    the spec's single "primary null" by direction, a documented deviation.
    Also the fraction and relative position of each class, and the final-20%
    region (layers after the 40-80% window): per-layer slopes, region mean with
    its joint p, and the mass response if `mass` [n, L] is given."""
    rng = _rng(rng)
    classes = np.asarray(classes, int)
    Y = np.asarray(Y, float)
    if Y.ndim != 2 or Y.shape[1] < 2:
        raise ValueError(f"layer_map needs [n, L] with L >= 2 layers, got {Y.shape}")
    L = Y.shape[1]
    if classes.shape != (L,):
        raise ValueError(f"classes {classes.shape} do not match {L} layers")
    obs, null = _joint_slopes(Y, k, n_perm, rng)
    ch, run = _seq_stats(classes)
    Pn = _null_p_rows(null)
    kf = np.asarray(k, float)
    _, rnull = _joint_slopes(Y - np.outer(kf - kf.mean(), obs), kf, n_perm, rng)
    nulls = {
        "joint": _matched_classes(rnull, (classes > 0).sum(), (classes < 0).sum()),
        "joint_bh": np.where(_bh_rows(Pn, q), np.sign(null), 0).astype(int),
        "layer_shuffle": rng.permuted(np.tile(classes, (n_perm, 1)), axis=1),
    }
    tests = {}
    for name, C in nulls.items():
        st = np.array([_seq_stats(c) for c in C])
        tests[name] = {
            "p_fewer": float(((st[:, 0] <= ch).sum() + 1) / (len(st) + 1)),
            "p_more": float(((st[:, 0] >= ch).sum() + 1) / (len(st) + 1)),
            "p_longer": float(((st[:, 1] >= run).sum() + 1) / (len(st) + 1)),
            "null_changes_mean": float(st[:, 0].mean()),
            "null_run_mean": float(st[:, 1].mean()),
            "null_classified_mean": float((C != 0).sum(1).mean()),
        }
    x = np.arange(L) / (L - 1)
    sp, fu = np.nonzero(classes < 0)[0], np.nonzero(classes > 0)[0]
    fin = np.arange(L) > math.floor(final * (L - 1) + 1e-9)
    out = {
        "L": L, "signmap": "".join("+" if c > 0 else "-" if c < 0 else "." for c in classes),
        "n_spread": int(len(sp)), "n_funnel": int(len(fu)),
        "frac_spread": len(sp) / L, "frac_funnel": len(fu) / L,
        "pos_spread": [round(float(x[l]), 3) for l in sp],
        "pos_funnel": [round(float(x[l]), 3) for l in fu],
        "mean_pos_spread": float(x[sp].mean()) if len(sp) else None,
        "mean_pos_funnel": float(x[fu].mean()) if len(fu) else None,
        "sign_changes": ch, "longest_run": run, "tests": tests,
        "arrangement": {"p_contiguous": tests["joint"]["p_fewer"],
                        "p_contiguous_run": tests["joint"]["p_longer"],
                        "p_interleaved": tests["layer_shuffle"]["p_more"],
                        "nulls": "contiguity vs joint (matched counts, residual "
                                 "shuffle); interleaving vs layer shuffle"},
        "slopes": obs.tolist(), "p": _perm_p_cols(obs, null).tolist(),
        "final_region": {
            "layers": np.nonzero(fin)[0].tolist(),
            "slopes": obs[fin].tolist(),
            "mean_slope": float(obs[fin].mean()) if fin.any() else None,
            "p": _p_two_sided(obs[fin].mean(), null[:, fin].mean(1)) if fin.any() else None,
            "n_spread": int((classes[fin] < 0).sum()), "n_funnel": int((classes[fin] > 0).sum()),
        },
    }
    if mass is not None and fin.any():
        mo, mn = _joint_slopes(np.asarray(mass, float), k, n_perm, rng)
        base = np.asarray(mass, float).mean(0)
        out["final_region"]["mass_slope_rel"] = float((mo[fin] / base[fin]).mean())
        out["final_region"]["mass_p"] = _p_two_sided((mo[fin] / base[fin]).mean(),
                                                     (mn[:, fin] / base[fin]).mean(1))
        out["mass_slope_rel"] = (mo / base).tolist()
    return out


def band_rule(slope, p, window_=0.10, call=0.5, sustain=0.15):
    """The band from per-layer PR/H slopes and permutation p-values, under the
    contiguous rule and the superseded one, in relative depth l/(L-1).

    A layer is callable if p < call.  frac[l] = share of the callable layers
    within +/- window_ of l that disperse (slope > 0).
      contiguous   onset = first layer of the first run of layers with
                   frac > 1/2 whose span (last - first)/(L-1) >= sustain; the
                   band closes where that run ends
      superseded   onset = first layer followed by int(sustain*L) layers with
                   frac > 1/2 (a count, not a span); close = the last layer
                   with frac > 1/2 anywhere, which bridges gaps.  Kept only so
                   old results can be compared (sweep_analysis.py,
                   replicate_analysis.py)."""
    slope, p = np.asarray(slope, float), np.asarray(p, float)
    L = len(slope)
    if L < 2 or len(p) != L:
        raise ValueError("band_rule needs matching slope and p over >= 2 layers")
    x = np.arange(L) / (L - 1)
    callable_ = p < call
    frac = np.full(L, np.nan)
    for l in range(L):
        w = (np.abs(x - x[l]) <= window_ + 1e-12) & callable_
        if w.any():
            frac[l] = float((slope[w] > 0).mean())
    above = frac > 0.5                    # NaN (nothing callable nearby) is not above

    runs, start = [], None
    for l in range(L + 1):
        if l < L and above[l]:
            start = l if start is None else start
        elif start is not None:
            runs.append((start, l - 1))
            start = None
    qual = [(a, b) for a, b in runs if (b - a) / (L - 1) >= sustain - 1e-9]
    if qual:
        a, b = qual[0]
        cont = {"onset": a / (L - 1), "close": b / (L - 1), "width": (b - a) / (L - 1),
                "first_layer": a, "last_layer": b, "n_layers": b - a + 1}
    else:
        cont = {"onset": None, "close": None, "width": 0.0,
                "first_layer": None, "last_layer": None, "n_layers": 0}
    cont["runs"] = [[a / (L - 1), b / (L - 1)] for a, b in runs]

    need, on = int(sustain * L), None
    for l in range(L):
        if above[l] and all(above[j] for j in range(l, min(L, l + need))):
            on = l
            break
    cl = max(np.nonzero(above)[0]) if on is not None else None
    sup = ({"onset": on / (L - 1), "close": cl / (L - 1), "width": (cl - on) / (L - 1),
            "width_legacy": (cl - on + 1) / L, "first_layer": int(on), "last_layer": int(cl)}
           if on is not None else
           {"onset": None, "close": None, "width": 0.0, "width_legacy": 0.0,
            "first_layer": None, "last_layer": None})
    differs = (cont["onset"], cont["close"]) != (sup["onset"], sup["close"])
    return {"frac": frac.tolist(), "callable": int(callable_.sum()),
            "contiguous": cont, "superseded": sup, "differs": bool(differs)}


def band(S, k, rng=None, n_perm=2000, window_=0.10, call=0.5, sustain=0.15, exclude=None):
    """Contiguous band rule on per-layer PR/H slopes (item-level OLS on k, p from
    a joint k-permutation null), with the superseded rule for comparison."""
    if np.ndim(S) != 3 or np.shape(S)[1] < 2:
        raise ValueError(f"band needs activity [n, L, H] with L >= 2, got {np.shape(S)}")
    P = pr_h(S, exclude)
    obs, null = _joint_slopes(P, k, n_perm, _rng(rng))
    p = _perm_p_cols(obs, null)
    out = band_rule(obs, p, window_, call, sustain)
    out.update(slopes=obs.tolist(), p=p.tolist(), L=int(P.shape[1]))
    return out


# ────────────────────────────── loading ─────────────────────────────────────
_RULE = re.compile(r"(\d) goes to (\d)")


def _pointer_parse(text):
    """(rule table, start, written k) of the query in a pointer-chase prompt.
    The last "Rules:" block is the query; earlier blocks are worked examples."""
    q = (text or "").split("Rules:")[-1]
    table = {int(a): int(b) for a, b in _RULE.findall(q)}
    s = re.search(r"Start at (\d+)", q)
    kk = re.search(r"Take (\d+) steps", q)
    return table, (int(s.group(1)) if s else None), (int(kk.group(1)) if kk else None)


def _ring_min_hops(table, s, g):
    """Fewest lookups from s to g walking a permutation table forward or
    backward (serial_tasks._min_hops): on a cycle of length c with g f steps
    ahead, min(f, c - f).  None when s or g is not on a cycle of the table."""
    if s not in table or g is None:
        return None
    ring, x = [s], table.get(s)
    while x is not None and x != s and len(ring) <= len(table):
        ring.append(x)
        x = table.get(x)
    if x != s or g not in ring:
        return None
    f = ring.index(g)
    return min(f, len(ring) - f)


def _item_min_hops(t, family):
    """An item's shortest lookup path: recorded by serial_tasks (sweep_probe
    copies it), else recomputed from the prompt for the pointer-chase families
    (Sep 22 and 70B files predate the field).  NaN where undefined."""
    v = t.get("min_hops")
    if v is None:
        v = (t.get("setting") or {}).get("min_hops")
    if v is None and family in ("pointer", "ladder", "fixed_digit"):
        table, s, _ = _pointer_parse(t.get("text"))
        g = t.get("gold")
        try:
            g = int(g) if g is not None else None
        except (TypeError, ValueError):
            g = None
        v = _ring_min_hops(table, s, g) if table and s is not None else None
    return np.nan if v is None else float(v)


def _legacy_k(label, text):
    """Depth of an item without a "k" field.  D{k}...: k.  K{k}R{rule}
    (the original fixed_digit control): the number of distinct states the
    k-step walk visits AFTER the start (ident 0, cyc2 1, cyc3 2, cyc5 up to
    4), serial_tasks' fixed_digit convention len(set(s_0..s_K)) - 1, so the
    depths of old and new numeral controls line up."""
    m = re.match(r"D(\d+)", label or "")
    if m:
        return int(m.group(1))
    m = re.match(r"K(\d+)R", label or "")
    if m:
        table, s, _ = _pointer_parse(text)
        if not table or s is None:
            raise RunError(f"label {label!r}: cannot parse the rule table")
        path = [s]
        for _ in range(int(m.group(1))):
            if path[-1] not in table:          # the walk leaves the table: not a permutation
                raise RunError(f"label {label!r}: state {path[-1]} has no rule in the table")
            path.append(table[path[-1]])
        return len(set(path)) - 1
    raise RunError(f"item label {label!r} carries no depth and the item has no 'k'")


def _legacy_family(suite):
    s = str(suite or "").lower()
    if s.startswith("fixed_digit"):
        return "fixed_digit"
    if s == "serial_depth":
        return "ladder"
    if s.startswith("serial_depth") or s.startswith("serial_replicates"):
        return "pointer"
    return s or "unknown"


def _norm_precision(p):
    s = str(p or "").lower().replace("torch.", "")
    return {"bfloat16": "bf16", "bf16": "bf16", "float32": "fp32", "fp32": "fp32",
            "float16": "fp16", "fp16": "fp16", "nf4": "nf4"}.get(s, s or None)


def load_run(path):
    """One capture file as a dict: arrays (activity etc.), meta, and per-item
    ids, k, gold, answer-set sizes, correctness flags, gold_prob, digit_mass.

    `correct` holds the correctness used by the analysis: correct_strict when
    the run generated, else correct_constrained (else the legacy `correct`)."""
    path = str(path)
    if path.endswith(".json"):
        with open(path) as f:
            d = json.load(f)
        if "tasks" not in d:
            raise RunError(f"{path}: not a deep_probe file (no 'tasks')")
        # content fingerprint: only files with identical contents are copies
        content = hashlib.sha1(json.dumps(d, sort_keys=True).encode()).hexdigest()
        tasks = d["tasks"]
        meta = {k: v for k, v in d.items() if k != "tasks"}
        drop = {"contribution", "anchor_pos", "anchor_mass", "attn_entropy", "bos_mass",
                "hidden_sim", "tokens"}
        items = [{k: v for k, v in t.items() if k not in drop} for t in tasks]
        arrays = {}
        if tasks and all("contribution" in t for t in tasks):
            try:
                arrays["contribution"] = np.array([t["contribution"] for t in tasks], float)
            except ValueError as e:
                raise RunError(f"{path}: ragged contribution arrays ({e})")
        fmt = "deep"
    elif path.endswith(".npz"):
        with np.load(path, allow_pickle=False) as z:
            if "meta" not in z.files:
                raise RunError(f"{path}: no meta")
            meta = json.loads(str(z["meta"]))
            # E5 globs every output on Drive, E4's included.  Refuse ablation
            # files by what they are, not only because their per-item keys
            # happen to differ from a capture's: the half-suite measurement
            # would otherwise read as a capture of the model.
            kind = str(meta.get("kind") or "")
            if kind.startswith("ablation"):
                raise RunError(f"{path}: an ablation_probe output (kind {kind!r}, E4), not a "
                               f"capture; analyse it with ablation_probe.py --analyse")
            arrays = {k: z[k] for k in z.files if k != "meta" and k not in _SKIP_ARRAYS}
            content = _npz_fingerprint(z, meta, arrays)
        items = meta.get("items")
        fmt = "align" if path.endswith("_align.npz") else "sweep"
    else:
        raise RunError(f"{path}: unknown file type")
    if not items:
        raise RunError(f"{path}: no per-item records")
    n = len(items)

    sm = meta.get("suite") if isinstance(meta.get("suite"), dict) else {}
    suite = sm.get("name") if sm else meta.get("suite")
    family = meta.get("family") or sm.get("family") or _legacy_family(suite)
    wording = meta.get("wording") or sm.get("wording") or "w0"
    setting = meta.get("setting") or sm.get("setting") or {}
    li = meta.get("load_info") or {}

    def col(name, conv):
        """Per-item field from the item records, else from a 1-D array."""
        if all(name in t for t in items):
            return [conv(t[name]) for t in items]
        a = arrays.get(name)
        if a is not None and a.ndim == 1 and len(a) == n:
            return [conv(v) for v in a.tolist()]
        return None

    ids, ks, golds, asz, echo, fams, words = [], [], [], [], [], [], []
    for i, t in enumerate(items):
        label, text = t.get("label") or "", t.get("text") or ""
        k = t.get("k")
        k = _legacy_k(label, text) if k is None else k
        if isinstance(k, bool) or int(k) != k or k < 0:
            raise RunError(f"{path}: item {i} has depth {k!r}")
        lm = re.match(r"D(\d+)", label)
        if t.get("k") is not None and lm and int(lm.group(1)) != int(k):
            raise RunError(f"{path}: item {i} label {label!r} but k = {k}")
        ks.append(int(k))
        g = t.get("gold")
        golds.append(None if g is None else str(g).strip())
        aset = t.get("answer_set")
        if aset is None:
            table, _, _ = _pointer_parse(text)
            aset = [str(s) for s in sorted(table)] if table else None
        if aset is not None and golds[-1] is not None and golds[-1] not in [str(a) for a in aset]:
            raise RunError(f"{path}: item {i} gold {golds[-1]!r} not in answer set {aset}")
        asz.append(len(aset) if aset else 0)
        if "echo" in t:
            echo.append(t["echo"])
        else:
            _, _, kw = _pointer_parse(text)
            echo.append(None if kw is None else str(kw))
        ids.append(str(t.get("id") or f"{suite}/{label or i}"))
        fams.append(t.get("family") or family)
        words.append(t.get("wording") or wording)
    if len(set(ids)) != n:
        raise RunError(f"{path}: duplicate item ids")
    ks = np.array(ks, int)
    if "k" in arrays and not np.array_equal(np.asarray(arrays["k"], int), ks):
        raise RunError(f"{path}: k array and item records disagree")
    if 0 in asz:                          # no table and no answer_set: the golds seen
        seen = sorted({g for g in golds if g is not None})
        asz = [a or len(seen) for a in asz]
    asz = np.array(asz, int)

    flags = {}
    for name in ("correct_strict", "correct_constrained", "correct"):
        v = col(name, lambda x: bool(x))
        if v is not None:
            flags[name] = np.array(v, bool)
            a = arrays.get(name)
            if (a is not None and a.ndim == 1 and len(a) == n
                    and not np.array_equal(a.astype(bool), flags[name])):
                raise RunError(f"{path}: {name} array and item records disagree")
    key = next((k for k in ("correct_strict", "correct_constrained", "correct") if k in flags), None)
    if key is None:
        raise RunError(f"{path}: no correctness field")
    _check_predictions(path, items, golds, arrays, flags, n)
    fl = lambda v: np.nan if v is None else float(v)
    gold_prob = np.array(col("gold_prob", fl) or [np.nan] * n, float)
    digit_mass = np.array(col("digit_mass", fl) or [np.nan] * n, float)
    for name, a in (("gold_prob", gold_prob), ("digit_mass", digit_mass)):
        f = a[np.isfinite(a)]
        if len(f) and (f.min() < -1e-4 or f.max() > 1 + 1e-3):   # probabilities
            raise RunError(f"{path}: {name} outside [0, 1] ({f.min():.4g}..{f.max():.4g})")
    hb = [t.get("has_bos") for t in items if t.get("has_bos") is not None]
    position = _measured_position(items, digit_mass)

    # activity: shapes agree with the records and nothing is NaN
    L, H = meta.get("n_layers"), meta.get("n_heads")
    C = arrays.get("contribution")
    if C is not None:
        if C.ndim != 3 or C.shape[0] != n:
            raise RunError(f"{path}: contribution of shape {C.shape} for {n} items")
        if (L is not None and C.shape[1] != L) or (H is not None and C.shape[2] != H):
            raise RunError(f"{path}: contribution {C.shape} but meta says L={L} H={H}")
        L, H = C.shape[1], C.shape[2]
        if not np.isfinite(C).all():
            raise RunError(f"{path}: contribution contains NaN/inf (head_dim bug in the "
                           f"capture); re-run it")
        if (C < 0).any():
            raise RunError(f"{path}: negative contribution; not a norm")
    sandwich = [int(l) for l in (meta.get("sandwich_layers") or [])]
    Wn = arrays.get("write_norm")
    if Wn is not None:
        if C is None or Wn.shape != C.shape:
            raise RunError(f"{path}: write_norm without matching contribution")
        Wn = Wn.astype(float)
        plain = np.ones(L, bool)
        plain[sandwich] = False
        # layers without a post-attention norm write the pre-norm vector itself
        Wn[:, plain] = np.where(np.isfinite(Wn[:, plain]), Wn[:, plain], C[:, plain])
        if not np.isfinite(Wn).all():
            raise RunError(f"{path}: write_norm NaN/inf on post-norm layers")
        arrays["write_norm"] = Wn
    A0 = arrays.get("pos0_attn")
    if A0 is not None:
        if C is None or A0.shape != C.shape:
            raise RunError(f"{path}: pos0_attn of shape {A0.shape} does not match activity")
        if not np.isfinite(A0).all() or A0.min() < -1e-3 or A0.max() > 1 + 1e-3:
            raise RunError(f"{path}: pos0_attn outside [0, 1] or NaN")

    chance = meta.get("chance") if meta.get("chance") is not None else sm.get("chance")
    if chance is None:
        chance = float(np.mean(1.0 / np.maximum(asz, 1)))
    # what makes two files with the same name different captures (sweep_probe
    # names every output <model>__<suite>.npz whatever the split or --limit)
    splits = sorted({str(t["split"]) for t in items if t.get("split")})
    split = meta.get("split") or (splits[0] if len(splits) == 1 else "all" if splits else None)
    return {
        "path": path, "format": fmt, "model": meta.get("model") or Path(path).stem,
        "suite": suite, "family": family, "wording": wording, "setting": setting,
        "split": split, "limit": meta.get("limit"),
        "accuracy_only": bool(meta.get("accuracy_only")) or "contribution" not in arrays,
        "suite_sha256": meta.get("suite_sha256"),
        "chance": float(chance),
        "attn_impl": meta.get("attn_impl") or li.get("attn_impl"),
        "precision": _norm_precision(meta.get("precision") or li.get("precision")),
        "L": L, "H": H, "n": n, "arrays": arrays, "meta": meta, "items": items,
        "ids": np.array(ids), "k": ks, "gold": golds, "answer_set_size": asz,
        "item_family": np.array(fams), "item_wording": np.array(words),
        "echo": echo, "correctness": key, "correct": flags[key],
        "correct_strict": flags.get("correct_strict"),
        "correct_constrained": flags.get("correct_constrained"),
        "gold_prob": gold_prob, "digit_mass": digit_mass,
        "has_bos": (sum(bool(b) for b in hb) / len(hb)) if hb else None,
        "sandwich_layers": sandwich, "position": position, "content_sha1": content,
        # on a cycle, k steps forward = n - k steps back, so depth is clean only
        # where min_hops == k (NaN where the family has no such shortcut data)
        "min_hops": np.array([_item_min_hops(t, t.get("family") or family) for t in items], float),
    }


def _npz_fingerprint(z, meta, arrays):
    """SHA-1 over a capture's meta and every array, so that only files with
    identical contents count as copies (two captures of one suite share all
    their metadata but not their numbers).  Arrays not loaded (_SKIP_ARRAYS)
    enter by the CRC-32 and size the zip directory holds for them."""
    h = hashlib.sha1(json.dumps(meta, sort_keys=True).encode())
    for name in sorted(arrays):
        a = np.ascontiguousarray(arrays[name])
        h.update(f"|{name}|{a.dtype.str}|{a.shape}|".encode())
        h.update(a.tobytes())
    zf = getattr(z, "zip", None)
    for name in sorted(n for n in z.files if n in _SKIP_ARRAYS):
        try:
            info = zf.getinfo(name + ".npy")
            h.update(f"|{name}|crc{info.CRC}|{info.file_size}|".encode())
        except (AttributeError, KeyError):
            a = np.ascontiguousarray(z[name])
            h.update(f"|{name}|{a.dtype.str}|{a.shape}|".encode())
            h.update(a.tobytes())
    return h.hexdigest()


def _measured_position(items, digit_mass):
    """Was activity measured at the position whose next token is the answer
    digit?  Returns {"status", "evidence"}:
      aligned     the items record deep_probe.align_answer ('appended' or
                  'measured_token'): the position is right by construction
      verified    no record, but the position predicts a digit (median digit
                  mass >= 0.5, or top-1 next token a digit on most items);
                  GPT-2 and Pythia keep " 6" whole and are right at the last
                  prompt token
      misaligned  no record and the file's own predictions show a position
                  that does not predict the answer: top-1 next token is
                  whitespace on at least half the items (the space that
                  split-digit tokenizers put before the digit: Llama-3, Qwen2.5,
                  OLMo-2 'Ġ'; Mistral, Yi, Phi-3, Gemma-2 '▁', which decodes
                  to ''), or median digit mass < 0.5.  The 17 September
                  deep_probe runs look like this
      unverified  no record and no evidence either way"""
    n = len(items)
    rec = sum(("appended" in t) or ("measured_token" in t) for t in items)
    if rec == n:
        return {"status": "aligned", "evidence": "align_answer record on every item"}
    top = []
    for t in items:
        t5 = t.get("top5")
        if isinstance(t5, (list, tuple)) and t5 and isinstance(t5[0], (list, tuple)) and t5[0]:
            top.append(str(t5[0][0]))
    ws = sum(s.strip() == "" for s in top)
    dg = sum(s.strip().isdigit() for s in top)
    dm = np.asarray(digit_mass, float)
    dm = dm[np.isfinite(dm)]
    what = "no align_answer record" + (f" (only {rec}/{n} items)" if rec else "")
    if top and 2 * ws >= len(top):
        return {"status": "misaligned",
                "evidence": f"{what}; top-1 next token is whitespace on {ws}/{len(top)} items"}
    if len(dm) and np.median(dm) < 0.5:
        return {"status": "misaligned", "evidence": f"{what}; median digit mass "
                                                    f"{np.median(dm):.2f} at the measured position"}
    if len(dm):
        return {"status": "verified",
                "evidence": f"{what}; median digit mass {np.median(dm):.2f}"}
    if top and 2 * dg > len(top):
        return {"status": "verified",
                "evidence": f"{what}; top-1 next token is a digit on {dg}/{len(top)} items"}
    return {"status": "unverified",
            "evidence": what + ("; no top-5 predictions or digit mass" if not top else
                                f"; top-1 next token a digit on {dg}/{len(top)} items, "
                                f"whitespace on {ws}")}


def _dig(x):
    """A stored digit (str or int) as a string; None if absent."""
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return None
    if isinstance(x, (int, np.integer)) and not isinstance(x, bool):
        return str(int(x))
    return str(x).strip()


def _check_predictions(path, items, golds, arrays, flags, n):
    """The correctness flags drive the correct-item analysis, so they must
    agree with the predictions stored beside them (both probes define
    correct_constrained = pred_constrained == gold and correct_strict =
    correct_constrained and first generated digit == gold).  A contradiction
    means a corrupted or hand-edited file."""
    def one_d(name):
        a = arrays.get(name)
        return a if a is not None and a.ndim == 1 and len(a) == n else None

    gold = list(golds)
    for name in ("gold", "pred_constrained"):             # arrays and records agree
        a = one_d(name)
        if a is None:
            continue
        for i, t in enumerate(items):
            v = gold[i] if name == "gold" else _dig(t.get(name))
            if v is not None and v.isdigit() and int(v) != int(a[i]):
                raise RunError(f"{path}: item {i} {name} {v!r} but the {name} array holds "
                               f"{int(a[i])}")
        if name == "gold":
            gold = [g if g is not None else str(int(a[i])) for i, g in enumerate(gold)]
    pred = [_dig(t.get("pred_constrained")) for t in items]
    if all(p is None for p in pred) and one_d("pred_constrained") is not None:
        pred = [str(int(v)) for v in one_d("pred_constrained")]
    cc, cs = flags.get("correct_constrained"), flags.get("correct_strict")
    for i in range(n):
        g = gold[i]
        if cc is not None and g is not None and pred[i] is not None and bool(cc[i]) != (pred[i] == g):
            raise RunError(f"{path}: item {i} correct_constrained={bool(cc[i])} but "
                           f"pred_constrained {pred[i]!r} and gold {g!r}")
        if cs is not None and cc is not None:
            if cs[i] and not cc[i]:
                raise RunError(f"{path}: item {i} correct_strict without correct_constrained")
            if g is not None and "pred_first_digit" in items[i]:
                want = bool(cc[i]) and _dig(items[i]["pred_first_digit"]) == g
                if bool(cs[i]) != want:
                    raise RunError(f"{path}: item {i} correct_strict={bool(cs[i])} but first "
                                   f"generated digit {items[i]['pred_first_digit']!r}, gold {g!r}")


def _subset(run, idx):
    """The run restricted to the items idx (every per-item field and array)."""
    idx = np.asarray(idx, int)
    n = run["n"]
    out = dict(run)
    out["arrays"] = {k: (v[idx] if isinstance(v, np.ndarray) and v.ndim >= 1 and len(v) == n else v)
                     for k, v in run["arrays"].items()}
    for f in ("ids", "k", "answer_set_size", "item_family", "item_wording", "correct",
              "gold_prob", "digit_mass", "correct_strict", "correct_constrained"):
        if run.get(f) is not None:
            out[f] = np.asarray(run[f])[idx]
    for f in ("gold", "echo", "items"):
        out[f] = [run[f][i] for i in idx]
    out["n"] = int(len(idx))
    return out


def _activity(run, metric):
    return run["arrays"].get(metric)


def _run_label(run):
    return f"{_short(run['model'])} / {run['suite']}"


# ────────────────────────────── per-run analyses ────────────────────────────
def _by_k(k, ok, y, yp):
    rows = []
    for d in np.unique(k):
        m = k == d
        rows.append({"k": int(d), "n": int(m.sum()), "correct": int(ok[m].sum()),
                     "acc": float(ok[m].mean()), "top_share": float(_nanmean(y[m])),
                     "pr_h": float(_nanmean(yp[m]))})
    return rows


def _pct(y, k, lo_k, hi_k):
    a, b = _nanmean(y[k == lo_k]), _nanmean(y[k == hi_k])
    return float(100 * (b / a - 1)) if np.isfinite(a) and a != 0 else float("nan")


def _correct_vs_wrong(y, k, ok, n_perm, rng):
    """Correct minus wrong at fixed k, averaged over depths with >= 3 of each;
    null: correctness shuffled within each depth."""
    strata = [np.nonzero((k == d) & np.isfinite(y))[0] for d in np.unique(k)]
    strata = [ix for ix in strata if ok[ix].sum() >= 3 and (~ok[ix]).sum() >= 3]
    if not strata:
        return None
    def diff(o):
        return float(np.mean([y[ix][o[ix]].mean() - y[ix][~o[ix]].mean() for ix in strata]))
    obs, null, o = diff(ok), np.empty(n_perm), ok.copy()
    for j in range(n_perm):
        for ix in strata:
            o[ix] = rng.permutation(ok[ix])
        null[j] = diff(o)
    return {"diff": obs, "p": _p_two_sided(obs, null),
            "depths": [int(k[ix[0]]) for ix in strata]}


def _adjusted_slope(y, k, ok, gp):
    """Descriptive slope on k with correctness and gold probability as
    covariates (outline: all-items results reported with both as covariates)."""
    m = np.isfinite(y) & np.isfinite(gp)
    if m.sum() < 5:
        return None
    X = [np.ones(m.sum()), k[m].astype(float)]
    if 0 < ok[m].sum() < m.sum():
        X.append(ok[m].astype(float))
    X.append(gp[m])
    beta, *_ = np.linalg.lstsq(np.stack(X, 1), y[m], rcond=None)
    return float(beta[1])


def signature(run, metric="contribution", *, alpha=0.05, n_perm=10000, rng=None,
              lo=0.4, hi=0.8, min_correct=20, head_mask=None):
    """A run's signature on the primary statistic (see module docstring).

    head_mask [n, L, H] (True = drop the head from the statistic) is how the
    sink control removes sink-dominated heads.  Classification:
      classification_all_items  spread / funnel / flat, or mixed when the
                                top-share and PR/H slopes are both significant
                                with the same sign
      classification            missing without a correct-item analysis
                                (fewer than 3 eligible depths) or without the
                                metric; inconclusive when the correct-item and
                                all-items slopes disagree in sign; else the
                                all-items class"""
    run = _as_run(run)
    rng = _rng(rng)
    base = {"model": run["model"], "suite": run["suite"], "family": run["family"],
            "wording": run["wording"], "metric": metric, "n": run["n"], "L": run["L"],
            "H": run["H"], "attn_impl": run["attn_impl"], "precision": run["precision"],
            "correctness": run["correctness"], "chance": run["chance"],
            "acc_threshold": 2 * run["chance"] / (1 + run["chance"])}
    S = _activity(run, metric)
    if S is None:
        return {**base, "classification": "missing", "classification_all_items": "missing",
                "reason": f"no {metric} in this run (accuracy-only?)"}
    L, H = run["L"], run["H"]
    w = window(L, lo, hi)
    TS, PR = top_share(S, head_mask), pr_h(S, head_mask)
    fin = TS[np.isfinite(TS)]
    if len(fin) and (fin.min() < 1 / H - 1e-6 or fin.max() > 1 + 1e-6):
        raise RunError(f"{_run_label(run)}: top share outside [1/H, 1]")
    y, yp = _nanmean(TS[:, w], 1), _nanmean(PR[:, w], 1)
    k, ok = run["k"], run["correct"]
    kmin, kmax = int(k.min()), int(k.max())
    base.update(window=[w.start, w.stop - 1], by_k=_by_k(k, ok, y, yp))
    if kmin == kmax:
        return {**base, "classification": "missing", "classification_all_items": "missing",
                "reason": "k does not vary"}

    s, p = slope_perm(y, k, n_perm, rng)
    spr, ppr = slope_perm(yp, k, n_perm, rng)
    if not np.isfinite(s):                  # e.g. every head masked out
        return {**base, "classification": "missing", "classification_all_items": "missing",
                "reason": "primary statistic undefined on too many items"}
    allr = {"slope": s, "p": p, "pct_change": _pct(y, k, kmin, kmax), "k_from": kmin,
            "k_to": kmax, "mean": float(_nanmean(y)),
            "slope_adjusted": _adjusted_slope(y, k, ok, run["gold_prob"])}
    clean = _clean_depth(y, k, run.get("min_hops"), run["family"], n_perm, rng)
    elig = eligible_depths(k, ok, run["chance"], min_correct)
    if len(elig) >= 3:
        sel = ok & np.isin(k, elig)
        sc, pc = slope_perm(y[sel], k[sel], n_perm, rng)
        corr = {"status": "ok", "slope": sc, "p": pc, "n": int(sel.sum()), "depths": elig,
                "pct_change": _pct(y[sel], k[sel], min(elig), max(elig))}
    else:
        corr = {"status": "missing", "depths": elig,
                "reason": f"{len(elig)} depth(s) with >= {min_correct} correct and accuracy "
                          f">= {base['acc_threshold']:.3f}; need 3"}

    sig, sig_pr = p < alpha, ppr < alpha
    if sig and sig_pr and _sign(s) == _sign(spr):
        cls_all = "mixed"
    elif sig:
        cls_all = "spread" if s < 0 else "funnel"
    else:
        cls_all = "flat"
    if corr["status"] != "ok":
        cls, reason = "missing", corr["reason"]
    elif _sign(corr["slope"]) != _sign(s):
        cls, reason = "inconclusive", "correct-item and all-items slopes disagree in sign"
    else:
        cls, reason = cls_all, None
    return {**base, "all_items": allr, "pr_h": {"slope": spr, "p": ppr},
            "eligible_depths": elig, "correct_items": corr, "clean_depth": clean,
            "correct_vs_wrong": _correct_vs_wrong(y, k, ok, min(n_perm, 2000), rng),
            "classification_all_items": cls_all, "classification": cls, "reason": reason}


def _clean_depth(y, k, mh, family, n_perm, rng):
    """The primary statistic's slope on items whose shortest lookup path equals
    k.  On an n-cycle, k steps forward land where n - k steps backward do, so
    for k > n/2 the written depth overstates the work a model that can read
    the rules backwards must do; the slope over the clean items is the depth
    effect free of that shortcut.  Also the slope on min_hops over all items."""
    if mh is None or not len(mh) or not np.isfinite(mh).all():
        why = ("no cycle shortcut in this family" if family in ("arith", "binding", "relational")
               else "min_hops not recorded and not recoverable from the prompts")
        return {"status": "not_applicable", "reason": why}
    sel = mh == k
    out = {"confounded_depths": sorted({int(d) for d in k[~sel]}),
           "min_hops_by_k": {int(d): sorted({int(x) for x in mh[k == d]}) for d in np.unique(k)}}
    if sel.all():
        return {**out, "status": "all_clean"}
    ds = sorted({int(d) for d in k[sel]})
    if len(ds) >= 2 and sel.sum() >= 6:
        s, p = slope_perm(y[sel], k[sel], n_perm, rng)
        out.update(status="ok", slope=s, p=p, n=int(sel.sum()), depths=ds,
                   pct_change=_pct(y[sel], k[sel], min(ds), max(ds)))
    else:
        out.update(status="missing", depths=ds, reason="fewer than 2 clean depths")
    if len(np.unique(mh)) >= 2:
        sm, pm = slope_perm(y, mh, n_perm, rng)
        out["slope_on_min_hops"] = {"slope": sm, "p": pm}
    return out


def power(run, delta=0.02, alpha=0.05, power=0.8, *, metric="contribution", lo=0.4, hi=0.8):
    """Items per depth needed to detect a relative change `delta` in the primary
    statistic between the lowest and highest depth, with the design's slope test
    (two-sided, level alpha) at the given power.  Normal approximation with the
    pilot's OLS residual variance (y on k) and a linear change across depths;
    on the Sep 22 pilots this gives 7 items for Qwen2.5-7B and 82 for
    OLMo-2-7B-Instruct."""
    run = _as_run(run)
    S = _activity(run, metric)
    if S is None:
        raise ValueError(f"{_run_label(run)}: no {metric}")
    y, k = top_share_window(S, run["L"], lo, hi), run["k"].astype(float)
    m = np.isfinite(y)
    y, k = y[m], k[m]
    depths = np.unique(k)
    if len(depths) < 2 or len(y) < 4:
        raise ValueError("power needs at least two depths and four items")
    res = y - y.mean() - _ols_slope(y, k) * (k - k.mean())
    var = float(res @ res / (len(y) - 2))
    beta = delta * y[k == depths[0]].mean() / (depths[-1] - depths[0])
    z = NormalDist().inv_cdf(1 - alpha / 2) + NormalDist().inv_cdf(power)
    sxx = float(((depths - depths.mean()) ** 2).sum())
    return int(max(2, math.ceil(z * z * var / (beta * beta * sxx))))


def sink_control(run, threshold=0.9, *, metric="contribution", **kw):
    """Signature with sink-dominated heads (attention from the measured position
    to position 0 above `threshold`) removed per item and layer.  Qwen2.5 has
    no BOS, so its position 0 is a content token; Llama-3's is a BOS."""
    run = _as_run(run)
    A0 = run["arrays"].get("pos0_attn")
    if A0 is None:
        return {"status": "missing", "reason": "no pos0_attn (run without --pos0)",
                "model": run["model"], "suite": run["suite"]}
    ex = A0 > threshold
    w = window(run["L"], kw.get("lo", 0.4), kw.get("hi", 0.8))
    # one seed for both: with nothing removed the two signatures must be
    # identical, not differ by Monte Carlo noise that can flip a class near alpha
    seed = int(_rng(kw.pop("rng", None)).integers(1 << 62))
    orig = signature(run, metric, rng=seed, **kw)
    masked = signature(run, metric, head_mask=ex, rng=seed, **kw)
    s0 = (orig.get("all_items") or {}).get("slope")
    s1 = (masked.get("all_items") or {}).get("slope")
    return {"status": "ok", "model": run["model"], "suite": run["suite"],
            "threshold": threshold, "has_bos": run["has_bos"],
            "frac_removed": float(ex.mean()), "frac_removed_window": float(ex[:, w].mean()),
            "frac_removed_by_layer": ex.mean((0, 2)).tolist(),
            "signature": orig, "signature_masked": masked,
            "sign_agree": _sign(s0) == _sign(s1),
            "class_agree": orig["classification_all_items"] == masked["classification_all_items"]}


def mass_centroid(run, metric="contribution", n_perm=5000, rng=None):
    """Centre of the attention-output mass sum_h s over layers, per item, and
    its slope on k: does the work move earlier or later as k rises?"""
    run = _as_run(run)
    S = _activity(run, metric)
    if S is None:
        return {"status": "missing", "reason": f"no {metric}"}
    if run["L"] is None or run["L"] < 2:
        raise ValueError(f"mass_centroid needs L >= 2 layers, got {run['L']}")
    M = np.asarray(S, float).sum(-1)
    c = (M * np.arange(run["L"])).sum(1) / M.sum(1)
    k = run["k"]
    s, p = slope_perm(c, k, n_perm, _rng(rng))
    lo, hi = k.min(), k.max()
    shift = float(c[k == hi].mean() - c[k == lo].mean())
    return {"status": "ok", "model": run["model"], "suite": run["suite"],
            "by_k": {int(d): float(c[k == d].mean()) for d in np.unique(k)},
            "slope": s, "p": p, "shift_layers": shift, "shift_rel": shift / (run["L"] - 1),
            "centroid": c}


# ────────────────────────────── cross-run analyses ──────────────────────────
def _item_fields(run, ix):
    """(prompt text, gold) of the run's items ix; None where not recorded."""
    items, gold = run.get("items"), run.get("gold")
    out = []
    for j in ix:
        t = items[j].get("text") if items is not None and isinstance(items[j], dict) else None
        g = gold[j] if gold is not None else None
        out.append((None if t is None else str(t).strip(), None if g is None else str(g)))
    return out


def _check_same_items(runs, idx, ids):
    """The same id must mean the same item in every run: same depth, prompt
    and gold wherever recorded.  Ids alone are not enough: legacy files get
    ids from the suite name and label ('serial_depth_fs/D1N00'), which two
    files of prompts drawn differently (another seed or generator version)
    share although no item is the same."""
    ref_k = runs[0]["k"][idx[0]]
    ref = [[None, None] for _ in ids]             # first recorded text / gold per id
    src = [[None, None] for _ in ids]
    for r, ix in zip(runs, idx):
        if not np.array_equal(r["k"][ix], ref_k):
            raise RunError(f"{_run_label(r)}: the same item ids carry different depths")
        bad = []
        for i, fields in enumerate(_item_fields(r, ix)):
            for f, v in enumerate(fields):
                if v is None:
                    continue
                if ref[i][f] is None:
                    ref[i][f], src[i][f] = v, r
                elif v != ref[i][f]:
                    bad.append((ids[i], ("prompt", "gold")[f], src[i][f]))
        if bad:
            i0, what, other = bad[0]
            raise RunError(f"{_run_label(r)}: {len({b[0] for b in bad})} of {len(ids)} shared item "
                           f"ids hold a different prompt or gold than in {_run_label(other)} "
                           f"(e.g. {i0!r}: {what}); the runs do not measure the same items")


def _align_items(runs, metric, lo=0.4, hi=0.8):
    """Primary statistic of each run on the item ids all runs share, in the
    first run's order.  The same id must mean the same item everywhere
    (depth, prompt and gold; _check_same_items)."""
    shared = set(runs[0]["ids"])
    for r in runs[1:]:
        shared &= set(r["ids"])
    ids = [i for i in runs[0]["ids"] if i in shared]
    if len(ids) < 3:
        raise ValueError(f"runs share {len(ids)} item ids; need >= 3")
    Y, idx = [], []
    for r in runs:
        S = _activity(r, metric)
        if S is None:
            raise ValueError(f"{_run_label(r)}: no {metric}")
        pos = {i: j for j, i in enumerate(r["ids"])}
        ix = np.array([pos[i] for i in ids])
        idx.append(ix)
        Y.append(top_share_window(S, r["L"], lo, hi)[ix])
    _check_same_items(runs, idx, ids)
    return ids, runs[0]["k"][idx[0]], np.stack(Y, 1)


def joint_relabel(runs, metric="contribution", n_perm=5000, *, rng=None, pairs=None,
                  scale="mean", lo=0.4, hi=0.8):
    """Cross-model test on shared items under ONE relabelling of k.

    Per run: OLS slope of the primary statistic on k, scale-normalised (divided
    by the run's mean, i.e. relative change per step; scale="sd" divides by
    its SD).  Each null draw shuffles k once and applies that shuffle to every
    run, the design-respecting null when models see the same items.  Reports
    each run's p, the pooled mean slope and its p, and paired differences
    (runs[j] - runs[i]; default for two runs: second minus first, e.g.
    instruct minus base)."""
    runs = [_as_run(r) for r in runs]
    rng = _rng(rng)
    ids, k, Y = _align_items(runs, metric, lo, hi)
    if not np.isfinite(Y).all():
        raise ValueError("non-finite primary statistic on shared items")
    norm = Y.mean(0) if scale == "mean" else Y.std(0, ddof=1)
    kc = k - k.mean()
    ss = kc @ kc
    if ss == 0:
        raise ValueError("k does not vary on the shared items")
    Yc = (Y - Y.mean(0)) / norm
    obs = kc @ Yc / ss
    null = np.concatenate([P @ Yc / ss for P in _perm_rows(kc, n_perm, rng)])  # [n_perm, R]
    names = [f"{_short(r['model'])}" for r in runs]
    if pairs is None:
        pairs = [(0, 1)] if len(runs) == 2 else []
    return {
        "n_items": len(ids), "models": [r["model"] for r in runs], "suite": runs[0]["suite"],
        "scale": scale, "slopes": obs.tolist(), "p_each": _perm_p_cols(obs, null).tolist(),
        "pooled": float(obs.mean()), "p_pooled": _p_two_sided(obs.mean(), null.mean(1)),
        "diffs": [{"a": names[i], "b": names[j], "diff": float(obs[j] - obs[i]),
                   "p": _p_two_sided(obs[j] - obs[i], null[:, j] - null[:, i])}
                  for i, j in pairs],
    }


def kernel_compare(run_a, run_b, *, metric="contribution", late=0.8, **kw):
    """Same model and items under two kernels (or precisions; pass the
    reference, e.g. fp32, as run_b): score correlation, median relative
    difference in the final 20% of layers, per-layer slope agreement for top
    share and PR/H, and both signatures.

    Everything, the two signatures included, is computed on the item ids the
    runs share (a kernel-check re-run may cover a subset, e.g. --limit), and
    both signatures use the same permutation seed, so identical activity gives
    identical signatures."""
    a, b = _as_run(run_a), _as_run(run_b)
    if not _model_match(a["model"], b["model"]):
        raise RunError(f"kernel_compare compares ONE model under two configurations; got "
                       f"{a['model']} and {b['model']}")
    if _activity(a, metric) is None or _activity(b, metric) is None:
        raise ValueError(f"kernel_compare needs {metric} in both runs")
    if (a["L"], a["H"]) != (b["L"], b["H"]):
        raise ValueError("runs differ in shape; not the same model")
    shared = set(a["ids"]) & set(b["ids"])
    ids = [i for i in a["ids"] if i in shared]
    if len(ids) < 3:
        raise ValueError("fewer than 3 shared items")
    pa = {i: j for j, i in enumerate(a["ids"])}
    pb = {i: j for j, i in enumerate(b["ids"])}
    ia, ib = np.array([pa[i] for i in ids]), np.array([pb[i] for i in ids])
    _check_same_items([a, b], [ia, ib], ids)       # same depth, prompt and gold per id
    Sa, Sb = _activity(a, metric)[ia].astype(float), _activity(b, metric)[ib].astype(float)
    k, L = a["k"][ia], a["L"]
    fin = np.arange(L) > math.floor(late * (L - 1) + 1e-9)
    den = np.abs(Sa) + np.abs(Sb)
    rel = np.where(den > 0, 2 * np.abs(Sa - Sb) / np.where(den > 0, den, 1), 0.0)
    per_layer = {}
    for name, f in (("top_share", top_share), ("pr_h", pr_h)):
        kc = k - k.mean()
        la = kc @ (f(Sa) - f(Sa).mean(0)) / (kc @ kc)
        lb = kc @ (f(Sb) - f(Sb).mean(0)) / (kc @ kc)
        per_layer[name] = {"r": float(np.corrcoef(la, lb)[0, 1]),
                           "same_sign": int((np.sign(la) == np.sign(lb)).sum()), "L": L}
    # one seed for both: the comparison is paired, not two independent nulls
    seed = int(_rng(kw.pop("rng", None)).integers(1 << 62))
    sa = signature(_subset(a, ia), metric, rng=seed, **kw)
    sb = signature(_subset(b, ib), metric, rng=seed, **kw)
    s1 = (sa.get("all_items") or {}).get("slope")
    s2 = (sb.get("all_items") or {}).get("slope")
    sign_agree = _sign(s1) == _sign(s2)
    class_agree = sa["classification_all_items"] == sb["classification_all_items"]
    outside = {"a": a["n"] - len(ids), "b": b["n"] - len(ids)}
    warn = ([f"run {s} has {v} item(s) outside the shared set; compared on the {len(ids)} "
             f"shared items" for s, v in outside.items() if v]) or None
    return {"model": a["model"], "suite": a["suite"], "n_items": len(ids),
            "n_items_a": a["n"], "n_items_b": b["n"], "warning": warn,
            "a": {"attn_impl": a["attn_impl"], "precision": a["precision"], "path": a["path"]},
            "b": {"attn_impl": b["attn_impl"], "precision": b["precision"], "path": b["path"]},
            "score_r": float(np.corrcoef(Sa.ravel(), Sb.ravel())[0, 1]),
            "late_rel_diff_median": float(np.median(rel[:, fin])) if fin.any() else None,
            "rel_diff_median": float(np.median(rel)),
            "per_layer": per_layer,
            "signature_a": sa, "signature_b": sb,
            "sign_agree": sign_agree, "class_agree": class_agree,
            "agree": bool(sign_agree and class_agree)}


def _sign_of(pre):
    if isinstance(pre, (int, float, np.integer, np.floating)) and not isinstance(pre, bool):
        if pre == 0:
            raise ValueError("preregistered sign must be non-zero")
        return int(np.sign(pre))
    s = str(pre).strip().lower()
    if s in ("funnel", "funnelling", "+", "+1", "1", "positive", "up"):
        return 1
    if s in ("spread", "spreading", "-", "-1", "negative", "down"):
        return -1
    raise ValueError(f"preregistered sign {pre!r}: use funnel/spread or +1/-1")


_NOT_PREREGISTERED = ("none", "skip", "exclude")


def _prereg_value(v):
    """A --preregistered value: +1 / -1, or None (none / skip / exclude) for a
    model that gets no RQ1 verdict."""
    if v is None or (isinstance(v, str) and v.strip().lower() in _NOT_PREREGISTERED):
        return None
    return _sign_of(v)


def _post_marks(name):
    return {m.lower() for m in _POST_MARK.findall(str(name))}


def _prereg_match(model, prereg):
    """(value, pattern) of the --preregistered entry that covers `model`, or
    (None, None).  value None: excluded on purpose (MODEL=none).

      text       occurs in the model name (case-insensitive).  A post-trained
                 model (-Instruct, -SFT, -DPO, -it, -chat, -RLVR) is covered
                 only by a pattern naming its marker: the outline pre-registers
                 base and post-trained models separately ("Qwen2.5 base:
                 funnel", "post-trained OLMo: spread"), so qwen2.5 must not
                 reach Qwen2.5-7B-Instruct
      text$      the same, anchored at the end of the name
      re:REGEX   regular expression searched in the name; no marker rule
    Precedence: a pattern equal to the full or short model name, then the
    longest pattern.  Equally specific patterns with different values raise."""
    full = str(model).lower()
    short = _short(full)
    marks = _post_marks(short)
    hits = []
    for pat, val in prereg.items():
        p = str(pat).strip()
        pl = p.lower()
        if pl.startswith("re:"):
            ok, exact = re.search(p[3:], str(model), re.I) is not None, False
        else:
            body = pl[:-1] if pl.endswith("$") else pl
            if not body:
                continue
            ok = ((full.endswith(body) if pl.endswith("$") else body in full)
                  and marks <= _post_marks(body))
            exact = body in (full, short)
        if ok:
            hits.append((exact, len(p), p, val))
    if not hits:
        return None, None
    best = max(h[:2] for h in hits)
    top = [h for h in hits if h[:2] == best]
    if len({h[3] for h in top}) > 1:
        raise ValueError(f"{model}: equally specific --preregistered patterns disagree: "
                         f"{', '.join(h[2] for h in top)}")
    return top[0][3], top[0][2]


def _pre_sign_for(model, prereg):
    """The pre-registered sign of `model` (None: not pre-registered)."""
    v, _ = _prereg_match(model, {p: _prereg_value(s) for p, s in prereg.items()})
    return v


def _stratified_boot_idx(ks, n_boot, rng):
    idx = np.empty((n_boot, len(ks)), int)
    pos = 0
    for d in np.unique(ks):
        ix = np.nonzero(ks == d)[0]
        idx[:, pos:pos + len(ix)] = ix[rng.integers(0, len(ix), (n_boot, len(ix)))]
        pos += len(ix)
    return idx


def _mixedlm(present):
    """Secondary: random intercepts and slopes by family, variance components
    for wording (cell) intercepts and slopes within family.  Reported only if
    statsmodels imports and the optimiser converges.  A variance on the
    boundary (zero), which is the MLE when families agree, keeps status "ok"
    but sets boundary=True: the Hessian is then not positive definite and the
    slope's SE ignores between-family variation, so the report flags it."""
    fams = {c[0] for c in present}
    if len(fams) < 2:
        return {"status": "skipped", "reason": "fewer than 2 families"}
    try:
        import pandas as pd
        import statsmodels.formula.api as smf
    except Exception as e:
        return {"status": "unavailable", "reason": str(e)}
    # present is keyed by (family, cell label); a cell is one wording of one suite
    rows = [(float(y), float(k), f, cell) for (f, cell), (ys, ks) in present.items()
            for y, k in zip(ys, ks)]
    df = pd.DataFrame(rows, columns=["y", "k", "family", "cell"])
    sd = float(df["y"].std()) or 1.0
    # unit scale and centred k: the optimiser fails far more often without
    df["y"] = (df["y"] - df["y"].mean()) / sd
    df["k"] = df["k"] - df["k"].mean()
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            md = smf.mixedlm("y ~ k", df, groups=df["family"], re_formula="~k",
                             vc_formula={"cell": "0 + C(cell)", "cell_k": "0 + C(cell):k"})
            fit = md.fit(reml=True, method=["powell", "lbfgs"])
        msgs = sorted({str(w.message).split(".")[0] for w in caught})
        failed = [m for m in msgs if "failed" in m.lower()]
        if not fit.converged or failed or not np.isfinite(fit.bse["k"]):
            return {"status": "not converged", "warnings": msgs[:4]}
        boundary = any(w in m.lower() for m in msgs
                       for w in ("not positive definite", "boundary", "singular"))
        return {"status": "ok", "boundary": bool(boundary),
                "slope": float(fit.fe_params["k"] * sd),
                "se": float(fit.bse["k"] * sd), "p": float(fit.pvalues["k"]), "notes": msgs[:4]}
    except Exception as e:
        return {"status": "failed", "reason": f"{type(e).__name__}: {e}"}


def _cell_labels(keys):
    """Report names of RQ1 cells keyed (family, wording, suite, suite_sha256):
    'family/wording', with the suite added when one family and wording has
    several suites (and a hash prefix if their names also coincide)."""
    fw = Counter((f, w) for f, w, _, _ in keys)
    fws = Counter((f, w, s) for f, w, s, _ in keys)
    out = {}
    for f, w, s, h in keys:
        lab = f"{f}/{w}"
        if fw[(f, w)] > 1:
            lab += f" [{s}]"
            if fws[(f, w, s)] > 1:
                lab += f" sha {str(h)[:8]}"
        out[(f, w, s, h)] = lab
    return out


def _invariance_core(cells, mode, sign, alpha, n_boot, n_perm, rng, min_correct, family_frac,
                     min_families):
    """cells {(family, wording, suite, suite_sha256): (y, k, correct, chance)};
    a (family, wording) key is one cell of an unnamed suite."""
    cells = {(tuple(c) + (None, None) if len(c) == 2 else tuple(c)): v for c, v in cells.items()}
    labels = _cell_labels(list(cells))
    cell_out, present, where = {}, {}, {}
    for key in sorted(cells, key=lambda c: (c[0], c[1], str(c[2]), str(c[3]))):
        fam, wd, suite, _ = key
        y, k, ok, ch = cells[key]
        lab = labels[key]
        info = {"family": fam, "wording": wd, "suite": suite}
        fin = np.isfinite(y)
        if mode == "correct":
            elig = eligible_depths(k[fin], ok[fin], float(np.mean(ch)), min_correct)
            if len(elig) < 3:
                cell_out[lab] = {"status": "missing", "eligible_depths": elig, **info}
                continue
            sel = fin & ok & np.isin(k, elig)
        else:
            elig, sel = sorted(int(d) for d in np.unique(k[fin])), fin
            if len(elig) < 2:
                cell_out[lab] = {"status": "missing", "eligible_depths": elig, **info}
                continue
        present[(fam, lab)] = (y[sel], k[sel].astype(float))
        where[lab] = (wd, suite)
        cell_out[lab] = {"status": "ok", "eligible_depths": elig, **info,
                         "n": int(sel.sum()), "slope": _ols_slope(y[sel], k[sel])}
    all_fams = sorted({c[0] for c in cells})
    if not present:
        return {"verdict": "missing", "cells": cell_out, "families": {},
                "families_missing": all_fams, "n_families_present": 0}

    keys = list(present)
    obs = np.array([_ols_slope(*present[c]) for c in keys])
    boot = np.empty((n_boot, len(keys)))
    perm = np.empty((n_perm, len(keys)))
    for j, c in enumerate(keys):
        ys, ks = present[c]
        bi = _stratified_boot_idx(ks, n_boot, rng)
        boot[:, j] = _row_slopes(ys[bi], ks[bi])
        pk = np.concatenate(list(_perm_rows(ks, n_perm, rng)))
        perm[:, j] = _row_slopes(np.broadcast_to(ys, pk.shape), pk)
    pooled = float(obs.mean())
    ci = [float(np.percentile(boot.mean(1), 2.5)), float(np.percentile(boot.mean(1), 97.5))]
    p_pooled = _p_two_sided(pooled, perm.mean(1))

    fams = sorted({c[0] for c in keys})
    fres = {}
    for f in fams:
        cols = [j for j, c in enumerate(keys) if c[0] == f]
        s = float(obs[cols].mean())                # mean over the family's (wording, suite) cells
        cl = [keys[j][1] for j in cols]
        fres[f] = {"slope": s, "p": _p_two_sided(s, perm[:, cols].mean(1)),
                   "wordings": sorted({where[c][0] for c in cl}), "cells": cl,
                   "suites": [where[c][1] for c in cl]}
    ph = holm([fres[f]["p"] for f in fams])
    for f, h in zip(fams, ph):
        fres[f]["p_holm"] = float(h)
    n_present = len(fams)
    n_with = sum(_sign(fres[f]["slope"]) == sign for f in fams)
    need = math.ceil(family_frac * n_present - 1e-9)
    opposite = [f for f in fams if fres[f]["p_holm"] < alpha and _sign(fres[f]["slope"]) == -sign]
    a = (ci[0] > 0) if sign > 0 else (ci[1] < 0)
    # one family cannot show invariance ACROSS families, whatever its sign
    b = n_with >= need and n_present >= min_families
    c = not opposite
    verdict = ("task-dependent" if opposite else "invariant" if (a and b and c)
               else "inconclusive")
    return {"verdict": verdict, "pooled": pooled, "ci95": ci, "p_pooled": p_pooled,
            "criteria": {"a_pooled_ci_excludes_0_with_sign": bool(a),
                         "b_families_with_sign": f"{n_with}/{n_present} (need {need}, "
                                                 f"and >= {min_families} families)",
                         "b": bool(b), "c_no_significant_opposite_family": bool(c)},
            "opposite_families": opposite, "cells": cell_out, "families": fres,
            "families_missing": [f for f in all_fams if f not in fams],
            "n_families_present": n_present, "_present": present}


def invariance(runs_of_one_model, preregistered_sign, alpha=0.05, *, metric="contribution",
               n_boot=2000, n_perm=5000, rng=None, min_correct=20, lo=0.4, hi=0.8,
               family_frac=0.75, min_families=2, mixedlm=True, task_families=TASK_FAMILIES):
    """RQ1 for one model: is its pre-registered direction invariant across task
    families and wordings?

    Pass the runs of ONE configuration (kernel, precision).  Only items of the
    task families (`task_families`, default pointer / arith / binding /
    relational; None keeps every family) form cells: the numeral control
    (fixed_digit) and the single-prompt ladder are not task families, and
    counting them would change both the verdict and the 3-of-4 rule; their
    cells are listed under excluded_cells.

    Items are pooled into (family, wording, suite) cells: two suites of one
    family and wording (E1 picks a setting per family AND comparison group, so
    a model on the matched pair and a ladder can run two settings) are two
    cells, never one regression, whose pooled OLS could show a slope that
    neither suite has (Simpson's paradox from different levels and k ranges).
    Per cell: OLS slope of the primary statistic on k over the items correct
    at eligible depths (cells with fewer than 3 eligible depths are missing
    and do not count).  Pooled slope = mean of cell slopes, 95% CI by a
    bootstrap that resamples items within each (cell, k) stratum; family slope
    = mean over its cells (wordings x suites), p from shuffling k within
    cells.  Verdict:
      invariant       (a) pooled CI excludes 0 with the pre-registered sign,
                      (b) at least ceil(3/4 of the families present) have
                      that sign, and at least min_families (2) are present,
                      (c) no family significant in the opposite sign after
                      Holm across families
      task-dependent  some family significant in the opposite sign (Holm)
      inconclusive    otherwise;  missing: no cell meets the rules
    The same on all items is reported, with a flag when the verdicts differ.
    p_pooled (permutation) is the per-model primary p for Holm across models."""
    rng = _rng(rng)
    sign = _sign_of(preregistered_sign)
    runs = [_as_run(r) for r in runs_of_one_model]
    seen = [i for r in runs for i in r["ids"]]
    if len(set(seen)) != len(seen):
        raise RunError("the same item ids appear in two runs of this model; one run per "
                       "suite and configuration")
    tf = None if task_families is None else {str(f) for f in task_families}
    cells, skipped, excluded = {}, [], {}
    for r in runs:
        S = _activity(r, metric)
        if S is None:
            skipped.append(_run_label(r))
            continue
        y = top_share_window(S, r["L"], lo, hi)
        ch = 1.0 / np.maximum(r["answer_set_size"], 1)
        suite, sha = str(r.get("suite")), r.get("suite_sha256")
        for fam, wd in sorted(set(zip(r["item_family"], r["item_wording"]))):
            if tf is not None and fam not in tf:
                excluded[f"{fam}/{wd}"] = CONTROL_FAMILIES.get(fam, "not a task family")
                continue
            m = (r["item_family"] == fam) & (r["item_wording"] == wd)
            parts = (y[m], r["k"][m], r["correct"][m], ch[m])
            key = (fam, wd, suite, sha)
            if key in cells:          # one suite over two captures (its calib and test halves)
                parts = tuple(np.concatenate([o, p]) for o, p in zip(cells[key], parts))
            cells[key] = parts
    args = (sign, alpha, n_boot, n_perm, rng, min_correct, family_frac, min_families)
    corr = _invariance_core(cells, "correct", *args)
    alli = _invariance_core(cells, "all", *args)
    present = corr.pop("_present", None)
    alli.pop("_present", None)
    out = {"models": sorted({r["model"] for r in runs}), "metric": metric,
           "preregistered_sign": sign, "alpha": alpha, "skipped_runs": skipped,
           "task_families": None if tf is None else sorted(tf), "excluded_cells": excluded,
           "correct_items": corr, "all_items": alli,
           "verdict": corr["verdict"], "verdict_all_items": alli["verdict"],
           "disagree": (corr["verdict"] != alli["verdict"]
                        and "missing" not in (corr["verdict"], alli["verdict"]))}
    if mixedlm and present:
        out["mixedlm"] = _mixedlm(present)
    return out


# ────────────────────────────── E1 calibration ──────────────────────────────
def _model_match(run_model, name):
    a, b = str(run_model).lower(), str(name).lower()
    return a == b or _short(a) == _short(b)


def group_by_setting(runs):
    """{setting key: [runs]} from each run's suite setting (canonical JSON), the
    input choose_setting expects."""
    out = {}
    for r in runs:
        r = _as_run(r)
        out.setdefault(json.dumps(r["setting"], sort_keys=True), []).append(r)
    return out


def _test_per_k(run):
    """Items per depth on the TEST split of the run's suite, for a capture of
    the calibration split whose setting records n_per and calib_frac
    (serial_tasks puts round(calib_frac * n_per) items of every depth, or of
    every shared body, in calib); None otherwise (the run's own counts are
    then judged, as for a test-split or legacy run)."""
    if run.get("split") != "calib":
        return None
    s = run.get("setting") or {}
    try:
        n_per, f = int(s["n_per"]), float(s["calib_frac"])
    except (KeyError, TypeError, ValueError):
        return None
    return n_per - int(round(f * n_per))


def _cell_eligibility(k, ok, chance, min_correct, n_test):
    """Eligible depths of one (family, wording) cell and its by-k rows.  With
    n_test (items per depth on the test split) the rule is judged for the
    test split the setting is chosen for: calibration accuracy >= 2c/(1+c)
    and projected correct items acc * n_test >= min_correct.  (Applied to the
    calibration split itself, the absolute count could never be met: 12
    calibration items per depth at n_per 40.)"""
    if n_test is None:
        elig = eligible_depths(k, ok, chance, min_correct)
    thr = 2 * chance / (1 + chance)
    rows, proj = [], []
    for d in np.unique(k):
        m = k == d
        nc, acc = int(ok[m].sum()), float(ok[m].mean())
        row = {"k": int(d), "n": int(m.sum()), "correct": nc, "acc": acc}
        if n_test is not None:
            row["projected_test_correct"] = round(acc * n_test, 1)
            if acc * n_test >= min_correct - 1e-9 and acc >= thr - 1e-12:
                proj.append(int(d))
        rows.append(row)
    return (elig if n_test is None else proj), rows


def choose_setting(calib_runs_by_setting, groups, *, min_depths=3, min_correct=20, wordings=None):
    """E1: one setting per task family and comparison group.

    calib_runs_by_setting {setting: [runs or paths]} (accuracy-only runs on the
    calibration split are enough); groups {name: [model, ...]}.  Eligibility
    is judged per (family, wording) cell, as invariance() does, and for the
    TEST split: a calibration-split run whose setting has n_per and calib_frac
    is projected to the test split's n_per - round(calib_frac * n_per) items
    per depth (_cell_eligibility); other runs are judged on their own counts.
    A model's number of eligible depths in a family is that of its weakest
    wording, over every wording the family needs: `wordings` ({family:
    [wording, ...]} or one list for all families; default: every wording seen
    for the family under any setting and model).  A wording with no run for
    the model under a setting counts as 0 eligible depths (listed under
    missing_wordings): E2 runs every wording, so a setting must not win
    because a harder wording's calibration run crashed or has not run yet.
    For each family and group, the setting under which the WEAKEST model of
    the group (fewest eligible depths, then lowest accuracy) has the most
    eligible depths, at least `min_depths`; ties go to more serial depth (the
    weakest model's deepest eligible depth, then the setting's largest k).  A
    setting where a group model has no run is not a candidate.  All
    settings' accuracy tables are returned, with the projected counts."""
    tables, pooled = {}, {}
    for setting, runs in calib_runs_by_setting.items():
        for r in (_as_run(x) for x in runs):
            nt = _test_per_k(r)
            for fam, wd in sorted(set(zip(r["item_family"], r["item_wording"]))):
                m = (r["item_family"] == fam) & (r["item_wording"] == wd)
                cells = pooled.setdefault((setting, fam, r["model"]), {})
                parts = (r["k"][m], r["correct"][m], 1.0 / np.maximum(r["answer_set_size"][m], 1))
                ntw = nt
                if wd in cells:
                    old = cells[wd]
                    parts = tuple(np.concatenate([o, p]) for o, p in zip(old[:3], parts))
                    ntw = min((x for x in (old[3], nt) if x is not None), default=None)
                cells[wd] = (*parts, ntw)
    seen = {}
    for (_, fam, _), cells in pooled.items():
        seen.setdefault(fam, set()).update(cells)

    def needed(fam, cells):
        if wordings is None:
            want = seen.get(fam, set())
        elif isinstance(wordings, dict):
            want = set(wordings.get(fam, seen.get(fam, set())))
        else:
            want = set(wordings)
        return sorted(want | set(cells))

    for (setting, fam, model), cells in pooled.items():
        k = np.concatenate([c[0] for c in cells.values()])
        ok = np.concatenate([c[1] for c in cells.values()])
        ch = np.concatenate([c[2] for c in cells.values()])
        wout, missing = {}, []
        for wd in needed(fam, cells):
            if wd not in cells:
                missing.append(wd)
                wout[wd] = {"status": "missing", "chance": None, "acc": None,
                            "eligible_depths": [], "n_test_per_k": None,
                            "basis": "no run under this setting (counts as 0 eligible depths)",
                            "by_k": []}
                continue
            kw, okw, chw, nt = cells[wd]
            e, rows = _cell_eligibility(kw, okw, float(chw.mean()), min_correct, nt)
            wout[wd] = {"status": "ok", "chance": float(chw.mean()), "acc": float(okw.mean()),
                        "eligible_depths": e, "n_test_per_k": nt,
                        "basis": "projected to the test split" if nt is not None else "own counts",
                        "by_k": rows}
        sets = [set(v["eligible_depths"]) for v in wout.values()]
        tables.setdefault(setting, {}).setdefault(fam, {})[model] = {
            "chance": float(ch.mean()), "acc": float(ok.mean()),
            "eligible_depths": sorted(set.intersection(*sets)),    # in every wording
            "n_eligible": min(len(s) for s in sets),               # weakest wording
            "max_eligible_depth": (None if any(not s for s in sets)
                                   else min(max(s) for s in sets)),
            "missing_wordings": missing,
            "k_max": int(k.max()), "wordings": wout,
            "by_k": [{"k": int(d), "n": int((k == d).sum()), "correct": int(ok[k == d].sum()),
                      "acc": float(ok[k == d].mean())} for d in np.unique(k)]}
    families = sorted({f for t in tables.values() for f in t})
    choices = {}
    for fam in families:
        for g, models in groups.items():
            cands = []
            for setting, t in tables.items():
                rows = t.get(fam, {})
                got = {}
                for mname in models:
                    hit = [v for rm, v in rows.items() if _model_match(rm, mname)]
                    if hit:
                        got[mname] = hit[0]
                if len(got) < len(models):
                    continue
                weak = min(got, key=lambda mm: (got[mm]["n_eligible"], got[mm]["acc"]))
                wv = got[weak]
                cands.append({"setting": setting, "weakest": weak,
                              "n_eligible": wv["n_eligible"],
                              "max_eligible_depth": wv["max_eligible_depth"],
                              "k_max": max(v["k_max"] for v in got.values()),
                              "acc_weakest": wv["acc"],
                              "per_model": {mm: {wd: c["eligible_depths"]
                                                 for wd, c in v["wordings"].items()}
                                            for mm, v in got.items()},
                              "missing_wordings": {mm: v["missing_wordings"]
                                                   for mm, v in got.items()
                                                   if v["missing_wordings"]}})
            cands.sort(key=lambda c: (c["n_eligible"], c["max_eligible_depth"] or -1,
                                      c["k_max"], c["acc_weakest"]), reverse=True)
            if not cands:
                res = {"setting": None, "status": "missing",
                       "reason": "no setting has runs for every model of the group"}
            elif cands[0]["n_eligible"] < min_depths:
                mw = cands[0]["missing_wordings"]
                res = {"setting": None, "status": "fails",
                       "reason": f"best setting gives the weakest model "
                                 f"{cands[0]['n_eligible']} eligible depth(s); need {min_depths}"
                                 + (f" (no run for wording(s) {mw})" if mw else "")}
            else:
                res = {"setting": cands[0]["setting"], "status": "ok",
                       "weakest": cands[0]["weakest"], "n_eligible": cands[0]["n_eligible"]}
            res["candidates"] = cands
            choices.setdefault(fam, {})[g] = res
    return {"choices": choices, "tables": tables}


# ────────────────────────────── report ──────────────────────────────────────
def _fp(p):
    if p is None or not np.isfinite(p):
        return "-"
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def _fs(x, fmt="{:+.2e}"):
    return "-" if x is None or not np.isfinite(x) else fmt.format(x)


def _jsonable(o):
    if isinstance(o, dict):
        return {str(k): _jsonable(v) for k, v in o.items() if not str(k).startswith("_")}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return _jsonable(o.tolist())
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o


def _provenance():
    here = Path(__file__).resolve()
    h = hashlib.sha256(here.read_bytes()).hexdigest()
    commit = None
    try:
        r = subprocess.run(["git", "rev-parse", "HEAD"], cwd=here.parent,
                           capture_output=True, text=True, timeout=10)
        commit = r.stdout.strip() if r.returncode == 0 else None
    except Exception:
        pass
    return {"module_sha256": h, "git_commit": commit or os.environ.get("NEXT_RUNS_COMMIT"),
            "numpy_version": np.__version__, "python_version": sys.version.split()[0]}


def model_family(name):
    s = re.sub(r"^meta-", "", _short(name).lower())     # meta-llama/Meta-Llama-3-70B
    for f in ("qwen", "llama", "olmo", "gemma", "mixtral", "mistral", "phi", "yi", "gpt2", "pythia"):
        if s.startswith(f):
            return f
    m = re.match(r"[a-z]+", s)           # unknown: the leading word ("m-1b-instruct" -> "m")
    return m.group(0) if m else s


def _post_pairs(models):
    out = []
    for a in models:
        for b in models:
            sa, sb = _short(a).lower(), _short(b).lower()
            if sb.startswith(sa) and sb[len(sa):] in POST_TRAINING_SUFFIXES:
                out.append((a, b))
    return out


def _size_b(name):
    """Parameter count in billions from a model name (Qwen2.5-0.5B -> 0.5)."""
    m = re.findall(r"(\d+(?:\.\d+)?)[bB](?![a-zA-Z])", _short(name))
    return float(m[-1]) if m else float("nan")


def _config(r):
    return (r["attn_impl"], r["precision"])


def _has_activity(r):
    return "contribution" in r["arrays"]


def _run_key(r):
    """A run's identity for its random streams (not its path: moving a file
    must not move its p-values)."""
    return "|".join(str(x) for x in (r["model"], r["suite"], r["attn_impl"], r["precision"]))


_SPLIT_RANK = {"test": 2, "calib": 0}          # "all" / unknown: 1


def _capture_identity(r):
    """Two files are the same capture only if all of this matches, contents
    included (an output copied to two folders).  The name alone does not tell
    apart the E1 calibration run, a --limit smoke run and the E2 capture of
    one suite (sweep_probe names all of them <model>__<suite>.npz), and the
    metadata alone does not tell apart two captures of the same suite (a
    re-run after a probe fix, saved in another folder): content_sha1 does."""
    return (r["model"], r["suite"], r["attn_impl"], r["precision"], r.get("split"),
            r.get("limit"), bool(r.get("accuracy_only")), r.get("suite_sha256"),
            _has_activity(r), frozenset(r["ids"].tolist()), r.get("content_sha1"))


def _capture_rank(r):
    """Which capture of one model, suite and configuration to analyse: with
    activity, then without --limit, then the test split, then more items."""
    return (_has_activity(r), r.get("limit") is None, _SPLIT_RANK.get(r.get("split"), 1), r["n"])


def _why_superseded(r, best):
    a, b = _capture_rank(r), _capture_rank(best)
    if a[0] != b[0]:
        return "no activity (accuracy-only capture)"
    if a[1] != b[1]:
        return f"--limit {r.get('limit')} capture"
    if a[2] != b[2]:
        return f"split {r.get('split')} (the test split is analysed)"
    return f"{r['n']} items against {best['n']}"


def _capture_diff(a, b):
    """What differs between two captures of one model, suite and
    configuration, so a conflict says where to look."""
    out = []
    ia, ib = set(a["ids"].tolist()), set(b["ids"].tolist())
    if ia != ib:
        out.append(f"item ids ({len(ia ^ ib)} not shared)")
    if a.get("suite_sha256") != b.get("suite_sha256"):
        out.append("suite hash")
    for key in ("git_commit", "module_sha256"):
        if a["meta"].get(key) != b["meta"].get(key):
            out.append(key)
    shared = [i for i in a["ids"].tolist() if i in ib]
    if shared:
        pa = {i: j for j, i in enumerate(a["ids"].tolist())}
        pb = {i: j for j, i in enumerate(b["ids"].tolist())}
        xa, xb = np.array([pa[i] for i in shared]), np.array([pb[i] for i in shared])
        nd = int((a["correct"][xa] != b["correct"][xb]).sum())
        if nd:
            out.append(f"correctness on {nd} shared item(s)")
        Sa, Sb = _activity(a, "contribution"), _activity(b, "contribution")
        if Sa is not None and Sb is not None and Sa.shape[1:] == Sb.shape[1:]:
            d = float(np.abs(Sa[xa].astype(float) - Sb[xb].astype(float)).max())
            if d > 0:
                out.append(f"activity (max |difference| {d:.3g})")
    return ", ".join(out) or "contents"


def _select_captures(loaded, rejected, log):
    """One run per (model, suite, kernel, precision).  Files with identical
    contents are rejected as duplicates; among the rest the best-ranked
    capture (_capture_rank) is analysed and the others are listed as
    superseded.  Several best-ranked captures with different contents are a
    conflict: nothing recorded in the files (sweep_probe stores no wall-clock
    time) says which is current, and taking the first in glob order would
    analyse an obsolete capture silently, so none of them is analysed and each
    is listed as rejected with what differs."""
    groups = {}
    for r in loaded:
        groups.setdefault((r["model"], r["suite"]) + _config(r), []).append(r)
    keep, superseded = set(), []
    for rs in groups.values():
        uniq = []
        for r in rs:
            twin = next((u for u in uniq if _capture_identity(u) == _capture_identity(r)), None)
            if twin is not None:
                rejected.append({"path": r["path"], "reason": f"duplicate of {twin['path']}"})
                log(f"  SKIPPED {r['path']}: duplicate of {twin['path']}")
                continue
            uniq.append(r)
        top = max(_capture_rank(u) for u in uniq)
        best = [u for u in uniq if _capture_rank(u) == top]
        if len(best) > 1:
            for u in best:
                others = [b for b in best if b is not u]
                why = (f"conflicting capture: same model, suite and configuration as "
                       f"{', '.join(repr(b['path']) for b in others)} with different contents "
                       f"({_capture_diff(u, others[0])}); the files do not say which is current, "
                       f"so none is analysed: pass only the current one to --runs")
                rejected.append({"path": u["path"], "reason": why})
                log(f"  CONFLICT {u['path']}: {why}")
        else:
            keep.add(id(best[0]))
        for r in uniq:
            if any(r is b for b in best):          # identity: run dicts hold arrays
                continue
            why = _why_superseded(r, best[0])
            kept = best[0]["path"] if len(best) == 1 else None
            if kept is None:
                why += "; the better-ranked captures conflict, so none is analysed"
            superseded.append({"path": r["path"], "kept": kept, "reason": why})
            log(f"  SUPERSEDED {r['path']}: {why}" + (f"; analysing {kept}" if kept else ""))
    return [r for r in loaded if id(r) in keep], superseded


def _pin_configs(runs, pinned=None, task_families=TASK_FAMILIES):
    """{model: (attn_impl, precision)}: the one configuration per model that
    RQ1 tests and whose report rows carry no [kernel precision] tag.
    `pinned` (E3's choice) when the model has runs under it; else the
    configuration holding the most task-family cells, then the most cells and
    items, then the one most runs share (the sweep's kernel), then bf16 and
    eager (the outline's pinned environment).  Other configurations are kernel
    / precision checks: not extra models in any Holm correction."""
    act = [r for r in runs if _has_activity(r)]
    freq = Counter(_config(r) for r in act)
    tf = None if task_families is None else set(task_families)
    stats = {}
    for r in act:
        s = stats.setdefault(r["model"], {}).setdefault(_config(r), [set(), set(), 0])
        cells = set(zip(r["item_family"].tolist(), r["item_wording"].tolist()))
        s[0] |= {c for c in cells if tf is None or c[0] in tf}
        s[1] |= cells
        s[2] += r["n"]
    pins = {}
    for model, cfgs in stats.items():
        if pinned is not None and tuple(pinned) in cfgs:
            pins[model] = tuple(pinned)
            continue
        pins[model] = max(cfgs, key=lambda c: (len(cfgs[c][0]), len(cfgs[c][1]), cfgs[c][2],
                                               freq[c], c[1] == "bf16", c[0] == "eager", str(c)))
    return pins


def _load_runs(paths, rejected, notes, log):
    """load_run on every path.  A file that cannot be read or is inconsistent
    goes to `rejected` (a truncated .npz from a Drive sync raises BadZipFile,
    not RunError), as does one measured where the next token is not the
    answer (_measured_position 'misaligned'); a file whose measured position
    cannot be checked is kept with a note."""
    loaded = []
    for p in paths:
        try:
            r = load_run(p)
        except Exception as e:
            rejected.append({"path": str(p), "reason": f"{type(e).__name__}: {e}"})
            log(f"  REJECTED {p}: {e}")
            continue
        pos = r["position"]
        if pos["status"] == "misaligned":
            why = (f"measured position does not predict the answer digit ({pos['evidence']}): a "
                   f"capture made before deep_probe.align_answer, measured at the last prompt "
                   f"token, which predicts the space before the digit for split-digit tokenizers "
                   f"(the outline's 17 September runs, not evidence); re-run it with the current "
                   f"probe")
            rejected.append({"path": str(p), "reason": why})
            log(f"  REJECTED {p}: {why}")
            continue
        if pos["status"] == "unverified":
            notes.append(f"`{p}`: measured position unverified ({pos['evidence']}); a capture "
                         f"made before deep_probe.align_answer was measured at the last prompt "
                         f"token, which predicts a space, not the answer, for tokenizers that "
                         f"split ' 6' (Llama-3, Qwen2.5, OLMo-2, Mistral, Yi, Phi-3, Gemma-2)")
            log(f"  NOTE {p}: measured position unverified ({pos['evidence']})")
        loaded.append(r)
        log(f"  loaded {p}")
    return loaded


def analyse(paths, out_dir, *, preregistered=None, pairs=None, n_perm=5000, n_boot=2000,
            alpha=0.05, q=0.05, seed=0, mixedlm=True, log=print, task_families=TASK_FAMILIES,
            pinned=None, min_families=2, notes=None):
    """Everything the report needs, from a list of capture files.

    task_families   the families RQ1 pools; runs of other families (the
                    numeral control fixed_digit, the ladder) are reported in
                    their own table, never as families
    pinned          (attn_impl, precision) chosen by E3; RQ1 and its Holm
                    correction across models use one configuration per model
                    (_pin_configs), re-runs under another kernel or precision
                    go to the kernel checks
    min_families    passed to invariance() (criterion (b))"""
    prereg = {str(p): _prereg_value(v) for p, v in (preregistered or {}).items()}
    tf = None if task_families is None else tuple(task_families)
    if pinned is not None:
        pinned = (pinned[0], _norm_precision(pinned[1]))
    t0 = time.time()
    rejected, notes = [], list(notes or [])
    loaded = _load_runs(paths, rejected, notes, log)
    runs, superseded = _select_captures(loaded, rejected, log)
    pins = _pin_configs(runs, pinned, tf)
    # resolved before any number is computed: conflicting patterns abort here
    pre = {m: _prereg_match(m, prereg) for m in sorted({r["model"] for r in runs})}
    R = {"provenance": _provenance(), "settings": {
            "n_perm": n_perm, "n_boot": n_boot, "alpha": alpha, "q": q, "seed": seed,
            "preregistered": prereg, "task_families": list(tf) if tf else None,
            "pinned": list(pinned) if pinned else None, "min_families": min_families},
         "rejected": rejected, "superseded": superseded, "notes": notes,
         "runs": [], "signatures": [], "layer_maps": [], "bands": [], "power": [], "sink": [],
         "centroid": [], "preregistration": [], "invariance": [], "rq1_excluded": [],
         "joint": [], "pairs": [], "kernel": [], "errors": []}
    if pinned is not None:
        R["notes"] += [f"{m}: no run under the pinned {pinned[0]} {pinned[1]}; its RQ1 uses "
                       f"{c[0]} {c[1]}" for m, c in sorted(pins.items()) if c != pinned]

    # a model and suite under several configurations: tag all but the pinned one
    multi = Counter((r["model"], r["suite"]) for r in runs)

    def label(r):
        lab = _run_label(r)
        if multi[(r["model"], r["suite"])] > 1 and pins.get(r["model"]) != _config(r):
            lab += f" [{r['attn_impl']} {r['precision']}]"
        return lab

    def attempt(section, what, fn):
        try:
            return fn()
        except Exception as e:           # one odd file must not cost the whole report
            R["errors"].append({"section": section, "what": what,
                                "reason": f"{type(e).__name__}: {e}"})
            log(f"  refused {section} / {what}: {type(e).__name__}: {e}")
            return None

    for r in runs:
        lab, rid = label(r), _run_key(r)
        R["runs"].append({"label": lab, "path": r["path"], "format": r["format"],
                          "model": r["model"], "suite": r["suite"], "family": r["family"],
                          "wording": r["wording"], "n": r["n"], "L": r["L"], "H": r["H"],
                          "attn_impl": r["attn_impl"], "precision": r["precision"],
                          "split": r["split"], "limit": r["limit"],
                          "correctness": r["correctness"], "chance": r["chance"],
                          "has_bos": r["has_bos"], "sandwich_layers": len(r["sandwich_layers"]),
                          "activity": _has_activity(r), "position": r["position"]["status"],
                          "content_sha1": r["content_sha1"]})
        metrics = ["contribution"] + (["write_norm"] if r["sandwich_layers"]
                                      and "write_norm" in r["arrays"] else [])
        sigs = {}
        for m in metrics:
            s = attempt("signature", lab, lambda: signature(
                r, m, alpha=alpha, n_perm=n_perm, rng=_sub_rng(seed, rid, "signature", m)))
            if s:
                sigs[m] = s
                R["signatures"].append({"label": lab, **s})
        if len(sigs) == 2:
            a, b = sigs["contribution"], sigs["write_norm"]
            sa_, sb_ = (a.get("all_items") or {}).get("slope"), (b.get("all_items") or {}).get("slope")
            R["signatures"][-1]["definition_dependent"] = _sign(sa_) != _sign(sb_)
        S = _activity(r, "contribution")
        if S is None or len(np.unique(r["k"])) < 2:
            continue
        TS = top_share(S)

        def lm():
            g = _sub_rng(seed, rid, "layer_map")
            cls = classify_layers(TS, r["k"], q, n_perm, g)
            out = {"label": lab, **layer_map(cls, TS, r["k"], n_perm, g, q, mass=S.sum(-1))}
            if 1 / (n_perm + 1) > q / r["L"]:
                out["bh_note"] = (f"{n_perm} permutations: the smallest p, 1/{n_perm + 1}, is above "
                                  f"q/L = {q / r['L']:.2g}, so BH cannot classify a layer on its "
                                  f"own; use --n-perm >= {math.ceil(r['L'] / q)}")
            return out
        x = attempt("layer_map", lab, lm)
        if x:
            R["layer_maps"].append(x)
        x = attempt("band", lab, lambda: {"label": lab, **band(
            S, r["k"], _sub_rng(seed, rid, "band"), n_perm)})
        if x:
            R["bands"].append(x)
        x = attempt("power", lab, lambda: {"label": lab, "items_per_depth_2pct": power(r)})
        if x:
            R["power"].append(x)
        if "pos0_attn" in r["arrays"]:
            x = attempt("sink", lab, lambda: {"label": lab, **sink_control(
                r, alpha=alpha, n_perm=n_perm, rng=_sub_rng(seed, rid, "sink"))})
            if x:
                R["sink"].append(x)
        x = attempt("centroid", lab, lambda: {"label": lab, **mass_centroid(
            r, n_perm=n_perm, rng=_sub_rng(seed, rid, "centroid"))})
        if x:
            x.pop("centroid", None)
            R["centroid"].append(x)

    # Holm across models within each suite and configuration (a kernel-check
    # re-run is not another model), on the primary all-items p
    by_suite = {}
    for s in R["signatures"]:
        if s["metric"] == "contribution" and s.get("all_items"):
            by_suite.setdefault((s["suite"], s["attn_impl"], s["precision"]), []).append(s)
    for ss in by_suite.values():
        for s, h in zip(ss, holm([x["all_items"]["p"] for x in ss])):
            s["all_items"]["p_holm_models"] = float(h)

    # RQ1: one configuration per model, task families only
    for model in sorted(pins):
        cfg = pins[model]
        val, pat = pre.get(model, (None, None))
        R["preregistration"].append({"model": model, "sign": val, "pattern": pat,
                                     "attn_impl": cfg[0], "precision": cfg[1]})
        rs = []
        for r in runs:
            if r["model"] != model or not _has_activity(r):
                continue
            fams = set(r["item_family"].tolist())
            if _config(r) != cfg:
                why = (f"{r['attn_impl']} {r['precision']} re-run; the model's RQ1 configuration "
                       f"is {cfg[0]} {cfg[1]} (see kernel checks)")
            elif tf is not None and not fams & set(tf):
                why = "; ".join(CONTROL_FAMILIES.get(f, f"family {f!r} is not a task family")
                                for f in sorted(fams))
            else:
                rs.append(r)
                continue
            R["rq1_excluded"].append({"label": label(r), "model": model, "suite": r["suite"],
                                      "family": r["family"], "attn_impl": r["attn_impl"],
                                      "precision": r["precision"], "reason": why})
        if val is None or not rs:
            continue
        x = attempt("invariance", model, lambda: invariance(
            rs, val, alpha, n_boot=n_boot, n_perm=n_perm, mixedlm=mixedlm,
            rng=_sub_rng(seed, "invariance", model, *cfg), task_families=tf,
            min_families=min_families))
        if x:
            x.update(model=model, attn_impl=cfg[0], precision=cfg[1], preregistered_pattern=pat)
            R["invariance"].append(x)
    ph = holm([x["correct_items"].get("p_pooled", np.nan) for x in R["invariance"]])
    pa = holm([x["all_items"].get("p_pooled", np.nan) for x in R["invariance"]])
    for x, h, h2 in zip(R["invariance"], ph, pa):
        x["correct_items"]["p_pooled_holm_models"] = float(h)
        x["all_items"]["p_pooled_holm_models"] = float(h2)
        # the pooled slope is each model's one primary outcome: an invariant
        # verdict also needs it to survive Holm across models
        for mode, key, hp in (("correct_items", "verdict_holm", h),
                              ("all_items", "verdict_all_items_holm", h2)):
            v = x[mode]["verdict"]
            x[key] = "inconclusive" if v == "invariant" and not hp < alpha else v

    # E6 ladders: per suite and model family, models in order of size
    lad = {}
    for s in R["signatures"]:
        if s["metric"] == "contribution" and s.get("all_items"):
            lad.setdefault((s["suite"], model_family(s["model"]), s["attn_impl"], s["precision"]),
                           []).append(s)
    R["ladders"] = [{"suite": k[0], "family": k[1], "attn_impl": k[2], "precision": k[3],
                     "models": [{"model": s["model"], "size_b": _size_b(s["model"]),
                                 "pct_change": s["all_items"]["pct_change"],
                                 "slope": s["all_items"]["slope"], "p": s["all_items"]["p"],
                                 "classification_all_items": s["classification_all_items"],
                                 "classification": s["classification"]}
                                for s in sorted(v, key=lambda s: (_size_b(s["model"]), s["model"]))]}
                    for k, v in sorted(lad.items(), key=lambda kv: str(kv[0])) if len(v) >= 2]

    # RQ4: runs sharing items (same suite and configuration)
    groups = {}
    for r in runs:
        if _has_activity(r):
            groups.setdefault((r["suite"],) + _config(r), []).append(r)
    for (suite, ai, pr), rs in sorted(groups.items(), key=lambda kv: str(kv[0])):
        fams = {}
        for r in rs:
            fams.setdefault(model_family(r["model"]), []).append(r)
        for fam, fr in sorted(fams.items()):
            if len(fr) >= 2:
                x = attempt("joint", f"{suite}/{fam}", lambda: joint_relabel(
                    fr, n_perm=n_perm, pairs=[], rng=_sub_rng(
                        seed, "joint", suite, ai, pr, *sorted(y["model"] for y in fr))))
                if x:
                    x.update(family=fam, attn_impl=ai, precision=pr)
                    R["joint"].append(x)
        byname = {r["model"]: r for r in rs}
        want = _post_pairs(list(byname)) + [
            (a, b) for pa_, pb_ in (pairs or []) for a in byname for b in byname
            if pa_.lower() in a.lower() and pb_.lower() in b.lower() and a != b]
        for a, b in dict.fromkeys(want):
            x = attempt("pairs", f"{suite}: {a} vs {b}", lambda: joint_relabel(
                [byname[a], byname[b]], n_perm=n_perm,
                rng=_sub_rng(seed, "pair", suite, ai, pr, a, b)))
            if x:
                x.update(attn_impl=ai, precision=pr)
                R["pairs"].append(x)

    # kernel / precision checks: same model and suite, different configuration
    km = {}
    for r in runs:
        if _has_activity(r):
            km.setdefault((r["model"], r["suite"]), []).append(r)
    for (model, suite), rs in sorted(km.items(), key=lambda kv: str(kv[0])):
        for i in range(len(rs)):
            for j in range(i + 1, len(rs)):
                a, b = rs[i], rs[j]
                if _config(a) == _config(b):
                    continue
                if a["precision"] == "fp32":           # the reference goes second
                    a, b = b, a
                x = attempt("kernel", f"{model}/{suite}", lambda: kernel_compare(
                    a, b, alpha=alpha, n_perm=n_perm,
                    rng=_sub_rng(seed, "kernel", _run_key(a), _run_key(b))))
                if x:
                    R["kernel"].append(x)
    R["elapsed_s"] = round(time.time() - t0, 1)

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    (out / "report.json").write_text(json.dumps(_jsonable(R), indent=1))
    (out / "report.md").write_text(render_markdown(R))
    log(f"  -> {out / 'report.md'}   {out / 'report.json'}   ({R['elapsed_s']} s)")
    return R


def render_markdown(R):
    L = []
    P = R["provenance"]
    st = R["settings"]
    L += ["# Signature analysis report", "",
          f"signature_analysis.py sha256 `{P['module_sha256'][:16]}`, commit "
          f"`{P['git_commit']}`; permutations {st['n_perm']}, bootstrap {st['n_boot']}, "
          f"alpha {st['alpha']}, BH q {st['q']}, seed {st['seed']}.", "",
          "Primary statistic: top-head share at 40-80% depth (activity = `contribution`, "
          "pre-norm). Slopes are per unit k; p-values two-sided, permutation.", ""]
    L += ["## Runs", "",
          "A `[kernel precision]` tag marks a re-run outside the model's pinned configuration "
          "(RQ1 uses the pinned one).", "",
          "Position: `aligned` (deep_probe.align_answer recorded), `verified` (no record; the "
          "position predicts a digit), `unverified` (no evidence either way; see notes).", "",
          "| run | family/wording | split | n | L x H | kernel | precision | correctness | chance "
          "| BOS | position |", "|---|---|---|---|---|---|---|---|---|---|---|"]
    for r in R["runs"]:
        sp = (r.get("split") or "-") + (f" (limit {r['limit']})" if r.get("limit") else "")
        L.append(f"| {r['label']} | {r['family']}/{r['wording']} | {sp} | {r['n']} | "
                 f"{r['L']}x{r['H']} | {r['attn_impl']} | {r['precision']} | {r['correctness']} | "
                 f"{r['chance']:.3f} | {_fs(r['has_bos'], '{:.2f}')} | {r.get('position', '-')} |")
    if R["rejected"]:
        L += ["", "**Rejected files (no numbers reported):**", ""]
        L += [f"- `{x['path']}`: {x['reason']}" for x in R["rejected"]]
    if R.get("superseded"):
        L += ["", "**Superseded captures (same model, suite and configuration; not analysed):**", ""]
        L += [f"- `{x['path']}`: {x['reason']}"
              + (f"; analysed `{x['kept']}`" if x.get("kept") else "") for x in R["superseded"]]
    if R.get("notes"):
        L += ["", "**Notes:**", ""] + [f"- {x}" for x in R["notes"]]
    if R["errors"]:
        L += ["", "**Analyses refused:**", ""]
        L += [f"- {x['section']} / {x['what']}: {x['reason']}" for x in R["errors"]]

    L += ["", "## 1. Signatures", "",
          "| run | metric | all-items slope | p | p Holm (models) | % change | PR/H slope | p "
          "| eligible depths | correct slope | p | all items | signature |",
          "|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for s in R["signatures"]:
        a, c = s.get("all_items") or {}, s.get("correct_items") or {}
        pct = (f"{a['pct_change']:+.1f}% (k={a['k_from']}->{a['k_to']})"
               if a.get("pct_change") is not None and np.isfinite(a["pct_change"]) else "-")
        L.append(f"| {s['label']} | {s['metric']} | {_fs(a.get('slope'))} | {_fp(a.get('p'))} | "
                 f"{_fp(a.get('p_holm_models'))} | {pct} | "
                 f"{_fs((s.get('pr_h') or {}).get('slope'))} | {_fp((s.get('pr_h') or {}).get('p'))} | "
                 f"{s.get('eligible_depths', '-')} | {_fs(c.get('slope'))} | {_fp(c.get('p'))} | "
                 f"{s['classification_all_items']} | **{s['classification']}** |")
    cd = [s for s in R["signatures"] if (s.get("clean_depth") or {}).get("status") == "ok"]
    if cd:
        L += ["", "**Depth confound.** On a cycle, k steps forward land where n - k steps back do, "
              "so depths above n/2 need fewer lookups than k for a model that reads the rules "
              "backwards. Slope of the primary statistic on the items whose shortest lookup "
              "path equals k:", "",
              "| run | metric | clean depths | n | slope | p | % change | confounded depths | "
              "slope on min_hops (p) |", "|---|---|---|---|---|---|---|---|---|"]
        for s in cd:
            c = s["clean_depth"]
            smh = c.get("slope_on_min_hops") or {}
            L.append(f"| {s['label']} | {s['metric']} | {c['depths']} | {c['n']} | {_fs(c['slope'])} | "
                     f"{_fp(c['p'])} | {c['pct_change']:+.1f}% | {c['confounded_depths']} | "
                     f"{_fs(smh.get('slope'))} ({_fp(smh.get('p'))}) |")
    dd = [s for s in R["signatures"] if s.get("definition_dependent")]
    if dd:
        L += ["", "Definition-dependent (pre-norm and post-norm slopes differ in sign): "
              + ", ".join(s["label"] for s in dd)]
    L += ["", "Accuracy by k (correct / n):", ""]
    for s in R["signatures"]:
        if s["metric"] == "contribution" and s.get("by_k"):
            L.append(f"- {s['label']}: " + "  ".join(
                f"k{b['k']} {b['correct']}/{b['n']}" for b in s["by_k"])
                + f"  (threshold {s['acc_threshold']:.2f})")

    L += ["", "## 2. Invariance (RQ1)", ""]
    tfs = st.get("task_families")
    L += [f"Task families: {', '.join(tfs) if tfs else 'all'}. One configuration per model "
          f"(pinned: {' '.join(map(str, st['pinned'])) if st.get('pinned') else 'the one holding most task-family cells'}"
          "); Holm across models counts each model once.", ""]
    if R.get("preregistration"):
        L += ["| model | RQ1 configuration | pre-registered | by pattern |", "|---|---|---|---|"]
        for p in R["preregistration"]:
            v = ("funnel (+)" if p["sign"] == 1 else "spread (-)" if p["sign"] == -1
                 else "excluded (none)" if p["pattern"] is not None else "not pre-registered")
            L.append(f"| {p['model']} | {p['attn_impl']} {p['precision']} | {v} | "
                     f"{'`' + p['pattern'] + '`' if p['pattern'] else '-'} |")
        L.append("")
    if not R["invariance"]:
        L.append("No model with a pre-registered sign.")
    for x in R["invariance"]:
        sgn = "funnel (+)" if x["preregistered_sign"] > 0 else "spread (-)"
        L += [f"### {x['model']}  ({x['attn_impl']}, {x['precision']}); pre-registered {sgn} "
              f"(pattern `{x.get('preregistered_pattern')}`)", ""]
        if x.get("excluded_cells"):
            L += ["Not task families (excluded): " + ", ".join(
                f"{c} ({why})" for c, why in x["excluded_cells"].items()), ""]
        for mode in ("correct_items", "all_items"):
            v = x[mode]
            L.append(f"**{mode.replace('_', ' ')}: {v['verdict']}**")
            if v.get("pooled") is not None:
                L.append(f"pooled slope {_fs(v['pooled'])}, 95% CI [{_fs(v['ci95'][0])}, "
                         f"{_fs(v['ci95'][1])}], p {_fp(v['p_pooled'])}, Holm across models "
                         f"{_fp(v.get('p_pooled_holm_models'))}; criteria {v['criteria']}")
            L += ["", "| family | slope | p | p Holm | cells (wording: suite) |",
                  "|---|---|---|---|---|"]
            for f, fr in v.get("families", {}).items():
                cells = [f"{v['cells'][c]['wording']}: `{v['cells'][c]['suite']}`"
                         for c in fr.get("cells", [])] or fr["wordings"]
                L.append(f"| {f} | {_fs(fr['slope'])} | {_fp(fr['p'])} | {_fp(fr['p_holm'])} "
                         f"| {', '.join(cells)} |")
            miss = [c for c, cv in v["cells"].items() if cv["status"] != "ok"]
            if miss:
                L.append(f"\nmissing cells: {', '.join(miss)}")
            L.append("")
        L.append(f"After Holm across models: correct items **{x.get('verdict_holm')}**, "
                 f"all items **{x.get('verdict_all_items_holm')}**.")
        if x["disagree"]:
            L.append("**Correct-item and all-items verdicts disagree.**")
        mx = x.get("mixedlm")
        if mx:
            tag = (" (boundary: a variance component is estimated at 0, so the SE ignores "
                   "between-family variation)" if mx.get("boundary") else "")
            L.append(f"MixedLM (secondary){tag}: {mx}")
        L.append("")
    if R.get("rq1_excluded"):
        sig = {s["label"]: s for s in R["signatures"] if s["metric"] == "contribution"}
        L += ["### Outside RQ1", "",
              "Numeral control, ladder and kernel / precision re-runs: reported here, never "
              "counted as a task family or as another model.", "",
              "| run | why not in RQ1 | all-items slope | p | correct slope | p | signature |",
              "|---|---|---|---|---|---|---|"]
        for e in R["rq1_excluded"]:
            s = sig.get(e["label"], {})
            a, c = s.get("all_items") or {}, s.get("correct_items") or {}
            L.append(f"| {e['label']} | {e['reason']} | {_fs(a.get('slope'))} | {_fp(a.get('p'))} "
                     f"| {_fs(c.get('slope'))} | {_fp(c.get('p'))} | {s.get('classification', '-')} |")
        L.append("")

    L += ["## 3. Layer maps (RQ3)", "",
          "`+` funnelling, `-` spreading, `.` neither (BH q within model and task). "
          "Contiguity (fewer sign changes, longer runs than chance) is tested against the joint "
          "k-permutation null, which keeps the noise correlation between adjacent layers; "
          "interleaving (more sign changes) against the layer-order shuffle, since the joint "
          "null over-calls interleaving when layer noise is correlated. The BH-reclassified "
          "joint null (`joint_bh` in report.json) only tests whether any layer responds.", "",
          "| run | map | spread / funnel | sign changes | longest run | contiguous p (joint) | "
          "longest-run p (joint) | interleaved p (layer shuffle) | contiguous p (layer shuffle) "
          "| final 20% slope | p |",
          "|---|---|---|---|---|---|---|---|---|---|---|"]
    for m in R["layer_maps"]:
        t, fr = m["tests"], m["final_region"]
        L.append(f"| {m['label']} | `{m['signmap']}` | {m['n_spread']} / {m['n_funnel']} | "
                 f"{m['sign_changes']} | {m['longest_run']} | {_fp(t['joint']['p_fewer'])} | "
                 f"{_fp(t['joint']['p_longer'])} | {_fp(t['layer_shuffle']['p_more'])} | "
                 f"{_fp(t['layer_shuffle']['p_fewer'])} | "
                 f"{_fs(fr['mean_slope'])} | {_fp(fr['p'])} |")
    notes = [f"- {m['label']}: {m['bh_note']}" for m in R["layer_maps"] if m.get("bh_note")]
    if notes:
        L += [""] + notes

    L += ["", "## 4. Bands (PR/H, contiguous rule; superseded rule for comparison)", "",
          "| run | onset | close | width | superseded onset-close | differs |",
          "|---|---|---|---|---|---|"]
    for b in R["bands"]:
        c, s = b["contiguous"], b["superseded"]
        sup = f"{s['onset']:.2f}-{s['close']:.2f}" if s["onset"] is not None else "none"
        L.append(f"| {b['label']} | {_fs(c['onset'], '{:.2f}')} | {_fs(c['close'], '{:.2f}')} | "
                 f"{c['width']:.2f} | {sup} | {b['differs']} |")

    L += ["", "## 5. Ladders and joint relabelling (RQ4)", ""]
    for lad in R.get("ladders", []):
        L += [f"**{lad['family']} on {lad['suite']}** ({lad['attn_impl']}, {lad['precision']}): "
              + ", ".join(f"{_short(m['model'])} {_fs(m['pct_change'], '{:+.1f}%')} "
                          f"(p {_fp(m['p'])}, {m['classification_all_items']})"
                          for m in lad["models"]), ""]
    L += ["",
          "Scale-normalised slopes (relative change per step), one shuffle of k shared by all "
          "runs.", "", "| suite | family | models | pooled | p | per-model slopes |",
          "|---|---|---|---|---|---|"]
    for j in R["joint"]:
        L.append(f"| {j['suite']} | {j['family']} | {', '.join(_short(m) for m in j['models'])} | "
                 f"{_fs(j['pooled'])} | {_fp(j['p_pooled'])} | "
                 + ", ".join(f"{_fs(s)} (p {_fp(p)})" for s, p in zip(j["slopes"], j["p_each"])) + " |")
    if R["pairs"]:
        L += ["", "| suite | pair (b - a) | difference | p |", "|---|---|---|---|"]
        for j in R["pairs"]:
            for d in j["diffs"]:
                L.append(f"| {j['suite']} | {d['b']} - {d['a']} | {_fs(d['diff'])} | {_fp(d['p'])} |")

    L += ["", "## 6. Kernel checks", ""]
    if not R["kernel"]:
        L.append("No model and suite measured under two configurations.")
    else:
        L += ["Everything, both signatures included, on the items the two runs share.", "",
              "| model / suite | a vs b (ref) | shared items | score r | late rel. diff | "
              "top-share slope r (same sign) | PR/H slope r | signatures | agree |",
              "|---|---|---|---|---|---|---|---|---|"]
        for x in R["kernel"]:
            pl = x["per_layer"]
            L.append(f"| {_short(x['model'])} / {x['suite']} | {x['a']['attn_impl']} "
                     f"{x['a']['precision']} vs {x['b']['attn_impl']} {x['b']['precision']} | "
                     f"{x['n_items']} | "
                     f"{x['score_r']:.3f} | {_fs(x['late_rel_diff_median'], '{:.3f}')} | "
                     f"{pl['top_share']['r']:.2f} ({pl['top_share']['same_sign']}/{pl['top_share']['L']}) | "
                     f"{pl['pr_h']['r']:.2f} | {x['signature_a']['classification_all_items']} / "
                     f"{x['signature_b']['classification_all_items']} | {x['agree']} |")
        warn = [f"- {_short(x['model'])} / {x['suite']}: {w}" for x in R["kernel"]
                for w in (x.get("warning") or [])]
        if warn:
            L += [""] + warn

    L += ["", "## 7. Power targets", "",
          "Items per depth to detect a 2% change in the primary statistic between the lowest "
          "and highest depth (alpha 0.05, power 0.8, pilot OLS residual variance).", ""]
    L += [f"- {p['label']}: {p['items_per_depth_2pct']}" for p in R["power"]]

    L += ["", "## 8. Sink control", ""]
    if not R["sink"]:
        L.append("No run recorded position-0 attention (--pos0).")
    for s in R["sink"]:
        a = s["signature"].get("all_items") or {}
        b = s["signature_masked"].get("all_items") or {}
        L.append(f"- {s['label']}: BOS {_fs(s['has_bos'], '{:.2f}')}, heads removed "
                 f"{100 * s['frac_removed_window']:.1f}% at 40-80%; slope {_fs(a.get('slope'))} "
                 f"(p {_fp(a.get('p'))}) -> {_fs(b.get('slope'))} (p {_fp(b.get('p'))}); "
                 f"{s['signature']['classification_all_items']} -> "
                 f"{s['signature_masked']['classification_all_items']}; sign agrees {s['sign_agree']}")

    L += ["", "## 9. Mass centroid", "",
          "| run | slope (layers per step) | p | shift k_min->k_max (layers) |", "|---|---|---|---|"]
    for c in R["centroid"]:
        L.append(f"| {c['label']} | {_fs(c['slope'])} | {_fp(c['p'])} | "
                 f"{_fs(c['shift_layers'], '{:+.2f}')} |")
    L += ["", f"_{R.get('elapsed_s')} s_", ""]
    return "\n".join(L)


def render_choice_markdown(res):
    L = ["# E1 setting choice", "",
         "Eligibility is judged for the test split: a calibration-split run is projected to "
         "n_per - round(calib_frac * n_per) test items per depth (calibration accuracy >= "
         "2c/(1+c) and accuracy x test items >= the correct-item minimum), per family and "
         "wording; a model counts the eligible depths of its weakest wording, and a wording "
         "the family needs but the model has no run for under a setting counts as 0.", ""]
    for fam, gs in res["choices"].items():
        L += [f"## {fam}", ""]
        for g, c in gs.items():
            L.append(f"- **{g}**: {c['status']}" + (f", setting `{c['setting']}` (weakest "
                     f"{c['weakest']}, {c['n_eligible']} eligible depths)" if c["setting"] else
                     f" ({c.get('reason')})"))
        L.append("")
    L += ["## Accuracy tables", "",
          "Per depth: correct / calibration items, and in brackets the projected correct items "
          "on the test split.", ""]
    for setting, t in res["tables"].items():
        L += [f"### setting `{setting}`", ""]
        for fam, rows in t.items():
            for model, v in rows.items():
                for wd, c in v.get("wordings", {}).items():
                    if c.get("status") == "missing":
                        L.append(f"- {fam}/{wd} / {_short(model)}: **no run under this setting** "
                                 f"(counts as 0 eligible depths)")
                        continue
                    L.append(f"- {fam}/{wd} / {_short(model)}: " + "  ".join(
                        f"k{b['k']} {b['correct']}/{b['n']}"
                        + (f" ({b['projected_test_correct']:g})" if "projected_test_correct" in b
                           else "") for b in c["by_k"])
                        + f"  eligible {c['eligible_depths']} ({c['basis']})")
        L.append("")
    return "\n".join(L)


def _expand(patterns, unmatched=None):
    """Files matching the globs, in order, without repeats; '**' matches any
    number of folders (runs/**/*.npz reaches runs/ itself and every subfolder).
    A pattern that matches nothing is reported (a typo must not silently drop a
    family of runs) and appended to `unmatched`."""
    out = []
    for p in patterns:
        hits = sorted(glob.glob(p, recursive=True))
        hits = hits if hits else ([p] if Path(p).exists() else [])
        if not hits:
            print(f"  WARNING: {p!r} matches no file", file=sys.stderr)
            if unmatched is not None:
                unmatched.append(p)
        out += hits
    return list(dict.fromkeys(out))


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("--runs", nargs="+", default=[],
                    help="capture files or quoted globs (*.npz from sweep_probe or "
                         "head_alignment_probe, *_deep.json from deep_probe)")
    ap.add_argument("--out", required=True, help="report directory")
    ap.add_argument("--preregistered", nargs="*", default=[], metavar="MODEL=SIGN",
                    help="e.g. qwen2.5=funnel llama=spread OLMo-2-1124-7B-Instruct=spread; "
                         "SIGN funnel/spread or none (no RQ1 verdict).  MODEL occurs in the "
                         "model name (case-insensitive); a post-trained model (-Instruct, -SFT, "
                         "-DPO, -it, -chat, -RLVR) is covered only by a pattern naming its "
                         "marker; MODEL$ anchors at the end, re:REGEX is a regular expression; "
                         "an exact name wins, then the longest pattern")
    ap.add_argument("--pair", nargs="*", default=[], metavar="A:B",
                    help="extra paired tests B minus A (name substrings); base/post-trained "
                         "pairs such as X and X-Instruct are found automatically")
    ap.add_argument("--task-families", nargs="+", default=list(TASK_FAMILIES),
                    help="families RQ1 pools (default: %(default)s); fixed_digit and the ladder "
                         "are reported beside it")
    ap.add_argument("--pinned", nargs=2, metavar=("ATTN", "PRECISION"),
                    help="E3's configuration, e.g. eager bf16; RQ1 uses it for every model that "
                         "has runs under it (default: per model, the configuration holding "
                         "most task-family cells)")
    ap.add_argument("--min-families", type=int, default=2,
                    help="RQ1 criterion (b) needs at least this many families present "
                         "(default 2, the spec's 'of those present'; 3 requires three "
                         "families with the sign whatever is missing)")
    ap.add_argument("--n-perm", type=int, default=5000)
    ap.add_argument("--n-boot", type=int, default=2000)
    ap.add_argument("--alpha", type=float, default=0.05)
    ap.add_argument("--q", type=float, default=0.05)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--no-mixedlm", action="store_true",
                    help="skip the secondary MixedLM fit in the invariance test (slow on CPU)")
    ap.add_argument("--calib", nargs="+", help="E1: calibration runs; writes the setting choice")
    ap.add_argument("--groups", help="E1: JSON file {group: [model, ...]}")
    ap.add_argument("--min-correct", type=int, default=20,
                    help="E1: correct items a depth needs on the test split (default 20)")
    ap.add_argument("--min-depths", type=int, default=3,
                    help="E1: eligible depths the weakest model needs (default 3)")
    a = ap.parse_args(argv)
    unmatched = []
    if a.calib:
        if not a.groups:
            ap.error("--calib needs --groups")
        paths = _expand(a.calib, unmatched)
        if not paths:
            ap.error("no calibration files match")
        with open(a.groups) as f:
            groups = json.load(f)
        rej, notes = [], []
        # an unreadable file (a Drive sync cut short) is listed, not fatal; a
        # calibration file copied to two folders must not count twice
        runs, sup = _select_captures(_load_runs(paths, rej, notes, print), rej, print)
        res = choose_setting(group_by_setting(runs), groups, min_depths=a.min_depths,
                             min_correct=a.min_correct)
        res["skipped"] = rej + sup
        res["notes"] = notes
        res["unmatched_patterns"] = unmatched
        out = Path(a.out)
        out.mkdir(parents=True, exist_ok=True)
        (out / "choose_setting.json").write_text(json.dumps(_jsonable(res), indent=1))
        (out / "choose_setting.md").write_text(render_choice_markdown(res))
        print(f"  -> {out / 'choose_setting.md'}")
        return res
    paths = _expand(a.runs, unmatched)
    if not paths:
        ap.error("no run files match --runs")
    prereg = {}
    for x in a.preregistered:
        if "=" not in x:
            ap.error(f"--preregistered {x!r}: use MODEL=funnel|spread|none")
        k, v = x.rsplit("=", 1)
        try:
            prereg[k] = _prereg_value(v)
        except ValueError as e:
            ap.error(f"--preregistered {x!r}: {e}")
        if k.strip().lower().startswith("re:"):
            try:                          # before any file is loaded, not deep in analyse()
                re.compile(k.strip()[3:])
            except re.error as e:
                ap.error(f"--preregistered {x!r}: invalid regular expression ({e})")
    bad = [x for x in a.pair if ":" not in x or not all(s.strip() for s in x.split(":", 1))]
    if bad:
        ap.error(f"--pair {' '.join(bad)}: use A:B (name substrings, B minus A)")
    pairs = [tuple(x.split(":", 1)) for x in a.pair]
    notes = [f"--runs pattern {p!r} matches no file" for p in unmatched]
    return analyse(paths, a.out, preregistered=prereg, pairs=pairs, n_perm=a.n_perm,
                   n_boot=a.n_boot, alpha=a.alpha, q=a.q, seed=a.seed, mixedlm=not a.no_mixedlm,
                   task_families=a.task_families, pinned=a.pinned, min_families=a.min_families,
                   notes=notes)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ablation_probe.py
#!/usr/bin/env python3
"""
Head ablation at the measured position: E4 of paper_manuscript/research_outline.tex
(RQ2, what the signature means), on one suite (serial_tasks.py JSON) per run.

Design, fixed before running (outline E4):

  items         the chosen split of the suite, divided 50/50 into a selection
                and a measurement half, stratified by k and seeded (at each
                depth the selection half gets floor(n/2) items)
  pass 1        selection half, one full forward per item.  At the measured
                position (the one whose next token is the answer digit;
                deep_probe.align_answer): per-head activity ||z_h W_O^h|| on
                every layer (the paper's `contribution`) and the heads' o_proj
                inputs z on the window layers (40-80% depth by default).  From
                these: the per-head mean z (the mean-ablation value), the mean
                activity (heads ranked per layer), and the layer classes:
                signature_analysis.classify_layers on per-layer top share, -1
                spreading / +1 funnelling, BH over all layers of the model and
                task as in the outline's rule; the target sets are the
                classified layers inside the window.
  pass 2        measurement half.  The prefix (every token but the last) runs
                once with a KV cache; each condition then runs only the last
                token against that cache, which is cropped back to the prefix
                afterwards (DynamicCache.crop).  Ablating a head at the measured
                position cannot change earlier positions, so this is exact.
  conditions    clean; for each set (spread, funnel) that has layers: the top 1
                and the top 3 heads of every layer of the set by selection-half
                activity, ablated jointly across the set, by mean- and by
                zero-ablation; a random baseline of 1 or 3 heads per layer drawn
                from the mid-ranked heads (activity between the layer's 25th
                and 75th percentile, top 3 excluded), R draws, mean-ablation
                only; with --per-layer also the top-1 mean-ablation of one layer
                at a time.  An empty set is skipped and reported, as is a
                random baseline a layer has too few mid-ranked heads for.
  intervention  the head's slice of the o_proj input at the last position is
                replaced by its selection-half mean (mean) or by zero (zero),
                inside the patched o_proj forward (_old_forward under
                accelerate offload, else forward, as deep_probe patches it)
  outcome       per item and condition: logit difference = gold logit minus
                the mean logit of the competitors (answer_set minus gold),
                absolute and as a fraction of clean; clean-correct items
                (digit-constrained, gold the unique maximum over answer_set, as
                sweep_probe scores it) are flagged

Verification (the run aborts rather than write numbers):
  - n_heads / head_dim against the model config and attention modules, and the
    per-head split against the output projection on every captured forward
    (sweep_probe's checks: exact fp32 map, module output within kernel
    rounding, Linear or GPT-2 Conv1D layout)
  - every output projection runs exactly once per forward; model in eval mode;
    no NaN / inf logits; no layer whose heads all write zero at the measured
    position of a selection item (its top share, and so its class, is
    undefined)
  - the cache holds exactly the prefix after every crop, and the first
    condition group re-run at the end of every item reproduces its first run
  - on the first measured item of a session and every VERIFY_EVERY-th after:
    the clean last-token logits through the cache reproduce a full forward
    (max relative error <= 1e-2)
  - on the first measured item of a session: replacing every head of every
    window layer with the item's own z reproduces clean; zero-ablating each
    window layer's largest head changes the logits; for every ablation on that
    item the o_proj input equals the intended one, rebuilt from the condition
    table (its heads and values, per-head column slices) bit for bit, so a
    replacement that hits another head, row or position aborts; and each
    o_proj output changes by exactly the listed heads' writes predicted from
    that intended input (twice the kernel tolerance, as two rounded outputs are
    compared)

Two run options leave the design unchanged.  --batch-conditions B runs B
conditions per forward (the prefix cache is repeated B times and cropped as
above; chunks are padded with clean rows, which must reproduce the clean row):
under CPU offload (70B / 72B on one A100) every forward streams the offloaded
weights once, so B conditions cost about one forward.  Per measured item a run
makes 1 prefix forward, ceil(C/B) condition forwards and 1 repeat of the first
chunk (plus the checks); at the 70B pair with --per-layer C is about 46, so the
default is B = 16 on cuda (5 forwards per item instead of 48) and 1 on the CPU.
Rows at different batch positions may round differently in bf16, so B is
recorded in the output.  A checkpoint next to the output (<output>.ckpt, not
.npz, so no '*.npz' glob picks it up) holds pass 1, the finished pass-2 items
and the weight layout; a disconnected Colab session resumes from it when every
setting and this file's version match, and aborts otherwise.

    python3 ablation_probe.py --model meta-llama/Llama-3.1-70B --suite suites/pointer_w0.json \\
        --out runs/e4 --device cuda --attn-impl eager --window 0.4 0.8 --split test \\
        --random-draws 5 --seed 0 --per-layer
    python3 ablation_probe.py --analyse runs/e4/meta-llama_Llama-3.1-70B__S__ablation.npz \\
        runs/e4/Qwen_Qwen2.5-72B__S__ablation.npz --out runs/e4/report

Output: OUT/<model with / -> _>__<suite name>__ablation.npz (arrays plus `meta`,
a JSON string with kind = "ablation", the design, the conditions and their
heads, provenance and checks); skipped when it exists with the same settings,
refused when it exists with others.  The per-item records are
meta["measurement_items"] and the measurement half's activity is
`meas_contribution`, never `items` / `contribution`: signature_analysis reads
those, and would otherwise take the half-suite ablation file for a capture of
the model (signature_analysis.load_run rejects these files, so a glob over
all outputs is safe, but keeping them apart keeps its report clean).

--analyse writes ablation_report.md and ablation_report.json: per model,
condition and k the mean fractional drop 1 - LD/LD_clean with bootstrap CIs on
clean-correct measurement items, the sub-additivity ratio top3 / (3 top1), the
excess over the random baseline, the slope of the drop on k, and the same slope
on the absolute scale (LD_clean - LD) with the clean LD by k: the clean LD
falls with k, so a fractional slope without an absolute one is flagged.  Slope
p-values come from a studentised permutation test (slope / HC2 SE, k
shuffled; slope_test): fewer items are correct at high k and the fraction is
noisier there, and a plain k permutation then rejects a zero slope far more
often than alpha.  For two different models on the same suite (the only
combination compared; any other is listed as not compared) the ablation x k x
model interaction with a bootstrap 90% CI and the outline's equivalence rule (a
null only if the CI lies within +/-0.10 of clean LD; a CI outside that
excluding 0 is a difference; anything else inconclusive), plus the
absolute-scale interaction.  Each model's layer classes come from its own
selection half, so a condition the two files hold only for different classes
(Llama spreading, Qwen funnelling) is compared across them, each model's own
target layers ('spread-vs-funnel/top1/mean'); a pair with no comparable
condition gets a note instead of an empty table, and a pair run with another
design (window, dtype, kernel, ...) or on other items gets a warning.  Glob
patterns in --analyse are expanded here (quote them for Drive paths).  Depths: the
eligible ones (outline rule on both halves' clean correctness) when there are
>= 3, else the depths solved above chance, reported as exploratory (the
outline's fallback); either way a depth needs --min-items clean-correct
measurement items, and depths below 20 analysed items are flagged.  Random
draws are pooled per item.  Every group's permutations and every pair's
bootstrap come from generators of their own, so a group's numbers do not
depend on the other groups in the file.
"""
import argparse, glob, hashlib, json, math, os, sys, time, zlib
from pathlib import Path
import numpy as np

from deep_probe import align_answer, _above_chance
from head_alignment_probe import _find_proj, _rel_err
import signature_analysis as sa
import sweep_probe as sp

DIGITS = "0123456789"
# Tolerances, max |error| / max |reference| (the repository's convention).
CACHE_TOL = 1e-2      # cached last-token logits vs a full forward (spec).  fp32
                      # reaches ~1e-6; a model that ignores the cache is O(1)
CACHE_TOL_BF16 = 0.25 # bf16: the cached and full forwards sum in a different
                      # order, and on 70B models with 400-token prompts they differ
                      # by a few percent (Llama-3.1-70B 3.8e-2, Qwen2.5-72B 6.6e-2,
                      # E4 Sep 24); a broken cache (wrong positions, stale rows)
                      # is still O(1).  Every condition, clean included, runs
                      # through the cached path, so comparisons stay consistent.
                      # Errors above CACHE_TOL are recorded and reported.
IDENT_TOL = 1e-3      # the same computation twice at the same batch position
                      # (repeat, own-z replacement): 0 on deterministic kernels;
                      # a stale cache or a misplaced slice is far larger
EXACT_TOL = sp.EXACT_TOL
VERIFY_EVERY = 25     # full-forward cache check: first item of a session, then every N
SETS = {"spread": -1, "funnel": 1}          # signature_analysis.classify_layers codes
TOP_SIZES = (1, 3)
MODES = ("mean", "zero")
MID_PCT = (25.0, 75.0)
EXCLUDE_TOP = 3
EQUIV = 0.10          # outline: a null needs the 90% CI within +/-10% of clean LD
SLOPE_TEST = "studentised permutation (slope / HC2 SE, k shuffled)"
# settings a pair's two files should share (the design is fixed before running
# and applies to both models); a mismatch is flagged in the interaction
PAIR_DESIGN_KEYS = ("window", "split", "max_items_per_k", "seed", "random_draws", "n_perm",
                    "q", "attn_impl", "dtype", "requested_precision")
# condition kinds paired across layer classes when the pair's classes differ
CROSS_KINDS = ("top", "rand")
# --batch-conditions default on CUDA.  Per measured item a run makes 1 prefix
# forward + ceil(C/B) condition forwards + 1 repeat of the first chunk (+ 1
# full forward on cache-check items, + 2 on the first item).  At the 70B pair
# (window 31-63, 5 draws, --per-layer) C is about 46: B = 1 is 48 forwards per
# item, each streaming the CPU-offloaded weights (~5 s), B = 16 is 5.  The KV
# cache grows B-fold (Llama-3.1-70B: 328 KB/token x B, ~2 GB at 400 tokens).
BATCH_CUDA = 16
SUFFIX = "__ablation.npz"
# settings that define an output; one written with any other value is not "done"
RESUME_KEYS = ("model", "suite", "suite_sha256", "split", "window", "max_items_per_k",
               "random_draws", "seed", "per_layer", "n_perm", "q", "batch_conditions",
               "attn_impl", "requested_precision", "measurement_version")
# An output or checkpoint is "done" only if measured the same way.
# MEASUREMENT_VERSION is bumped by any change that can alter the numbers; edits
# that cannot (a tolerance of a check, messages) keep it.  Files written before
# the field existed are mapped from their module_sha256.
MEASUREMENT_VERSION = 1
_COMPATIBLE_MODULES = {
    # Sep 24 (first E4 runs); differs from version 1 only in the bf16 cache-check tolerance
    "5ee967dee55448ca24ccf134407a04ba8118d0f49db1730462156c75b8c17585": 1,
}


def _meas_version(m):
    v = m.get("measurement_version")
    return v if v is not None else _COMPATIBLE_MODULES.get(m.get("module_sha256"))


def _abort(msg):
    raise SystemExit(msg)


def _rng(seed, tag):
    """A generator that depends only on the seed and what it is for, so the
    halves, the classification and each random baseline have their own
    streams (adding draws or conditions never moves the others)."""
    return np.random.default_rng([int(seed) % (1 << 63), zlib.crc32(str(tag).encode())])


def _mod_sha():
    return sp._sha256(Path(__file__).resolve())


# ───────────────────────────── design pieces ────────────────────────────────
def split_halves(items, split="test", seed=0, max_items_per_k=None):
    """(selection, measurement) suite indices: the items of `split`, per depth
    shuffled by a generator seeded with `seed`, capped at max_items_per_k,
    then halved (selection floor(n/2), measurement the rest).  Both lists are
    sorted.  Aborts when a depth has fewer than 2 items or there is only one
    depth (the design needs k to vary in both halves)."""
    idx = sp._select(items, split)
    byk = {}
    for i in idx:
        byk.setdefault(int(items[i]["k"]), []).append(i)
    if len(byk) < 2:
        _abort(f"split {split!r} has one depth ({sorted(byk)}); the ablation x k design "
               f"needs at least two")
    if max_items_per_k is not None and int(max_items_per_k) < 2:
        _abort("max_items_per_k must be >= 2 (each depth is split in two)")
    rng = _rng(seed, "halves")
    sel, meas = [], []
    for k in sorted(byk):
        ix = np.asarray(byk[k])[rng.permutation(len(byk[k]))]
        if max_items_per_k is not None:
            ix = ix[:int(max_items_per_k)]
        if len(ix) < 2:
            _abort(f"k={k}: {len(ix)} item in split {split!r}; each depth needs at least 2 "
                   f"to be divided into selection and measurement halves")
        h = len(ix) // 2
        sel += ix[:h].tolist()
        meas += ix[h:].tolist()
    return sorted(int(i) for i in sel), sorted(int(i) for i in meas)


def check_competitors(items, idx):
    """Abort unless every item in idx has a competitor (answer_set minus gold):
    the outcome, gold minus the mean competitor logit, is undefined without
    one.  sweep_probe._check_items accepts answer_set == [gold], so this runs
    before the model is loaded and before pass 1."""
    bad = [items[i]["id"] for i in idx
           if not [d for d in items[i]["answer_set"] if str(d) != str(items[i]["gold"])]]
    if bad:
        _abort(f"{len(bad)} item(s) have no competitor in their answer_set (e.g. {bad[:3]}); "
               f"the logit difference needs at least one admissible answer besides gold")


def rank_heads(mean_activity):
    """[L, H] head indices per layer by mean activity, largest first (ties to
    the lower index)."""
    return np.argsort(-np.asarray(mean_activity, float), axis=-1, kind="stable")


def mid_rank_heads(activity, lo=MID_PCT[0], hi=MID_PCT[1], exclude_top=EXCLUDE_TOP):
    """Heads of one layer whose mean activity lies between its lo-th and hi-th
    percentile (numpy's linear interpolation, inclusive), minus the
    exclude_top most active heads: the random baseline's pool.  Excluding the
    top heads matters when H is small (at H=4 the percentile band holds the
    2nd and 3rd heads)."""
    a = np.asarray(activity, float)
    p_lo, p_hi = np.percentile(a, [lo, hi])
    top = set(np.argsort(-a, kind="stable")[:exclude_top].tolist())
    return np.array([h for h in range(len(a)) if p_lo <= a[h] <= p_hi and h not in top], int)


def build_conditions(layer_class, window_layers, mean_activity, *, random_draws=5, seed=0,
                     per_layer=False):
    """(conditions, skipped, sets).

    conditions: list of dicts {name, set, kind (clean|top|rand|top_layer),
    size, mode (mean|zero), draw, layer, heads: [[layer, [head, ...]], ...]},
    clean first.  sets: {"spread": [layers], "funnel": [layers]} (window
    layers classed -1 / +1).  Random draws for (set, size) come from their own
    generator seeded by (seed, set, size), so adding draws keeps earlier ones."""
    lc = np.asarray(layer_class, int)
    A = np.asarray(mean_activity, float)
    L, H = A.shape
    if lc.shape != (L,):
        raise ValueError(f"layer_class {lc.shape} does not match activity {A.shape}")
    rank = rank_heads(A)
    wl = [int(l) for l in window_layers]
    conds = [dict(name="clean", set=None, kind="clean", size=0, mode=None, draw=None,
                  layer=None, heads=[])]
    skipped, sets = [], {}
    for s, code in SETS.items():
        layers = [l for l in wl if lc[l] == code]
        sets[s] = layers
        if not layers:
            skipped.append({"set": s, "condition": f"{s}/*",
                            "reason": f"no {s} layer in the window (layers {wl[0]}-{wl[-1]})"})
            continue
        for n in TOP_SIZES:
            if n > H:
                skipped.append({"set": s, "condition": f"{s}/top{n}/*",
                                "reason": f"only {H} heads per layer"})
                continue
            for mode in MODES:
                conds.append(dict(name=f"{s}/top{n}/{mode}", set=s, kind="top", size=n,
                                  mode=mode, draw=None, layer=None,
                                  heads=[[l, sorted(int(h) for h in rank[l, :n])] for l in layers]))
        for n in TOP_SIZES:
            if random_draws < 1:
                break
            pool = {l: mid_rank_heads(A[l]) for l in layers}
            short = {l: len(p) for l, p in pool.items() if len(p) < n}
            if short:
                skipped.append({"set": s, "condition": f"{s}/rand{n}/mean",
                                "reason": f"layers with fewer than {n} mid-ranked heads "
                                          f"(25th-75th percentile, top {EXCLUDE_TOP} "
                                          f"excluded): {short}"})
                continue
            rng = _rng(seed, f"random/{s}/{n}")
            for r in range(int(random_draws)):
                conds.append(dict(name=f"{s}/rand{n}/mean/r{r}", set=s, kind="rand", size=n,
                                  mode="mean", draw=r, layer=None,
                                  heads=[[l, sorted(int(h) for h in
                                                    rng.choice(pool[l], n, replace=False))]
                                         for l in layers]))
        if per_layer:
            for l in layers:
                conds.append(dict(name=f"{s}/top1/mean/L{l}", set=s, kind="top_layer", size=1,
                                  mode="mean", draw=None, layer=l, heads=[[l, [int(rank[l, 0])]]]))
    return conds, skipped, sets


def ablation_tables(conditions, mean_z, window_layers):
    """Per condition {layer: ([heads], values [n, d_head] float32 tensor, or
    None for zero-ablation)}; the clean condition is {}."""
    import torch
    wi = {int(l): i for i, l in enumerate(window_layers)}
    mz = np.asarray(mean_z, np.float32)
    out = []
    for c in conditions:
        t = {}
        for l, heads in c["heads"]:
            if int(l) not in wi:
                raise ValueError(f"{c['name']}: layer {l} is not a window layer")
            heads = [int(h) for h in heads]
            vals = (None if c["mode"] == "zero" else
                    torch.tensor(mz[wi[int(l)]][heads], dtype=torch.float32))
            t[int(l)] = (heads, vals)
        out.append(t)
    return out


def logit_diff(digit_logits, gold, answer_set):
    """Gold logit minus the mean logit of the competitors (answer_set - gold);
    digit_logits [..., 10] indexed by digit."""
    dl = np.asarray(digit_logits, float)
    comp = [int(d) for d in answer_set if str(d) != str(gold)]
    if not comp:
        raise ValueError(f"answer_set {answer_set} has no competitor for gold {gold}")
    return dl[..., int(gold)] - dl[..., comp].mean(-1)


# ─────────────────────────────── cache ──────────────────────────────────────
def _as_cache(pkv):
    if pkv is None:
        _abort("the model returned no KV cache (use_cache=True); cannot run the last token "
               "against the prefix")
    if hasattr(pkv, "crop") and hasattr(pkv, "get_seq_length"):
        return pkv
    try:                                     # legacy tuple caches (older transformers)
        from transformers import DynamicCache
        return DynamicCache.from_legacy_cache(pkv)
    except Exception as e:
        _abort(f"KV cache of type {type(pkv).__name__} cannot be cropped ({e})")


def _crop(cache, P):
    try:
        cache.crop(P)
    except Exception as e:
        _abort(f"cannot crop the KV cache back to the prefix ({type(e).__name__}: {e}); "
               f"sliding-window or static caches are not supported")
    n = int(cache.get_seq_length())
    if n != P:
        _abort(f"KV cache holds {n} positions after cropping, not the prefix's {P}; "
               f"conditions would see each other's last token")


# ─────────────────────────────── hooks ──────────────────────────────────────
class Ablator:
    """The model's output projections, patched: capture per-head activity and
    head inputs z at the last position, and replace head inputs there.

    Use as a context manager (patches removed on exit).  `model` is an
    _HFHookedModel-like wrapper (.cfg.n_layers/.n_heads/.d_head, ._layers(),
    ._get_attn(layer), ._model, ._tok).  Model-agnostic so the tests can drive
    it with small random models."""

    def __init__(self, model, window_layers, log=print):
        import torch
        self.torch, self.model, self.log = torch, model, log
        self.hf, self.tok = model._model, model._tok
        self.L, self.H, self.dh = model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_head
        self.layers = model._layers()
        if len(self.layers) != self.L:
            _abort(f"config says {self.L} layers, model has {len(self.layers)}")
        self.window = [int(l) for l in window_layers]
        self.wset = set(self.window)
        self.dev = sp._input_device(self.hf)
        _dt = next(self.hf.parameters()).dtype
        self.cache_tol = CACHE_TOL if _dt == torch.float32 else CACHE_TOL_BF16
        self.layout, self.patched, self.pname, self.head_checks = {}, [], None, []
        self.checks = {"split_err": 0.0, "split_exact_err": 0.0, "split_tol": None,
                       "delta_err": 0.0, "delta_checks": 0, "input_checks": 0, "cache_err": 0.0,
                       "cache_checks": 0, "ownz_err": 0.0, "pad_err": 0.0,
                       "repeat_err": 0.0, "items": 0, "cache_tol": self.cache_tol}
        self._clear()

    def _clear(self):
        self.capture_on, self.check_delta = False, False
        self.ablate = {}       # layer -> [(row, [heads], values or None)]
        self.calls, self.contrib, self.z = {}, {}, {}

    # ── patch management ──
    def __enter__(self):
        torch = self.torch
        training = [n or type(self.hf).__name__ for n, m in self.hf.named_modules() if m.training]
        if training:
            _abort(f"model is in train mode ({len(training)} modules, e.g. {training[0]!r}): "
                   f"dropout would randomise every number; call .eval() first")
        self.head_checks = sp._check_heads(self.model, self.layers, self.H, self.dh)
        try:
            for l in range(self.L):
                o, nm = _find_proj(self.model._get_attn(self.layers[l]))
                if o is None:
                    _abort(f"layer {l}: no output projection; unsupported architecture")
                self.pname = self.pname or nm
                sp._patch(o, self._factory(l, o), self.patched)
        except BaseException:
            sp._unpatch(self.patched)
            raise
        # the exact split check compares fp32 products at EXACT_TOL; TF32 would
        # round both to 10 mantissa bits (restored on exit)
        self._mm = torch.get_float32_matmul_precision()
        torch.set_float32_matmul_precision("highest")
        return self

    def __exit__(self, *exc):
        sp._unpatch(self.patched)
        self.torch.set_float32_matmul_precision(self._mm)
        self._clear()
        return False

    def _factory(self, l, module):
        torch, H, dh = self.torch, self.H, self.dh

        def factory(orig):
            def f(*a, **kw):
                self.calls[l] = self.calls.get(l, 0) + 1
                if a:
                    x = a[0]
                    call = lambda t: orig(t, *a[1:], **kw)
                else:
                    key = next(iter(kw))
                    x = kw[key]
                    call = lambda t: orig(**{**kw, key: t})
                rows = self.ablate.get(l)
                if rows:
                    if x.dim() != 3 or x.shape[-1] != H * dh:
                        _abort(f"layer {l}: o_proj input of shape {tuple(x.shape)}, expected "
                               f"[batch, seq, {H}*{dh}]; cannot locate head slices")
                    xm = x.clone(memory_format=torch.contiguous_format)
                    for row, heads, vals in rows:
                        v = xm[row, -1].view(H, dh)
                        if vals is None:
                            v[heads] = 0
                        else:
                            v[heads] = vals.to(device=x.device, dtype=x.dtype)
                    out = call(xm)
                else:
                    out = call(x)
                # row 0 is never ablated while capturing, so its output is clean
                if self.capture_on:
                    self._capture(l, module, x, out)
                if rows and self.check_delta:
                    self._check_delta(l, module, x, xm, out, call(x), rows)
                return out
            return f
        return factory

    def _capture(self, l, module, x, out):
        torch, H, dh = self.torch, self.H, self.dh
        W = module.weight
        if x.dim() != 3 or W.device.type == "meta":
            return                       # not recorded -> the capture check aborts
        ref = out[0, -1].float()
        if not torch.isfinite(ref).all():
            _abort(f"layer {l}: non-finite attention output at the measured position")
        b = getattr(module, "bias", None)
        b = 0 if b is None else b.float()
        ktol = sp._kernel_tol(x.dtype, W.dtype, out.dtype)
        self.checks["split_tol"] = max(self.checks["split_tol"] or 0.0, ktol)
        if l not in self.layout:
            # shape cannot tell Linear [out, in] from Conv1D [in, out] when the
            # matrix is square: keep the layout that reproduces the output
            best = ("none", float("inf"))
            for mode in ("linear", "conv1d"):
                try:
                    e = _rel_err(sp._split(x[0, -1], W, mode, H, dh)[0].sum(0) + b, ref)
                except Exception:
                    e = float("inf")
                e = e if math.isfinite(e) else float("inf")
                if e < best[1]:
                    best = (mode, e)
            if not sp._ok(best[1], ktol):
                _abort(f"layer {l}: per-head split does not reproduce {type(module).__name__} "
                       f"under either weight layout (best relative error {best[1]:.2e} > "
                       f"{ktol}, n_heads={H}, head_dim={dh}); refusing")
            self.layout[l] = best
        v, whole = sp._split(x[0, -1], W, self.layout[l][0], H, dh)
        tot = v.sum(0) + b
        e_exact = _rel_err(tot, whole + b)
        if not sp._ok(e_exact, EXACT_TOL):
            _abort(f"layer {l}: per-head split does not sum to the projection recomputed in "
                   f"fp32 (error {e_exact:.2e} > {EXACT_TOL}); refusing")
        e = _rel_err(tot, ref)
        if not sp._ok(e, ktol):
            _abort(f"layer {l}: per-head split error {e:.2e} > {ktol} "
                   f"(layout {self.layout[l][0]}); refusing")
        self.checks["split_err"] = max(self.checks["split_err"], e)
        self.checks["split_exact_err"] = max(self.checks["split_exact_err"], e_exact)
        self.contrib[l] = v.norm(dim=-1).cpu()
        if l in self.wset:
            self.z[l] = x[0, -1].float().reshape(H, dh).cpu()

    def _check_delta(self, l, module, x, xm, out, out_clean, rows):
        """The ablation the projection saw must be the one the table asks for,
        and it must change the output by exactly those heads' writes.

        (1) The intended input is rebuilt from `rows` (the condition's heads
        and values) with explicit column slices h*d_head:(h+1)*d_head, not
        from the replaced input: it must equal the input the projection
        received bit for bit, so a wrong head, a wrong value, a wrong row or a
        touched earlier position aborts (the own-z check replaces every head,
        so it cannot see which head a replacement hit).  (2) The ablated
        output minus the clean one must be the listed heads' (r_h - z_h)
        W_O^h, predicted from the intended input: both outputs went through
        the model's kernel, so the bound is twice the kernel tolerance,
        relative to the clean output's scale."""
        torch, H, dh = self.torch, self.H, self.dh
        W = module.weight
        if l not in self.layout:
            _abort(f"layer {l}: weight layout unknown at the ablation check")
        xe = x.clone(memory_format=torch.contiguous_format)
        for row, heads, vals in rows:
            for i, h in enumerate(heads):
                if not 0 <= int(h) < H:
                    _abort(f"layer {l}: head {h} outside 0..{H - 1} in the ablation table")
                cols = slice(int(h) * dh, (int(h) + 1) * dh)
                xe[row, -1, cols] = (0 if vals is None else
                                     vals[i].to(device=x.device, dtype=x.dtype))
        if xm.shape != xe.shape or not torch.equal(xm, xe):
            where = []
            if xm.shape == xe.shape:
                diff = (xm[:, -1] != xe[:, -1]).reshape(xm.shape[0], H, dh).any(-1)
                where = [(int(r), int(h)) for r, h in diff.nonzero().tolist()][:8]
                if xm.shape[1] > 1 and not torch.equal(xm[:, :-1], xe[:, :-1]):
                    where.append("an earlier position")
            _abort(f"layer {l}: the o_proj input differs from the intended ablation at "
                   f"(row, head) {where or 'shape ' + str(tuple(xm.shape))}; the replacement "
                   f"does not hit the heads and values of the condition table, refusing")
        self.checks["input_checks"] += 1
        ktol = 2 * sp._kernel_tol(x.dtype, W.dtype, out.dtype)
        rs = sorted({r for r, _, _ in rows})
        # one fp32 copy of W_O per call (268 MB at 8192 x 8192), not per row;
        # sum_h (r_h - z_h) W_O^h is W_O applied to the intended input change
        Wm = (W if self.layout[l][0] == "linear" else W.t()).float()
        d_pred_all = (xe[rs, -1].float() - x[rs, -1].float()) @ Wm.t()
        del Wm, xe
        for i, row in enumerate(rs):
            d_obs = out[row, -1].float() - out_clean[row, -1].float()
            d_pred = d_pred_all[i]
            scale = float(out_clean[row, -1].float().abs().max().clamp_min(1e-9))
            e = float((d_obs - d_pred).abs().max()) / scale
            if not sp._ok(e, ktol):
                _abort(f"layer {l}: ablating heads changed the o_proj output by something "
                       f"other than their write (error {e:.2e} > {ktol}); refusing")
            self.checks["delta_err"] = max(self.checks["delta_err"], e)
            self.checks["delta_checks"] += 1

    # ── forwards ──
    def forward(self, ids, cache=None, use_cache=False):
        """One model forward; (output, last-position logits [B, V] float32)."""
        torch = self.torch
        self.calls.clear()
        with torch.no_grad():
            o = self.hf(input_ids=ids, past_key_values=cache, use_cache=use_cache)
        bad = {l: self.calls.get(l, 0) for l in range(self.L) if self.calls.get(l, 0) != 1}
        if bad:
            _abort(f"output projection calls per layer {bad}: expected exactly one per "
                   f"forward on every layer")
        lg = o.logits[:, -1].float()
        if not torch.isfinite(lg).all():
            _abort("non-finite logits at the measured position")
        return o, lg

    def _collect(self, what):
        miss = [l for l in range(self.L) if l not in self.contrib]
        missz = [l for l in self.window if l not in self.z]
        if miss or missz:
            _abort(f"{what}: activity recorded on {self.L - len(miss)}/{self.L} layers, z on "
                   f"{len(self.window) - len(missz)}/{len(self.window)} window layers")
        C = self.torch.stack([self.contrib[l] for l in range(self.L)]).numpy()
        Z = (self.torch.stack([self.z[l] for l in self.window]).numpy() if self.window
             else np.zeros((0, self.H, self.dh), np.float32))
        if not (np.isfinite(C).all() and np.isfinite(Z).all()):
            _abort(f"{what}: activity or z has NaN/inf")
        return C.astype(np.float32), Z.astype(np.float32)

    def capture(self, feed):
        """Full forward of `feed` (token ids): (activity [L, H], z [len(window),
        H, d_head], last-position logits [V] float32 on the CPU)."""
        ids = self.torch.tensor([list(feed)], device=self.dev)
        self.contrib.clear(); self.z.clear()
        self.capture_on = True
        try:
            _, lg = self.forward(ids)
        finally:
            self.capture_on = False
        C, Z = self._collect("capture")
        return C, Z, lg[0].cpu()

    def measure(self, feed, tables, ans_ids, *, batch=1, verify=False, verify_cache=None):
        """Run every condition on one item.

        feed: token ids (align_answer's), measured at the last one.  tables:
        ablation_tables(...) output, tables[0] the clean condition ({}).
        ans_ids: the ten digit answer tokens.  Returns (digit logits [C, 10],
        clean last-position logits [V] (CPU), clean activity [L, H], clean z
        [len(window), H, d_head]).  verify runs the first-item checks (module
        docstring); verify_cache the full-forward cache check (default:
        verify)."""
        torch = self.torch
        if not tables or tables[0]:
            raise ValueError("tables[0] must be the clean condition ({})")
        batch = int(batch)
        if batch < 1:
            raise ValueError("batch must be >= 1")
        verify_cache = verify if verify_cache is None else verify_cache
        C = len(tables)
        ids = torch.tensor([list(feed)], device=self.dev)
        P = ids.shape[1] - 1
        if P < 1:
            _abort("the prompt is a single token; there is no prefix to cache")
        o, _ = self.forward(ids[:, :P], use_cache=True)
        cache = _as_cache(o.past_key_values)
        del o
        if int(cache.get_seq_length()) != P:
            _abort(f"prefix cache holds {int(cache.get_seq_length())} positions, not {P}")
        if batch > 1:
            cache.batch_repeat_interleave(batch)
        last = ids[:, P:].expand(batch, 1).contiguous()
        ans = list(ans_ids)
        dl = np.full((C, 10), np.nan, np.float32)

        def run(tabs, capture=False):
            tabs = list(tabs) + [{}] * (batch - len(tabs))       # pad with clean rows
            abl = {}
            for r, t in enumerate(tabs):
                for l, (heads, vals) in t.items():
                    abl.setdefault(int(l), []).append((r, heads, vals))
            if capture:
                self.contrib.clear(); self.z.clear()
            self.ablate, self.capture_on, self.check_delta = abl, capture, verify
            try:
                _, lg = self.forward(last, cache=cache, use_cache=True)
            finally:
                self.ablate, self.capture_on, self.check_delta = {}, False, False
                _crop(cache, P)
            return lg

        def pad_check(lg, n_used, clean):
            # padded rows are clean at other batch positions: a batched forward
            # must reproduce the clean row (bf16 may round rows differently)
            for r in range(n_used, batch):
                e = _rel_err(lg[r], clean)
                if not sp._ok(e, self.cache_tol):
                    _abort(f"batched clean row {r} differs from the clean row (error {e:.2e} "
                           f"> {self.cache_tol}); rows of one forward are not independent")
                self.checks["pad_err"] = max(self.checks["pad_err"], e)

        chunks = [list(range(i, min(i + batch, C))) for i in range(0, C, batch)]
        lg0 = run([tables[m] for m in chunks[0]], capture=True)
        clean = lg0[0].clone()
        act, z = self._collect("clean condition")
        pad_check(lg0, len(chunks[0]), clean)
        for r, m in enumerate(chunks[0]):
            dl[m] = lg0[r, ans].cpu().numpy()

        if verify_cache:
            _, lf = self.forward(ids)
            e = _rel_err(clean, lf[0])
            if not sp._ok(e, self.cache_tol):
                _abort(f"clean last-token logits through the KV cache do not reproduce a full "
                       f"forward (relative error {e:.2e} > {self.cache_tol}); the cached path is "
                       f"not exact here, refusing")
            self.checks["cache_err"] = max(self.checks["cache_err"], e)
            self.checks["cache_checks"] += 1
        if verify:
            zt = {l: self.torch.from_numpy(z[i]) for i, l in enumerate(self.window)}
            own = {l: (list(range(self.H)), zt[l]) for l in self.window}
            zero = {l: ([int(np.argmax(act[l]))], None) for l in self.window}
            # own z at row 0, the clean row's batch position: same computation
            lo = run([own])
            e = _rel_err(lo[0], clean)
            if not sp._ok(e, IDENT_TOL):
                _abort(f"replacing every window head's input with its own value changed the "
                       f"logits (error {e:.2e} > {IDENT_TOL}): the replacement does not hit "
                       f"the measured position's head slices")
            self.checks["ownz_err"] = max(self.checks["ownz_err"], e)
            lz = run([zero])
            if self.window and torch.equal(lz[0], clean):
                _abort("zero-ablating the largest head of every window layer left the logits "
                       "unchanged: the ablation does not reach the model")

        for ch in chunks[1:]:
            lg = run([tables[m] for m in ch])
            pad_check(lg, len(ch), clean)
            for r, m in enumerate(ch):
                dl[m] = lg[r, ans].cpu().numpy()

        # the first chunk again: a cache the crops did not restore changes it
        lr = run([tables[m] for m in chunks[0]])
        e = _rel_err(lr[:len(chunks[0])], lg0[:len(chunks[0])])
        if not sp._ok(e, IDENT_TOL):
            _abort(f"re-running the first conditions after the others changed their logits "
                   f"(error {e:.2e} > {IDENT_TOL}): the KV cache was not restored")
        self.checks["repeat_err"] = max(self.checks["repeat_err"], e)
        self.checks["items"] += 1
        del cache
        if not np.isfinite(dl).all():
            _abort("digit logits contain NaN/inf")
        return dl, clean.cpu(), act, z


# ────────────────────────────── the run ─────────────────────────────────────
def _provenance():
    import torch, transformers
    here = Path(__file__).resolve()
    commit, src = sp._git_commit()
    deps = {}
    for n in ("deep_probe.py", "head_alignment_probe.py", "active_subgraph_dot.py",
              "sweep_probe.py", "signature_analysis.py", "serial_tasks.py"):
        p = here.parent / n
        deps[n] = sp._sha256(p) if p.exists() else None
    return {"torch_version": torch.__version__, "transformers_version": transformers.__version__,
            "numpy_version": np.__version__, "python_version": sys.version.split()[0],
            "git_commit": commit, "git_commit_source": src,
            "next_runs_commit_env": os.environ.get("NEXT_RUNS_COMMIT"),
            "module_sha256": _mod_sha(), "dependency_sha256": deps,
            "measurement_version": MEASUREMENT_VERSION}


def _ids_sha(items, idx):
    return hashlib.sha256(json.dumps([items[i]["id"] for i in idx]).encode()).hexdigest()


def _load_checkpoint(path, fingerprint, log):
    if path is None or not Path(path).exists():
        return None
    try:
        A, meta = sp.load_output(path)
    except Exception as e:
        _abort(f"checkpoint {path} exists but cannot be read ({e}); delete it to start over")
    have = dict(meta.get("fingerprint", {}))
    have["measurement_version"] = _meas_version(have)
    diff = [f"{k}: have {have.get(k)!r}, want {v!r}" for k, v in fingerprint.items()
            if have.get(k) != v]
    if diff:
        _abort(f"checkpoint {path} was written with other settings ({'; '.join(diff)}); "
               f"delete it or use another --out")
    log(f"  resuming from {Path(path).name}: pass 1 done, {meta['done']} measurement items done")
    return A, meta


def run_ablation(model, suite, *, attn_impl, window=(0.4, 0.8), split="test",
                 max_items_per_k=None, random_draws=5, seed=0, per_layer=False, n_perm=5000,
                 q=0.05, batch_conditions=1, layer_classes=None, checkpoint=None,
                 checkpoint_every=5, log=print):
    """E4 on one model and suite; returns (arrays, meta) (module docstring).

    model: an _HFHookedModel-like wrapper.  layer_classes [L] of -1/0/+1
    replaces the selection-half classification (tests; recorded in meta).
    checkpoint: path of a resumable checkpoint (written after pass 1 and every
    checkpoint_every measurement items, deleted by the caller when done)."""
    import torch
    t_start = time.time()
    hf, tok = model._model, model._tok
    L, H, dh = model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_head
    name = getattr(model.cfg, "model_name", None)
    sp._check_items(suite)
    actual = getattr(hf.config, "_attn_implementation", None)
    if actual is not None and actual != attn_impl:
        _abort(f"model runs {actual!r} attention but attn_impl={attn_impl!r}; the recorded "
               f"kernel must be the one that ran")
    lo, hi = (float(w) for w in window)
    if not 0 <= lo < hi <= 1:
        _abort(f"window must satisfy 0 <= lo < hi <= 1, got {window}")
    if int(batch_conditions) < 1 or int(random_draws) < 0:
        _abort("batch_conditions must be >= 1 and random_draws >= 0")
    items_all = suite["items"]
    sel, meas = split_halves(items_all, split, seed, max_items_per_k)
    check_competitors(items_all, sel + meas)
    wl = list(range(L))[sa.window(L, lo, hi)]
    if not wl:
        _abort(f"window {window} holds no layer of {L}")
    if 1 / (n_perm + 1) > q / L:
        log(f"  note: {n_perm} permutations, smallest p 1/{n_perm + 1} > q/L = {q / L:.2g}; "
            f"BH can then classify a layer only together with many others "
            f"(use --n-perm >= {math.ceil(L / q)})")
    asd = sys.modules.get("active_subgraph_dot")
    load_info = dict(getattr(asd, "LAST_LOAD_INFO", None) or {})
    if load_info.get("model") != name:
        load_info = {"note": "model not loaded through active_subgraph_dot.load_model "
                             "in this process"}
    dtype = str(next(hf.parameters()).dtype).replace("torch.", "")
    fingerprint = {"model": name, "suite_sha256": sp.suite_sha256(suite), "split": split,
                   "window": [lo, hi], "max_items_per_k": max_items_per_k,
                   "random_draws": int(random_draws), "seed": int(seed),
                   "per_layer": bool(per_layer), "n_perm": int(n_perm), "q": float(q),
                   "batch_conditions": int(batch_conditions), "attn_impl": attn_impl,
                   "dtype": dtype, "measurement_version": MEASUREMENT_VERSION,
                   "selection_sha256": _ids_sha(items_all, sel),
                   "measurement_sha256": _ids_sha(items_all, meas),
                   "layer_classes": None if layer_classes is None
                   else [int(c) for c in layer_classes]}
    ck = _load_checkpoint(checkpoint, fingerprint, log)
    nS, nM = len(sel), len(meas)
    log(f"  {name}: L={L} H={H} d_head={dh}   window layers {wl[0]}-{wl[-1]} ({len(wl)})   "
        f"selection {nS} / measurement {nM} items   batch {batch_conditions}")

    with Ablator(model, wl, log=log) as ab:
        # ── pass 1: selection half ──────────────────────────────────────────
        if ck is None:
            P1 = {"sel_index": np.array(sel, np.int32),
                  "sel_k": np.array([items_all[i]["k"] for i in sel], np.int16),
                  "sel_contribution": np.zeros((nS, L, H), np.float32),
                  "sel_digit_logits": np.zeros((nS, 10), np.float32),
                  "sel_correct": np.zeros(nS, bool)}
            zsum = np.zeros((len(wl), H, dh), np.float64)
            for j, i in enumerate(sel):
                t = items_all[i]
                log(f"  [select {j + 1}/{nS}] {t['id']}")
                feed, ANS, _ = align_answer(tok, t["text"])
                C, Z, lg = ab.capture(feed)
                P1["sel_contribution"][j] = C
                zsum += Z
                dlj = np.array([float(lg[ANS[d]]) for d in DIGITS], np.float32)
                P1["sel_digit_logits"][j] = dlj
                P1["sel_correct"][j] = sp._constrained(dlj, t["answer_set"], t["gold"])[1]
            P1["mean_z"] = (zsum / nS).astype(np.float32)
            P1["mean_activity"] = P1["sel_contribution"].mean(0)
            if layer_classes is None:
                Y = sa.top_share(P1["sel_contribution"])
                bad = ~np.isfinite(Y)
                if bad.any():
                    # top share is 0/0 where every head of a layer writes zero;
                    # the joint null needs every item on every layer, and no
                    # trained layer is silent at the answer position
                    ls = np.nonzero(bad.any(0))[0]
                    _abort(f"layer(s) {ls.tolist()}: every head's activity is zero at the "
                           f"measured position on {int(bad.any(1).sum())}/{nS} selection "
                           f"items (e.g. {items_all[sel[int(np.nonzero(bad.any(1))[0][0])]]['id']})"
                           f", so the top share is undefined and the layers cannot be "
                           f"classified; a zeroed or bypassed output projection? refusing")
                cls = sa.classify_layers(Y, P1["sel_k"].astype(float), q, n_perm,
                                         _rng(seed, "classify"))
                src = "signature_analysis.classify_layers (selection half, all layers)"
            else:
                cls, src = np.asarray(layer_classes, int), "given"
                if cls.shape != (L,) or not set(np.unique(cls)) <= {-1, 0, 1}:
                    _abort(f"layer_classes must be [L={L}] of -1/0/+1")
            P1["layer_class"] = np.asarray(cls, np.int8)
            P1["head_rank"] = rank_heads(P1["mean_activity"]).astype(np.int16)
            P1["window_layers"] = np.array(wl, np.int16)
            p1meta = {"classes_source": src}
            done, recs, P2 = 0, [], None
            prev_checks = {}
        else:
            A0, m0 = ck
            P1 = {k: A0[k] for k in ("sel_index", "sel_k", "sel_contribution",
                                      "sel_digit_logits", "sel_correct", "mean_z",
                                      "mean_activity", "layer_class", "head_rank",
                                      "window_layers")}
            p1meta, done, recs = m0["pass1"], int(m0["done"]), list(m0["measurement_items"])
            P2 = {k: A0[k] for k in A0 if k not in P1}
            prev_checks = m0.get("checks", {})
        conds, skipped, sets = build_conditions(P1["layer_class"], wl, P1["mean_activity"],
                                                random_draws=random_draws, seed=seed,
                                                per_layer=per_layer)
        if ck is not None and [c["name"] for c in conds] != ck[1]["condition_names"]:
            _abort("checkpoint conditions differ from the ones rebuilt from its pass 1")
        tables = ablation_tables(conds, P1["mean_z"], wl)
        nC = len(conds)
        for s in SETS:
            log(f"  {s} layers in the window: {sets[s] or 'none'}")
        for sk in skipped:
            log(f"  skipped {sk['condition']}: {sk['reason']}")
        log(f"  {nC} conditions ({p1meta['classes_source']})")

        # ── pass 2: measurement half ────────────────────────────────────────
        A = {"index": np.array(meas, np.int32),
             "k": np.array([items_all[i]["k"] for i in meas], np.int16),
             "gold": np.array([int(items_all[i]["gold"]) for i in meas], np.int8),
             "answer_mask": np.zeros((nM, 10), bool),
             "digit_logits": np.full((nM, nC, 10), np.nan, np.float32),
             "logit_diff": np.full((nM, nC), np.nan, np.float32),
             "frac": np.full((nM, nC), np.nan, np.float32),
             "clean_correct": np.zeros(nM, bool),
             "meas_contribution": np.zeros((nM, L, H), np.float32),
             "n_tokens": np.zeros(nM, np.int32)}
        for j, i in enumerate(meas):
            A["answer_mask"][j, [int(d) for d in items_all[i]["answer_set"]]] = True
        if P2 is not None:
            for key in ("digit_logits", "logit_diff", "frac", "clean_correct",
                        "meas_contribution", "n_tokens"):
                A[key][:done] = P2[key][:done]

        # the weight layout found by the split check in any session; a session
        # resumed with nothing left to measure captures nothing and would
        # otherwise record none
        ck_layout = {} if ck is None else dict(ck[1].get("head_layout") or {})

        def layout_now():
            cur = {str(l): m for l, (m, _) in ab.layout.items()}
            clash = {l: (ck_layout[l], m) for l, m in cur.items()
                     if l in ck_layout and ck_layout[l] != m}
            if clash:
                _abort(f"weight layout differs from the checkpoint's on layers {clash}; the "
                       f"checkpoint was written for other weights")
            out = {**ck_layout, **cur}
            return {l: out[l] for l in sorted(out, key=int)}

        def save_ck(n_done):
            if checkpoint is None:
                return
            arr = dict(P1)
            arr.update({k: v for k, v in A.items()})
            sp.save_output(checkpoint, arr, {
                "kind": "ablation_checkpoint", "fingerprint": fingerprint, "done": n_done,
                "pass1": p1meta, "measurement_items": recs,
                "condition_names": [c["name"] for c in conds],
                "head_layout": layout_now(), "proj_name": ab.pname,
                "checks": _merge_checks(prev_checks, ab.checks)})

        if ck is None:
            save_ck(0)
        session = 0
        for j in range(done, nM):
            t = items_all[meas[j]]
            t0 = time.time()
            log(f"  [measure {j + 1}/{nM}] {t['id']}")
            feed, ANS, appended = align_answer(tok, t["text"])
            ans_ids = [ANS[d] for d in DIGITS]
            dl, _, act, _ = ab.measure(feed, tables, ans_ids, batch=batch_conditions,
                                       verify=(session == 0),
                                       verify_cache=(session % VERIFY_EVERY == 0))
            session += 1
            gold, aset = t["gold"], list(t["answer_set"])
            ld = logit_diff(dl, gold, aset)
            pred, correct, tied = sp._constrained(dl[0], aset, gold)
            A["digit_logits"][j], A["logit_diff"][j] = dl, ld
            A["frac"][j] = ld / ld[0] if ld[0] != 0 else np.nan
            A["clean_correct"][j] = correct
            A["meas_contribution"][j] = act
            A["n_tokens"][j] = len(feed)
            toks = tok.convert_ids_to_tokens(feed)
            recs.append({"id": t["id"], "suite_index": meas[j], "k": int(t["k"]),
                         "gold": gold, "answer_set": aset, "echo": t["echo"],
                         "n_tokens": len(feed), "measured_token": toks[-1],
                         "appended": appended, "pred_constrained": pred,
                         "correct_constrained": correct, "n_tied_at_max": len(tied),
                         "clean_logit_diff": float(ld[0]),
                         "time_s": round(time.time() - t0, 3)})
            if (j + 1) % max(1, int(checkpoint_every)) == 0:
                save_ck(j + 1)
            if ab.dev.type == "cuda":
                torch.cuda.empty_cache()
        layout = layout_now()
        if len(layout) != L:
            _abort(f"weight layout known for {len(layout)} of {L} layers; the provenance "
                   f"record would be incomplete")
        checks = _merge_checks(prev_checks, ab.checks)
        pname, head_checks = ab.pname, ab.head_checks

    A.update(P1)
    times = [r["time_s"] for r in recs] or [0.0]
    meta = {
        "kind": "ablation",
        "model": name, "suite": suite.get("name"), "family": suite.get("family"),
        "wording": suite.get("wording"), "setting": suite.get("setting"),
        "chance": suite.get("chance"), "suite_sha256": fingerprint["suite_sha256"],
        "split": split, "window": [lo, hi], "window_layers": wl,
        "max_items_per_k": max_items_per_k, "random_draws": int(random_draws),
        "seed": int(seed), "per_layer": bool(per_layer), "n_perm": int(n_perm), "q": float(q),
        "batch_conditions": int(batch_conditions), "attn_impl": attn_impl, "dtype": dtype,
        "n_layers": L, "n_heads": H, "d_head": dh, "load_info": load_info,
        **_provenance(),
        "proj_name": pname, "head_layout": layout, "head_boundaries_checked": head_checks,
        "classes_source": p1meta["classes_source"],
        "sets": sets, "conditions": conds, "skipped": skipped,
        "n_selection": nS, "n_measurement": nM,
        "selection_ids": [items_all[i]["id"] for i in sel],
        "tolerances": {"cache": CACHE_TOL, "cache_bf16": CACHE_TOL_BF16, "identical": IDENT_TOL, "split_exact": EXACT_TOL,
                       "split": sp.TOL, "split_fp32_kernel": sp.ROUNDING_TOL,
                       "delta": "2 x split tolerance", "input": "exact (intended ablation)",
                       "padded_rows": "as cache",
                       "verify_every": VERIFY_EVERY},
        "checks": checks, "measurement_items": recs,
        "timing": {"total_s": round(time.time() - t_start, 2),
                   "per_item_mean_s": round(float(np.mean(times)), 3)},
    }
    return A, meta


_COUNTS = ("delta_checks", "input_checks", "cache_checks", "items")


def _merge_checks(a, b):
    """Checks of an earlier session (a checkpoint) and this one: counts add,
    errors and tolerances take the maximum."""
    out = dict(a or {})
    for k, v in b.items():
        old = out.get(k)
        if k in _COUNTS:
            out[k] = (old or 0) + v
        elif isinstance(v, (int, float)) and isinstance(old, (int, float)):
            out[k] = max(old, v)
        elif v is not None or k not in out:
            out[k] = v
    return out


# ────────────────────────────── outputs ─────────────────────────────────────
def default_batch(device):
    """--batch-conditions when not given: BATCH_CUDA on a GPU (the offloaded
    70B runs, where every forward streams the weights), 1 on the CPU."""
    return BATCH_CUDA if str(device).split(":")[0] == "cuda" else 1


def output_path(out_dir, model_name, suite_name):
    return (Path(out_dir) /
            f"{model_name.replace('/', '_')}__{str(suite_name).replace('/', '_')}{SUFFIX}")


def checkpoint_path(out):
    out = Path(out)
    return out.with_name(out.name[:-len(".npz")] + ".ckpt")


def _summary(A, meta, log=print):
    ks = A["k"]
    log("  clean constrained by k: " + "  ".join(
        f"k{k} {int(A['clean_correct'][ks == k].sum())}/{int((ks == k).sum())}"
        for k in np.unique(ks)) + f"   chance {meta.get('chance')}")
    ok = A["clean_correct"]
    if ok.any():
        for i, c in enumerate(meta["conditions"]):
            if c["kind"] in ("top",):
                log(f"    {c['name']:<22} mean LD fraction of clean on correct items "
                    f"{np.nanmean(A['frac'][ok, i]):+.3f}")
    ch = meta["checks"]
    log(f"  checks: cache {ch.get('cache_err', 0):.1e} (abort {ch.get('cache_tol', CACHE_TOL)}"
        + (f"; above {CACHE_TOL}, the bf16 rounding band" if ch.get('cache_err', 0) > CACHE_TOL else "")
        + ", "
        f"{ch.get('cache_checks')} items)   own-z {ch.get('ownz_err', 0):.1e}   repeat "
        f"{ch.get('repeat_err', 0):.1e}   o_proj delta {ch.get('delta_err', 0):.1e} "
        f"({ch.get('input_checks', 0)} intended-input checks)   split "
        f"{ch.get('split_err', 0):.1e}")


def run(args, log=print):
    with open(args.suite) as f:
        suite = json.load(f)
    if not isinstance(suite, dict) or not isinstance(suite.get("name"), str) or not suite["name"]:
        _abort(f"{args.suite}: not a suite with a name (serial_tasks.save_suite)")
    # validate before the load (minutes to an hour for a 70B)
    try:
        sp._check_items(suite)
        sel, meas = split_halves(suite["items"], args.split, args.seed, args.max_items_per_k)
        check_competitors(suite["items"], sel + meas)
    except SystemExit as e:
        _abort(f"suite {args.suite}: {e}")
    B = getattr(args, "batch_conditions", None)
    B = default_batch(args.device) if B is None else int(B)
    out = output_path(args.out, args.model, suite["name"])
    want = {"model": args.model, "suite": suite["name"], "suite_sha256": sp.suite_sha256(suite),
            "split": args.split, "window": [float(w) for w in args.window],
            "max_items_per_k": args.max_items_per_k, "random_draws": args.random_draws,
            "seed": args.seed, "per_layer": bool(args.per_layer), "n_perm": args.n_perm,
            "q": float(args.q), "batch_conditions": B,
            "attn_impl": args.attn_impl, "requested_precision": args.precision,
            "measurement_version": MEASUREMENT_VERSION}
    if out.exists():
        try:
            _, m = sp.load_output(out)
        except Exception as e:
            _abort(f"{out} exists but cannot be read ({e}); delete it to re-run")
        m = dict(m, measurement_version=_meas_version(m))
        diff = [f"{k}: have {m.get(k)!r}, want {want[k]!r}" for k in RESUME_KEYS
                if m.get(k) != want[k]]
        if diff:
            _abort(f"{out} exists with different settings ({'; '.join(diff)}); use another "
                   f"--out or delete it")
        log(f"  skip {out.name}: already done")
        return None

    sp._git_commit()                 # before any tokenizer threads exist
    os.environ["ASD_FORCE_STRATEGY"] = "B"
    os.environ["ASD_FORCE_PRECISION"] = "bf16" if args.precision == "bf16" else ""
    os.environ["ASD_ATTN_IMPL"] = args.attn_impl
    import torch
    try:
        want_dev = torch.device(args.device).type
    except Exception:
        _abort(f"--device {args.device!r} is not a torch device")
    if want_dev == "cuda" and not torch.cuda.is_available():
        _abort("--device cuda but CUDA is not available: select a GPU runtime, or pass "
               "--device cpu explicitly")
    if args.precision == "fp32" and args.device != "cpu" and torch.cuda.is_available():
        _abort("load_model supports fp32 on CPU only; use --device cpu")
    import active_subgraph_dot as asd
    t0 = time.time()
    model = asd.load_model(args.model, device=args.device)
    load_s = time.time() - t0
    if not hasattr(model, "_model"):
        _abort("needs the _HFHookedModel wrapper (ASD_FORCE_STRATEGY=B)")
    on = sorted({p.device.type for p in model._model.parameters()})
    if want_dev != "cpu" and want_dev not in on:
        _abort(f"--device {args.device} requested but the model's weights are on {on} "
               f"(load_model fell back); refusing to run there")
    li = asd.LAST_LOAD_INFO
    want_prec = {"bf16": "bfloat16", "fp32": "float32"}[args.precision]
    if li.get("strategy") != "B" or li.get("precision") != want_prec:
        _abort(f"loaded with strategy {li.get('strategy')!r} precision "
               f"{li.get('precision')!r}; requested B / {want_prec}")
    log(f"  {args.model}: load_info={dict(li)}")
    ck = checkpoint_path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    A, meta = run_ablation(model, suite, attn_impl=args.attn_impl,
                           window=tuple(args.window), split=args.split,
                           max_items_per_k=args.max_items_per_k,
                           random_draws=args.random_draws, seed=args.seed,
                           per_layer=args.per_layer, n_perm=args.n_perm, q=args.q,
                           batch_conditions=B, checkpoint=ck,
                           checkpoint_every=args.checkpoint_every, log=log)
    meta.update(want)
    meta.update(model=args.model, suite_path=str(args.suite), load_s=round(load_s, 1),
                cli={k: v for k, v in vars(args).items()})
    sp.save_output(out, A, meta)
    if ck.exists():
        ck.unlink()
    _summary(A, meta, log)
    log(f"  -> {out}  ({out.stat().st_size / 1e6:.1f} MB)")
    return out


# ───────────────────────────── analysis ─────────────────────────────────────
class RunError(ValueError):
    """An ablation output that cannot be analysed without emitting wrong numbers."""


def load_ablation(path):
    """One ablation output as a dict, after checking that its stored logit
    differences follow from its digit logits."""
    try:
        A, meta = sp.load_output(path)
    except Exception as e:
        raise RunError(f"{path}: cannot read ({e})")
    need = ("k", "gold", "answer_mask", "digit_logits", "logit_diff", "frac", "clean_correct",
            "sel_k", "sel_correct")
    miss = [k for k in need if k not in A]
    if miss or "conditions" not in meta:
        raise RunError(f"{path}: not an ablation_probe output (missing {miss or ['conditions']})")
    conds = meta["conditions"]
    n, C = A["logit_diff"].shape
    if C != len(conds) or conds[0]["kind"] != "clean" or A["digit_logits"].shape != (n, C, 10):
        raise RunError(f"{path}: {C} logit-difference columns for {len(conds)} conditions")
    ld = np.full((n, C), np.nan)
    for i in range(n):
        aset = [str(d) for d in np.nonzero(A["answer_mask"][i])[0]]
        gold = str(int(A["gold"][i]))
        ld[i] = logit_diff(A["digit_logits"][i], gold, aset)
        if sp._constrained(A["digit_logits"][i, 0], aset, gold)[1] != bool(A["clean_correct"][i]):
            raise RunError(f"{path}: item {i}: clean_correct contradicts its clean digit logits")
    if not np.allclose(ld, A["logit_diff"], rtol=1e-4, atol=1e-4, equal_nan=True):
        raise RunError(f"{path}: stored logit differences do not follow from the digit logits")
    with np.errstate(divide="ignore", invalid="ignore"):
        frac = np.where(ld[:, :1] != 0, ld / ld[:, :1], np.nan)
    if not np.allclose(frac, A["frac"], rtol=1e-3, atol=1e-4, equal_nan=True):
        raise RunError(f"{path}: stored fractions do not follow from the logit differences")
    return {"path": str(path), "model": meta.get("model"), "suite": meta.get("suite"),
            "family": meta.get("family"), "meta": meta,
            # 1/|answer set| as the suite defines it; from the items if absent
            "chance": meta.get("chance") or 1 / int(A["answer_mask"].sum(1).max()),
            "conditions": conds, "k": A["k"].astype(int),
            "correct": A["clean_correct"].astype(bool), "ld": ld, "frac": frac,
            "sel_k": A["sel_k"].astype(int), "sel_correct": A["sel_correct"].astype(bool)}


def _groups(conds):
    """Condition group -> column indices; the random draws of one set and size
    form one group (an item's drop is its mean over the draws)."""
    g = {}
    for i, c in enumerate(conds):
        if c["kind"] == "clean":
            continue
        key = c["name"].rsplit("/r", 1)[0] if c["kind"] == "rand" else c["name"]
        g.setdefault(key, []).append(i)
    return g


def _strat_idx(k, n_boot, rng):
    """[n_boot, n] indices resampled with replacement within each depth; the
    columns of depth d are cols[d]."""
    k = np.asarray(k)
    idx = np.empty((n_boot, len(k)), int)
    cols, pos = {}, 0
    for d in np.unique(k):
        ix = np.nonzero(k == d)[0]
        idx[:, pos:pos + len(ix)] = ix[rng.integers(0, len(ix), (n_boot, len(ix)))]
        cols[int(d)] = slice(pos, pos + len(ix))
        pos += len(ix)
    return idx, cols


def _row_slopes(Y, K):
    Kc = K - K.mean(-1, keepdims=True)
    ss = (Kc * Kc).sum(-1)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(ss > 0, (Kc * (Y - Y.mean(-1, keepdims=True))).sum(-1) / ss, np.nan)


def _hc2_t(yc, KC, ss):
    """[B] studentised OLS slopes b / SE_HC2(b) of centred y [n] on each row of
    KC [B, n] (centred depths, sum of squares ss).  HC2 weighs each squared
    residual by 1 / (1 - leverage), which is unbiased when the variance
    differs by depth.  An item alone at one of only two depths is fitted
    exactly (leverage 1) and carries no residual.  A zero SE gives +-inf (a
    perfect fit), or 0 when the slope is 0 too."""
    n = KC.shape[-1]
    b = KC @ yc / ss
    e = yc[None, :] - b[:, None] * KC            # the intercept is mean(y): KC is centred
    lev = 1.0 / n + KC * KC / ss
    w = np.where(lev < 1 - 1e-9, 1.0 / np.maximum(1 - lev, 1e-9), 0.0)
    se = np.sqrt((KC * KC * e * e * w).sum(-1)) / ss
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(se > 0, b / np.where(se > 0, se, 1.0),
                        np.where(b != 0, np.copysign(np.inf, b), 0.0))


def slope_test(y, k, n_perm=5000, rng=None):
    """(OLS slope of y on k, two-sided p) that stays valid when the spread of
    y and the number of items change with k: a studentised permutation test,
    |b / SE_HC2(b)| against the same statistic with k shuffled across items;
    p = (count + 1) / (n_perm + 1).  Non-finite items dropped.

    Why not signature_analysis.slope_perm (the raw slope with k shuffled):
    that tests whether the drop's whole distribution depends on k, and E4
    changes its spread with k by construction (fewer clean-correct items at
    high k, and a noisier fraction where the clean margin is smaller), so it
    rejects a zero mean slope far more often than alpha (about 25% at 5% with
    40/28/17/10 items and SD 0.1-0.8).  Studentising makes the permutation
    test asymptotically valid for the slope alone (DiCiccio & Romano 2017)
    and keeps it exact when the drop does not depend on k at all."""
    y, k = np.asarray(y, float), np.asarray(k, float)
    ok = np.isfinite(y) & np.isfinite(k)
    y, k = y[ok], k[ok]
    if len(y) < 3 or np.ptp(k) == 0:
        return float("nan"), float("nan")
    rng = rng if isinstance(rng, np.random.Generator) else np.random.default_rng(rng)
    kc = k - k.mean()
    ss, yc = float(kc @ kc), y - y.mean()
    t0 = abs(float(_hc2_t(yc, kc[None, :], ss)[0]))
    cnt = 0
    for P in sa._perm_rows(kc, n_perm, rng):
        T = np.abs(_hc2_t(yc, P, ss))
        # sa._ge counts ties within rounding; an infinite t (perfect fit) only
        # ties another infinite one
        cnt += int(((T >= t0) | sa._ge(T, t0)).sum())
    return float(kc @ yc / ss), (cnt + 1) / (n_perm + 1)


def _ci(x, level):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if not len(x):
        return [None, None]
    a = (100 - level) / 2
    return [float(np.percentile(x, a)), float(np.percentile(x, 100 - a))]


def _depths(run, min_correct, min_items):
    """(depths analysed, mode, eligible depths, candidate depths).

    Correctness of both halves (clean) decides which depths qualify: the
    eligible ones (outline: >= min_correct correct, accuracy >= 2c/(1+c)) when
    there are at least 3; otherwise, as the outline's fallback for a pair
    below the rules, the depths solved above chance (one-sided binomial
    p < .05, deep_probe's rule), labelled exploratory.  Either way a
    candidate enters only with at least min_items clean-correct MEASUREMENT
    items: the bootstrap resamples items within a depth, so a depth with one
    or two items would enter the slope as if its mean were known exactly.
    The analysed n per depth is reported, and depths with fewer than
    min_correct analysed items are flagged."""
    kk = np.concatenate([run["sel_k"], run["k"]])
    ok = np.concatenate([run["sel_correct"], run["correct"]])
    c = run["chance"]
    n_meas = {int(d): int((run["correct"] & (run["k"] == d)).sum()) for d in np.unique(run["k"])}
    elig = [int(d) for d in sa.eligible_depths(kk, ok, c, min_correct)]
    if len(elig) >= 3:
        cand, mode = elig, "eligible"
    else:
        cand, mode = [int(d) for d in _above_chance(kk, ok, c)], "exploratory"
    need = max(1, int(min_items))
    return [d for d in cand if n_meas.get(d, 0) >= need], mode, elig, cand


def _significant_same_sign(s, p, s_ref, alpha):
    return bool(np.isfinite(p) and p < alpha and np.sign(s) == np.sign(s_ref) and s != 0)


def analyse_run(run, *, n_boot=2000, n_perm=5000, seed=0, min_correct=20, min_items=5,
                alpha=0.05):
    """Per condition group, on clean-correct measurement items at the analysed
    depths: the mean fractional drop (1 - LD/LD_clean) per k and overall with
    95% bootstrap CIs (items resampled within k) and its slope on k (95% CI;
    p from slope_test, a studentised permutation test, because the drop's
    spread and the number of clean-correct items change with k); the same on
    the absolute scale (LD_clean - LD, logits) and the ratio of means (sum of
    drops / sum of clean LD) per k; the sub-additivity ratio top3/(3 top1);
    the excess over the random baseline.  The slope CI is the percentile of
    the stratified bootstrap; with few items at a depth it runs a little
    narrow (about 9% outside at nominal 5% with 10 items and a spread 8x the
    k=1 spread), so decisions key on p.

    Scale check: the fractional drop divides by the clean LD, which falls with
    k (outline: the answer margin falls with k in both 70B models), so a head
    whose absolute effect is constant shows a fractional drop that grows with
    k, which is what specialisation predicts.  A group whose fractional slope
    is significant (p < alpha) while the absolute slope is not significant
    with the same sign is flagged `fractional_only`; the clean LD's own slope
    on k is reported.

    A group's statistics depend only on its own data and the seed: the
    bootstrap indices depend only on the items' depths (and are shared by all
    groups, as the ratios combine groups), and each group draws its
    permutations from a generator of its own, so adding a condition group
    (another set, --per-layer, random draws) never moves another group's p."""
    run = load_ablation(run) if isinstance(run, (str, Path)) else run
    tag = f"run/{run['model']}/{run['suite']}"
    depths, mode, elig, cand = _depths(run, min_correct, min_items)
    k, ok = run["k"], run["correct"]
    use = ok & np.isin(k, depths)
    K = k[use].astype(float)
    drop_all = 1 - run["frac"][use]
    ld = np.asarray(run["ld"], float)[use]
    groups = _groups(run["conditions"])
    n_by_k = {int(d): int((K == d).sum()) for d in depths}
    res = {"model": run["model"], "suite": run["suite"], "path": run["path"],
           "depth_mode": mode, "depths": [int(d) for d in depths],
           "eligible_depths": [int(d) for d in elig],
           "dropped_depths": [int(d) for d in cand if d not in depths],
           "analysed_n": n_by_k,
           "below_min_correct": [d for d, n in n_by_k.items() if n < min_correct],
           "min_items": int(min_items), "min_correct": int(min_correct), "alpha": float(alpha),
           "n_items": int(use.sum()),
           "accuracy": {int(d): [int(ok[k == d].sum()), int((k == d).sum())] for d in np.unique(k)},
           "sets": run["meta"].get("sets"), "skipped": run["meta"].get("skipped"),
           "clean_ld": {}, "groups": {}, "subadditivity": {}, "excess_over_random": {}}
    if use.sum() < 2 or len(depths) < 1:
        res["note"] = "no clean-correct items at the analysed depths"
        return res
    idx, cols = _strat_idx(K, n_boot, _rng(seed, f"{tag}/boot"))
    Kb = K[idx]
    clean = ld[:, 0]
    res["clean_ld"] = {"mean": float(clean.mean()),
                       "by_k": {d: float(clean[K == d].mean()) for d in cols}}
    if len(cols) >= 2:
        s, p = slope_test(clean, K, n_perm, _rng(seed, f"{tag}/perm/clean_ld"))
        res["clean_ld"]["slope"] = {"slope": s, "p": p, "test": SLOPE_TEST,
                                    "ci95": _ci(_row_slopes(clean[idx], Kb), 95)}
    G = {g: drop_all[:, ix].mean(1) for g, ix in groups.items()}
    labs = {g: (clean[:, None] - ld[:, ix]).mean(1) for g, ix in groups.items()}
    for g, y in G.items():
        a = labs[g]
        yb, ab_ = y[idx], a[idx]
        # the median guards against items whose clean LD is near zero, where
        # the fraction is large and noisy; the mean is the preregistered outcome
        ent = {"n_draws": len(groups[g]), "mean": float(np.mean(y)),
               "ci95": _ci(yb.mean(1), 95), "median": float(np.median(y)),
               "abs_drop": float(np.mean(a)), "abs_ci95": _ci(ab_.mean(1), 95),
               "ratio_of_means": float(a.sum() / clean.sum()), "by_k": {}}
        for d, sl in cols.items():
            m = K == d
            ent["by_k"][d] = {"n": int(m.sum()), "mean": float(y[m].mean()),
                              "ci95": _ci(yb[:, sl].mean(1), 95),
                              "median": float(np.median(y[m])),
                              "abs_drop": float(a[m].mean()),
                              "ratio_of_means": float(a[m].sum() / clean[m].sum())}
        if len(cols) >= 2:
            s, p = slope_test(y, K, n_perm, _rng(seed, f"{tag}/perm/{g}"))
            ent["slope"] = {"slope": s, "p": p, "test": SLOPE_TEST,
                            "ci95": _ci(_row_slopes(yb, Kb), 95)}
            s_a, p_a = slope_test(a, K, n_perm, _rng(seed, f"{tag}/perm_abs/{g}"))
            ent["abs_slope"] = {"slope": s_a, "p": p_a, "test": SLOPE_TEST,
                                "ci95": _ci(_row_slopes(ab_, Kb), 95)}
            ent["fractional_only"] = bool(np.isfinite(p) and p < alpha and
                                          not _significant_same_sign(s_a, p_a, s, alpha))
        if len(groups[g]) > 1:
            per_draw = drop_all[:, groups[g]].mean(0)
            ent["draw_means"] = [float(v) for v in per_draw]
        res["groups"][g] = ent
    for s in SETS:
        for m in MODES:
            g1, g3 = f"{s}/top1/{m}", f"{s}/top3/{m}"
            if g1 in G and g3 in G:
                r = _ratio(G[g3], G[g1], idx)
                r["by_k"] = {d: _ratio(G[g3][K == d], G[g1][K == d], None)["ratio"]
                             for d in cols}
                res["subadditivity"][f"{s}/{m}"] = r
        for n in TOP_SIZES:
            gt, gr = f"{s}/top{n}/mean", f"{s}/rand{n}/mean"
            if gt in G and gr in G:
                dlt = G[gt] - G[gr]
                res["excess_over_random"][f"{s}/{n}"] = {
                    "mean": float(dlt.mean()), "ci95": _ci(dlt[idx].mean(1), 95)}
    return res


def _ratio(y3, y1, idx):
    """top3 / (3 top1) on mean drops; undefined unless the top-1 drop is > 0."""
    def f(a, b):
        return a / (3 * b) if b > 0 else np.nan
    out = {"ratio": _nan_none(f(y3.mean(), y1.mean())), "top3": float(y3.mean()),
           "top1": float(y1.mean())}
    if idx is not None:
        m3, m1 = y3[idx].mean(1), y1[idx].mean(1)
        with np.errstate(divide="ignore", invalid="ignore"):
            out["ci95"] = _ci(np.where(m1 > 0, m3 / (3 * m1), np.nan), 95)
    if not (y1.mean() > 0):
        out["note"] = "top-1 drop not positive; ratio undefined"
    return out


def _nan_none(x):
    return None if x is None or not np.isfinite(x) else float(x)


def pair_reason(run_a, run_b):
    """None when two ablation outputs form the outline's pair (two models on
    one suite); otherwise why they are not compared.  One model on two
    suites, or two models on different suites, is not the ablation x k x
    model interaction and gets no verdict."""
    if run_a["model"] == run_b["model"]:
        return f"same model ({run_a['model']}); the interaction compares two models"
    if run_a["suite"] != run_b["suite"]:
        return (f"different suites ({run_a['suite']} / {run_b['suite']}); the interaction "
                f"compares two models on one suite")
    ha, hb = run_a["meta"].get("suite_sha256"), run_b["meta"].get("suite_sha256")
    if ha and hb and ha != hb:
        return (f"suite {run_a['suite']!r} has different contents in the two files "
                f"(suite_sha256 differs)")
    return None


def _same_items(run_a, run_b):
    """Whether both files measured the same items (None when a file does not
    record them)."""
    ids = []
    for r in (run_a, run_b):
        recs = r["meta"].get("measurement_items")
        if not recs:
            return None
        ids.append([t.get("id") for t in recs])
    return ids[0] == ids[1]


def design_mismatch(run_a, run_b):
    """{setting: [a, b]} for the PAIR_DESIGN_KEYS both files record with
    different values (another --window selects other target layers; another
    dtype or kernel rounds differently)."""
    ma, mb = run_a["meta"], run_b["meta"]
    return {k: [ma[k], mb[k]] for k in PAIR_DESIGN_KEYS
            if k in ma and k in mb and ma[k] != mb[k]}


def _group_info(conds, groups):
    """Group name -> (set, kind, rest of the name after the set), e.g.
    'spread/top1/mean' -> ('spread', 'top', 'top1/mean')."""
    out = {}
    for g, ix in groups.items():
        c = conds[ix[0]]
        s, _, rest = g.partition("/")
        out[g] = (c.get("set") or s, c["kind"], rest)
    return out


def pair_groups(run_a, run_b):
    """(primary, cross, not compared) pairings of the two files' condition
    groups, each pairing (key, group in a, group in b).

    primary: every group both files hold (same name, so the same layer
    class); and, for a condition (top / random baseline, size and mode) the
    files hold only for different layer classes, each model's own target
    layers: key 'SETa-vs-SETb/top1/mean' (e.g. Llama's spreading against Qwen's
    funnelling layers, which the pair's classes may well be).  cross: the
    other pairings of different classes of one condition (both files hold
    that condition for a common class too), reported apart.  Per-layer
    conditions are paired by name only.  not compared: groups in no
    pairing."""
    ga, gb = _groups(run_a["conditions"]), _groups(run_b["conditions"])
    ia, ib = _group_info(run_a["conditions"], ga), _group_info(run_b["conditions"], gb)
    primary = [(g, g, g) for g in sorted(set(ga) & set(gb))]
    cross = []
    rests = sorted({(kd, r) for _, kd, r in list(ia.values()) + list(ib.values())
                    if kd in CROSS_KINDS})
    for kd, rest in rests:
        sa_ = sorted(s for g, (s, k2, r) in ia.items() if (k2, r) == (kd, rest))
        sb_ = sorted(s for g, (s, k2, r) in ib.items() if (k2, r) == (kd, rest))
        pairs = [(s1, s2) for s1 in sa_ for s2 in sb_ if s1 != s2]
        dest = cross if set(sa_) & set(sb_) else primary
        dest += [(f"{s1}-vs-{s2}/{rest}", f"{s1}/{rest}", f"{s2}/{rest}") for s1, s2 in pairs]
    used_a = {x for _, x, _ in primary + cross}
    used_b = {y for _, _, y in primary + cross}
    rest_ = sorted((set(ga) - used_a) | (set(gb) - used_b))
    return primary, cross, rest_


def interaction(run_a, run_b, *, n_boot=2000, seed=0, equiv=EQUIV, min_correct=20,
                min_items=5):
    """Ablation x k x model, per pairing of the two runs' condition groups
    (pair_groups): the difference of the two models' slopes of fractional
    drop on k over their common analysed depths, scaled to that span (the
    change of the drop from the lowest to the highest common depth, in units
    of clean LD), with a bootstrap 90% CI (items resampled within k, each
    model separately).  Verdict (outline): 'null' only when the 90% CI lies
    within +/-equiv; 'difference' when it excludes 0; otherwise
    'inconclusive'.

    Pairings: `groups` holds the groups both files have (same layer class)
    and, where the pair's layer classes differ (each model's classes come
    from its own selection half; Llama spreading and Qwen funnelling is what
    E4 tests), each model's own target layers under 'SETa-vs-SETb/...';
    `cross_groups` the other cross-class pairings; every entry records the
    set each side used.  With no pairing at all, a note says why there is no
    verdict.

    Only two different models on one suite are compared (pair_reason); any
    other combination is returned with compared=False and a note.  Files
    that measured other items, or record another design (PAIR_DESIGN_KEYS:
    window, dtype, kernel, ...), get a warning.  The same interaction on the
    absolute scale (LD_clean - LD, logits; no margin, as the outline's is in
    units of clean LD) is reported beside it, and a 'difference' whose
    absolute CI does not exclude 0 on the same side is flagged
    fractional_only (analyse_run explains why).  The bootstrap indices
    depend only on the two runs' analysed depths and the seed, so a group's
    CI does not depend on which other groups the files hold."""
    ra = load_ablation(run_a) if isinstance(run_a, (str, Path)) else run_a
    rb = load_ablation(run_b) if isinstance(run_b, (str, Path)) else run_b
    out = {"a": ra["model"], "b": rb["model"], "suite_a": ra["suite"], "suite_b": rb["suite"],
           "compared": True, "depths": [], "depth_mode": None, "equivalence_margin": equiv,
           "min_correct": int(min_correct), "sets_a": ra["meta"].get("sets"),
           "sets_b": rb["meta"].get("sets"), "groups": {}, "cross_groups": {},
           "not_compared": []}
    why = pair_reason(ra, rb)
    if why:
        out.update(compared=False, note=f"not compared: {why}")
        return out
    same = _same_items(ra, rb)
    out["same_items"] = same
    warn = []
    if same is False:
        warn.append("the two files measured different items (another --seed, --split or "
                    "--max-items-per-k): the interaction compares independent samples of the "
                    "suite, not the same items")
    mism = design_mismatch(ra, rb)
    out["design_mismatch"] = mism
    if mism:
        warn.append("the two files were run with different designs (" + "; ".join(
            f"{k}: {v[0]!r} vs {v[1]!r}" for k, v in mism.items()) + "): E4's design is "
            "fixed before running and applies to both models")
    if warn:
        out["warnings"], out["warning"] = warn, "; ".join(warn)
    da, ma, _, _ = _depths(ra, min_correct, min_items)
    db, mb, _, _ = _depths(rb, min_correct, min_items)
    common = sorted(set(da) & set(db))
    out.update(depths=common, depth_mode="eligible" if ma == mb == "eligible" else "exploratory")
    if len(common) < 2:
        out["note"] = f"fewer than 2 common analysed depths ({common})"
        return out
    span = float(max(common) - min(common))
    ga, gb = _groups(ra["conditions"]), _groups(rb["conditions"])
    primary, cross, out["not_compared"] = pair_groups(ra, rb)
    if not primary:
        out["note"] = (f"no condition group common to both files, nor a condition both hold "
                       f"for different layer classes (sets {ra['model']}: "
                       f"{ra['meta'].get('sets')}, {rb['model']}: {rb['meta'].get('sets')}); "
                       f"no interaction verdict")
        return out
    ua = ra["correct"] & np.isin(ra["k"], common)
    ub = rb["correct"] & np.isin(rb["k"], common)
    ka, kb = ra["k"][ua].astype(float), rb["k"][ub].astype(float)
    out["n_by_k"] = {"a": {d: int((ka == d).sum()) for d in common},
                     "b": {d: int((kb == d).sum()) for d in common}}
    out["below_min_correct"] = [d for d in common
                                if min(out["n_by_k"]["a"][d], out["n_by_k"]["b"][d]) < min_correct]
    if len(np.unique(ka)) < 2 or len(np.unique(kb)) < 2:
        out["note"] = "a model has correct items at fewer than two common depths"
        return out
    tag = f"pair/{ra['model']}/{ra['suite']}/{rb['model']}/{rb['suite']}"
    ia, _ = _strat_idx(ka, n_boot, _rng(seed, f"{tag}/boot/a"))
    ib, _ = _strat_idx(kb, n_boot, _rng(seed, f"{tag}/boot/b"))
    lda, ldb = np.asarray(ra["ld"], float)[ua], np.asarray(rb["ld"], float)[ub]
    fa, fb = ra["frac"][ua], rb["frac"][ub]
    for dest, pairs in ((out["groups"], primary), (out["cross_groups"], cross)):
        for key, g_a, g_b in pairs:
            ca, cb = ga[g_a], gb[g_b]
            e = _pair_entry(fa[:, ca], fb[:, cb], lda[:, :1] - lda[:, ca],
                            ldb[:, :1] - ldb[:, cb], ka, kb, ia, ib, span, equiv)
            e.update(group_a=g_a, group_b=g_b, set_a=ra["conditions"][ca[0]].get("set"),
                     set_b=rb["conditions"][cb[0]].get("set"), cross_set=g_a != g_b)
            dest[key] = e
    return out


def _excludes_zero(ci):
    return ci[0] is not None and (ci[0] > 0 or ci[1] < 0)


def _pair_entry(fa, fb, da, db, ka, kb, ia, ib, span, equiv):
    """One pairing's interaction (interaction() documents the numbers).  fa /
    fb: [n, c] fractions LD/LD_clean of each model's columns of its group
    (random draws pooled per item); da / db: the matching absolute drops
    LD_clean - LD; ka / kb depths; ia / ib the stratified bootstrap indices."""
    Ka, Kb = ka[ia], kb[ib]
    ya, yb = (1 - fa).mean(1), (1 - fb).mean(1)
    sa_, sb_ = _row_slopes(ya, ka), _row_slopes(yb, kb)
    bsa, bsb = _row_slopes(ya[ia], Ka), _row_slopes(yb[ib], Kb)
    est = (sa_ - sb_) * span
    ci = _ci((bsa - bsb) * span, 90)
    main = float(ya.mean() - yb.mean())
    main_ci = _ci(ya[ia].mean(1) - yb[ib].mean(1), 90)
    within = ci[0] is not None and ci[0] >= -equiv and ci[1] <= equiv
    excl = _excludes_zero(ci)
    verdict = "null" if within else "difference" if excl else "inconclusive"
    # absolute scale (logits): LD_clean - LD, averaged over the group's columns
    xa, xb = da.mean(1), db.mean(1)
    abs_est = (_row_slopes(xa, ka) - _row_slopes(xb, kb)) * span
    abs_ci = _ci((_row_slopes(xa[ia], Ka) - _row_slopes(xb[ib], Kb)) * span, 90)
    abs_excl = _excludes_zero(abs_ci)
    abs_agrees = abs_excl and excl and (abs_ci[0] > 0) == (ci[0] > 0)
    return {
        "slope_a": float(sa_), "slope_b": float(sb_), "n_a": int(len(ya)), "n_b": int(len(yb)),
        "interaction_per_step": float(sa_ - sb_), "interaction": float(est), "ci90": ci,
        "within_margin": bool(within), "excludes_zero": bool(excl), "verdict": verdict,
        "main_effect": main, "main_ci90": main_ci,
        "abs_slope_a": float(_row_slopes(xa, ka)), "abs_slope_b": float(_row_slopes(xb, kb)),
        "abs_interaction": float(abs_est), "abs_ci90": abs_ci,
        "abs_excludes_zero": bool(abs_excl),
        "fractional_only": bool(verdict == "difference" and not abs_agrees)}


def _fmt(x, f="+.3f"):
    return "–" if x is None or (isinstance(x, float) and not np.isfinite(x)) else format(x, f)


def _fci(ci, f="+.3f"):
    return "–" if not ci or ci[0] is None else f"[{format(ci[0], f)}, {format(ci[1], f)}]"


def report_md(res):
    L = ["# E4 ablation report", "",
         "Drop = 1 − LD/LD_clean (LD: gold logit minus the mean competitor logit), on "
         "clean-correct measurement items; 0 = no effect, 1 = the clean logit difference "
         "removed.  Absolute drop = LD_clean − LD, in logits.  CIs: bootstrap over items "
         "within k (95%; interactions 90%).  Slope p: studentised permutation test (slope / "
         "HC2 standard error, k shuffled across items), valid when the drop's spread and the "
         "number of correct items change with k.  † = fractional_only: the fractional slope "
         "or interaction is significant but the absolute one is not, on the same side; the "
         "clean LD changes with k, so the fraction's denominator may drive it.", ""]
    for r in res["runs"]:
        an = r.get("analysed_n") or {}
        L += [f"## {r['model']} — {r['suite']}", "",
              f"- depths analysed: {r['depths']} ({r['depth_mode']}; eligible "
              f"{r['eligible_depths']}), {r['n_items']} items",
              "- clean accuracy (measurement half) by k: " + ", ".join(
                  f"k{d} {a[0]}/{a[1]}" for d, a in
                  sorted(r["accuracy"].items(), key=lambda kv: int(kv[0]))),
              f"- sets: {r['sets']}"]
        if an:
            L.append("- analysed clean-correct items by k: " + ", ".join(
                f"k{d} {n}" for d, n in sorted(an.items(), key=lambda kv: int(kv[0]))))
        if r.get("below_min_correct"):
            L.append(f"- depths with fewer than {r.get('min_correct')} analysed items (the "
                     f"outline's rule counts both halves): {r['below_min_correct']}")
        if r.get("dropped_depths"):
            L.append(f"- qualifying depths left out, fewer than {r.get('min_items')} analysed "
                     f"items: {r['dropped_depths']}")
        cl = r.get("clean_ld") or {}
        if cl.get("by_k"):
            sl = cl.get("slope") or {}
            L.append("- clean LD by k: " + ", ".join(
                f"k{d} {_fmt(v, '.3f')}" for d, v in
                sorted(cl["by_k"].items(), key=lambda kv: int(kv[0])))
                + (f"; slope {_fmt(sl.get('slope'))} {_fci(sl.get('ci95'))}, "
                   f"p {_fmt(sl.get('p'), '.4f')}" if sl else ""))
        for sk in r.get("skipped") or []:
            L.append(f"- skipped {sk['condition']}: {sk['reason']}")
        if r.get("note"):
            L += ["", f"_{r['note']}_", ""]
            continue
        ds = r["depths"]
        L += ["", "| condition | mean drop [95% CI] | median | slope on k [95% CI] | p | "
              "abs slope (logits) [95% CI] | p (abs) | "
              + " | ".join(f"k={d}" for d in ds) + " |",
              "|---|---|---|---|---|---|---|" + "---|" * len(ds)]
        for g, e in r["groups"].items():
            sl, sa_ = e.get("slope") or {}, e.get("abs_slope") or {}
            L.append(f"| {g}{' (' + str(e['n_draws']) + ' draws)' if e['n_draws'] > 1 else ''} "
                     f"| {_fmt(e['mean'])} {_fci(e['ci95'])} | {_fmt(e['median'])} "
                     f"| {_fmt(sl.get('slope'))} "
                     f"{_fci(sl.get('ci95'))}{' †' if e.get('fractional_only') else ''} "
                     f"| {_fmt(sl.get('p'), '.4f')} "
                     f"| {_fmt(sa_.get('slope'))} {_fci(sa_.get('ci95'))} "
                     f"| {_fmt(sa_.get('p'), '.4f')} | "
                     + " | ".join(_fmt((e["by_k"].get(str(d)) or {}).get("mean")) for d in ds)
                     + " |")
        if r["subadditivity"]:
            L += ["", "Sub-additivity top3 / (3 × top1) (outline: consensus predicts a top-3 drop well "
                  "below three times the top-1 drop, not growing with k):", "",
                  "| set/mode | ratio [95% CI] | top1 | top3 |", "|---|---|---|---|"]
            for key, e in r["subadditivity"].items():
                L.append(f"| {key} | {_fmt(e['ratio'], '.3f')} {_fci(e.get('ci95'), '.3f')} | "
                         f"{_fmt(e['top1'])} | {_fmt(e['top3'])} |")
        if r["excess_over_random"]:
            L += ["", "Top heads minus the random mid-rank baseline (mean-ablation):", "",
                  "| set/size | excess drop [95% CI] |", "|---|---|"]
            for key, e in r["excess_over_random"].items():
                L.append(f"| {key} | {_fmt(e['mean'])} {_fci(e['ci95'])} |")
        L.append("")
    skipped_pairs = [p for p in res["pairs"] if p.get("compared") is False]
    for p in res["pairs"]:
        if p.get("compared") is False:
            continue
        L += [f"## Interaction: {p['a']} ({p['suite_a']}) minus {p['b']} ({p['suite_b']})", "",
              f"Common depths {p['depths']} ({p['depth_mode']}).  Interaction = difference of "
              f"the slopes of the drop on k, times the depth span: the change of the drop over "
              f"the common depths in units of clean LD.  Null claimed only if the 90% CI lies "
              f"within ±{p['equivalence_margin']}.  The absolute interaction (logits) has no "
              f"margin.", ""]
        for w in p.get("warnings") or ([p["warning"]] if p.get("warning") else []):
            L += [f"**Warning:** {w}.", ""]
        L += [f"Layer classes (window): {p['a']} {p.get('sets_a')}; "
              f"{p['b']} {p.get('sets_b')}", ""]
        if p.get("n_by_k"):
            L += ["Analysed items by k: " + "; ".join(
                f"{who} " + ", ".join(f"k{d} {n}" for d, n in
                                      sorted(p["n_by_k"][ab].items(), key=lambda kv: int(kv[0])))
                for who, ab in ((p["a"], "a"), (p["b"], "b"))), ""]
        if p.get("below_min_correct"):
            L += [f"Common depths with fewer than {p.get('min_correct')} analysed items in a "
                  f"model: {p['below_min_correct']}", ""]
        if p.get("note"):
            L += [f"_{p['note']}_", ""]
            continue
        if any(e.get("cross_set") for e in p["groups"].values()):
            L += [f"Rows named SETa-vs-SETb pair a condition the two files hold only for "
                  f"different layer classes: {p['a']}'s SETa layers against {p['b']}'s SETb "
                  f"layers (each model's own target layers).", ""]
        L += _pair_table(p["groups"])
        if p.get("cross_groups"):
            L += ["", "Cross-class pairings of conditions both files also hold for a common "
                      "class (secondary; SETa is " + str(p["a"]) + "'s layers, SETb "
                      + str(p["b"]) + "'s):", ""]
            L += _pair_table(p["cross_groups"])
        if p.get("not_compared"):
            L += ["", f"Condition groups in no pairing (not compared): "
                      f"{', '.join(p['not_compared'])}"]
        L.append("")
    if skipped_pairs:
        L += ["## Files not compared", "",
              "The interaction is defined for two models on one suite; no verdict for:", ""]
        for p in skipped_pairs:
            L.append(f"- {p['a']} ({p['suite_a']}) and {p['b']} ({p['suite_b']}): "
                     f"{p['note'].removeprefix('not compared: ')}")
        L.append("")
    return "\n".join(L) + "\n"


def _pair_table(groups):
    L = ["| condition | slope a | slope b | interaction [90% CI] | verdict | "
         "abs interaction (logits) [90% CI] | a − b mean drop [90% CI] |",
         "|---|---|---|---|---|---|---|"]
    for g, e in groups.items():
        # the outline's rule: within the margin is a null even when the CI
        # excludes 0 (equivalent, not identical); say so
        v = e["verdict"] + (" (CI excludes 0)" if e["verdict"] == "null"
                            and e["excludes_zero"] else "")
        v += " †" if e.get("fractional_only") else ""
        L.append(f"| {g} | {_fmt(e['slope_a'])} | {_fmt(e['slope_b'])} | "
                 f"{_fmt(e['interaction'])} {_fci(e['ci90'])} | {v} | "
                 f"{_fmt(e.get('abs_interaction'))} {_fci(e.get('abs_ci90'))} | "
                 f"{_fmt(e['main_effect'])} {_fci(e['main_ci90'])} |")
    return L


def _plain(x):
    if isinstance(x, dict):
        return {str(k): _plain(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_plain(v) for v in x]
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating, float)):
        return None if not np.isfinite(x) else float(x)
    if isinstance(x, np.bool_):
        return bool(x)
    if isinstance(x, np.ndarray):
        return _plain(x.tolist())
    return x


def expand_paths(paths, log=print):
    """--analyse arguments as files.  A glob pattern is expanded here (Drive
    folders hold spaces, so the notebook quotes the pattern and the shell
    leaves it), keeping ablation outputs (*__ablation.npz) only, so 'DIR/*.npz'
    over a folder that also holds sweep captures works; '**' matches any number
    of folders, as in signature_analysis; a pattern matching none aborts.
    Plain paths are kept as given (load_ablation checks them).
    Duplicates are dropped, first occurrence kept."""
    out = []
    for p in (str(x) for x in paths):
        if any(c in p for c in "*?[") and not Path(p).exists():
            # recursive, as in signature_analysis: one Drive pattern such as
            # 'runs/**/*.npz' must reach the same files in both analyses
            hits = sorted(glob.glob(p, recursive=True))
            keep = [h for h in hits if h.endswith(SUFFIX)]
            if not keep:
                _abort(f"--analyse {p!r}: no ablation output (*{SUFFIX}) matches "
                       f"({len(hits)} other file(s))")
            if len(keep) < len(hits):
                log(f"  {p}: {len(hits) - len(keep)} match(es) not ending in {SUFFIX} left out")
            out += keep
        else:
            out.append(p)
    return list(dict.fromkeys(out))


def analyse(paths, out_dir=None, *, n_boot=2000, n_perm=5000, seed=0, equiv=EQUIV,
            min_correct=20, min_items=5, log=print):
    """Analyse ablation outputs (paths or glob patterns, expand_paths).  Every
    file gets analyse_run; every two files get an entry in `pairs`, but only
    two different models on the same suite (pair_reason) get the interaction
    and a verdict: one model on two suites or a cross-suite combination is
    listed as not compared.  Writes ablation_report.md / .json to out_dir
    (default: the first file's folder) and returns the report dict."""
    paths = expand_paths(paths, log)
    runs = [load_ablation(p) for p in paths]
    kw = dict(n_boot=n_boot, seed=seed, min_correct=min_correct, min_items=min_items)
    res = {"runs": [analyse_run(r, n_perm=n_perm, **kw) for r in runs], "pairs": []}
    for i in range(len(runs)):
        for j in range(i + 1, len(runs)):
            x = interaction(runs[i], runs[j], equiv=equiv, **kw)
            if not x["compared"]:
                log(f"  {x['a']} ({x['suite_a']}) and {x['b']} ({x['suite_b']}): {x['note']}")
            res["pairs"].append(x)
    res = _plain(res)
    out = Path(out_dir) if out_dir else Path(paths[0]).parent
    out.mkdir(parents=True, exist_ok=True)
    (out / "ablation_report.json").write_text(json.dumps(res, indent=1))
    (out / "ablation_report.md").write_text(report_md(res))
    log(f"  -> {out / 'ablation_report.md'}")
    return res


# ─────────────────────────────── CLI ────────────────────────────────────────
def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0],
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model")
    ap.add_argument("--suite", help="suite JSON (serial_tasks.save_suite)")
    ap.add_argument("--out", help="output directory (with --analyse: the report's)")
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--attn-impl", default="eager", choices=["eager", "sdpa"])
    ap.add_argument("--precision", default="bf16", choices=["bf16", "fp32"],
                    help="fp32 is supported on CPU only (load_model)")
    ap.add_argument("--window", nargs=2, type=float, default=[0.4, 0.8], metavar=("LO", "HI"),
                    help="target layers floor(LO(L-1))..floor(HI(L-1))")
    ap.add_argument("--split", default="test", choices=list(sp.SPLITS))
    ap.add_argument("--max-items-per-k", type=int, default=None)
    ap.add_argument("--random-draws", type=int, default=5)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--per-layer", action="store_true",
                    help="also top-1 mean-ablation one layer at a time")
    ap.add_argument("--n-perm", type=int, default=5000,
                    help="permutations for the layer classification (and --analyse slopes)")
    ap.add_argument("--q", type=float, default=0.05, help="BH level of the layer classes")
    ap.add_argument("--batch-conditions", type=int, default=None,
                    help=f"conditions per forward (default {BATCH_CUDA} on cuda, 1 otherwise). "
                         f"Forwards per measured item: 1 prefix + ceil(C/B) conditions + 1 "
                         f"repeat of the first chunk (+1 on cache-check items, +2 on the "
                         f"first); C is about 46 for the 70B pair with --per-layer, so B=1 is "
                         f"about 10x slower than B={BATCH_CUDA} under CPU offload")
    ap.add_argument("--checkpoint-every", type=int, default=5)
    ap.add_argument("--analyse", nargs="+", metavar="NPZ",
                    help="ablation outputs; quoted glob patterns are expanded here")
    ap.add_argument("--n-boot", type=int, default=2000)
    ap.add_argument("--equiv", type=float, default=EQUIV)
    ap.add_argument("--min-correct", type=int, default=20)
    ap.add_argument("--min-items", type=int, default=5)
    a = ap.parse_args(argv)
    if a.analyse:
        if a.model or a.suite:
            ap.error("--analyse takes ablation outputs only")
        try:
            return analyse(a.analyse, a.out, n_boot=a.n_boot, n_perm=a.n_perm, seed=a.seed,
                           equiv=a.equiv, min_correct=a.min_correct, min_items=a.min_items)
        except RunError as e:
            _abort(f"--analyse: {e}")
    if not (a.model and a.suite and a.out):
        ap.error("--model, --suite and --out are required (or --analyse)")
    if not 0 <= a.window[0] < a.window[1] <= 1:
        ap.error("--window needs 0 <= LO < HI <= 1")
    if a.random_draws < 0 or (a.batch_conditions is not None and a.batch_conditions < 1) \
            or a.n_perm < 1:
        ap.error("--random-draws >= 0, --batch-conditions >= 1, --n-perm >= 1")
    if a.batch_conditions is None:
        a.batch_conditions = default_batch(a.device)
    if a.max_items_per_k is not None and a.max_items_per_k < 2:
        ap.error("--max-items-per-k must be >= 2")
    return run(a)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile head_alignment_probe.py
#!/usr/bin/env python3
"""
Head alignment and direct logit attribution at the answer position.

Separates two readings of the dispersion contrast (Llama spreads, Qwen funnels):

  consensus       heads that gain share as D rises write in the same direction
                  as the layer's leading head and push toward the answer, so
                  their writes add constructively
  specialisation  one head carries the answer; the others write elsewhere and
                  the leader's share of answer support grows with D

Norms alone cannot tell these apart: evenly spread norms are equally
consistent with heads writing one signal and with heads writing unrelated
ones.  This records directions.  Per item, at the measured position (the one
whose next token is the answer digit; deep_probe.align_answer), per head:

  contribution  ||z_h W_O^h||, exactly as deep_probe records it
  write         the head's write to the residual stream: z_h W_O^h, passed
                through the post-attention norm at its frozen scale on
                architectures that normalise the attention output before
                adding it (OLMo-2, Gemma-2)
  cos_top       cosine of the write with the layer's largest write
  cos_sum       cosine of the write with the layer's summed write
  dla           the write through the final norm at its frozen scale,
                projected on each digit's unembedding row          [L, H, 10]

and per layer sum_norm = ||sum of writes||, so coherence = sum_norm / sum of
write norms: 1 when every head writes one direction, ~1/sqrt(n) when unrelated.

Every linearisation is checked against the model on each forward and the run
aborts rather than emit numbers when one fails: the per-head split must
reproduce o_proj's output, the frozen post-attention norm its norm's output,
and the frozen final norm plus unembedding the model's digit logits.

    python3 head_alignment_probe.py --model Qwen/Qwen2.5-0.5B --suite serial_depth_fs \
        --out probe/align
    python3 head_alignment_probe.py --analyse probe/align/<model>_align.npz \
        [--band 0.30 0.94] [--plot align.png]
"""
import argparse, json, os, re
from pathlib import Path
import numpy as np

from deep_probe import align_answer, _above_chance

DIGITS = "0123456789"
# max |error| / max |reference|.  bf16 rounding reaches ~2e-2 on GPT-2's large
# logits; a wrong linearisation (missing centring, (1+w) vs w) is O(1).
WARN, FAIL = 2e-2, 1e-1


# ────────────────────────────── linear pieces ───────────────────────────────
def _rel_err(pred, ref):
    return float((pred - ref).abs().max() / ref.abs().max().clamp_min(1e-9))


def fit_norm(x, y, w, b, eps, family=None):
    """Identify a norm's form from one input/output pair and return it frozen.

    Tries RMSNorm (x/rms * w), Gemma's RMSNorm (x/rms * (1+w)) and LayerNorm
    ((x-mean)/std * w + b), keeping whichever reproduces y.  Returns
    (kind, scale[d], center, bias, err): the frozen map is
    v -> (v - mean(v) if center else v) * scale, with bias added only to the
    whole output (it belongs to no head).
    """
    import torch
    x, y = x.float(), y.float()
    w = torch.ones_like(x) if w is None else w.float()
    rms = torch.sqrt((x * x).mean() + eps)
    mu = x.mean(); sd = torch.sqrt(((x - mu) ** 2).mean() + eps)
    bb = torch.zeros_like(x) if b is None else b.float()
    cands = {"rms": (w / rms, False, None), "rms1": ((1 + w) / rms, False, None),
             "ln": (w / sd, True, bb)}
    # When the residual's mean is near zero LayerNorm and RMSNorm fit one
    # input equally well, and bf16 rounding can pick the wrong one; the total
    # logit still matches but per-head attributions do not.  Use the module's
    # class to decide the family where it says.
    if family == "rms":
        cands.pop("ln")
    elif family == "rms1":
        cands = {"rms1": cands["rms1"]}
    elif family == "ln":
        cands = {"ln": cands["ln"]}
    best = None
    for kind, (g, center, bias) in cands.items():
        pred = ((x - mu) if center else x) * g + (0 if bias is None else bias)
        e = _rel_err(pred, y)
        if best is None or e < best[4]:
            best = (kind, g, center, bias, e)
    return best


def norm_family(module):
    """The norm's form from its class, for fit_norm.  Gemma's RMSNorm scales by
    (1 + w): where |w| is large, w and 1 + w fit one input equally well in bf16,
    and a per-item fit can flip between them (seen on Gemma-2-27B, layer 42), so
    the form is taken from the architecture rather than fitted."""
    cls = type(module).__name__.lower()
    if "rms" in cls:
        return "rms1" if "gemma" in cls else "rms"
    return "ln" if "layernorm" in cls else None


def _eps(m):
    for n in ("variance_epsilon", "eps", "epsilon"):
        v = getattr(m, n, None)
        if isinstance(v, (int, float)):
            return float(v)
    return 1e-6


def _find_final_norm(hf):
    for base in ("model", "transformer", "gpt_neox"):
        b = getattr(hf, base, None)
        for n in ("norm", "ln_f", "final_layer_norm", "final_layernorm", "final_norm"):
            m = getattr(b, n, None) if b is not None else None
            if m is not None and hasattr(m, "weight"):
                return m, f"{base}.{n}"
    raise SystemExit("final norm not found; unsupported architecture")


def _find_proj(attn):
    for n in ("o_proj", "dense", "c_proj", "out_proj", "wo"):
        p = getattr(attn, n, None)
        if p is not None and hasattr(p, "weight"):
            return p, n
    return None, None


def _patch(module, fn_factory, patched):
    at = "_old_forward" if hasattr(module, "_old_forward") else "forward"
    orig = getattr(module, at)
    setattr(module, at, fn_factory(orig)); patched.append((module, at, orig))


# ────────────────────────────── capture ─────────────────────────────────────
def capture(hf, tok, layers, get_attn, H, dh, tasks, labels, golds, log=print):
    """Run every task once and return per-item arrays.  Model-agnostic so the
    tests can drive it with small random models."""
    import torch
    L = len(layers)
    st = {"vec": {}, "post": {}, "final": None, "unemb": None}
    layout = {}                                   # layer -> ("linear"|"conv1d", err)
    ANS_IDS = []                                  # digit token ids, set per item

    def per_head_last(x, W, mode):
        Wm = W if mode == "linear" else W.t()
        Wr = Wm.reshape(Wm.shape[0], H, dh).float()
        xh = x[0, -1].reshape(H, dh).float()
        return torch.einsum("hd,ohd->ho", xh, Wr)             # [H, d_model]

    def proj_hook(l, module):
        def factory(orig):
            def f(*a, **kw):
                x = a[0] if a else next(iter(kw.values()))
                out = orig(*a, **kw)
                W = module.weight
                if x.dim() == 3 and W.device.type != "meta":
                    ref = out[0, -1].float()
                    bias = getattr(module, "bias", None)
                    bias = None if bias is None else bias.float()
                    if l not in layout:
                        best = None
                        for mode in ("linear", "conv1d"):
                            try:
                                rec = per_head_last(x, W, mode).sum(0)
                                e = _rel_err(rec + (0 if bias is None else bias), ref)
                            except Exception:
                                e = float("inf")
                            if best is None or e < best[1]:
                                best = (mode, e)
                        if best[1] > FAIL:
                            raise RuntimeError(
                                f"layer {l}: per-head split does not reproduce "
                                f"{type(module).__name__} (error {best[1]:.2e}); refusing")
                        layout[l] = best
                    v = per_head_last(x, W, layout[l][0])
                    st["vec"][l] = (v.cpu(), None if bias is None else bias.cpu(),
                                    ref.cpu())
                return out
            return f
        return factory

    def norm_hook(key, module):
        """Freeze a norm at the measured position.  key is a layer index for a
        post-attention norm, or "final"."""
        def factory(orig):
            def f(*a, **kw):
                x = a[0] if a else next(iter(kw.values()))
                out = orig(*a, **kw)
                if x.dim() == 3 and module.weight.device.type != "meta":
                    fam = norm_family(module)
                    fit = fit_norm(x[0, -1], out[0, -1], module.weight,
                                   getattr(module, "bias", None), _eps(module), fam)
                    rec = (fit[0], fit[1].cpu(), fit[2],
                           None if fit[3] is None else fit[3].cpu(), fit[4],
                           x[0, -1].float().cpu(), out[0, -1].float().cpu())
                    if key == "final":
                        st["final"] = rec
                    else:
                        st["post"][key] = rec
                return out
            return f
        return factory

    def unemb_hook(module):
        def factory(orig):
            def f(*a, **kw):
                out = orig(*a, **kw)
                if module.weight.device.type != "meta" and ANS_IDS:
                    idx = torch.tensor(ANS_IDS, device=module.weight.device)
                    b = getattr(module, "bias", None)
                    st["unemb"] = (module.weight[idx].float().cpu(),
                                   None if b is None else b[idx].float().cpu())
                return out
            return f
        return factory

    patched, pname, sandwich = [], None, []
    try:
        for l in range(L):
            o, n = _find_proj(get_attn(layers[l]))
            if o is None:
                raise SystemExit(f"layer {l}: no output projection; unsupported architecture")
            pname = pname or n
            _patch(o, proj_hook(l, o), patched)
            # OLMo-2 and Gemma-2 normalise the attention output before the
            # residual add; they are the architectures that also carry a
            # post_feedforward_layernorm.  (In Llama/Qwen, post_attention_layernorm
            # is the pre-MLP norm and must not be applied to the attention write.)
            if hasattr(layers[l], "post_feedforward_layernorm"):
                _patch(layers[l].post_attention_layernorm,
                       norm_hook(l, layers[l].post_attention_layernorm), patched)
                sandwich.append(l)
        fnorm, fname = _find_final_norm(hf)
        _patch(fnorm, norm_hook("final", fnorm), patched)
        lm = hf.get_output_embeddings()
        _patch(lm, unemb_hook(lm), patched)
        softcap = getattr(hf.config, "final_logit_softcapping", None)
        log(f"  output projection: {pname}   final norm: {fname}   "
            f"post-attention norm on {len(sandwich)}/{L} layers"
            + (f"   logit softcap {softcap}" if softcap else ""))

        n = len(tasks)
        A = {k: np.zeros((n, L, H), np.float32)
             for k in ("contribution", "write_norm", "cos_top", "cos_sum")}
        A["top_idx"] = np.zeros((n, L), np.int16)
        A["sum_norm"] = np.zeros((n, L), np.float32)
        A["dla"] = np.zeros((n, L, H, 10), np.float32)
        A["digit_logit"] = np.zeros((n, 10), np.float32)
        checks = {"post_err": 0.0, "final_err": 0.0, "logit_err": 0.0}
        items = []
        dev = next(hf.parameters()).device if not hasattr(hf, "hf_device_map") else hf.device

        for i, (text, label) in enumerate(zip(tasks, labels)):
            log(f"  [{i+1}/{n}] {label}")
            st["vec"].clear(); st["post"].clear(); st["final"] = st["unemb"] = None
            feed, ANS, appended = align_answer(tok, text)
            ANS_IDS[:] = [ANS[d] for d in DIGITS]
            ids = torch.tensor([feed], device=dev)
            with torch.no_grad():
                res = hf(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)
            logits = res.logits[0, -1].float().cpu()
            del res
            if len(st["vec"]) < L or st["final"] is None or st["unemb"] is None:
                raise SystemExit(f"hooks fired on {len(st['vec'])}/{L} layers, final norm "
                                 f"{st['final'] is not None}, unembedding {st['unemb'] is not None}")

            # final norm, frozen at this position, then the digit rows
            kind, gF, centerF, biasF, eF, xF, _ = st["final"]
            WU, bU = st["unemb"]                                   # [10, d], [10]|None
            checks["final_err"] = max(checks["final_err"], eF)
            WUg = WU * gF                                          # [10, d]
            yF = ((xF - xF.mean()) if centerF else xF) * gF + (0 if biasF is None else biasF)
            lg = yF @ WU.T + (0 if bU is None else bU)
            if softcap:
                lg = softcap * torch.tanh(lg / softcap)
            ref = logits[ANS_IDS]
            checks["logit_err"] = max(checks["logit_err"], _rel_err(lg, ref))
            A["digit_logit"][i] = ref.numpy()

            for l in range(L):
                v, bias, ref_o = st["vec"][l]
                A["contribution"][i, l] = v.norm(dim=-1).numpy()
                if l in st["post"]:
                    kP, gP, cP, bP, eP, xP, yP = st["post"][l]
                    # the norm's input must be this layer's attention output
                    tot = v.sum(0) + (0 if bias is None else bias)
                    e_in = _rel_err(tot, xP)
                    checks["post_err"] = max(checks["post_err"], eP, e_in)
                    w = ((v - v.mean(-1, keepdim=True)) if cP else v) * gP
                else:
                    w = v
                wn = w.norm(dim=-1)
                s = w.sum(0)
                top = int(wn.argmax())
                A["write_norm"][i, l] = wn.numpy()
                A["top_idx"][i, l] = top
                A["sum_norm"][i, l] = float(s.norm())
                A["cos_top"][i, l] = (w @ w[top] / (wn * wn[top]).clamp_min(1e-12)).numpy()
                A["cos_sum"][i, l] = (w @ s / (wn * s.norm()).clamp_min(1e-12)).numpy()
                wc = (w - w.mean(-1, keepdim=True)) if centerF else w
                A["dla"][i, l] = (wc @ WUg.T).numpy()              # [H, 10]

            if max(checks.values()) > FAIL:
                raise SystemExit(f"linearisation check failed on item {i}: {checks}")
            probs = torch.softmax(logits, -1)
            gold = None if golds[i] is None else str(golds[i]).strip()
            pred = max(DIGITS, key=lambda d: float(probs[ANS[d]]))
            items.append({"label": label, "text": text, "gold": gold,
                          "pred_constrained": pred,
                          "correct_constrained": gold is not None and pred == gold,
                          "gold_prob": None if gold is None else float(probs[ANS[gold]]),
                          "digit_mass": float(sum(probs[a] for a in ANS.values())),
                          "appended": appended})
    finally:
        for m, at, orig in patched:
            setattr(m, at, orig)

    worst = max([e for _, e in layout.values()] + list(checks.values()))
    if worst > WARN:
        log(f"  WARNING: a linearisation check reached {worst:.1e} (warn {WARN}, abort {FAIL}); "
            f"expected from bf16 rounding only if well below {FAIL}")
    log(f"  checks (max relative error): per-head split "
        f"{max(e for _, e in layout.values()):.1e}   post-attn norm {checks['post_err']:.1e}"
        f"   final norm {checks['final_err']:.1e}   digit logits {checks['logit_err']:.1e}")
    meta = {"n_layers": L, "n_heads": H, "proj_name": pname, "sandwich_layers": sandwich,
            "final_norm": fname, "final_norm_kind": st["final"][0] if st["final"] else None,
            "softcap": softcap, "checks": checks,
            "head_layout": {str(l): m for l, (m, _) in layout.items()}, "items": items}
    return A, meta


def run(args):
    os.environ.setdefault("ASD_FORCE_STRATEGY", "B")
    os.environ.setdefault("ASD_FORCE_PRECISION", "bf16")
    os.environ.setdefault("ASD_ATTN_IMPL", args.attn_impl)
    import active_subgraph_dot as asd
    from active_subgraph_dot import load_model, TASK_SUITES

    model = load_model(args.model, device=args.device, hf_token=args.hf_token)
    if not hasattr(model, "_model"):
        raise SystemExit("needs the _HFHookedModel wrapper; set ASD_FORCE_STRATEGY=B")
    suite = TASK_SUITES[args.suite]
    tasks, labels = suite["tasks"], suite["labels"]
    golds = suite.get("_answers", [None] * len(tasks))
    if args.limit:
        tasks, labels, golds = tasks[:args.limit], labels[:args.limit], golds[:args.limit]
    print(f"  n_layers={model.cfg.n_layers} n_heads={model.cfg.n_heads}  "
          f"load_info={asd.LAST_LOAD_INFO}")
    A, meta = capture(model._model, model._tok, model._layers(), model._get_attn,
                      model.cfg.n_heads, model.cfg.d_head, tasks, labels, golds)
    meta.update(model=args.model, suite=args.suite, load_info=dict(asd.LAST_LOAD_INFO))
    p = Path(args.out); p.mkdir(parents=True, exist_ok=True)
    f = p / f"{args.model.replace('/', '_')}_align.npz"
    np.savez_compressed(f, meta=np.array(json.dumps(meta)), **A)
    acc = np.mean([t["correct_constrained"] for t in meta["items"]])
    print(f"  digit-constrained accuracy {acc:.2f}   -> {f}  ({f.stat().st_size/1e6:.1f} MB)")


# ───────────────────────────── analysis ─────────────────────────────────────
# Per item and layer, from the arrays above (s = write norms, m = answer margin):
#   PR          participation ratio of s / H           spread of activity
#   top_share   max s / sum s                          dominance of the leader
#   coherence   ||sum w|| / sum s                      do the writes add up?  Falls
#               as ~1/sqrt(PR) when activity spreads even at fixed directions,
#               so it is descriptive only; pair_cos is the test.
#   pair_cos    s-weighted mean cosine between distinct heads' writes, exact:
#               (||sum w||^2 - sum s^2) / ((sum s)^2 - sum s^2)
#   align       s-weighted cos_top over non-top heads  do others follow the leader?
#   margin      sum_h m_h, m_h = dla[gold] - mean dla[competing states]
#   support_PR  PR of the positive m_h / H             how many heads carry the answer
#   support_top max m+ / sum m+                        does one head carry it?
#   agreement   sum m+ / sum |m|                       do heads agree on the answer?
PREDICT = {   # sign of d(metric)/dD inside the band under each reading
    "PR":          ("+", "0/-"), "top_share":  ("-", "+"),
    "coherence":   ("+", "0"),   "pair_cos":   ("+", "0"),
    "align":       ("+", "0"),
    "support_PR":  ("+", "-"),   "support_top": ("-", "+"),
    "agreement":   ("+", "0"),   "margin":     ("", ""),
}


def _pr(x):
    s, s2 = x.sum(-1), (x * x).sum(-1)
    return np.where(s2 > 0, s * s / np.where(s2 > 0, s2, 1), 0) / x.shape[-1]


def load(path):
    z = np.load(path)
    meta = json.loads(str(z["meta"]))
    A = {k: z[k] for k in z.files if k != "meta"}
    return A, meta


def layer_metrics(A, meta):
    items = meta["items"]
    s = A["write_norm"].astype(float)                        # [n, L, H]
    n, L, H = s.shape
    gold = np.array([int(t["gold"]) for t in items])
    # competitors: the states named in the item's final rule table
    comp = np.zeros((n, 10), bool)
    for i, t in enumerate(items):
        states = {int(d) for d in re.findall(r"(\d) goes to", t["text"].split("Rules:")[-1])}
        comp[i, sorted(states - {gold[i]})] = True
    dla = A["dla"].astype(float)                             # [n, L, H, 10]
    g = np.take_along_axis(dla, gold[:, None, None, None], -1)[..., 0]
    o = (dla * comp[:, None, None, :]).sum(-1) / comp.sum(1)[:, None, None]
    m = g - o                                                # [n, L, H]
    mp = np.clip(m, 0, None)
    top = A["top_idx"].astype(int)
    notop = np.ones_like(s, bool)
    np.put_along_axis(notop, top[..., None], False, -1)
    sw = s * notop
    M = dict(
        PR=_pr(s),
        top_share=s.max(-1) / np.clip(s.sum(-1), 1e-12, None),
        coherence=A["sum_norm"] / np.clip(s.sum(-1), 1e-12, None),
        pair_cos=(A["sum_norm"] ** 2 - (s * s).sum(-1))
                 / np.clip(s.sum(-1) ** 2 - (s * s).sum(-1), 1e-12, None),
        align=(sw * A["cos_top"]).sum(-1) / np.clip(sw.sum(-1), 1e-12, None),
        margin=m.sum(-1),
        support_PR=_pr(mp),
        support_top=mp.max(-1) / np.clip(mp.sum(-1), 1e-12, None),
        agreement=mp.sum(-1) / np.clip(np.abs(m).sum(-1), 1e-12, None),
    )
    return M, m, s


def _slopes(Y, D, rng, nperm):
    """OLS slope of each column of Y on D, and a joint permutation null
    (one shuffle of D applied to every column)."""
    dc = D - D.mean(); ss = dc @ dc; Yc = Y - Y.mean(0)
    obs = dc @ Yc / ss
    P = np.stack([rng.permutation(dc) for _ in range(nperm)])
    return obs, P @ Yc / ss


def _p(obs, null):
    return float(((np.abs(null) >= abs(obs) - 1e-15).sum() + 1) / (len(null) + 1))


def analyse_one(path, band=None, correct_only=False, nperm=2000, seed=0):
    rng = np.random.default_rng(seed)
    A, meta = load(path)
    items = meta["items"]
    L, H = meta["n_layers"], meta["n_heads"]
    D = np.array([int(re.match(r"D(\d+)", t["label"]).group(1)) for t in items], float)
    ok = np.array([t["correct_constrained"] for t in items])
    nstates = len(set(re.findall(r"(\d) goes to", items[0]["text"].split("Rules:")[-1])))
    M, m, s = layer_metrics(A, meta)

    print(f"\n{'#'*78}\n# {meta['model']}   suite={meta['suite']}   L={L} H={H}\n{'#'*78}")
    print(f"  checks: {meta['checks']}   post-attn norm layers: {len(meta['sandwich_layers'])}")
    print("  accuracy by D: " + "  ".join(
        f"D{int(k)} {ok[D == k].mean():.2f}" for k in np.unique(D)) + f"   chance {1/nstates:.2f}")
    sel = np.ones(len(D), bool)
    if correct_only:
        above = _above_chance(D.astype(int), ok, 1 / nstates)
        sel = ok & np.isin(D, above)
        print(f"  correct items at above-chance depths {above}: n={sel.sum()}")
        if len(above) < 3 or sel.sum() < 12:
            print("  not enough correct items for a depth slope"); return None

    if band:
        lo, hi = band
        inb = np.zeros(L, bool); inb[int(lo * (L - 1)):int(hi * (L - 1)) + 1] = True
        regions = [(f"band {lo:.2f}-{hi:.2f}", inb), ("outside", ~inb)]
    else:
        regions = [(f"{20*q}-{20*q+20}%", (np.arange(L) >= q * L / 5) & (np.arange(L) < (q + 1) * L / 5))
                   for q in range(5)]

    res = {"model": meta["model"], "L": L, "slopes": {}}
    print(f"\n  change per extra step, mean over layers in each region (p: joint D-permutation)")
    hdr = "".join(f"{r:>22}" for r, _ in regions)
    print(f"  {'metric':<12}{'consensus':>10}{'special.':>9}{hdr}")
    for k, Y in M.items():
        obs, null = _slopes(Y[sel], D[sel], rng, nperm)
        res["slopes"][k] = obs
        cells = []
        for _, mask in regions:
            o = obs[mask].mean()
            cells.append(f"{o:>+11.2e} (p={_p(o, null[:, mask].mean(1)):.3f})")
        c, sp = PREDICT[k]
        print(f"  {k:<12}{c:>10}{sp:>9}" + "".join(f"{x:>22}" for x in cells))

    # head level: do heads that gain share gain answer support and follow the leader?
    # In the band if given, else 40-80% depth, where dispersion rises in every
    # model measured so far (the first quintile is not where the question is).
    if band:
        hl_name, mask = regions[0]
    else:
        mask = np.zeros(L, bool); mask[int(.4 * (L - 1)):int(.8 * (L - 1)) + 1] = True
        hl_name = "40-80%"
    share = s / np.clip(s.sum(-1, keepdims=True), 1e-12, None)
    Ys = share[sel][:, mask].reshape(sel.sum(), -1)
    Ym = m[sel][:, mask].reshape(sel.sum(), -1)
    ct = A["cos_top"][sel][:, mask].reshape(sel.sum(), -1).astype(float)
    isnt = (A["top_idx"][sel][:, mask][..., None] != np.arange(H)).reshape(sel.sum(), -1)
    cnt = isnt.sum(0)
    ct_mean = np.where(cnt > 0, (ct * isnt).sum(0) / np.maximum(cnt, 1), np.nan)
    Dsel = D[sel]
    dc = Dsel - Dsel.mean(); ss = dc @ dc
    ds, dm = dc @ (Ys - Ys.mean(0)) / ss, dc @ (Ym - Ym.mean(0)) / ss
    base_m = Ym.mean(0)
    ok_h = np.isfinite(ct_mean)
    r_sm = float(np.corrcoef(ds, dm)[0, 1])
    r_sc = float(np.corrcoef(ds[ok_h], ct_mean[ok_h])[0, 1])
    r_sb = float(np.corrcoef(ds, base_m)[0, 1])
    null_sm, null_sc, null_sb = [], [], []
    for _ in range(min(nperm, 1000)):
        pc = rng.permutation(dc)
        a, b = pc @ (Ys - Ys.mean(0)) / ss, pc @ (Ym - Ym.mean(0)) / ss
        null_sm.append(np.corrcoef(a, b)[0, 1])
        null_sc.append(np.corrcoef(a[ok_h], ct_mean[ok_h])[0, 1])
        null_sb.append(np.corrcoef(a, base_m)[0, 1])
    print(f"\n  head level, {hl_name} ({int(mask.sum())} layers x {H} heads):")
    print(f"    r(d share/dD, d margin/dD)   = {r_sm:+.3f}  (p={_p(r_sm, np.array(null_sm)):.3f})"
          f"   consensus +, specialisation 0")
    print(f"    r(d share/dD, mean cos_top)  = {r_sc:+.3f}  (p={_p(r_sc, np.array(null_sc)):.3f})"
          f"   consensus + (gainers follow the leader)")
    print(f"    r(d share/dD, mean margin)   = {r_sb:+.3f}  (p={_p(r_sb, np.array(null_sb)):.3f})"
          f"   consensus + (gainers support the answer)")
    res.update(head=dict(r_share_margin=r_sm, r_share_cos=r_sc, r_share_base=r_sb))
    return res


def plot(results, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    keys = ["PR", "pair_cos", "align", "support_PR", "support_top", "margin"]
    fig, ax = plt.subplots(len(results), len(keys), figsize=(2.9 * len(keys), 2.4 * len(results)),
                           squeeze=False, constrained_layout=True)
    for i, r in enumerate(results):
        xs = np.linspace(0, 1, r["L"])
        for j, k in enumerate(keys):
            a = ax[i, j]
            a.plot(xs, r["slopes"][k], lw=1.2); a.axhline(0, c="k", lw=.5)
            a.set_title(f"{r['model'].split('/')[-1]}: {k}" if j == 0 else k, fontsize=8)
            a.tick_params(labelsize=6)
    for a in ax[-1]:
        a.set_xlabel("relative depth", fontsize=7)
    fig.suptitle("change per extra step, by layer", fontsize=10)
    fig.savefig(path, dpi=110); plt.close(fig)
    print(f"\n  figure -> {path}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model"); ap.add_argument("--suite", default="serial_depth_fs")
    ap.add_argument("--device", default="cuda"); ap.add_argument("--hf_token", default=None)
    ap.add_argument("--out", default="probe/align")
    ap.add_argument("--attn-impl", default="eager", choices=["eager", "sdpa"])
    ap.add_argument("--limit", type=int, default=None, help="first N items only (smoke test)")
    ap.add_argument("--analyse", nargs="+", metavar="NPZ")
    ap.add_argument("--band", nargs=2, type=float, default=None,
                    help="relative depths, e.g. --band 0.30 0.94; default: quintiles")
    ap.add_argument("--correct-only", action="store_true",
                    help="correct items at above-chance depths only")
    ap.add_argument("--nperm", type=int, default=2000)
    ap.add_argument("--plot")
    a = ap.parse_args()
    if a.analyse:
        rs = [r for r in (analyse_one(p, a.band, a.correct_only, a.nperm) for p in a.analyse) if r]
        if a.plot and rs:
            plot(rs, a.plot)
    elif a.model:
        run(a)
    else:
        ap.error("--model or --analyse required")


In [ ]:
%%writefile ablation_window.py
#!/usr/bin/env python3
"""
Top-10% head ablation at the answer position, in fixed depth bands, for many
models: does a model whose head activity is more concentrated depend more on
its most active heads?  One model, several suites (serial_tasks.py JSON); the
model loads once.

Design, fixed before running:

  items       the suite's test split, halved by k as ablation_probe does
              (same seed, so the halves are E4's): a selection and a
              measurement half
  pass 1      selection half, one full forward per item: per-head activity
              ||z_h W_O^h|| at the measured position on every layer (the
              paper's activity), and each head's o_proj input z there.  From
              these: mean activity (heads ranked per layer), mean z (the
              mean-ablation value), and the top-q share of every layer
  heads       q = ceil(0.1 H) per layer, the paper's top-10% group, ranked by
              selection-half mean activity; random baseline: q heads per layer
              from the mid-ranked pool (activity between the layer's 25th and
              75th percentile, the top max(q, 3) excluded), ablation_probe's rule
  sets        win = the 40-80% depth window (ablation_probe's, the E4 window),
              and five depth bands b0..b4 by relative depth l/(L-1) in
              [0, .2), [.2, .4), [.4, .6), [.6, .8), [.8, 1]; every layer of a
              set is ablated jointly, whatever its response to k
  conditions  clean; per set topq/mean, topq/zero, randq/mean (--draws-win
              draws for win, --draws-band per band); for win also top1/mean
              and top3/mean (E4's sizes)
  outcome     per measurement item and condition: digit logits and the logit
              difference gold minus mean competitor (ablation_probe's
              logit_diff), at the measured position only (prefix KV cache,
              ablation_probe.Ablator: every check of E4 runs, the full
              per-item verification on each suite's first item and the
              full-forward cache check every 25th)

Analysis (--analyse), fixed before running:

  share       per model x task: top-q share of the win layers, averaged over
              selection items (all k) and layers; per band the same
  drop        per clean-correct measurement item, 1 - LD / LD_clean; averaged
              within each depth with >= MIN_CORRECT such items, then over
              those depths (depth-balanced).  Primary: win/topq/mean minus
              the mean of the win/randq/mean draws ("excess drop")
  within task across models: Spearman(share, excess drop) over the models,
              per task; a permutation null shuffles models within each task
  across tasks within a model: Spearman over its four tasks, averaged over
              models, reported beside it

    python3 ablation_window.py --model Qwen/Qwen2.5-0.5B --suites s1.json s2.json --out DIR
    python3 ablation_window.py --analyse DIR/*__abwin.npz --out REPORT_DIR
"""
import argparse, json, math, os, sys, time
from pathlib import Path

import numpy as np

import ablation_probe as ap
import signature_analysis as sa
import sweep_probe as sp
from deep_probe import align_answer

DIGITS = "0123456789"
Q = 0.10
WINDOW = (0.4, 0.8)
N_BANDS = 5
SUFFIX = "__abwin.npz"
MIN_CORRECT = 5
MEASUREMENT_VERSION = 1


def top_q(H, q=Q):
    return max(1, int(math.ceil(q * H)))


def band_of(L):
    x = np.arange(L) / max(L - 1, 1)
    return np.minimum((x * N_BANDS).astype(int), N_BANDS - 1)


def window_layers(L, lo=WINDOW[0], hi=WINDOW[1]):
    return list(range(L))[sa.window(L, lo, hi)]


def top_share(S, m):
    """[..., H] -> [...]: summed share of the m largest entries."""
    S = np.asarray(S, float)
    return -np.sort(-S, -1)[..., :m].sum(-1) / np.clip(S.sum(-1), 1e-30, None)


def build_conditions(mean_activity, *, draws_win=5, draws_band=2, seed=0):
    """(conditions, sets, skipped).  Condition dicts as ablation_probe's
    (name, set, kind, size, mode, draw, layer, heads=[[layer, [heads]], ...]),
    clean first; names use 'topq' / 'randq' so they are the same for every
    model (the head count is in 'size')."""
    A = np.asarray(mean_activity, float)
    L, H = A.shape
    m = top_q(H)
    rank = ap.rank_heads(A)
    b = band_of(L)
    sets = {"win": window_layers(L)}
    sets.update({f"b{i}": [int(l) for l in np.nonzero(b == i)[0]] for i in range(N_BANDS)})
    conds = [dict(name="clean", set=None, kind="clean", size=0, mode=None, draw=None, layer=None,
                  heads=[])]
    skipped = []

    def top(s, n, label, mode):
        conds.append(dict(name=f"{s}/{label}/{mode}", set=s, kind="top", size=n, mode=mode,
                          draw=None, layer=None,
                          heads=[[l, sorted(int(h) for h in rank[l, :n])] for l in sets[s]]))

    for s, layers in sets.items():
        if not layers:
            skipped.append({"set": s, "reason": "no layer"})
            continue
        top(s, m, "topq", "mean")
        top(s, m, "topq", "zero")
        if s == "win":
            top(s, 1, "top1", "mean")
            if H > 3:
                top(s, 3, "top3", "mean")
        pool = {l: ap.mid_rank_heads(A[l], exclude_top=max(m, ap.EXCLUDE_TOP)) for l in layers}
        short = {l: len(p) for l, p in pool.items() if len(p) < m}
        if short:
            skipped.append({"set": s, "reason": f"mid-ranked pool smaller than {m}: {short}"})
            continue
        rng = ap._rng(seed, f"abwin/random/{s}/{m}")
        for r in range(draws_win if s == "win" else draws_band):
            conds.append(dict(name=f"{s}/randq/mean/r{r}", set=s, kind="rand", size=m, mode="mean",
                              draw=r, layer=None,
                              heads=[[l, sorted(int(h) for h in rng.choice(pool[l], m, replace=False))]
                                     for l in layers]))
    return conds, sets, skipped


def run_window(model, suite, *, attn_impl, split="test", seed=0, draws_win=5, draws_band=2,
               batch=None, max_items_per_k=None, log=print):
    """One model x suite; returns (arrays, meta)."""
    import torch
    t_start = time.time()
    tok = model._tok
    L, H, dh = model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_head
    name = getattr(model.cfg, "model_name", None)
    sp._check_items(suite)
    actual = getattr(model._model.config, "_attn_implementation", None)
    if actual is not None and actual != attn_impl:
        ap._abort(f"model runs {actual!r} attention but attn_impl={attn_impl!r}")
    items = suite["items"]
    sel, meas = ap.split_halves(items, split, seed, max_items_per_k)
    ap.check_competitors(items, sel + meas)
    allL = list(range(L))
    nS, nM = len(sel), len(meas)
    with ap.Ablator(model, allL, log=log) as ab:
        # ── pass 1 ──
        P1 = {"sel_index": np.array(sel, np.int32),
              "sel_k": np.array([items[i]["k"] for i in sel], np.int16),
              "sel_contribution": np.zeros((nS, L, H), np.float32),
              "sel_digit_logits": np.zeros((nS, 10), np.float32),
              "sel_correct": np.zeros(nS, bool)}
        zsum = np.zeros((L, H, dh), np.float64)
        for j, i in enumerate(sel):
            t = items[i]
            if j == 0 or (j + 1) % 25 == 0 or j + 1 == nS:
                log(f"  [select {j + 1}/{nS}] {t['id']}")
            feed, ANS, _ = align_answer(tok, t["text"])
            C, Z, lg = ab.capture(feed)
            P1["sel_contribution"][j] = C
            zsum += Z
            dlj = np.array([float(lg[ANS[d]]) for d in DIGITS], np.float32)
            P1["sel_digit_logits"][j] = dlj
            P1["sel_correct"][j] = sp._constrained(dlj, t["answer_set"], t["gold"])[1]
        P1["mean_z"] = (zsum / nS).astype(np.float32)
        P1["mean_activity"] = P1["sel_contribution"].mean(0)
        P1["head_rank"] = ap.rank_heads(P1["mean_activity"]).astype(np.int16)
        conds, sets, skipped = build_conditions(P1["mean_activity"], draws_win=draws_win,
                                                draws_band=draws_band, seed=seed)
        tables = ap.ablation_tables(conds, P1["mean_z"], allL)
        nC = len(conds)
        B = int(batch) if batch else min(nC, 32)
        log(f"  {name} | {suite['name']}: L={L} H={H} q={top_q(H)}  win layers "
            f"{sets['win'][0]}-{sets['win'][-1]}  {nC} conditions, {B} per forward"
            + (f"  skipped {skipped}" if skipped else ""))
        # ── pass 2 ──
        A = {"index": np.array(meas, np.int32),
             "k": np.array([items[i]["k"] for i in meas], np.int16),
             "gold": np.array([int(items[i]["gold"]) for i in meas], np.int8),
             "answer_mask": np.zeros((nM, 10), bool),
             "digit_logits": np.full((nM, nC, 10), np.nan, np.float32),
             "logit_diff": np.full((nM, nC), np.nan, np.float32),
             "frac": np.full((nM, nC), np.nan, np.float32),
             "clean_correct": np.zeros(nM, bool),
             "meas_contribution": np.zeros((nM, L, H), np.float32),
             "n_tokens": np.zeros(nM, np.int32)}
        for j, i in enumerate(meas):
            t = items[i]
            A["answer_mask"][j, [int(d) for d in t["answer_set"]]] = True
            if j == 0 or (j + 1) % 25 == 0 or j + 1 == nM:
                log(f"  [measure {j + 1}/{nM}] {t['id']}")
            feed, ANS, _ = align_answer(tok, t["text"])
            dl, _, act, _ = ab.measure(feed, tables, [ANS[d] for d in DIGITS], batch=B,
                                       verify=(j == 0), verify_cache=(j % ap.VERIFY_EVERY == 0))
            ld = ap.logit_diff(dl, t["gold"], list(t["answer_set"]))
            A["digit_logits"][j], A["logit_diff"][j] = dl, ld
            A["frac"][j] = ld / ld[0] if ld[0] != 0 else np.nan
            A["clean_correct"][j] = sp._constrained(dl[0], list(t["answer_set"]), t["gold"])[1]
            A["meas_contribution"][j] = act
            A["n_tokens"][j] = len(feed)
            if ab.dev.type == "cuda" and (j + 1) % 25 == 0:
                torch.cuda.empty_cache()
        checks = dict(ab.checks)
        layout = {str(l): md for l, (md, _) in ab.layout.items()}
        pname, head_checks = ab.pname, ab.head_checks
    A.update(P1)
    dtype = str(next(model._model.parameters()).dtype).replace("torch.", "")
    meta = {"kind": "ablation_window", "model": name, "suite": suite.get("name"),
            "family": suite.get("family"), "wording": suite.get("wording"),
            "setting": suite.get("setting"), "chance": suite.get("chance"),
            "suite_sha256": sp.suite_sha256(suite), "split": split, "seed": int(seed),
            "max_items_per_k": max_items_per_k, "q": Q, "q_heads": top_q(H), "window": list(WINDOW),
            "n_bands": N_BANDS, "draws_win": int(draws_win), "draws_band": int(draws_band),
            "batch_conditions": B, "attn_impl": attn_impl, "dtype": dtype,
            "n_layers": L, "n_heads": H, "d_head": dh, "sets": sets, "conditions": conds,
            "skipped": skipped, "n_selection": nS, "n_measurement": nM,
            "proj_name": pname, "head_layout": layout, "head_boundaries_checked": head_checks,
            "checks": checks, "measurement_version": MEASUREMENT_VERSION,
            "module_sha256": sp._sha256(Path(__file__).resolve()),
            "ablation_probe_sha256": ap._mod_sha(),
            "timing": {"total_s": round(time.time() - t_start, 1)}}
    return A, meta


def output_path(out_dir, model_name, suite_name):
    return Path(out_dir) / f"{model_name.replace('/', '_')}__{suite_name}{SUFFIX}"


def load_model(name, device, attn_impl, precision="bf16"):
    """ablation_probe.run's load path: native hooks, the given precision and kernel."""
    os.environ["ASD_FORCE_STRATEGY"] = "B"
    os.environ["ASD_FORCE_PRECISION"] = "bf16" if precision == "bf16" else ""
    os.environ["ASD_ATTN_IMPL"] = attn_impl
    import torch
    import active_subgraph_dot as asd
    if device.startswith("cuda") and not torch.cuda.is_available():
        ap._abort("--device cuda but CUDA is not available")
    model = asd.load_model(name, device=device)
    if not hasattr(model, "_model"):
        ap._abort("needs the _HFHookedModel wrapper (ASD_FORCE_STRATEGY=B)")
    on = sorted({p.device.type for p in model._model.parameters()})
    if device.split(":")[0] not in on:
        ap._abort(f"weights on {on}, not {device}; refusing")
    li = dict(asd.LAST_LOAD_INFO)
    want = {"bf16": "bfloat16", "fp32": "float32"}[precision]
    if li.get("strategy") != "B" or li.get("precision") != want:
        ap._abort(f"loaded with {li.get('strategy')!r} / {li.get('precision')!r}, not B / {want}")
    return model, li


def run(args, log=print):
    suites = []
    for p in args.suites:
        with open(p) as f:
            s = json.load(f)
        ap.split_halves(s["items"], args.split, args.seed, args.max_items_per_k)   # validate early
        suites.append((p, s))
    todo = [(p, s) for p, s in suites if not output_path(args.out, args.model, s["name"]).exists()]
    for p, s in suites:
        if (p, s) not in todo:
            log(f"  skip {s['name']}: done")
    if not todo:
        return []
    sp._git_commit()
    t0 = time.time()
    model, li = load_model(args.model, args.device, args.attn_impl, args.precision)
    load_s = round(time.time() - t0, 1)
    log(f"  {args.model} loaded in {load_s} s: {li}")
    Path(args.out).mkdir(parents=True, exist_ok=True)
    outs = []
    for p, s in todo:
        A, meta = run_window(model, s, attn_impl=args.attn_impl, split=args.split, seed=args.seed,
                             draws_win=args.draws_win, draws_band=args.draws_band,
                             batch=args.batch_conditions, max_items_per_k=args.max_items_per_k,
                             log=log)
        meta.update(model=args.model, suite_path=str(p), load_s=load_s, load_info=li,
                    requested_precision=args.precision,
                    next_runs_commit_env=os.environ.get("NEXT_RUNS_COMMIT"))
        out = output_path(args.out, args.model, s["name"])
        sp.save_output(out, A, meta)
        ok = A["clean_correct"]
        ci = {c["name"]: i for i, c in enumerate(meta["conditions"])}
        drop = lambda n: float(np.nanmean(1 - A["frac"][ok, ci[n]])) if n in ci and ok.any() else float("nan")
        rnd = [n for n in ci if n.startswith("win/randq/")]
        log(f"  -> {out.name}: clean correct {int(ok.sum())}/{len(ok)}; win share "
            f"{top_share(A['sel_contribution'][:, meta['sets']['win']], meta['q_heads']).mean():.3f}; "
            f"drop topq/mean {drop('win/topq/mean'):+.3f} topq/zero {drop('win/topq/zero'):+.3f} "
            f"randq {np.nanmean([drop(n) for n in rnd]) if rnd else float('nan'):+.3f}   checks: "
            f"split {meta['checks']['split_err']:.1e} delta {meta['checks']['delta_err']:.1e} "
            f"cache {meta['checks']['cache_err']:.1e} repeat {meta['checks']['repeat_err']:.1e}   "
            f"{meta['timing']['total_s']} s")
        outs.append(out)
    return outs


# ─────────────────────────────── analysis ───────────────────────────────────
def summarise(path, min_correct=MIN_CORRECT):
    A, meta = sp.load_output(path)
    ci = {c["name"]: i for i, c in enumerate(meta["conditions"])}
    ok, k = A["clean_correct"], A["k"]
    ld = A["logit_diff"].astype(float)
    depths = [int(d) for d in np.unique(k) if (ok & (k == d)).sum() >= min_correct]
    m = int(meta["q_heads"])

    def drop(cols, absolute=False):
        if not depths:
            return float("nan")
        v = (ld[:, [0]] - ld[:, cols]) if absolute else (1 - ld[:, cols] / ld[:, [0]])
        v = v.mean(1)
        return float(np.mean([v[ok & (k == d)].mean() for d in depths]))

    row = {"model": meta["model"].split("/")[-1], "task": meta["family"], "H": meta["n_heads"],
           "L": meta["n_layers"], "q_heads": m, "depths": depths,
           "n_correct": int(ok[np.isin(k, depths)].sum()), "accuracy": float(ok.mean()),
           "clean_ld": float(np.mean([ld[ok & (k == d), 0].mean() for d in depths])) if depths else float("nan")}
    for s, layers in meta["sets"].items():
        if not layers:
            continue
        sh = top_share(A["sel_contribution"][:, layers], m)
        row[f"share_{s}"] = float(sh.mean())
        row[f"share_{s}_k1"] = float(sh[A["sel_k"] == A["sel_k"].min()].mean())
        for n in ("topq/mean", "topq/zero", "top1/mean", "top3/mean"):
            if f"{s}/{n}" in ci:
                row[f"drop_{s}_{n.replace('/', '_')}"] = drop([ci[f"{s}/{n}"]])
        rc = [i for nm, i in ci.items() if nm.startswith(f"{s}/randq/")]
        if rc:
            row[f"drop_{s}_randq"] = drop(rc)
            row[f"excess_{s}"] = row[f"drop_{s}_topq_mean"] - row[f"drop_{s}_randq"]
            row[f"excess_{s}_abs"] = drop([ci[f"{s}/topq/mean"]], True) - drop(rc, True)
    return row


def _spearman(x, y):
    from scipy.stats import spearmanr
    ok = np.isfinite(x) & np.isfinite(y)
    return (float(spearmanr(x[ok], y[ok])[0]), int(ok.sum())) if ok.sum() >= 4 else (float("nan"), int(ok.sum()))


def _md_table(df):
    cols = list(df.columns)
    fmt = lambda v: f"{v:.3f}" if isinstance(v, float) else str(v)
    return "\n".join(["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)] +
                     ["| " + " | ".join(fmt(v) for v in r) + " |" for r in df.itertuples(index=False)])


def analyse(paths, out_dir, n_perm=10000, seed=0, x="share_win", y="excess_win", min_correct=MIN_CORRECT):
    import pandas as pd
    rows = [summarise(p, min_correct) for p in paths]
    df = pd.DataFrame(rows)
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    df.to_csv(out / "abwin_rows.csv", index=False)
    rng = np.random.default_rng(seed)
    res = {"x": x, "y": y, "within_task": {}, "across_tasks_within_model": {}}
    tasks = sorted(df.task.unique())
    obs = []
    for t in tasks:
        d = df[df.task == t]
        r, n = _spearman(d[x].to_numpy(float), d[y].to_numpy(float))
        res["within_task"][t] = {"rho": r, "n_models": n}
        obs.append(r)
    mean_obs = float(np.nanmean(obs))
    null = []
    for _ in range(n_perm):
        rr = []
        for t in tasks:
            d = df[df.task == t]
            rr.append(_spearman(d[x].to_numpy(float), rng.permutation(d[y].to_numpy(float)))[0])
        null.append(np.nanmean(rr))
    null = np.array(null)
    res["within_task_mean_rho"] = mean_obs
    res["within_task_p_one_sided_positive"] = float((np.sum(null >= mean_obs) + 1) / (n_perm + 1))
    per_model = {}
    for mdl, d in df.groupby("model"):
        per_model[mdl] = _spearman(d[x].to_numpy(float), d[y].to_numpy(float))[0] if len(d) >= 4 else float("nan")
    res["across_tasks_within_model"] = per_model
    res["across_tasks_within_model_mean_rho"] = float(np.nanmean(list(per_model.values())))
    json.dump(res, open(out / "abwin_report.json", "w"), indent=1)
    lines = [f"# Top-10% ablation in fixed depth bands\n",
             f"x = {x} (top-q share of the 40-80% window, selection half), y = {y} (drop of the "
             f"top-q heads minus random mid-ranked heads, fraction of clean LD, clean-correct items, "
             f"depth-balanced)\n",
             "## Within task, across models\n", "| task | rho | models |", "|---|---|---|"]
    lines += [f"| {t} | {v['rho']:+.2f} | {v['n_models']} |" for t, v in res["within_task"].items()]
    lines += [f"\nmean rho {mean_obs:+.2f}, permutation p (models shuffled within task, one-sided "
              f"positive) {res['within_task_p_one_sided_positive']:.4f}\n",
              "## Across tasks, within a model\n", "| model | rho over its tasks |", "|---|---|"]
    lines += [f"| {m} | {v:+.2f} |" for m, v in per_model.items()]
    lines += [f"\nmean {res['across_tasks_within_model_mean_rho']:+.2f}\n", "## Rows\n",
              _md_table(df[["model", "task", "H", "q_heads", "accuracy", "n_correct", "share_win",
                            "drop_win_topq_mean", "drop_win_randq", "excess_win", "drop_win_topq_zero"]])]
    (out / "abwin_report.md").write_text("\n".join(lines))
    return df, res


def main(argv=None):
    ap_ = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap_.add_argument("--model")
    ap_.add_argument("--suites", nargs="+")
    ap_.add_argument("--out")
    ap_.add_argument("--device", default="cuda")
    ap_.add_argument("--attn-impl", default="sdpa", choices=["eager", "sdpa"])
    ap_.add_argument("--precision", default="bf16", choices=["bf16", "fp32"])
    ap_.add_argument("--split", default="test")
    ap_.add_argument("--seed", type=int, default=0)
    ap_.add_argument("--draws-win", type=int, default=5)
    ap_.add_argument("--draws-band", type=int, default=2)
    ap_.add_argument("--batch-conditions", type=int, default=None)
    ap_.add_argument("--max-items-per-k", type=int, default=None)
    ap_.add_argument("--analyse", nargs="+", metavar="NPZ")
    a = ap_.parse_args(argv)
    if a.analyse:
        df, res = analyse(a.analyse, a.out or ".")
        print(Path(a.out or ".", "abwin_report.md").read_text())
        return
    if not (a.model and a.suites and a.out):
        ap_.error("--model, --suites and --out are required (or --analyse)")
    run(a)


if __name__ == "__main__":
    main()


In [ ]:
# 0.3 Check the written modules and the clone, then record the code version
import hashlib, importlib.metadata as _im

def sha256_file(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

EXPECTED_SHA256 = {
    "serial_tasks.py": "53ff1714f1f48e4edc78d3c94ce793bd98eaf01eef5425a1b4bf193cd526da63",
    "sweep_probe.py": "f3bd7326724231ad91eb1fc65dbd17a3604a330606ad359e55dfac8ef8213301",
    "signature_analysis.py": "d58d38b05163925f23377623da0246ad14a9e0b098710d9d92c9e5ca3b027628",
    "ablation_probe.py": "70a18114d286885f8a3f56d5049e2c96c7f5fed4b4c22e5f5353cc8102a5568f",
    "head_alignment_probe.py": "e7a62b1988cf8927288010e2d43f78b7e89d7e97cead8620b3216a82a2e3b9f1",
    "ablation_window.py": "feb85e8a7055efb1ea86ef7ead9a08e5fb87a47a1400a783b83eab48d3303385"
}
TESTED_COMMIT = "d7763dd74865ef561233db4d1edb591ff9a329a8"      # the repository commit the modules were tested with
TESTED_DEPS = {
    "deep_probe.py": "9addf05ba05b8225b463ccac5b85c7943052c368b339b815b68de3266d0ca124",
    "active_subgraph_dot.py": "4e5d1d59a670ff7a4cbc8a8f1a9ff4d3ca4735c16cb8f84a0d937a2b92d3f98b",
    "path_analyzer.py": "37ddf41bd0f489497c6e10204955790f4dd7ed30a474e33ca5f108987b498c6b"
}
STRICT_DEPS = True       # False: run with the clone's versions anyway (outputs record the hashes)

bad = [f for f, h in EXPECTED_SHA256.items() if not os.path.exists(f) or sha256_file(f) != h]
if bad:
    raise RuntimeError(f"{bad}: not the embedded module; re-run its %%writefile cell unedited")
changed = [f for f, h in TESTED_DEPS.items() if not os.path.exists(f) or sha256_file(f) != h]
if changed:
    msg = (f"{changed} in the clone differ from the versions the modules were tested with "
           f"(commit {TESTED_COMMIT[:12]}); to use those: !git checkout {TESTED_COMMIT} -- "
           + " ".join(changed))
    if STRICT_DEPS:
        raise RuntimeError(msg + "   (or set STRICT_DEPS = False)")
    print("WARNING:", msg)
os.environ["NEXT_RUNS_COMMIT"] = CLONE_COMMIT + "+" + ",".join(
    f"{f[:-3]}:{sha256_file(f)[:12]}" for f in EXPECTED_SHA256)
print("NEXT_RUNS_COMMIT =", os.environ["NEXT_RUNS_COMMIT"])

def _version(pkg):
    try:
        return _im.version(pkg)
    except _im.PackageNotFoundError:
        return None
print({p: _version(p) for p in ("torch", "transformers", "accelerate", "numpy", "scipy",
                                "statsmodels")})

### 0.4 Google Drive
Every output of E1–E6 is written under `MyDrive/next_runs` (the same paths cell in
every notebook). Change `DRIVE_ROOT` here, identically in all five notebooks, to use
another folder.

In [ ]:
# 0.4 Google Drive: every output goes under DRIVE_ROOT
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/next_runs"
P = {
    "suites_grid":     f"{DRIVE_ROOT}/E1_suites/grid",          # E1: family x wording x setting
    "suites_chosen":   f"{DRIVE_ROOT}/E1_suites/chosen",        # E1's choice (informational)
    "suites_fixed":    f"{DRIVE_ROOT}/fixed_suites",            # E2, E4: the fixed recommended suites
    "chosen_manifest": f"{DRIVE_ROOT}/E1_suites/chosen_suites.json",
    "e1_checks":       f"{DRIVE_ROOT}/E1_suites/check_suite.json",
    "e1_calib":        f"{DRIVE_ROOT}/E1_calib",                # accuracy-only calibration captures
    "e1_choice":       f"{DRIVE_ROOT}/E1_choice",               # groups.json, choose_setting.md/.json
    "e3":              f"{DRIVE_ROOT}/E3_kernel_ref",           # ladder captures: fp32 / bf16 eager / bf16 sdpa
    "kernel_choice":   f"{DRIVE_ROOT}/E3_kernel_ref/kernel_choice.json",   # E3's report (not read)
    "e2":              f"{DRIVE_ROOT}/E2_sweep",                # one folder per <precision>_<kernel>
    "e4":              f"{DRIVE_ROOT}/E4_ablation",             # one folder per <precision>_<kernel>, smoke/
    "e4_report":       f"{DRIVE_ROOT}/E4_ablation/report",
    "report":          f"{DRIVE_ROOT}/E5_E6_report",
    "logs":            f"{DRIVE_ROOT}/logs",                    # full output of every run
}
for _k in ("suites_grid", "suites_chosen", "suites_fixed", "e1_calib", "e1_choice", "e3", "e2", "e4",
           "e4_report", "report", "logs"):
    os.makedirs(P[_k], exist_ok=True)
print("outputs ->", DRIVE_ROOT)

### 0.5 Hugging Face token
Llama 3.x and Gemma 2 are gated: accept their licences on huggingface.co first. The token
is typed here, kept only in this session's environment (`HF_TOKEN`) and passed to each
run; it is never printed or written to Drive. (`load_model` logs in with it, which stores
a copy under `/content/hf` on this VM's local disk; the last cell deletes it.) Press
Enter to skip if you only run ungated models.

In [ ]:
# 0.5 Hugging Face token (getpass only; never printed or saved)
from getpass import getpass
if not os.environ.get("HF_TOKEN"):
    _t = getpass("Hugging Face token (Enter to skip): ").strip()
    if _t:
        os.environ["HF_TOKEN"] = _t
    del _t
_have_token = bool(os.environ.get("HF_TOKEN"))
print("token:", "set for this session" if _have_token else "not set (gated models will fail)")

### 0.6 Helpers
`run_cmd` runs each module's command line in a fresh process (its memory is freed when
it exits), streams its output with per-item progress thinned to every 25th item plus
the rate and time left, and appends the full output to `logs/` on Drive. `run_models`
moves on to the next model when one fails and lists every failure at the end. Before a
long loop the run cells call `check_access` (every model that still has work can be
downloaded with this token: a licence not yet accepted stops the cell at once, not hours
in) and, before a 70B-class cell, `preflight` (a few items of the smallest model with the
same flags). A model's weights are deleted from local disk when its runs end, except after
an interrupt, so re-running the cell reuses them.

In [ ]:
# 0.6 Helpers shared by the run cells
import contextlib, gc, glob, json, re, shlex, shutil, tempfile, time
import pandas as pd
from IPython.display import Markdown, display
import sweep_probe as sp

DROP_WEIGHTS = True          # delete each model's weights from local disk after its runs
_PROGRESS = re.compile(r"^\s*\[(?:(select|measure) )?(\d+)/(\d+)\]")
_BAR = re.compile(r"\d+%\|")                 # tqdm bars (downloads, shard loading)

def safe(name):
    return str(name).replace("/", "_")

def run_cmd(args, log_name):
    """Run `python <args>` in the clone with this notebook's environment; raise if it fails."""
    cmd = [sys.executable, *map(str, args)]
    print("$", " ".join(shlex.quote(c) for c in cmd[1:]), flush=True)
    t0, phase, last_bar = time.time(), None, 0.0
    with open(os.path.join(P["logs"], log_name), "a") as log:
        log.write(f"\n$ {' '.join(cmd[1:])}   [{time.strftime('%Y-%m-%d %H:%M:%S')}]\n")
        # unbuffered: a child writing to a pipe otherwise holds ~8 kB of output (an hour of
        # 70B progress lines, which the s/item estimate below would misread) and loses it
        # if it is killed (out of memory); cwd: the modules are in the clone whatever the
        # notebook's working folder is now
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                bufsize=1, cwd=REPO_DIR, env=dict(os.environ, PYTHONUNBUFFERED="1"))
        try:
            for line in proc.stdout:
                log.write(line)
                m = _PROGRESS.match(line)
                if m:
                    i, n = int(m.group(2)), int(m.group(3))
                    if i == 1 or m.group(1) != phase:
                        t0, phase = time.time(), m.group(1)
                    if i == 1 or i % 25 == 0 or i == n:
                        rate = (time.time() - t0) / max(i - 1, 1)
                        print(f"{line.rstrip()}   ({rate:.1f} s/item, about "
                              f"{rate * (n - i) / 60:.0f} min left in this pass)", flush=True)
                    continue
                if _BAR.search(line) and "100%" not in line and time.time() - last_bar < 30:
                    continue
                last_bar = time.time() if _BAR.search(line) else last_bar
                print(line, end="", flush=True)
            rc = proc.wait()
        except BaseException:              # interrupted: do not leave the run behind
            proc.kill()
            proc.wait()
            raise
    if rc:
        raise RuntimeError(f"exit code {rc}; read the output above (full log: "
                           f"{P['logs']}/{log_name})")

def run_models(models, one):
    """one(model) for every model; a failure is reported and the next model runs, and all
    failures are raised together at the end."""
    failed = {}
    for m in models:
        print(f"\n===== {m} =====", flush=True)
        try:
            one(m)
        except Exception as e:
            failed[m] = f"{type(e).__name__}: {e}"
            print(f"\n!!! {m} FAILED: {failed[m]}\n", flush=True)
        gc.collect()
    if failed:
        raise RuntimeError("failed models (read their output above):\n" +
                           "\n".join(f"  {m}: {why}" for m, why in failed.items()))

def drop_weights(model, force=False):
    """Delete a model's downloaded weights (Colab's local disk holds one 70B at a time)."""
    if DROP_WEIGHTS or force:
        d = os.path.join(os.environ["HF_HOME"], "hub", "models--" + model.replace("/", "--"))
        shutil.rmtree(d, ignore_errors=True)

@contextlib.contextmanager
def model_weights(model):
    """Around one model's runs: delete its weights when they end, after a failure too (the
    next model needs the disk), but not after an interrupt, so re-running the cell reuses
    them instead of downloading a 70B again."""
    try:
        yield
    except KeyboardInterrupt:
        print(f"\ninterrupted: {model}'s weights stay in {os.environ['HF_HOME']} for the next "
              f"run (drop_weights({model!r}, force=True) frees the disk)", flush=True)
        raise
    except BaseException:
        drop_weights(model)
        raise
    drop_weights(model)

def cached_models():
    """{model: GB} of the weights in HF_HOME (blobs only; snapshots are links to them)."""
    out = {}
    for d in glob.glob(os.path.join(os.environ["HF_HOME"], "hub", "models--*")):
        b = os.path.join(d, "blobs")
        gb = sum(os.path.getsize(os.path.join(b, f)) for f in os.listdir(b)) / 1e9 \
            if os.path.isdir(b) else 0.0
        out[os.path.basename(d)[len("models--"):].replace("--", "/")] = round(gb, 1)
    return out

def check_access(models):
    """Stop before a long loop, not hours into it, when a model cannot be downloaded with
    this session's token: a licence not accepted (Llama 3.1 and 3.2 and Gemma 2 are
    separate), no token, a typo.  Local folders and offline sessions are not checked; a
    network error only warns."""
    models = [m for m in dict.fromkeys(models) if not os.path.isdir(m)]
    if not models or os.environ.get("HF_HUB_OFFLINE", "").lower() in ("1", "true", "yes", "on"):
        return
    from huggingface_hub import HfApi
    try:
        from huggingface_hub.errors import RepositoryNotFoundError
    except ImportError:
        from huggingface_hub.utils import RepositoryNotFoundError
    api, bad = HfApi(), {}
    for m in models:
        try:
            api.auth_check(m)
        except RepositoryNotFoundError as e:           # includes gated without access
            bad[m] = type(e).__name__
        except Exception as e:
            status = getattr(getattr(e, "response", None), "status_code", None)
            if status in (401, 403):
                bad[m] = f"HTTP {status}"
            else:
                print(f"WARNING: could not check {m} ({type(e).__name__}); continuing")
    if bad:
        raise RuntimeError("these models cannot be downloaded in this session (accept the licence "
                           "on huggingface.co with the account of the token from 0.5, re-run 0.5, "
                           "or remove them from the list):\n" +
                           "\n".join(f"  {m}: {why}" for m, why in bad.items()))
    print(f"access ok for {len(models)} model(s) with work left", flush=True)

def preflight(args, model, log_name="preflight.log"):
    """`python args[0] --model model args[1:]` into a temporary folder that is then deleted:
    a few items of a small model with the flags of the long run that follows, so a broken
    environment (a package upgrade, CUDA) costs a minute, not a 70B download and load."""
    tmp = tempfile.mkdtemp(prefix="preflight_")
    try:
        with model_weights(model):
            run_cmd([args[0], "--model", model, *args[1:], "--out", tmp], log_name)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    print(f"preflight ok on {model}; the long run starts now\n", flush=True)

def size_b(model):
    """Parameters in billions, from the model name (Qwen2.5-0.5B -> 0.5); None if absent."""
    m = re.findall(r"(\d+(?:\.\d+)?)[bB](?![a-zA-Z])", str(model).split("/")[-1])
    return float(m[-1]) if m else None

def need_gb(model):
    s = size_b(model)
    return None if s is None else 2.15 * s + 2          # bf16 weights plus activations

def hardware():
    import psutil, torch
    gpu = (torch.cuda.get_device_properties(0).total_memory / 1e9
           if torch.cuda.is_available() else 0.0)
    return gpu, psutil.virtual_memory().total / 1e9, shutil.disk_usage(os.environ["HF_HOME"]).free / 1e9

def fits_gpu(model, gpu_gb):
    need = need_gb(model)
    return need is None or need <= 0.95 * gpu_gb

def require_big_runtime(models=()):
    """Stop unless this runtime can hold a 70B-class model: GPU, GPU + RAM, and disk for the
    weights (weights of `models` already on disk, kept after an interrupt, count as free)."""
    gpu, ram, disk = hardware()
    held = cached_models()
    have = disk + sum(held.get(m, 0.0) for m in models)
    print(f"GPU {gpu:.0f} GB | RAM {ram:.0f} GB | free disk {disk:.0f} GB"
          + (f" (+{have - disk:.0f} GB of weights already here)" if have > disk else ""))
    if not (gpu >= 75 and gpu + ram >= 150):
        raise RuntimeError("70B-class runs need an 80 GB GPU (A100 80GB or H100) with High-RAM: "
                           "GPU + RAM above ~150 GB (what does not fit on the GPU is offloaded "
                           "to RAM) and ~150 GB free disk for one model's weights")
    if have < 150:
        raise RuntimeError(f"{disk:.0f} GB free disk; one 70B-class model needs ~150 GB. Weights "
                           f"on this VM: {held or 'none'}; free them with "
                           f"drop_weights('<model>', force=True)")

def read_meta(path):
    """The JSON metadata of a sweep_probe / ablation_probe output, without its arrays."""
    import numpy as np
    with np.load(path, allow_pickle=False) as z:
        return json.loads(str(z["meta"]))

def list_outputs(root, pattern="**/*", show=8):
    """Files under root per folder: count, size, and the first few names."""
    by_dir = {}
    for f in sorted(glob.glob(os.path.join(root, pattern), recursive=True)):
        if os.path.isfile(f):
            by_dir.setdefault(os.path.dirname(f), []).append(f)
    for d, fs in by_dir.items():
        mb = sum(os.path.getsize(f) for f in fs) / 1e6
        print(f"{os.path.relpath(d, DRIVE_ROOT)}/  {len(fs)} files, {mb:.1f} MB")
        for f in fs[:show]:
            print("    ", os.path.basename(f))
        if len(fs) > show:
            print(f"     ... {len(fs) - show} more")
    if not by_dir:
        print(f"(nothing under {root})")

def forget_token():
    """Remove the Hugging Face token from this session and from the VM's disk."""
    os.environ.pop("HF_TOKEN", None)
    for f in ("token", "stored_tokens"):
        p = os.path.join(os.environ["HF_HOME"], f)
        if os.path.exists(p):
            os.remove(p)
    print("Hugging Face token removed from this session and this VM; re-run 0.5 before "
          "running more models.")

## Configuration
Everything you may want to change is in this cell.

In [ ]:
# CONFIG ─ E7
MODELS = [
    'meta-llama/Llama-3.2-1B',
    'meta-llama/Llama-3.2-3B',
    'meta-llama/Llama-3.2-3B-Instruct',
    'meta-llama/Llama-3.1-8B',
    'meta-llama/Llama-3.1-8B-Instruct',
    'Qwen/Qwen2.5-0.5B',
    'Qwen/Qwen2.5-1.5B',
    'Qwen/Qwen2.5-3B',
    'Qwen/Qwen2.5-7B',
    'Qwen/Qwen2.5-7B-Instruct',
    'Qwen/Qwen2.5-14B',
    'allenai/OLMo-2-1124-7B',
    'allenai/OLMo-2-1124-7B-SFT',
    'allenai/OLMo-2-1124-7B-DPO',
    'allenai/OLMo-2-1124-7B-Instruct',
    'mistralai/Mistral-7B-v0.1',
    'mistralai/Mistral-Small-24B-Base-2501',
]
KERNEL = "sdpa"               # E3: bf16 sdpa is closest to fp32; "eager" matches E2's captures
KERNEL_PER_MODEL = {}         # e.g. {"Qwen/Qwen2.5-7B": "eager"} to override one model
PRECISION, DEVICE = "bf16", "cuda"
FAMILIES = ["pointer", "arith", "binding", "relational"]   # add "fixed_digit" for the numeral control
WORDINGS = ["w0"]
SPLIT, SEED = "test", 0
DRAWS_WIN, DRAWS_BAND = 5, 2
BATCH_CONDITIONS = None       # None: all conditions (30) in one forward
SMOKE_MODEL = "Qwen/Qwen2.5-0.5B"   # a few items first (None to skip)
PREFETCH = True               # download the next model while the current one runs
GROUP = "all"

E7_ROOT = f"{DRIVE_ROOT}/E7_ablation_window"
E7_REPORT = f"{E7_ROOT}/report"
os.makedirs(E7_REPORT, exist_ok=True)
kernel_of = lambda m: KERNEL_PER_MODEL.get(m, KERNEL)
out_dir = lambda m: f"{E7_ROOT}/{PRECISION}_{kernel_of(m)}"
print(len(MODELS), "models ->", E7_ROOT)

### Task suites and load path
The fixed suites (built and saved to Drive if missing; the same files as E2 and E4). *Check:* the
four families with wording w0.

In [ ]:
# Fixed task suites (no E1 needed): one recommended setting per family, the same for every
# model, built here and saved to Drive. To change them, set SETTINGS (and N_PER_FIXED,
# SEED_FIXED, WORDINGS_FIXED) in the configuration cell; otherwise these defaults are used.
import serial_tasks as st
_DEFAULT_SETTINGS = {
    # a 10-cycle with k = 1..5: the fewest lookups equal k (on an n-cycle k steps
    # forward land where n - k steps back do, so k must stay <= n/2)
    "pointer":     dict(k_range=(1, 5), n_states=10, shots=3),
    "arith":       dict(k_range=(1, 6), modulus=7, shots=3),     # mod 7: no k makes the answer trivial
    "binding":     dict(k_range=(1, 6), n_chains=3, shots=3),
    "relational":  dict(k_range=(1, 6), n_chains=3, shots=3),
    "fixed_digit": dict(k_range=(1, 6), n_states=7, shots=3),    # the numeral control
}
SETTINGS = globals().get("SETTINGS") or _DEFAULT_SETTINGS
N_PER_FIXED = globals().get("N_PER_FIXED", 60)          # 60 per depth: 42 test items
SEED_FIXED = globals().get("SEED_FIXED", 0)
WORDINGS_FIXED = tuple(globals().get("WORDINGS_FIXED", ("w0", "w1")))
_fixed = {}
for _fam, _kw in SETTINGS.items():
    for _w in WORDINGS_FIXED:
        _s = st.make_suite(_fam, wording=_w, n_per=N_PER_FIXED, seed=SEED_FIXED, **_kw)
        _dst = f"{P['suites_fixed']}/{_s['name']}.json"
        if not os.path.exists(_dst):
            st.save_suite(_s, _dst)
        elif st.load_suite(_dst) != _s:
            raise RuntimeError(f"{_dst} exists with other items (settings or code changed); "
                               f"delete it or change SEED_FIXED")
        _fixed[(_fam, _w)] = (_dst, {"file": f"{_s['name']}.json", "name": _s["name"], "wording": _w,
                                     "family": _fam, "status": "fixed", "setting": dict(_kw),
                                     "n_test": sum(it["split"] == "test" for it in _s["items"]),
                                     "k_range": list(_kw["k_range"])})

def chosen_suites(group, families, wordings):
    # [(path, entry)] of the fixed suites, in family then wording order; group is
    # ignored: every model runs the same items
    return [(p, dict(e, group=group)) for f in families for w in wordings
            if (f, w) in _fixed for p, e in [_fixed[(f, w)]]]

_all = [m for _n in ("MODELS_SMALL", "MODELS_BIG", "MODELS") for m in globals().get(_n, [])]
GROUPS = {"all": list(dict.fromkeys(_all))}
manifest = {"chosen": {}}
for (_f, _w), (_p, _e) in _fixed.items():
    print(f"  {_f:12s} {_w}  {_e['n_test']:4d} test items  {_e['setting']}")

In [ ]:
# Pin the load path; every subprocess inherits os.environ
os.environ["ASD_FORCE_STRATEGY"] = "B"                                  # native HF hooks
os.environ["ASD_FORCE_PRECISION"] = "bf16" if PRECISION == "bf16" else ""  # empty = fp32 (CPU only)
os.environ["ASD_ATTN_IMPL"] = KERNEL
print({k: os.environ[k] for k in ("ASD_FORCE_STRATEGY", "ASD_FORCE_PRECISION", "ASD_ATTN_IMPL",
                                  "NEXT_RUNS_COMMIT")})

## 1 · Plan
Suites per model and which outputs exist already (a finished model × suite is skipped).

In [ ]:
import ablation_window as aw
E7_SUITES = chosen_suites(GROUP, FAMILIES, WORDINGS)
SUITE_PATHS = [p for p, _ in E7_SUITES]
for p, s in E7_SUITES:
    print(f"  {s['name']}  ({s['n_test']} test items)")
def todo_suites(m):
    return [p for p, s in E7_SUITES if not aw.output_path(out_dir(m), m, s["name"]).exists()]
TODO = [m for m in MODELS if todo_suites(m)]
print(f"models with suites left: {len(TODO)} of {len(MODELS)}")
for m in TODO:
    print(f"  {m:42s} kernel {kernel_of(m)}  suites left {len(todo_suites(m))}")
if TODO:
    _g, _r, _d = hardware()
    print(f"GPU {_g:.0f} GB | RAM {_r:.0f} GB | free disk {_d:.0f} GB")
    big = [m for m in TODO if not fits_gpu(m, _g)]
    if big:
        print("WARNING: these need a larger GPU and will fail here:", big)

## 2 · Smoke test
`SMOKE_MODEL` on one suite with 4 items per depth into a temporary folder: loading, the KV-cache
path, the ablation checks and writing the output, in about a minute.

In [ ]:
if SMOKE_MODEL and TODO:
    check_access([SMOKE_MODEL])
    _tmp = tempfile.mkdtemp(prefix="e7smoke_")
    try:
        with model_weights(SMOKE_MODEL):
            run_cmd(["ablation_window.py", "--model", SMOKE_MODEL, "--suites", SUITE_PATHS[0], "--out", _tmp,
                     "--device", DEVICE, "--attn-impl", kernel_of(SMOKE_MODEL), "--precision", PRECISION,
                     "--split", SPLIT, "--seed", SEED, "--max-items-per-k", 4,
                     "--draws-win", 1, "--draws-band", 1], f"E7_smoke__{safe(SMOKE_MODEL)}.log")
    finally:
        shutil.rmtree(_tmp, ignore_errors=True)
    print("smoke test ok")
else:
    print("smoke test skipped")

## 3 · Ablation — **H100 / A100 80GB**
One process per model (it loads once for all its suites). While a model runs, the next one is
downloaded in the background (`PREFETCH`); each model's weights are deleted when its runs end. A
finished model × suite is skipped on re-running. *Check* per suite (the `->` line): clean-correct
items, the window's top-10% share, the drops of top-q (mean, zero) and random heads, and the checks
(split, delta, cache, repeat) all small.

In [ ]:
import threading
from huggingface_hub import snapshot_download
_PATTERNS = ["*.json", "model*.safetensors", "tokenizer*", "*.model", "*.tiktoken", "*.txt"]
_prefetch = {}

def prefetch(model):
    if not PREFETCH or model in _prefetch:
        return
    def job():
        try:
            snapshot_download(model, allow_patterns=_PATTERNS, token=os.environ.get("HF_TOKEN"))
        except Exception as e:            # the run downloads what is missing itself
            print(f"(prefetch of {model} failed: {type(e).__name__}; it will download when it runs)")
    th = threading.Thread(target=job, daemon=True); th.start(); _prefetch[model] = th

if TODO:
    check_access(TODO)

def e7_model(model):
    i = TODO.index(model)
    if i + 1 < len(TODO):
        prefetch(TODO[i + 1])
    th = _prefetch.get(model)
    if th is not None and th.is_alive():
        print(f"waiting for {model}'s download ...", flush=True); th.join()
    with model_weights(model):
        run_cmd(["ablation_window.py", "--model", model, "--suites", *todo_suites(model),
                 "--out", out_dir(model), "--device", DEVICE, "--attn-impl", kernel_of(model),
                 "--precision", PRECISION, "--split", SPLIT, "--seed", SEED,
                 "--draws-win", DRAWS_WIN, "--draws-band", DRAWS_BAND,
                 *(["--batch-conditions", BATCH_CONDITIONS] if BATCH_CONDITIONS else [])],
                f"E7__{safe(model)}.log")

t_all = time.time()
run_models(TODO, e7_model)
print(f"done in {(time.time() - t_all) / 60:.0f} min")

## 4 · Analyse
Every E7 output on Drive (both halves, if run in two runtimes). *Read:* within each task, the
Spearman correlation across models between the window's top-10% share and the excess drop (top-q
minus random), with a permutation p; and within each model, the same correlation across its four
tasks. The rows table has every model × task.

In [ ]:
files = sorted(glob.glob(f"{E7_ROOT}/*/*__abwin.npz"))
print(len(files), "outputs")
if files:
    run_cmd(["ablation_window.py", "--analyse", *files, "--out", E7_REPORT], "E7__analyse.log")
    display(Markdown(open(f"{E7_REPORT}/abwin_report.md").read()))

## Outputs
Everything under `E7_ablation_window/`. The Hugging Face token is removed from this VM.

In [ ]:
list_outputs(E7_ROOT)
forget_token()